In [ ]:
# Copyright (c) 2026 Daniyar Kuzekov, Li Yang, Ercan Engin Kuruoğlu, Wai Kin (Victor) Chan
# Licensed under the GNU Affero General Public License v3.0 (AGPL‑3.0)

In [1]:
import shutil
from pathlib import Path
import subprocess

# Change this to the location on /data you want to use
DOWNLOAD_ROOT = Path("/data/downloaded_models").resolve()

# Create it if needed
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

def gb(x):
    return x / (1024**3)

print(f"Checking server disk for: {DOWNLOAD_ROOT}\n")

total, used, free = shutil.disk_usage(DOWNLOAD_ROOT)

print("=== Python disk usage ===")
print(f"Mount/path : {DOWNLOAD_ROOT}")
print(f"Total      : {gb(total):.2f} GB")
print(f"Used       : {gb(used):.2f} GB")
print(f"Free       : {gb(free):.2f} GB")

needed_gb = 237.1
buffer_gb = 20.0
required_gb = needed_gb + buffer_gb

print("\n=== Space check for your model download ===")
print(f"Estimated needed : {needed_gb:.1f} GB")
print(f"Recommended min  : {required_gb:.1f} GB (with buffer)")

if gb(free) >= required_gb:
    print("Status           : Enough space")
else:
    print("Status           : NOT enough space")

print("\n=== df -h output ===")
_ = subprocess.run(["df", "-h", str(DOWNLOAD_ROOT)])

Checking server disk for: /data/downloaded_models

=== Python disk usage ===
Mount/path : /data/downloaded_models
Total      : 3666.45 GB
Used       : 1582.71 GB
Free       : 1897.42 GB

=== Space check for your model download ===
Estimated needed : 237.1 GB
Recommended min  : 257.1 GB (with buffer)
Status           : Enough space

=== df -h output ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/sdc1       3.6T  1.6T  1.9T  46% /data


In [ ]:
#Main Downloader ===============================================================================================

In [ ]:
import os
import sys
import time
import json
import queue
import threading
import subprocess
from pathlib import Path
from getpass import getpass

# =============================================================================
# CHANGE THIS
# =============================================================================
REPO_ID = "mistralai/Mixtral-8x7B-v0.1"
# Examples:
# REPO_ID = "Qwen/Qwen1.5-MoE-A2.7B"
# REPO_ID = "deepseek-ai/DeepSeek-V2-Lite"
# REPO_ID = "microsoft/Phi-3.5-MoE-instruct"

DOWNLOAD_ROOT = Path("/data/downloaded_models").resolve()
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = REPO_ID.split("/")[-1]
TARGET_DIR = DOWNLOAD_ROOT / MODEL_NAME

HF_HOME = (DOWNLOAD_ROOT / ".hf_home").resolve()
HF_HOME.mkdir(parents=True, exist_ok=True)

(HF_HOME / "hub").mkdir(parents=True, exist_ok=True)
(HF_HOME / "xet").mkdir(parents=True, exist_ok=True)
(HF_HOME / "assets").mkdir(parents=True, exist_ok=True)

LOG_DIR = (DOWNLOAD_ROOT / "_logs").resolve()
LOG_DIR.mkdir(parents=True, exist_ok=True)

HEARTBEAT_FILE = LOG_DIR / f"{MODEL_NAME}.heartbeat.json"
CHILD_SCRIPT = LOG_DIR / "hf_download_child.py"

HF_TOKEN = ""
if not HF_TOKEN:
    raise ValueError("No HF token provided.")

# =============================================================================
# Install / upgrade huggingface_hub
# =============================================================================
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"],
    check=True,
)

# =============================================================================
# Child script: one download attempt with one env config
# =============================================================================
child_code = r'''
import os
import sys
import time
import json
import logging
import threading
from pathlib import Path

# IMPORTANT: env vars are inherited from parent before this import
from huggingface_hub import HfApi, snapshot_download, logging as hf_logging

REPO_ID = os.environ["MY_REPO_ID"]
TARGET_DIR = Path(os.environ["MY_TARGET_DIR"]).resolve()
HF_HOME = Path(os.environ["HF_HOME"]).resolve()
HEARTBEAT_FILE = Path(os.environ["MY_HEARTBEAT_FILE"]).resolve()
HEARTBEAT_INTERVAL = int(os.environ.get("MY_HEARTBEAT_INTERVAL", "15"))
MAX_WORKERS = int(os.environ.get("MY_MAX_WORKERS", "8"))
TOKEN = os.environ.get("HF_TOKEN")

ALLOW_PATTERNS = [
    "*.safetensors",
    "*.json",
    "*.model",
    "*.py",
    "*.txt",
    "*.tiktoken",
    "*.merges",
    "*.vocab",
    "*.jinja",
    "tokenizer*",
    "special_tokens_map*",
    "generation_config*",
    "configuration_*",
    "modeling_*",
    "tokenization_*",
    "processing_*",
]

IGNORE_PATTERNS = [
    "consolidated*.pt",
    "*.pt",
    "*.pth",
    "*.ckpt",
    "*.onnx",
    "*.h5",
    "*.msgpack",
    "*.gguf",
    "original/*",
    "**/original/*",
]

def gb(n):
    return n / (1024 ** 3)

def mb(n):
    return n / (1024 ** 2)

def dir_size(path: Path) -> int:
    total = 0
    if not path.exists():
        return 0
    for p in path.rglob("*"):
        try:
            if p.is_file():
                total += p.stat().st_size
        except Exception:
            pass
    return total

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("hf_downloader")

hf_logging.set_verbosity_debug()

stop_flag = False

def heartbeat_loop():
    last_total = None
    while not stop_flag:
        target_bytes = dir_size(TARGET_DIR)
        hub_bytes = dir_size(HF_HOME / "hub")
        xet_bytes = dir_size(HF_HOME / "xet")
        total_bytes = target_bytes + hub_bytes + xet_bytes

        if last_total is None:
            delta = 0
            speed = 0.0
        else:
            delta = max(0, total_bytes - last_total)
            speed = delta / HEARTBEAT_INTERVAL / (1024 ** 2)

        payload = {
            "ts": time.time(),
            "target_bytes": target_bytes,
            "hub_bytes": hub_bytes,
            "xet_bytes": xet_bytes,
            "total_bytes": total_bytes,
            "delta_bytes": delta,
            "mb_per_s": speed,
        }
        try:
            HEARTBEAT_FILE.write_text(json.dumps(payload), encoding="utf-8")
        except Exception as e:
            logger.warning(f"[heartbeat-write-failed] {e}")

        logger.info(
            "[heartbeat] target=%.2f GB | hub=%.2f GB | xet=%.2f GB | total=%.2f GB | +%.1f MB | %.2f MB/s",
            gb(target_bytes), gb(hub_bytes), gb(xet_bytes), gb(total_bytes), mb(delta), speed
        )
        last_total = total_bytes
        time.sleep(HEARTBEAT_INTERVAL)

thread = threading.Thread(target=heartbeat_loop, daemon=True)
thread.start()

try:
    logger.info("Repo            : %s", REPO_ID)
    logger.info("Target dir      : %s", TARGET_DIR)
    logger.info("HF_HOME         : %s", HF_HOME)
    logger.info("Max workers     : %s", MAX_WORKERS)
    logger.info("HF_HUB_DISABLE_XET      : %s", os.environ.get("HF_HUB_DISABLE_XET"))
    logger.info("HF_XET_HIGH_PERFORMANCE : %s", os.environ.get("HF_XET_HIGH_PERFORMANCE"))
    logger.info("HF_XET_NUM_CONCURRENT_RANGE_GETS : %s", os.environ.get("HF_XET_NUM_CONCURRENT_RANGE_GETS"))

    api = HfApi(token=TOKEN)
    who = api.whoami(token=TOKEN)
    logger.info("[auth] OK | user=%s", who.get("name", "unknown") if isinstance(who, dict) else str(who))

    info = api.model_info(REPO_ID, token=TOKEN)
    logger.info("[repo] sha=%s | files=%d", getattr(info, "sha", "unknown"), len(getattr(info, "siblings", []) or []))

    local_path = snapshot_download(
        repo_id=REPO_ID,
        repo_type="model",
        local_dir=str(TARGET_DIR),
        token=TOKEN,
        allow_patterns=ALLOW_PATTERNS,
        ignore_patterns=IGNORE_PATTERNS,
        max_workers=MAX_WORKERS,
    )

    logger.info("[success] local_path=%s", local_path)
    print("__DOWNLOAD_OK__", flush=True)

except Exception as e:
    logger.exception("[download-failed] %s: %s", type(e).__name__, e)
    print("__DOWNLOAD_FAILED__", flush=True)
    raise

finally:
    stop_flag = True
    thread.join(timeout=2)
'''
CHILD_SCRIPT.write_text(child_code, encoding="utf-8")


def run_attempt(name: str, extra_env: dict, stall_seconds: int = 600):
    if HEARTBEAT_FILE.exists():
        HEARTBEAT_FILE.unlink()

    log_file = LOG_DIR / f"{MODEL_NAME}.{name}.log"

    env = os.environ.copy()
    env.update({
        "HF_TOKEN": HF_TOKEN,
        "HF_HOME": str(HF_HOME),
        "HF_HUB_CACHE": str(HF_HOME / "hub"),
        "HF_XET_CACHE": str(HF_HOME / "xet"),
        "HF_ASSETS_CACHE": str(HF_HOME / "assets"),
        "HF_HUB_VERBOSITY": "debug",
        "HF_DEBUG": "1",
        "HF_HUB_ETAG_TIMEOUT": "30",
        "HF_HUB_DOWNLOAD_TIMEOUT": "120",
        "MY_REPO_ID": REPO_ID,
        "MY_TARGET_DIR": str(TARGET_DIR),
        "MY_HEARTBEAT_FILE": str(HEARTBEAT_FILE),
        "MY_HEARTBEAT_INTERVAL": "15",
    })

    # Clear previous attempt flags
    for key in [
        "HF_HUB_DISABLE_XET",
        "HF_XET_HIGH_PERFORMANCE",
        "HF_XET_NUM_CONCURRENT_RANGE_GETS",
        "MY_MAX_WORKERS",
    ]:
        env.pop(key, None)

    env.update(extra_env)

    print("\n" + "=" * 100)
    print(f"ATTEMPT: {name}")
    print(f"LOG FILE: {log_file}")
    print("=" * 100)

    proc = subprocess.Popen(
        [sys.executable, str(CHILD_SCRIPT)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    q = queue.Queue()

    def reader():
        with open(log_file, "w", encoding="utf-8") as f:
            for line in proc.stdout:
                f.write(line)
                f.flush()
                q.put(line)
        q.put(None)

    threading.Thread(target=reader, daemon=True).start()

    last_growth_time = time.time()
    last_total_bytes = -1
    saw_success_marker = False

    while True:
        try:
            item = q.get(timeout=5)
            if item is None:
                break
            print(item, end="")
            if "__DOWNLOAD_OK__" in item:
                saw_success_marker = True
        except queue.Empty:
            pass

        if HEARTBEAT_FILE.exists():
            try:
                hb = json.loads(HEARTBEAT_FILE.read_text(encoding="utf-8"))
                total_bytes = int(hb.get("total_bytes", 0))
                if total_bytes > last_total_bytes:
                    last_total_bytes = total_bytes
                    last_growth_time = time.time()
                elif time.time() - last_growth_time > stall_seconds:
                    print(f"\n[watchdog] No byte growth for {stall_seconds}s. Terminating attempt: {name}")
                    proc.terminate()
                    try:
                        proc.wait(timeout=20)
                    except subprocess.TimeoutExpired:
                        proc.kill()
                    return False, f"stalled ({stall_seconds}s with no byte growth)", log_file
            except Exception:
                pass

        if proc.poll() is not None and q.empty():
            break

    rc = proc.wait()
    if rc == 0 and saw_success_marker:
        return True, "success", log_file
    return False, f"process exited rc={rc}", log_file


# =============================================================================
# Retry plan
# 1) Xet, but with safer concurrency
# 2) Xet, even lower concurrency
# 3) Disable Xet completely and use plain hub download path
# =============================================================================
attempts = [
    ("xet_safe", {
        "MY_MAX_WORKERS": "8",
        "HF_XET_NUM_CONCURRENT_RANGE_GETS": "8",
    }),
    ("xet_lower_concurrency", {
        "MY_MAX_WORKERS": "4",
        "HF_XET_NUM_CONCURRENT_RANGE_GETS": "4",
    }),
    ("http_fallback_no_xet", {
        "MY_MAX_WORKERS": "4",
        "HF_HUB_DISABLE_XET": "1",
    }),
]

for attempt_name, extra_env in attempts:
    ok, reason, log_path = run_attempt(attempt_name, extra_env, stall_seconds=600)
    print(f"\nResult: {attempt_name} -> {reason}")
    print(f"Logs  : {log_path}")
    if ok:
        print("\nDOWNLOAD FINISHED.")
        break
else:
    print("\nAll attempts failed. Inspect the latest log file above.")


ATTEMPT: xet_safe
LOG FILE: /data/downloaded_models/_logs/Mixtral-8x7B-v0.1.xet_safe.log
2026-04-20 18:39:11,532 | INFO | Repo            : mistralai/Mixtral-8x7B-v0.1
2026-04-20 18:39:11,533 | INFO | Target dir      : /data/downloaded_models/Mixtral-8x7B-v0.1
2026-04-20 18:39:11,533 | INFO | HF_HOME         : /data/downloaded_models/.hf_home
2026-04-20 18:39:11,533 | INFO | Max workers     : 8
2026-04-20 18:39:11,533 | INFO | HF_HUB_DISABLE_XET      : None
2026-04-20 18:39:11,533 | INFO | HF_XET_HIGH_PERFORMANCE : None
2026-04-20 18:39:11,533 | INFO | HF_XET_NUM_CONCURRENT_RANGE_GETS : 8
2026-04-20 18:39:11,539 | INFO | [heartbeat] target=20.68 GB | hub=0.00 GB | xet=0.09 GB | total=20.77 GB | +0.0 MB | 0.00 MB/s
Request 932902f2-2bf6-4ab2-8b96-5c9683e99b28: GET https://huggingface.co/api/whoami-v2 (authenticated: True)
2026-04-20 18:39:11,569 | DEBUG | Request 932902f2-2bf6-4ab2-8b96-5c9683e99b28: GET https://huggingface.co/api/whoami-v2 (authenticated: True)
Send: curl -X GET -H 'a

In [18]:
import os
import sys
import json
import time
import socket
import shutil
import threading
import subprocess
from pathlib import Path
from getpass import getpass

# =============================================================================
# CONFIG
# =============================================================================
DOWNLOAD_ROOT = Path("/data/downloaded_models").resolve()
HF_HOME = (DOWNLOAD_ROOT / ".hf_home").resolve()
LOG_DIR = (DOWNLOAD_ROOT / "_logs").resolve()

DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
(HF_HOME / "hub").mkdir(parents=True, exist_ok=True)
(HF_HOME / "xet").mkdir(parents=True, exist_ok=True)
(HF_HOME / "assets").mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

ENDPOINT = "https://hf-mirror.com"
HEARTBEAT_SECONDS = 15
STALL_SECONDS = 1200
MAX_WORKERS = 2

# Public repos only here, so mirror runs tokenless by default
MODELS = {
    "Mixtral-8x7B-v0.1": "mistralai/Mixtral-8x7B-v0.1",
    "Qwen1.5-MoE-A2.7B": "Qwen/Qwen1.5-MoE-A2.7B",
    "DeepSeek-V2-Lite": "deepseek-ai/DeepSeek-V2-Lite",
    "Phi-3.5-MoE-instruct": "microsoft/Phi-3.5-MoE-instruct",
}

ALLOW_PATTERNS = [
    "*.safetensors",
    "*.json",
    "*.model",
    "*.py",
    "*.txt",
    "*.tiktoken",
    "*.merges",
    "*.vocab",
    "*.jinja",
    "tokenizer*",
    "special_tokens_map*",
    "generation_config*",
    "configuration_*",
    "modeling_*",
    "tokenization_*",
    "processing_*",
]

IGNORE_PATTERNS = [
    "consolidated*.pt",
    "*.pt",
    "*.pth",
    "*.ckpt",
    "*.onnx",
    "*.h5",
    "*.msgpack",
    "*.gguf",
    "original/*",
    "**/original/*",
]

# For public mirror downloads, keep token disabled.
USE_TOKEN = False
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if USE_TOKEN and not HF_TOKEN:
    HF_TOKEN = getpass("HF token (hidden): ").strip()

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["HF_XET_CACHE"] = str(HF_HOME / "xet")
os.environ["HF_ASSETS_CACHE"] = str(HF_HOME / "assets")
os.environ["HF_ENDPOINT"] = ENDPOINT
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "20"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
if USE_TOKEN and HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    os.environ.pop("HF_TOKEN", None)

# =============================================================================
# INSTALL
# =============================================================================
def ensure_hf_installed():
    try:
        import huggingface_hub  # noqa: F401
        return
    except Exception:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"],
            check=True,
        )

ensure_hf_installed()

# =============================================================================
# HELPERS
# =============================================================================
def human_bytes(n: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)
    for unit in units:
        if x < 1024 or unit == units[-1]:
            return f"{x:.2f} {unit}"
        x /= 1024.0
    return f"{n} B"

def dir_size(path: Path) -> int:
    total = 0
    if not path.exists():
        return 0
    for p in path.rglob("*"):
        try:
            if p.is_file() and not p.is_symlink():
                total += p.stat().st_size
        except Exception:
            pass
    return total

def scan_local(target_dir: Path, top_n: int = 12):
    print(f"\n[local-scan] {target_dir}")
    if not target_dir.exists():
        print("  directory does not exist yet")
        return {"exists": False, "files": 0, "incomplete": 0, "locks": 0, "size": 0, "names": []}

    files = []
    names = []
    for p in target_dir.rglob("*"):
        try:
            if p.is_file():
                rel = p.relative_to(target_dir).as_posix()
                files.append((p.stat().st_size, rel))
                names.append(rel)
        except Exception:
            pass

    files.sort(reverse=True)
    total = sum(size for size, _ in files)
    incomplete = [(s, r) for s, r in files if r.endswith(".incomplete")]
    locks = [(s, r) for s, r in files if r.endswith(".lock")]

    print(f"  total size : {human_bytes(total)}")
    print(f"  files      : {len(files)}")
    print(f"  incomplete : {len(incomplete)}")
    print(f"  lock files : {len(locks)}")
    print("  top files  :")
    for size, rel in files[:top_n]:
        print(f"    {human_bytes(size):>10}  {rel}")

    return {
        "exists": True,
        "files": len(files),
        "incomplete": len(incomplete),
        "locks": len(locks),
        "size": total,
        "names": names,
    }

def remove_stale_locks(target_dir: Path):
    removed = []
    if not target_dir.exists():
        return removed
    for p in target_dir.rglob("*.lock"):
        try:
            p.unlink()
            removed.append(str(p))
        except Exception:
            pass
    return removed

def has_any_weight_files(target_dir: Path):
    if not target_dir.exists():
        return False
    for p in target_dir.rglob("*"):
        try:
            if p.is_file() and p.name.endswith(".safetensors") and "model-" in p.name:
                return True
        except Exception:
            pass
    return False

def looks_complete(model_name: str, target_dir: Path):
    info = scan_local(target_dir, top_n=8)
    if not info["exists"]:
        return False

    names = set(info["names"])

    # Generic completion rule
    generic_ok = (
        info["incomplete"] == 0
        and "config.json" in names
        and "model.safetensors.index.json" in names
        and has_any_weight_files(target_dir)
    )

    # Mixtral-specific stronger rule
    if model_name == "Mixtral-8x7B-v0.1":
        shard_names = {f"model-{i:05d}-of-00019.safetensors" for i in range(1, 20)}
        if generic_ok and shard_names.issubset(names):
            return True

    return generic_ok

def print_disk():
    total, used, free = shutil.disk_usage(DOWNLOAD_ROOT)
    print("=" * 100)
    print(f"DOWNLOAD_ROOT : {DOWNLOAD_ROOT}")
    print(f"HF_HOME       : {HF_HOME}")
    print(f"ENDPOINT      : {ENDPOINT}")
    print(f"Disk total    : {human_bytes(total)}")
    print(f"Disk used     : {human_bytes(used)}")
    print(f"Disk free     : {human_bytes(free)}")
    print("=" * 100)

# =============================================================================
# CHILD SCRIPT
# =============================================================================
CHILD_SCRIPT = LOG_DIR / "hf_download_child_multi_models.py"
CHILD_SCRIPT.write_text(r'''
import os
import sys
import ssl
import json
import time
import socket
import logging
import urllib.request
import urllib.error
import threading
from pathlib import Path

# Force IPv4 in child
_original_getaddrinfo = socket.getaddrinfo
def ipv4_only_getaddrinfo(host, port, family=0, type=0, proto=0, flags=0):
    results = _original_getaddrinfo(host, port, family, type, proto, flags)
    ipv4 = [r for r in results if r[0] == socket.AF_INET]
    return ipv4 or results
socket.getaddrinfo = ipv4_only_getaddrinfo

ENDPOINT = os.environ["MY_ENDPOINT"]
ENDPOINT_HOST = ENDPOINT.split("://", 1)[-1].split("/", 1)[0]
os.environ["HF_ENDPOINT"] = ENDPOINT
os.environ["HF_HUB_DISABLE_XET"] = "1"

from huggingface_hub import snapshot_download, logging as hf_logging
import huggingface_hub

REPO_ID = os.environ["MY_REPO_ID"]
TARGET_DIR = Path(os.environ["MY_TARGET_DIR"]).resolve()
HF_HOME = Path(os.environ["HF_HOME"]).resolve()
HEARTBEAT_FILE = Path(os.environ["MY_HEARTBEAT_FILE"]).resolve()
HEARTBEAT_INTERVAL = int(os.environ.get("MY_HEARTBEAT_INTERVAL", "15"))
MAX_WORKERS = int(os.environ.get("MY_MAX_WORKERS", "2"))
TOKEN = os.environ.get("MY_EFFECTIVE_TOKEN", "")

ALLOW_PATTERNS = json.loads(os.environ["MY_ALLOW_PATTERNS"])
IGNORE_PATTERNS = json.loads(os.environ["MY_IGNORE_PATTERNS"])

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("hf_downloader")
hf_logging.set_verbosity_debug()

def human_bytes(n: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)
    for unit in units:
        if x < 1024 or unit == units[-1]:
            return f"{x:.2f} {unit}"
        x /= 1024.0
    return f"{n} B"

def dir_size(path: Path) -> int:
    total = 0
    if not path.exists():
        return 0
    for p in path.rglob("*"):
        try:
            if p.is_file() and not p.is_symlink():
                total += p.stat().st_size
        except Exception:
            pass
    return total

def tcp_probe(host: str, port: int = 443, timeout: int = 8):
    out = {"host": host, "dns": [], "tcp_ok": False, "peer": None, "error": None}
    try:
        infos = socket.getaddrinfo(host, port, type=socket.SOCK_STREAM)
        out["dns"] = sorted({x[4][0] for x in infos})
    except Exception as e:
        out["error"] = f"DNS {type(e).__name__}: {e}"
        return out
    try:
        with socket.create_connection((host, port), timeout=timeout) as s:
            out["tcp_ok"] = True
            out["peer"] = str(s.getpeername())
    except Exception as e:
        out["error"] = f"TCP {type(e).__name__}: {e}"
    return out

def http_probe(url: str, timeout: int = 12):
    ctx = ssl.create_default_context()
    headers = {"User-Agent": "hf-mirror-multi-models/1.0"}
    req = urllib.request.Request(url, headers=headers, method="GET")
    out = {"url": url, "ok": False, "status": None, "reason": None, "error": None}
    try:
        with urllib.request.urlopen(req, timeout=timeout, context=ctx) as resp:
            out["ok"] = True
            out["status"] = getattr(resp, "status", None)
            out["reason"] = getattr(resp, "reason", None)
    except urllib.error.HTTPError as e:
        out["status"] = e.code
        out["reason"] = e.reason
        out["error"] = f"HTTPError: {e}"
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e}"
    return out

stop_flag = False

def heartbeat_loop():
    last_total = None
    beat_num = 0
    while not stop_flag:
        beat_num += 1
        target_bytes = dir_size(TARGET_DIR)
        hub_bytes = dir_size(HF_HOME / "hub")
        xet_bytes = dir_size(HF_HOME / "xet")
        total_bytes = target_bytes + hub_bytes + xet_bytes

        if last_total is None:
            delta = 0
            speed = 0.0
        else:
            delta = max(0, total_bytes - last_total)
            speed = delta / HEARTBEAT_INTERVAL / (1024 ** 2)

        payload = {
            "ts": time.time(),
            "beat_num": beat_num,
            "target_bytes": target_bytes,
            "hub_bytes": hub_bytes,
            "xet_bytes": xet_bytes,
            "total_bytes": total_bytes,
            "delta_bytes": delta,
            "mb_per_s": speed,
        }
        HEARTBEAT_FILE.write_text(json.dumps(payload), encoding="utf-8")

        logger.info(
            "[heartbeat #%d] target=%s | hub=%s | xet=%s | total=%s | +%s | %.2f MB/s",
            beat_num,
            human_bytes(target_bytes),
            human_bytes(hub_bytes),
            human_bytes(xet_bytes),
            human_bytes(total_bytes),
            human_bytes(delta),
            speed,
        )

        if beat_num == 1 or beat_num % 4 == 0:
            for host in [ENDPOINT_HOST, "huggingface.co", "cdn-lfs.huggingface.co", "cas-bridge.xethub.hf.co"]:
                logger.info("[probe-tcp] %s", json.dumps(tcp_probe(host), ensure_ascii=False))
            for url in [
                ENDPOINT,
                f"{ENDPOINT}/api/models/{REPO_ID}",
            ]:
                logger.info("[probe-http] %s", json.dumps(http_probe(url), ensure_ascii=False))

        last_total = total_bytes
        time.sleep(HEARTBEAT_INTERVAL)

thread = threading.Thread(target=heartbeat_loop, daemon=True)
thread.start()

try:
    logger.info("huggingface_hub version : %s", getattr(huggingface_hub, "__version__", "unknown"))
    logger.info("Repo                    : %s", REPO_ID)
    logger.info("Endpoint                : %s", ENDPOINT)
    logger.info("Target dir              : %s", TARGET_DIR)
    logger.info("HF_HOME                 : %s", HF_HOME)
    logger.info("Max workers             : %s", MAX_WORKERS)
    logger.info("Token enabled           : %s", bool(TOKEN))
    logger.info("HF_HUB_DISABLE_XET      : %s", os.environ.get("HF_HUB_DISABLE_XET"))

    before_target = dir_size(TARGET_DIR)
    before_total = before_target + dir_size(HF_HOME / "hub") + dir_size(HF_HOME / "xet")
    logger.info("[before] target=%s | total=%s", human_bytes(before_target), human_bytes(before_total))

    try:
        logger.info("[dry-run] starting")
        dry = snapshot_download(
            repo_id=REPO_ID,
            repo_type="model",
            token=(TOKEN or None),
            allow_patterns=ALLOW_PATTERNS,
            ignore_patterns=IGNORE_PATTERNS,
            dry_run=True,
            max_workers=MAX_WORKERS,
            endpoint=ENDPOINT,
        )
        infos = []
        for item in dry:
            name = (
                getattr(item, "file_name", None)
                or getattr(item, "filename", None)
                or getattr(item, "path", None)
                or str(item)
            )
            size = getattr(item, "size", 0) or 0
            will_download = getattr(item, "will_download", None)
            is_cached = getattr(item, "is_cached", None)
            infos.append((int(size), str(name), will_download, is_cached))
        logger.info("[dry-run] files=%d total=%s", len(infos), human_bytes(sum(x[0] for x in infos)))
        for size, name, will_download, is_cached in sorted(infos, reverse=True)[:20]:
            logger.info(
                "[dry-run-file] %s | will_download=%s | cached=%s | %s",
                human_bytes(size), will_download, is_cached, name
            )
    except Exception as e:
        logger.exception("[dry-run-failed] %s: %s", type(e).__name__, e)

    logger.info("[download] snapshot_download starting")
    local_path = snapshot_download(
        repo_id=REPO_ID,
        repo_type="model",
        local_dir=str(TARGET_DIR),
        token=(TOKEN or None),
        allow_patterns=ALLOW_PATTERNS,
        ignore_patterns=IGNORE_PATTERNS,
        max_workers=MAX_WORKERS,
        endpoint=ENDPOINT,
    )
    logger.info("[download] snapshot_download returned: %s", local_path)

    after_target = dir_size(TARGET_DIR)
    after_total = after_target + dir_size(HF_HOME / "hub") + dir_size(HF_HOME / "xet")
    delta = max(0, after_total - before_total)
    logger.info("[after] target=%s | total=%s | delta=%s", human_bytes(after_target), human_bytes(after_total), human_bytes(delta))

    if delta == 0:
        logger.warning("[no-progress] snapshot_download returned but no bytes changed")
        print("__DOWNLOAD_NO_PROGRESS__", flush=True)
        sys.exit(2)

    logger.info("[success] download made progress")
    print("__DOWNLOAD_OK__", flush=True)

except Exception as e:
    logger.exception("[download-failed] %s: %s", type(e).__name__, e)
    print("__DOWNLOAD_FAILED__", flush=True)
    raise

finally:
    stop_flag = True
    thread.join(timeout=2)
''', encoding="utf-8")

# =============================================================================
# RUNNER
# =============================================================================
def run_download(model_name: str, repo_id: str):
    target_dir = DOWNLOAD_ROOT / model_name
    target_dir.mkdir(parents=True, exist_ok=True)
    heartbeat_file = LOG_DIR / f"{model_name}.heartbeat.json"
    if heartbeat_file.exists():
        heartbeat_file.unlink()

    log_file = LOG_DIR / f"{model_name}.mirror_no_xet.endpoint.log"
    effective_token = HF_TOKEN if (USE_TOKEN and HF_TOKEN) else ""

    env = os.environ.copy()
    env.update({
        "PYTHONUNBUFFERED": "1",
        "HF_HOME": str(HF_HOME),
        "HF_HUB_CACHE": str(HF_HOME / "hub"),
        "HF_XET_CACHE": str(HF_HOME / "xet"),
        "HF_ASSETS_CACHE": str(HF_HOME / "assets"),
        "HF_HUB_VERBOSITY": "debug",
        "HF_DEBUG": "1",
        "HF_ENDPOINT": ENDPOINT,
        "HF_HUB_DISABLE_XET": "1",
        "MY_REPO_ID": repo_id,
        "MY_TARGET_DIR": str(target_dir),
        "MY_HEARTBEAT_FILE": str(heartbeat_file),
        "MY_HEARTBEAT_INTERVAL": str(HEARTBEAT_SECONDS),
        "MY_ALLOW_PATTERNS": json.dumps(ALLOW_PATTERNS),
        "MY_IGNORE_PATTERNS": json.dumps(IGNORE_PATTERNS),
        "MY_ENDPOINT": ENDPOINT,
        "MY_EFFECTIVE_TOKEN": effective_token,
        "MY_MAX_WORKERS": str(MAX_WORKERS),
    })
    if effective_token:
        env["HF_TOKEN"] = effective_token
    else:
        env.pop("HF_TOKEN", None)

    print("\n" + "=" * 100)
    print(f"MODEL    : {model_name}")
    print(f"REPO     : {repo_id}")
    print(f"ENDPOINT : {ENDPOINT}")
    print(f"TOKEN    : {'enabled' if effective_token else 'disabled'}")
    print(f"TARGET   : {target_dir}")
    print(f"LOG FILE : {log_file}")
    print("=" * 100)

    with open(log_file, "w", encoding="utf-8") as log_handle:
        proc = subprocess.Popen(
            [sys.executable, "-u", str(CHILD_SCRIPT)],
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            env=env,
        )

    last_print_pos = 0
    last_growth_time = time.time()
    last_total_bytes = -1
    saw_ok = False
    saw_no_progress = False
    saw_failed = False

    while True:
        time.sleep(5)

        if log_file.exists():
            with open(log_file, "r", encoding="utf-8", errors="replace") as f:
                f.seek(last_print_pos)
                chunk = f.read()
                if chunk:
                    print(chunk, end="")
                    last_print_pos = f.tell()
                    if "__DOWNLOAD_OK__" in chunk:
                        saw_ok = True
                    if "__DOWNLOAD_NO_PROGRESS__" in chunk:
                        saw_no_progress = True
                    if "__DOWNLOAD_FAILED__" in chunk:
                        saw_failed = True

        if heartbeat_file.exists():
            try:
                hb = json.loads(heartbeat_file.read_text(encoding="utf-8"))
                total_bytes = int(hb.get("total_bytes", 0))
                delta_bytes = int(hb.get("delta_bytes", 0))
                mbps = float(hb.get("mb_per_s", 0.0))
                beat_num = hb.get("beat_num", "?")
                print(
                    f"[parent-heartbeat {model_name} #{beat_num}] total={human_bytes(total_bytes)} "
                    f"| delta={human_bytes(delta_bytes)} | {mbps:.2f} MB/s"
                )
                if total_bytes > last_total_bytes:
                    last_total_bytes = total_bytes
                    last_growth_time = time.time()
                elif time.time() - last_growth_time > STALL_SECONDS:
                    print(f"\n[watchdog] No byte growth for {STALL_SECONDS}s. Terminating {model_name}.")
                    proc.terminate()
                    try:
                        proc.wait(timeout=20)
                    except subprocess.TimeoutExpired:
                        proc.kill()
                    break
            except Exception as e:
                print(f"[parent-heartbeat-read-failed] {e}")

        if proc.poll() is not None:
            time.sleep(1)
            if log_file.exists():
                with open(log_file, "r", encoding="utf-8", errors="replace") as f:
                    f.seek(last_print_pos)
                    chunk = f.read()
                    if chunk:
                        print(chunk, end="")
                        if "__DOWNLOAD_OK__" in chunk:
                            saw_ok = True
                        if "__DOWNLOAD_NO_PROGRESS__" in chunk:
                            saw_no_progress = True
                        if "__DOWNLOAD_FAILED__" in chunk:
                            saw_failed = True
            break

    rc = proc.poll()
    if saw_ok and rc == 0:
        return True, "success", log_file
    if saw_no_progress:
        return False, "no progress / inaccessible", log_file
    if saw_failed:
        return False, f"failed rc={rc}", log_file
    return False, f"ended rc={rc}", log_file

# =============================================================================
# MAIN
# =============================================================================
print_disk()

# Clean Mixtral stale locks after successful completion
mixtral_dir = DOWNLOAD_ROOT / "Mixtral-8x7B-v0.1"
if looks_complete("Mixtral-8x7B-v0.1", mixtral_dir):
    removed = remove_stale_locks(mixtral_dir)
    print(f"\n[Mixtral status] COMPLETE. Removed {len(removed)} stale lock files.")
else:
    print("\n[Mixtral status] NOT complete yet.")

# Download remaining models
for model_name, repo_id in MODELS.items():
    target_dir = DOWNLOAD_ROOT / model_name

    if looks_complete(model_name, target_dir):
        removed = remove_stale_locks(target_dir)
        print(f"\n[skip] {model_name} already looks complete. Removed {len(removed)} stale lock files.")
        continue

    print(f"\n[start] downloading {model_name}")
    ok, reason, log_path = run_download(model_name, repo_id)
    print(f"\nResult: {model_name} -> {reason}")
    print(f"Logs  : {log_path}")
    scan_local(target_dir)

    if not ok:
        print(f"\n[stop] {model_name} did not finish. Fix/retry this one before moving on.")
        break
else:
    print("\nAll requested models processed.")

DOWNLOAD_ROOT : /data/downloaded_models
HF_HOME       : /data/downloaded_models/.hf_home
ENDPOINT      : https://hf-mirror.com
Disk total    : 3.58 TB
Disk used     : 1.72 TB
Disk free     : 1.68 TB

[local-scan] /data/downloaded_models/Mixtral-8x7B-v0.1
  total size : 86.99 GB
  files      : 80
  incomplete : 0
  lock files : 26
  top files  :
       4.64 GB  model-00018-of-00019.safetensors
       4.64 GB  model-00016-of-00019.safetensors
       4.64 GB  model-00015-of-00019.safetensors
       4.64 GB  model-00013-of-00019.safetensors
       4.64 GB  model-00012-of-00019.safetensors
       4.64 GB  model-00011-of-00019.safetensors
       4.64 GB  model-00009-of-00019.safetensors
       4.64 GB  model-00008-of-00019.safetensors

[Mixtral status] COMPLETE. Removed 26 stale lock files.

[local-scan] /data/downloaded_models/Mixtral-8x7B-v0.1
  total size : 86.99 GB
  files      : 54
  incomplete : 0
  lock files : 0
  top files  :
       4.64 GB  model-00018-of-00019.safetensors
       4

In [ ]:
#Second ===========================================================================================================

In [21]:
import os
import sys
import time
import json
import socket
import shutil
import subprocess
from pathlib import Path
from getpass import getpass

# =========================
# CONFIG
# =========================
DOWNLOAD_ROOT = Path("/data/downloaded_models").resolve()
MODELS = {
    "Mixtral-8x7B-v0.1": "mistralai/Mixtral-8x7B-v0.1",
    "Qwen1.5-MoE-A2.7B": "Qwen/Qwen1.5-MoE-A2.7B",
    "DeepSeek-V2-Lite": "deepseek-ai/DeepSeek-V2-Lite",
    "Phi-3.5-MoE-instruct": "microsoft/Phi-3.5-MoE-instruct",
}

# Keep normal inference/loading files
ALLOW_PATTERNS = [
    "*.safetensors",
    "*.json",
    "*.model",
    "*.py",
    "*.txt",
    "*.tiktoken",
    "*.merges",
    "*.vocab",
    "*.jinja",
    "tokenizer*",
    "special_tokens_map*",
    "generation_config*",
    "configuration_*",
    "modeling_*",
    "tokenization_*",
    "processing_*",
]

# Skip alternate / legacy formats you likely do not need
IGNORE_PATTERNS = [
    "consolidated*.pt",
    "*.pt",
    "*.pth",
    "*.ckpt",
    "*.onnx",
    "*.h5",
    "*.msgpack",
    "*.gguf",
    "original/*",
    "**/original/*",
]

# Watchdog: if bytes on disk do not increase for this long, kill that attempt and try fallback
STALL_SECONDS = 900
HEARTBEAT_SECONDS = 15

# Safer than using very aggressive concurrency
ATTEMPTS = [
    ("xet_balanced", {"MY_MAX_WORKERS": "4", "HF_XET_NUM_CONCURRENT_RANGE_GETS": "4"}),
    ("xet_conservative", {"MY_MAX_WORKERS": "2", "HF_XET_NUM_CONCURRENT_RANGE_GETS": "2"}),
    ("http_no_xet", {"MY_MAX_WORKERS": "2", "HF_HUB_DISABLE_XET": "1"}),
]

# =========================
# SETUP
# =========================
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME = (DOWNLOAD_ROOT / ".hf_home").resolve()
(HF_HOME / "hub").mkdir(parents=True, exist_ok=True)
(HF_HOME / "xet").mkdir(parents=True, exist_ok=True)
(HF_HOME / "assets").mkdir(parents=True, exist_ok=True)

LOG_DIR = (DOWNLOAD_ROOT / "_logs").resolve()
LOG_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("Paste HF token (hidden): ").strip()
if not HF_TOKEN:
    raise ValueError("No HF token provided.")

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["HF_XET_CACHE"] = str(HF_HOME / "xet")
os.environ["HF_ASSETS_CACHE"] = str(HF_HOME / "assets")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

# install / upgrade latest hub
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"],
    check=True,
)

# =========================
# HELPERS
# =========================
def human_bytes(n: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)
    for unit in units:
        if x < 1024 or unit == units[-1]:
            return f"{x:.2f} {unit}"
        x /= 1024.0

def dir_size(path: Path) -> int:
    total = 0
    if not path.exists():
        return 0
    for p in path.rglob("*"):
        try:
            if p.is_file() and not p.is_symlink():
                total += p.stat().st_size
        except Exception:
            pass
    return total

def print_disk(path: Path) -> None:
    total, used, free = shutil.disk_usage(path)
    print("=" * 100)
    print(f"DOWNLOAD_ROOT : {DOWNLOAD_ROOT}")
    print(f"HF_HOME       : {HF_HOME}")
    print(f"Disk total    : {human_bytes(total)}")
    print(f"Disk used     : {human_bytes(used)}")
    print(f"Disk free     : {human_bytes(free)}")
    print("=" * 100)

def tcp_ok(host: str = "huggingface.co", port: int = 443, timeout: int = 10) -> bool:
    try:
        socket.getaddrinfo(host, port)
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except Exception:
        return False

def remove_stale_locks(target_dir: Path, older_than_sec: int = 1800):
    """
    Remove stale .lock files only if they are old.
    This avoids killing useful partial data while still clearing abandoned locks.
    """
    removed = []
    now = time.time()
    if not target_dir.exists():
        return removed
    for p in target_dir.rglob("*.lock"):
        try:
            if now - p.stat().st_mtime >= older_than_sec:
                p.unlink()
                removed.append(p)
        except Exception:
            pass
    return removed

def scan_local(target_dir: Path, top_n: int = 20) -> None:
    """
    Local-only scan. Does not call Hugging Face API.
    Useful even when the machine has no network.
    """
    print(f"\n[local-scan] {target_dir}")
    if not target_dir.exists():
        print("  directory does not exist yet")
        return

    files = []
    for p in target_dir.rglob("*"):
        try:
            if p.is_file():
                files.append((p.stat().st_size, p.relative_to(target_dir).as_posix()))
        except Exception:
            pass

    files.sort(reverse=True)
    total = sum(size for size, _ in files)
    incomplete = [(s, r) for s, r in files if r.endswith(".incomplete")]
    locks = [(s, r) for s, r in files if r.endswith(".lock")]

    print(f"  files        : {len(files)}")
    print(f"  total size   : {human_bytes(total)}")
    print(f"  incomplete   : {len(incomplete)}")
    print(f"  lock files   : {len(locks)}")
    print("  largest files:")
    for size, rel in files[:top_n]:
        print(f"    {human_bytes(size):>10}  {rel}")

def write_child_script() -> Path:
    """
    Runs the actual HF download in a fresh Python process so env vars
    like HF_HUB_DISABLE_XET / HF_XET_* are definitely applied per attempt.
    """
    child_code = r"""
import os
import sys
import time
import json
import socket
import logging
from pathlib import Path

from huggingface_hub import snapshot_download, logging as hf_logging

REPO_ID = os.environ["MY_REPO_ID"]
TARGET_DIR = Path(os.environ["MY_TARGET_DIR"]).resolve()
HF_HOME = Path(os.environ["HF_HOME"]).resolve()
HEARTBEAT_FILE = Path(os.environ["MY_HEARTBEAT_FILE"]).resolve()
HEARTBEAT_INTERVAL = int(os.environ.get("MY_HEARTBEAT_INTERVAL", "15"))
MAX_WORKERS = int(os.environ.get("MY_MAX_WORKERS", "4"))
TOKEN = os.environ.get("HF_TOKEN")

ALLOW_PATTERNS = json.loads(os.environ["MY_ALLOW_PATTERNS"])
IGNORE_PATTERNS = json.loads(os.environ["MY_IGNORE_PATTERNS"])

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("hf_downloader")
hf_logging.set_verbosity_debug()

def dir_size(path: Path) -> int:
    total = 0
    if not path.exists():
        return 0
    for p in path.rglob("*"):
        try:
            if p.is_file() and not p.is_symlink():
                total += p.stat().st_size
        except Exception:
            pass
    return total

def heartbeat() -> None:
    target_bytes = dir_size(TARGET_DIR)
    xet_bytes = dir_size(HF_HOME / "xet")
    payload = {
        "ts": time.time(),
        "target_bytes": target_bytes,
        "xet_bytes": xet_bytes,
        "total_bytes": target_bytes + xet_bytes,
    }
    HEARTBEAT_FILE.write_text(json.dumps(payload), encoding="utf-8")
    logger.info(
        "[heartbeat] target=%.2f GB | xet=%.2f GB | total=%.2f GB",
        target_bytes / (1024 ** 3),
        xet_bytes / (1024 ** 3),
        (target_bytes + xet_bytes) / (1024 ** 3),
    )

logger.info("Repo            : %s", REPO_ID)
logger.info("Target dir      : %s", TARGET_DIR)
logger.info("HF_HOME         : %s", HF_HOME)
logger.info("Max workers     : %s", MAX_WORKERS)
logger.info("HF_HUB_DISABLE_XET      : %s", os.environ.get("HF_HUB_DISABLE_XET"))
logger.info("HF_XET_HIGH_PERFORMANCE : %s", os.environ.get("HF_XET_HIGH_PERFORMANCE"))
logger.info("HF_XET_NUM_CONCURRENT_RANGE_GETS : %s", os.environ.get("HF_XET_NUM_CONCURRENT_RANGE_GETS"))

# Basic network reachability before we start
try:
    socket.getaddrinfo("huggingface.co", 443)
    with socket.create_connection(("huggingface.co", 443), timeout=10):
        pass
    logger.info("[network] huggingface.co:443 reachable")
except Exception as e:
    logger.exception("[network-failed] %s: %s", type(e).__name__, e)
    print("__DOWNLOAD_FAILED__", flush=True)
    raise

heartbeat()

try:
    # Optional manifest / file-size check
    try:
        dry = snapshot_download(
            repo_id=REPO_ID,
            repo_type="model",
            token=TOKEN,
            allow_patterns=ALLOW_PATTERNS,
            ignore_patterns=IGNORE_PATTERNS,
            dry_run=True,
            max_workers=MAX_WORKERS,
        )
        infos = []
        for item in dry:
            name = (
                getattr(item, "file_name", None)
                or getattr(item, "filename", None)
                or getattr(item, "path", None)
                or str(item)
            )
            size = getattr(item, "size", 0) or 0
            will_download = getattr(item, "will_download", None)
            is_cached = getattr(item, "is_cached", None)
            infos.append((int(size), str(name), will_download, is_cached))

        total_size = sum(x[0] for x in infos)
        todo = sum(1 for _, _, will_download, is_cached in infos if will_download is True or is_cached is False)
        logger.info("[dry-run] files=%d | total=%.2f GB | will_download=%d", len(infos), total_size / (1024 ** 3), todo)

        for size, name, will_download, is_cached in sorted(infos, reverse=True)[:15]:
            logger.info(
                "[dry-run-file] %8.2f GB | will_download=%s | cached=%s | %s",
                size / (1024 ** 3),
                will_download,
                is_cached,
                name,
            )
    except Exception as e:
        logger.warning("[dry-run-failed] %s: %s", type(e).__name__, e)

    heartbeat()

    # Actual download / resume
    local_path = snapshot_download(
        repo_id=REPO_ID,
        repo_type="model",
        local_dir=str(TARGET_DIR),
        token=TOKEN,
        allow_patterns=ALLOW_PATTERNS,
        ignore_patterns=IGNORE_PATTERNS,
        max_workers=MAX_WORKERS,
    )

    heartbeat()
    logger.info("[success] local_path=%s", local_path)
    print("__DOWNLOAD_OK__", flush=True)

except Exception as e:
    heartbeat()
    logger.exception("[download-failed] %s: %s", type(e).__name__, e)
    print("__DOWNLOAD_FAILED__", flush=True)
    raise
"""
    child_script = LOG_DIR / "hf_download_child.py"
    child_script.write_text(child_code, encoding="utf-8")
    return child_script

CHILD_SCRIPT = write_child_script()

def run_attempt(model_name: str, repo_id: str, attempt_name: str, extra_env: dict) -> bool:
    target_dir = DOWNLOAD_ROOT / model_name
    target_dir.mkdir(parents=True, exist_ok=True)

    heartbeat_file = LOG_DIR / f"{model_name}.heartbeat.json"
    log_file = LOG_DIR / f"{model_name}.{attempt_name}.log"
    if heartbeat_file.exists():
        heartbeat_file.unlink()

    env = os.environ.copy()
    env.update({
        "PYTHONUNBUFFERED": "1",
        "HF_TOKEN": HF_TOKEN,
        "HF_HOME": str(HF_HOME),
        "HF_HUB_CACHE": str(HF_HOME / "hub"),
        "HF_XET_CACHE": str(HF_HOME / "xet"),
        "HF_ASSETS_CACHE": str(HF_HOME / "assets"),
        "HF_HUB_VERBOSITY": "debug",
        "HF_DEBUG": "1",
        "HF_HUB_ETAG_TIMEOUT": "30",
        "HF_HUB_DOWNLOAD_TIMEOUT": "120",
        "MY_REPO_ID": repo_id,
        "MY_TARGET_DIR": str(target_dir),
        "MY_HEARTBEAT_FILE": str(heartbeat_file),
        "MY_HEARTBEAT_INTERVAL": str(HEARTBEAT_SECONDS),
        "MY_ALLOW_PATTERNS": json.dumps(ALLOW_PATTERNS),
        "MY_IGNORE_PATTERNS": json.dumps(IGNORE_PATTERNS),
    })

    # Clear flags from prior attempt, then apply this attempt's mode
    for key in [
        "HF_HUB_DISABLE_XET",
        "HF_XET_HIGH_PERFORMANCE",
        "HF_XET_NUM_CONCURRENT_RANGE_GETS",
        "MY_MAX_WORKERS",
    ]:
        env.pop(key, None)

    env.update(extra_env)

    print("\n" + "=" * 100)
    print(f"MODEL    : {model_name}")
    print(f"ATTEMPT  : {attempt_name}")
    print(f"REPO     : {repo_id}")
    print(f"TARGET   : {target_dir}")
    print(f"LOG FILE : {log_file}")
    print("=" * 100)

    with open(log_file, "w", encoding="utf-8") as log_handle:
        proc = subprocess.Popen(
            [sys.executable, "-u", str(CHILD_SCRIPT)],
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            env=env,
        )

    last_print_pos = 0
    last_growth_time = time.time()
    last_total_bytes = -1
    saw_success = False

    while True:
        time.sleep(5)

        # tail the log file live
        if log_file.exists():
            with open(log_file, "r", encoding="utf-8", errors="replace") as f:
                f.seek(last_print_pos)
                chunk = f.read()
                if chunk:
                    print(chunk, end="")
                    last_print_pos = f.tell()
                    if "__DOWNLOAD_OK__" in chunk:
                        saw_success = True

        # watchdog on bytes actually written to disk
        if heartbeat_file.exists():
            try:
                hb = json.loads(heartbeat_file.read_text(encoding="utf-8"))
                total_bytes = int(hb.get("total_bytes", 0))
                if total_bytes > last_total_bytes:
                    last_total_bytes = total_bytes
                    last_growth_time = time.time()
                elif time.time() - last_growth_time > STALL_SECONDS:
                    print(f"\n[watchdog] No byte growth for {STALL_SECONDS}s. Terminating {attempt_name}.")
                    proc.terminate()
                    try:
                        proc.wait(timeout=20)
                    except subprocess.TimeoutExpired:
                        proc.kill()
                    break
            except Exception:
                pass

        rc = proc.poll()
        if rc is not None:
            time.sleep(1)
            if log_file.exists():
                with open(log_file, "r", encoding="utf-8", errors="replace") as f:
                    f.seek(last_print_pos)
                    chunk = f.read()
                    if chunk:
                        print(chunk, end="")
                        last_print_pos = f.tell()
                        if "__DOWNLOAD_OK__" in chunk:
                            saw_success = True
            break

    rc = proc.poll()
    print(f"\n[attempt-result] rc={rc} | success={saw_success}")
    return bool(saw_success and rc == 0)

def run_model(model_name: str, repo_id: str) -> None:
    target_dir = DOWNLOAD_ROOT / model_name

    print("\n" + "#" * 100)
    print(f"STARTING MODEL : {model_name}")
    print("#" * 100)
    scan_local(target_dir)

    removed = remove_stale_locks(target_dir, older_than_sec=1800)
    if removed:
        print(f"[cleanup] removed {len(removed)} stale .lock files older than 1800s")

    # Do not waste time trying all modes if the box has no route to HF at all
    if not tcp_ok():
        print("[preflight] Network to huggingface.co:443 is not reachable from this machine right now.")
        print("[preflight] Skipping download attempts and showing local files only.")
        scan_local(target_dir)
        return

    for attempt_name, extra_env in ATTEMPTS:
        ok = run_attempt(model_name, repo_id, attempt_name, extra_env)
        scan_local(target_dir)
        if ok:
            print(f"\n[done] {model_name} finished successfully.")
            return

    print(f"\n[failed] all attempts failed for {model_name}")

print_disk(DOWNLOAD_ROOT)

for model_name, repo_id in MODELS.items():
    run_model(model_name, repo_id)

print("\n" + "#" * 100)
print("FINAL LOCAL CONTENT")
print("#" * 100)
for model_name in MODELS:
    scan_local(DOWNLOAD_ROOT / model_name, top_n=12)

ERROR: Operation cancelled by user

KeyboardInterrupt



In [17]:
import os
import sys
import json
import time
import socket
import shutil
import threading
import subprocess
from pathlib import Path

# =============================================================================
# CONFIG
# =============================================================================
REPO_ID = "mistralai/Mixtral-8x7B-v0.1"
ENDPOINT = "https://hf-mirror.com"   # mirror-only test
DOWNLOAD_ROOT = Path("/data/downloaded_models").resolve()
MODEL_NAME = REPO_ID.split("/")[-1]
TARGET_DIR = DOWNLOAD_ROOT / MODEL_NAME

HF_HOME = (DOWNLOAD_ROOT / ".hf_home").resolve()
LOG_DIR = (DOWNLOAD_ROOT / "_logs").resolve()
HEARTBEAT_FILE = LOG_DIR / f"{MODEL_NAME}.heartbeat.json"
CHILD_SCRIPT = LOG_DIR / "hf_download_child_mirror_only.py"

HEARTBEAT_SECONDS = 15
STALL_SECONDS = 900
MAX_WORKERS = 2

ALLOW_PATTERNS = [
    "*.safetensors",
    "*.json",
    "*.model",
    "*.py",
    "*.txt",
    "*.tiktoken",
    "*.merges",
    "*.vocab",
    "*.jinja",
    "tokenizer*",
    "special_tokens_map*",
    "generation_config*",
    "configuration_*",
    "modeling_*",
    "tokenization_*",
    "processing_*",
]

IGNORE_PATTERNS = [
    "consolidated*.pt",
    "*.pt",
    "*.pth",
    "*.ckpt",
    "*.onnx",
    "*.h5",
    "*.msgpack",
    "*.gguf",
    "original/*",
    "**/original/*",
]

# =============================================================================
# SETUP
# =============================================================================
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
TARGET_DIR.mkdir(parents=True, exist_ok=True)
(HF_HOME / "hub").mkdir(parents=True, exist_ok=True)
(HF_HOME / "xet").mkdir(parents=True, exist_ok=True)
(HF_HOME / "assets").mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# IMPORTANT:
# For this mirror test, do NOT send your HF token to the third-party endpoint.
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["HF_XET_CACHE"] = str(HF_HOME / "xet")
os.environ["HF_ASSETS_CACHE"] = str(HF_HOME / "assets")
os.environ["HF_HUB_ETAG_TIMEOUT"] = "20"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
os.environ["HF_ENDPOINT"] = ENDPOINT
os.environ["HF_HUB_DISABLE_XET"] = "1"  # mirror + no-xet
os.environ.pop("HF_TOKEN", None)

def ensure_hf_installed():
    try:
        import huggingface_hub  # noqa: F401
        return
    except Exception:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"],
            check=True,
        )

ensure_hf_installed()

# =============================================================================
# HELPERS
# =============================================================================
def human_bytes(n: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)
    for unit in units:
        if x < 1024 or unit == units[-1]:
            return f"{x:.2f} {unit}"
        x /= 1024.0
    return f"{n} B"

def dir_size(path: Path) -> int:
    total = 0
    if not path.exists():
        return 0
    for p in path.rglob("*"):
        try:
            if p.is_file() and not p.is_symlink():
                total += p.stat().st_size
        except Exception:
            pass
    return total

def scan_local(target_dir: Path, top_n: int = 12):
    print(f"\n[local-scan] {target_dir}")
    if not target_dir.exists():
        print("  directory does not exist yet")
        return
    files = []
    for p in target_dir.rglob("*"):
        try:
            if p.is_file():
                files.append((p.stat().st_size, p.relative_to(target_dir).as_posix()))
        except Exception:
            pass
    files.sort(reverse=True)
    total = sum(size for size, _ in files)
    incomplete = [(s, r) for s, r in files if r.endswith(".incomplete")]
    locks = [(s, r) for s, r in files if r.endswith(".lock")]
    print(f"  total size : {human_bytes(total)}")
    print(f"  files      : {len(files)}")
    print(f"  incomplete : {len(incomplete)}")
    print(f"  lock files : {len(locks)}")
    print("  top files  :")
    for size, rel in files[:top_n]:
        print(f"    {human_bytes(size):>10}  {rel}")

def print_disk():
    total, used, free = shutil.disk_usage(DOWNLOAD_ROOT)
    print("=" * 100)
    print(f"DOWNLOAD_ROOT : {DOWNLOAD_ROOT}")
    print(f"HF_HOME       : {HF_HOME}")
    print(f"ENDPOINT      : {ENDPOINT}")
    print(f"Disk total    : {human_bytes(total)}")
    print(f"Disk used     : {human_bytes(used)}")
    print(f"Disk free     : {human_bytes(free)}")
    print("=" * 100)

# =============================================================================
# CHILD SCRIPT
# =============================================================================
child_code = r'''
import os
import sys
import ssl
import json
import time
import socket
import logging
import urllib.request
import urllib.error
import threading
from pathlib import Path

# force IPv4 in child
_original_getaddrinfo = socket.getaddrinfo
def ipv4_only_getaddrinfo(host, port, family=0, type=0, proto=0, flags=0):
    results = _original_getaddrinfo(host, port, family, type, proto, flags)
    ipv4 = [r for r in results if r[0] == socket.AF_INET]
    return ipv4 or results
socket.getaddrinfo = ipv4_only_getaddrinfo

ENDPOINT = os.environ["MY_ENDPOINT"]
ENDPOINT_HOST = ENDPOINT.split("://", 1)[-1].split("/", 1)[0]
os.environ["HF_ENDPOINT"] = ENDPOINT
os.environ["HF_HUB_DISABLE_XET"] = "1"

from huggingface_hub import snapshot_download, logging as hf_logging
import huggingface_hub

REPO_ID = os.environ["MY_REPO_ID"]
TARGET_DIR = Path(os.environ["MY_TARGET_DIR"]).resolve()
HF_HOME = Path(os.environ["HF_HOME"]).resolve()
HEARTBEAT_FILE = Path(os.environ["MY_HEARTBEAT_FILE"]).resolve()
HEARTBEAT_INTERVAL = int(os.environ.get("MY_HEARTBEAT_INTERVAL", "15"))
MAX_WORKERS = int(os.environ.get("MY_MAX_WORKERS", "2"))

ALLOW_PATTERNS = json.loads(os.environ["MY_ALLOW_PATTERNS"])
IGNORE_PATTERNS = json.loads(os.environ["MY_IGNORE_PATTERNS"])

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("hf_downloader")
hf_logging.set_verbosity_debug()

def human_bytes(n: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)
    for unit in units:
        if x < 1024 or unit == units[-1]:
            return f"{x:.2f} {unit}"
        x /= 1024.0
    return f"{n} B"

def dir_size(path: Path) -> int:
    total = 0
    if not path.exists():
        return 0
    for p in path.rglob("*"):
        try:
            if p.is_file() and not p.is_symlink():
                total += p.stat().st_size
        except Exception:
            pass
    return total

def tcp_probe(host: str, port: int = 443, timeout: int = 8):
    out = {"host": host, "dns": [], "tcp_ok": False, "peer": None, "error": None}
    try:
        infos = socket.getaddrinfo(host, port, type=socket.SOCK_STREAM)
        out["dns"] = sorted({x[4][0] for x in infos})
    except Exception as e:
        out["error"] = f"DNS {type(e).__name__}: {e}"
        return out
    try:
        with socket.create_connection((host, port), timeout=timeout) as s:
            out["tcp_ok"] = True
            out["peer"] = str(s.getpeername())
    except Exception as e:
        out["error"] = f"TCP {type(e).__name__}: {e}"
    return out

def http_probe(url: str, timeout: int = 12):
    ctx = ssl.create_default_context()
    headers = {"User-Agent": "hf-mirror-heartbeat/1.0"}
    req = urllib.request.Request(url, headers=headers, method="GET")
    out = {"url": url, "ok": False, "status": None, "reason": None, "error": None}
    try:
        with urllib.request.urlopen(req, timeout=timeout, context=ctx) as resp:
            out["ok"] = True
            out["status"] = getattr(resp, "status", None)
            out["reason"] = getattr(resp, "reason", None)
    except urllib.error.HTTPError as e:
        out["status"] = e.code
        out["reason"] = e.reason
        out["error"] = f"HTTPError: {e}"
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e}"
    return out

stop_flag = False

def heartbeat_loop():
    last_total = None
    beat_num = 0
    while not stop_flag:
        beat_num += 1
        target_bytes = dir_size(TARGET_DIR)
        hub_bytes = dir_size(HF_HOME / "hub")
        xet_bytes = dir_size(HF_HOME / "xet")
        total_bytes = target_bytes + hub_bytes + xet_bytes

        if last_total is None:
            delta = 0
            speed = 0.0
        else:
            delta = max(0, total_bytes - last_total)
            speed = delta / HEARTBEAT_INTERVAL / (1024 ** 2)

        payload = {
            "ts": time.time(),
            "beat_num": beat_num,
            "target_bytes": target_bytes,
            "hub_bytes": hub_bytes,
            "xet_bytes": xet_bytes,
            "total_bytes": total_bytes,
            "delta_bytes": delta,
            "mb_per_s": speed,
        }
        HEARTBEAT_FILE.write_text(json.dumps(payload), encoding="utf-8")

        logger.info(
            "[heartbeat #%d] target=%s | hub=%s | xet=%s | total=%s | +%s | %.2f MB/s",
            beat_num,
            human_bytes(target_bytes),
            human_bytes(hub_bytes),
            human_bytes(xet_bytes),
            human_bytes(total_bytes),
            human_bytes(delta),
            speed,
        )

        if beat_num == 1 or beat_num % 4 == 0:
            for host in [ENDPOINT_HOST, "huggingface.co", "cdn-lfs.huggingface.co", "cas-bridge.xethub.hf.co"]:
                logger.info("[probe-tcp] %s", json.dumps(tcp_probe(host), ensure_ascii=False))
            for url in [
                ENDPOINT,
                f"{ENDPOINT}/api/models/{REPO_ID}",
            ]:
                logger.info("[probe-http] %s", json.dumps(http_probe(url), ensure_ascii=False))

        last_total = total_bytes
        time.sleep(HEARTBEAT_INTERVAL)

thread = threading.Thread(target=heartbeat_loop, daemon=True)
thread.start()

try:
    logger.info("huggingface_hub version : %s", getattr(huggingface_hub, "__version__", "unknown"))
    logger.info("Repo                    : %s", REPO_ID)
    logger.info("Endpoint                : %s", ENDPOINT)
    logger.info("Target dir              : %s", TARGET_DIR)
    logger.info("HF_HOME                 : %s", HF_HOME)
    logger.info("Max workers             : %s", MAX_WORKERS)
    logger.info("Token enabled           : False")
    logger.info("HF_HUB_DISABLE_XET      : %s", os.environ.get("HF_HUB_DISABLE_XET"))

    before_target = dir_size(TARGET_DIR)
    before_total = before_target + dir_size(HF_HOME / "hub") + dir_size(HF_HOME / "xet")
    logger.info("[before] target=%s | total=%s", human_bytes(before_target), human_bytes(before_total))

    try:
        logger.info("[dry-run] starting")
        dry = snapshot_download(
            repo_id=REPO_ID,
            repo_type="model",
            token=None,
            allow_patterns=ALLOW_PATTERNS,
            ignore_patterns=IGNORE_PATTERNS,
            dry_run=True,
            max_workers=MAX_WORKERS,
            endpoint=ENDPOINT,
        )
        infos = []
        for item in dry:
            name = (
                getattr(item, "file_name", None)
                or getattr(item, "filename", None)
                or getattr(item, "path", None)
                or str(item)
            )
            size = getattr(item, "size", 0) or 0
            will_download = getattr(item, "will_download", None)
            is_cached = getattr(item, "is_cached", None)
            infos.append((int(size), str(name), will_download, is_cached))
        logger.info("[dry-run] files=%d total=%s", len(infos), human_bytes(sum(x[0] for x in infos)))
        for size, name, will_download, is_cached in sorted(infos, reverse=True)[:20]:
            logger.info(
                "[dry-run-file] %s | will_download=%s | cached=%s | %s",
                human_bytes(size), will_download, is_cached, name
            )
    except Exception as e:
        logger.exception("[dry-run-failed] %s: %s", type(e).__name__, e)

    logger.info("[download] snapshot_download starting")
    local_path = snapshot_download(
        repo_id=REPO_ID,
        repo_type="model",
        local_dir=str(TARGET_DIR),
        token=None,
        allow_patterns=ALLOW_PATTERNS,
        ignore_patterns=IGNORE_PATTERNS,
        max_workers=MAX_WORKERS,
        endpoint=ENDPOINT,
    )
    logger.info("[download] snapshot_download returned: %s", local_path)

    after_target = dir_size(TARGET_DIR)
    after_total = after_target + dir_size(HF_HOME / "hub") + dir_size(HF_HOME / "xet")
    delta = max(0, after_total - before_total)
    logger.info("[after] target=%s | total=%s | delta=%s", human_bytes(after_target), human_bytes(after_total), human_bytes(delta))

    if delta == 0:
        logger.error("[no-progress] snapshot_download returned but no bytes changed")
        print("__DOWNLOAD_NO_PROGRESS__", flush=True)
        sys.exit(2)

    logger.info("[success] download made progress")
    print("__DOWNLOAD_OK__", flush=True)

except Exception as e:
    logger.exception("[download-failed] %s: %s", type(e).__name__, e)
    print("__DOWNLOAD_FAILED__", flush=True)
    raise

finally:
    stop_flag = True
    thread.join(timeout=2)
'''
CHILD_SCRIPT.write_text(child_code, encoding="utf-8")

# =============================================================================
# RUNNER
# =============================================================================
def run_attempt():
    if HEARTBEAT_FILE.exists():
        HEARTBEAT_FILE.unlink()

    log_file = LOG_DIR / f"{MODEL_NAME}.mirror_no_xet.endpoint.log"

    env = os.environ.copy()
    env.update({
        "PYTHONUNBUFFERED": "1",
        "HF_HOME": str(HF_HOME),
        "HF_HUB_CACHE": str(HF_HOME / "hub"),
        "HF_XET_CACHE": str(HF_HOME / "xet"),
        "HF_ASSETS_CACHE": str(HF_HOME / "assets"),
        "HF_HUB_VERBOSITY": "debug",
        "HF_DEBUG": "1",
        "MY_REPO_ID": REPO_ID,
        "MY_TARGET_DIR": str(TARGET_DIR),
        "MY_HEARTBEAT_FILE": str(HEARTBEAT_FILE),
        "MY_HEARTBEAT_INTERVAL": str(HEARTBEAT_SECONDS),
        "MY_ALLOW_PATTERNS": json.dumps(ALLOW_PATTERNS),
        "MY_IGNORE_PATTERNS": json.dumps(IGNORE_PATTERNS),
        "MY_ENDPOINT": ENDPOINT,
        "MY_MAX_WORKERS": str(MAX_WORKERS),
        "HF_ENDPOINT": ENDPOINT,
        "HF_HUB_DISABLE_XET": "1",
    })
    env.pop("HF_TOKEN", None)

    print("\n" + "=" * 100)
    print("ATTEMPT  : mirror_no_xet")
    print(f"ENDPOINT : {ENDPOINT}")
    print("TOKEN    : disabled")
    print(f"REPO     : {REPO_ID}")
    print(f"TARGET   : {TARGET_DIR}")
    print(f"LOG FILE : {log_file}")
    print("=" * 100)

    with open(log_file, "w", encoding="utf-8") as log_handle:
        proc = subprocess.Popen(
            [sys.executable, "-u", str(CHILD_SCRIPT)],
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            env=env,
        )

    last_print_pos = 0
    last_growth_time = time.time()
    last_total_bytes = -1
    saw_ok = False
    saw_no_progress = False
    saw_failed = False

    while True:
        time.sleep(5)

        if log_file.exists():
            with open(log_file, "r", encoding="utf-8", errors="replace") as f:
                f.seek(last_print_pos)
                chunk = f.read()
                if chunk:
                    print(chunk, end="")
                    last_print_pos = f.tell()
                    if "__DOWNLOAD_OK__" in chunk:
                        saw_ok = True
                    if "__DOWNLOAD_NO_PROGRESS__" in chunk:
                        saw_no_progress = True
                    if "__DOWNLOAD_FAILED__" in chunk:
                        saw_failed = True

        if HEARTBEAT_FILE.exists():
            hb = json.loads(HEARTBEAT_FILE.read_text(encoding="utf-8"))
            total_bytes = int(hb.get("total_bytes", 0))
            delta_bytes = int(hb.get("delta_bytes", 0))
            mbps = float(hb.get("mb_per_s", 0.0))
            beat_num = hb.get("beat_num", "?")
            print(
                f"[parent-heartbeat #{beat_num}] total={human_bytes(total_bytes)} "
                f"| delta={human_bytes(delta_bytes)} | {mbps:.2f} MB/s"
            )
            if total_bytes > last_total_bytes:
                last_total_bytes = total_bytes
                last_growth_time = time.time()
            elif time.time() - last_growth_time > STALL_SECONDS:
                print(f"\n[watchdog] No byte growth for {STALL_SECONDS}s. Terminating.")
                proc.terminate()
                try:
                    proc.wait(timeout=20)
                except subprocess.TimeoutExpired:
                    proc.kill()
                break

        if proc.poll() is not None:
            time.sleep(1)
            if log_file.exists():
                with open(log_file, "r", encoding="utf-8", errors="replace") as f:
                    f.seek(last_print_pos)
                    chunk = f.read()
                    if chunk:
                        print(chunk, end="")
                        if "__DOWNLOAD_OK__" in chunk:
                            saw_ok = True
                        if "__DOWNLOAD_NO_PROGRESS__" in chunk:
                            saw_no_progress = True
                        if "__DOWNLOAD_FAILED__" in chunk:
                            saw_failed = True
            break

    rc = proc.poll()
    if saw_ok and rc == 0:
        return True, "success", log_file
    if saw_no_progress:
        return False, "no progress / mirror inaccessible", log_file
    if saw_failed:
        return False, f"failed rc={rc}", log_file
    return False, f"ended rc={rc}", log_file

# =============================================================================
# MAIN
# =============================================================================
print_disk()
scan_local(TARGET_DIR)
removed = remove_stale_locks(TARGET_DIR)
if removed:
    print(f"[cleanup] removed {len(removed)} stale lock files")

ok, reason, log_path = run_attempt()
print(f"\nResult: mirror_no_xet -> {reason}")
print(f"Logs  : {log_path}")
scan_local(TARGET_DIR)

if ok:
    print("\nDOWNLOAD FINISHED.")
else:
    print("\nMirror attempt ended without progress. Inspect the log above.")

DOWNLOAD_ROOT : /data/downloaded_models
HF_HOME       : /data/downloaded_models/.hf_home
ENDPOINT      : https://hf-mirror.com
Disk total    : 3.58 TB
Disk used     : 1.69 TB
Disk free     : 1.70 TB

[local-scan] /data/downloaded_models/Mixtral-8x7B-v0.1
  total size : 64.62 GB
  files      : 65
  incomplete : 6
  lock files : 17
  top files  :
       4.64 GB  model-00015-of-00019.safetensors
       4.64 GB  model-00013-of-00019.safetensors
       4.64 GB  model-00012-of-00019.safetensors
       4.64 GB  model-00009-of-00019.safetensors
       4.64 GB  model-00008-of-00019.safetensors
       4.64 GB  model-00006-of-00019.safetensors
       4.64 GB  model-00005-of-00019.safetensors
       4.64 GB  model-00003-of-00019.safetensors
       4.64 GB  model-00002-of-00019.safetensors
       4.56 GB  model-00010-of-00019.safetensors
       4.56 GB  model-00007-of-00019.safetensors
       4.56 GB  model-00004-of-00019.safetensors
[cleanup] removed 17 stale lock files

ATTEMPT  : mirror_no_xet
E

In [ ]:
#MORE TESTS

In [12]:
import os
import sys
import json
import time
import socket
import shutil
import threading
import subprocess
from pathlib import Path
from getpass import getpass

# =============================================================================
# CONFIG
# =============================================================================
REPO_ID = "mistralai/Mixtral-8x7B-v0.1"
DOWNLOAD_ROOT = Path("/data/downloaded_models").resolve()
MODEL_NAME = REPO_ID.split("/")[-1]
TARGET_DIR = DOWNLOAD_ROOT / MODEL_NAME

HF_HOME = (DOWNLOAD_ROOT / ".hf_home").resolve()
LOG_DIR = (DOWNLOAD_ROOT / "_logs").resolve()
HEARTBEAT_FILE = LOG_DIR / f"{MODEL_NAME}.heartbeat.json"
CHILD_SCRIPT = LOG_DIR / "hf_download_child_verbose.py"

HEARTBEAT_SECONDS = 15
STALL_SECONDS = 900

# two modes, both fully logged
ATTEMPTS = [
    ("xet_balanced", {
        "MY_MAX_WORKERS": "2",
        "HF_XET_NUM_CONCURRENT_RANGE_GETS": "2",
    }),
    ("http_no_xet", {
        "MY_MAX_WORKERS": "2",
        "HF_HUB_DISABLE_XET": "1",
    }),
]

ALLOW_PATTERNS = [
    "*.safetensors",
    "*.json",
    "*.model",
    "*.py",
    "*.txt",
    "*.tiktoken",
    "*.merges",
    "*.vocab",
    "*.jinja",
    "tokenizer*",
    "special_tokens_map*",
    "generation_config*",
    "configuration_*",
    "modeling_*",
    "tokenization_*",
    "processing_*",
]

IGNORE_PATTERNS = [
    "consolidated*.pt",
    "*.pt",
    "*.pth",
    "*.ckpt",
    "*.onnx",
    "*.h5",
    "*.msgpack",
    "*.gguf",
    "original/*",
    "**/original/*",
]

# =============================================================================
# SETUP
# =============================================================================
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
TARGET_DIR.mkdir(parents=True, exist_ok=True)
(HF_HOME / "hub").mkdir(parents=True, exist_ok=True)
(HF_HOME / "xet").mkdir(parents=True, exist_ok=True)
(HF_HOME / "assets").mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("Paste HF token (hidden): ").strip()
if not HF_TOKEN:
    raise ValueError("No HF token provided.")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["HF_XET_CACHE"] = str(HF_HOME / "xet")
os.environ["HF_ASSETS_CACHE"] = str(HF_HOME / "assets")
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

def ensure_hf_installed():
    try:
        import huggingface_hub  # noqa: F401
        return
    except Exception:
        pass
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"],
        check=True,
    )

ensure_hf_installed()

# =============================================================================
# HELPERS
# =============================================================================
def human_bytes(n: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)
    for unit in units:
        if x < 1024 or unit == units[-1]:
            return f"{x:.2f} {unit}"
        x /= 1024.0

def dir_size(path: Path) -> int:
    total = 0
    if not path.exists():
        return 0
    for p in path.rglob("*"):
        try:
            if p.is_file() and not p.is_symlink():
                total += p.stat().st_size
        except Exception:
            pass
    return total

def scan_local(target_dir: Path, top_n: int = 12):
    print(f"\n[local-scan] {target_dir}")
    if not target_dir.exists():
        print("  directory does not exist yet")
        return
    files = []
    for p in target_dir.rglob("*"):
        try:
            if p.is_file():
                files.append((p.stat().st_size, p.relative_to(target_dir).as_posix()))
        except Exception:
            pass
    files.sort(reverse=True)
    total = sum(size for size, _ in files)
    incomplete = [(s, r) for s, r in files if r.endswith(".incomplete")]
    locks = [(s, r) for s, r in files if r.endswith(".lock")]
    print(f"  total size : {human_bytes(total)}")
    print(f"  files      : {len(files)}")
    print(f"  incomplete : {len(incomplete)}")
    print(f"  lock files : {len(locks)}")
    print("  top files  :")
    for size, rel in files[:top_n]:
        print(f"    {human_bytes(size):>10}  {rel}")

def remove_stale_locks(target_dir: Path, older_than_sec: int = 1800):
    removed = []
    now = time.time()
    if not target_dir.exists():
        return removed
    for p in target_dir.rglob("*.lock"):
        try:
            if now - p.stat().st_mtime >= older_than_sec:
                p.unlink()
                removed.append(str(p))
        except Exception:
            pass
    return removed

def print_disk():
    total, used, free = shutil.disk_usage(DOWNLOAD_ROOT)
    print("=" * 100)
    print(f"DOWNLOAD_ROOT : {DOWNLOAD_ROOT}")
    print(f"HF_HOME       : {HF_HOME}")
    print(f"Disk total    : {human_bytes(total)}")
    print(f"Disk used     : {human_bytes(used)}")
    print(f"Disk free     : {human_bytes(free)}")
    print("=" * 100)

# =============================================================================
# CHILD SCRIPT
# =============================================================================
child_code = r'''
import os
import sys
import ssl
import json
import time
import socket
import logging
import urllib.request
import urllib.error
from pathlib import Path

# -------------------------------------------------------------------------
# FORCE IPv4 FIRST/ONLY INSIDE CHILD PROCESS
# -------------------------------------------------------------------------
_original_getaddrinfo = socket.getaddrinfo
def ipv4_only_getaddrinfo(host, port, family=0, type=0, proto=0, flags=0):
    results = _original_getaddrinfo(host, port, family, type, proto, flags)
    ipv4 = [r for r in results if r[0] == socket.AF_INET]
    return ipv4 or results
socket.getaddrinfo = ipv4_only_getaddrinfo

from huggingface_hub import snapshot_download, logging as hf_logging
import huggingface_hub

REPO_ID = os.environ["MY_REPO_ID"]
TARGET_DIR = Path(os.environ["MY_TARGET_DIR"]).resolve()
HF_HOME = Path(os.environ["HF_HOME"]).resolve()
HEARTBEAT_FILE = Path(os.environ["MY_HEARTBEAT_FILE"]).resolve()
HEARTBEAT_INTERVAL = int(os.environ.get("MY_HEARTBEAT_INTERVAL", "15"))
MAX_WORKERS = int(os.environ.get("MY_MAX_WORKERS", "2"))
TOKEN = os.environ.get("HF_TOKEN")

ALLOW_PATTERNS = json.loads(os.environ["MY_ALLOW_PATTERNS"])
IGNORE_PATTERNS = json.loads(os.environ["MY_IGNORE_PATTERNS"])

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("hf_downloader")
hf_logging.set_verbosity_debug()

def human_bytes(n: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)
    for unit in units:
        if x < 1024 or unit == units[-1]:
            return f"{x:.2f} {unit}"
        x /= 1024.0
    return f"{n} B"

def dir_size(path: Path) -> int:
    total = 0
    if not path.exists():
        return 0
    for p in path.rglob("*"):
        try:
            if p.is_file() and not p.is_symlink():
                total += p.stat().st_size
        except Exception:
            pass
    return total

def tcp_probe(host: str, port: int = 443, timeout: int = 8):
    out = {"host": host, "dns": [], "tcp_ok": False, "peer": None, "error": None}
    try:
        infos = socket.getaddrinfo(host, port, type=socket.SOCK_STREAM)
        out["dns"] = sorted({x[4][0] for x in infos})
    except Exception as e:
        out["error"] = f"DNS {type(e).__name__}: {e}"
        return out

    try:
        with socket.create_connection((host, port), timeout=timeout) as s:
            out["tcp_ok"] = True
            out["peer"] = str(s.getpeername())
    except Exception as e:
        out["error"] = f"TCP {type(e).__name__}: {e}"
    return out

def http_probe(url: str, timeout: int = 12, token: str | None = None):
    ctx = ssl.create_default_context()
    headers = {"User-Agent": "hf-debug-heartbeat/1.0"}
    if token and "whoami" in url:
        headers["Authorization"] = f"Bearer {token}"
    req = urllib.request.Request(url, headers=headers, method="GET")
    out = {"url": url, "ok": False, "status": None, "reason": None, "error": None}
    try:
        with urllib.request.urlopen(req, timeout=timeout, context=ctx) as resp:
            out["ok"] = True
            out["status"] = getattr(resp, "status", None)
            out["reason"] = getattr(resp, "reason", None)
    except urllib.error.HTTPError as e:
        out["status"] = e.code
        out["reason"] = e.reason
        out["error"] = f"HTTPError: {e}"
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e}"
    return out

stop_flag = False

def heartbeat_loop():
    last_total = None
    beat_num = 0
    while not stop_flag:
        beat_num += 1
        target_bytes = dir_size(TARGET_DIR)
        hub_bytes = dir_size(HF_HOME / "hub")
        xet_bytes = dir_size(HF_HOME / "xet")
        total_bytes = target_bytes + hub_bytes + xet_bytes

        if last_total is None:
            delta = 0
            speed = 0.0
        else:
            delta = max(0, total_bytes - last_total)
            speed = delta / HEARTBEAT_INTERVAL / (1024 ** 2)

        payload = {
            "ts": time.time(),
            "beat_num": beat_num,
            "target_bytes": target_bytes,
            "hub_bytes": hub_bytes,
            "xet_bytes": xet_bytes,
            "total_bytes": total_bytes,
            "delta_bytes": delta,
            "mb_per_s": speed,
        }
        try:
            HEARTBEAT_FILE.write_text(json.dumps(payload), encoding="utf-8")
        except Exception as e:
            logger.warning("[heartbeat-write-failed] %s", e)

        logger.info(
            "[heartbeat #%d] target=%s | hub=%s | xet=%s | total=%s | +%s | %.2f MB/s",
            beat_num,
            human_bytes(target_bytes),
            human_bytes(hub_bytes),
            human_bytes(xet_bytes),
            human_bytes(total_bytes),
            human_bytes(delta),
            speed,
        )

        # full connectivity snapshot every 4 heartbeats
        if beat_num == 1 or beat_num % 4 == 0:
            for host in ["huggingface.co", "cdn-lfs.huggingface.co", "cas-bridge.xethub.hf.co"]:
                p = tcp_probe(host)
                logger.info("[probe-tcp] %s", json.dumps(p, ensure_ascii=False))
            for url in [
                "https://huggingface.co",
                f"https://huggingface.co/api/models/{REPO_ID}",
                "https://huggingface.co/api/whoami-v2",
                "https://cdn-lfs.huggingface.co",
            ]:
                p = http_probe(url, token=TOKEN)
                logger.info("[probe-http] %s", json.dumps(p, ensure_ascii=False))

        last_total = total_bytes
        time.sleep(HEARTBEAT_INTERVAL)

import threading
thread = threading.Thread(target=heartbeat_loop, daemon=True)
thread.start()

try:
    logger.info("huggingface_hub version : %s", getattr(huggingface_hub, "__version__", "unknown"))
    logger.info("Repo                    : %s", REPO_ID)
    logger.info("Target dir              : %s", TARGET_DIR)
    logger.info("HF_HOME                 : %s", HF_HOME)
    logger.info("Max workers             : %s", MAX_WORKERS)
    logger.info("HF_HUB_DISABLE_XET      : %s", os.environ.get("HF_HUB_DISABLE_XET"))
    logger.info("HF_XET_NUM_CONCURRENT_RANGE_GETS : %s", os.environ.get("HF_XET_NUM_CONCURRENT_RANGE_GETS"))

    before_target = dir_size(TARGET_DIR)
    before_total = before_target + dir_size(HF_HOME / "hub") + dir_size(HF_HOME / "xet")
    logger.info("[before] target=%s | total=%s", human_bytes(before_target), human_bytes(before_total))

    # try dry-run first for manifest visibility
    try:
        logger.info("[dry-run] starting")
        dry = snapshot_download(
            repo_id=REPO_ID,
            repo_type="model",
            token=TOKEN,
            allow_patterns=ALLOW_PATTERNS,
            ignore_patterns=IGNORE_PATTERNS,
            dry_run=True,
            max_workers=MAX_WORKERS,
        )
        infos = []
        for item in dry:
            name = (
                getattr(item, "file_name", None)
                or getattr(item, "filename", None)
                or getattr(item, "path", None)
                or str(item)
            )
            size = getattr(item, "size", 0) or 0
            will_download = getattr(item, "will_download", None)
            is_cached = getattr(item, "is_cached", None)
            infos.append((int(size), str(name), will_download, is_cached))
        logger.info("[dry-run] files=%d total=%s", len(infos), human_bytes(sum(x[0] for x in infos)))
        for size, name, will_download, is_cached in sorted(infos, reverse=True)[:20]:
            logger.info(
                "[dry-run-file] %s | will_download=%s | cached=%s | %s",
                human_bytes(size), will_download, is_cached, name
            )
    except Exception as e:
        logger.exception("[dry-run-failed] %s: %s", type(e).__name__, e)

    logger.info("[download] snapshot_download starting")
    local_path = snapshot_download(
        repo_id=REPO_ID,
        repo_type="model",
        local_dir=str(TARGET_DIR),
        token=TOKEN,
        allow_patterns=ALLOW_PATTERNS,
        ignore_patterns=IGNORE_PATTERNS,
        max_workers=MAX_WORKERS,
    )
    logger.info("[download] snapshot_download returned: %s", local_path)

    after_target = dir_size(TARGET_DIR)
    after_total = after_target + dir_size(HF_HOME / "hub") + dir_size(HF_HOME / "xet")
    delta = max(0, after_total - before_total)
    logger.info("[after] target=%s | total=%s | delta=%s", human_bytes(after_target), human_bytes(after_total), human_bytes(delta))

    if delta == 0:
        logger.error("[no-progress] snapshot_download returned but no bytes changed")
        print("__DOWNLOAD_NO_PROGRESS__", flush=True)
        sys.exit(2)

    logger.info("[success] download made progress")
    print("__DOWNLOAD_OK__", flush=True)

except Exception as e:
    logger.exception("[download-failed] %s: %s", type(e).__name__, e)
    print("__DOWNLOAD_FAILED__", flush=True)
    raise

finally:
    stop_flag = True
    thread.join(timeout=2)
'''
CHILD_SCRIPT.write_text(child_code, encoding="utf-8")

# =============================================================================
# RUNNER
# =============================================================================
def run_attempt(attempt_name: str, extra_env: dict):
    if HEARTBEAT_FILE.exists():
        HEARTBEAT_FILE.unlink()

    log_file = LOG_DIR / f"{MODEL_NAME}.{attempt_name}.verbose.log"

    env = os.environ.copy()
    env.update({
        "PYTHONUNBUFFERED": "1",
        "HF_TOKEN": HF_TOKEN,
        "HF_HOME": str(HF_HOME),
        "HF_HUB_CACHE": str(HF_HOME / "hub"),
        "HF_XET_CACHE": str(HF_HOME / "xet"),
        "HF_ASSETS_CACHE": str(HF_HOME / "assets"),
        "HF_HUB_VERBOSITY": "debug",
        "HF_DEBUG": "1",
        "MY_REPO_ID": REPO_ID,
        "MY_TARGET_DIR": str(TARGET_DIR),
        "MY_HEARTBEAT_FILE": str(HEARTBEAT_FILE),
        "MY_HEARTBEAT_INTERVAL": str(HEARTBEAT_SECONDS),
        "MY_ALLOW_PATTERNS": json.dumps(ALLOW_PATTERNS),
        "MY_IGNORE_PATTERNS": json.dumps(IGNORE_PATTERNS),
    })

    for key in [
        "HF_HUB_DISABLE_XET",
        "HF_XET_NUM_CONCURRENT_RANGE_GETS",
        "MY_MAX_WORKERS",
    ]:
        env.pop(key, None)

    env.update(extra_env)

    print("\n" + "=" * 100)
    print(f"ATTEMPT  : {attempt_name}")
    print(f"REPO     : {REPO_ID}")
    print(f"TARGET   : {TARGET_DIR}")
    print(f"LOG FILE : {log_file}")
    print("=" * 100)

    with open(log_file, "w", encoding="utf-8") as log_handle:
        proc = subprocess.Popen(
            [sys.executable, "-u", str(CHILD_SCRIPT)],
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            env=env,
        )

    last_print_pos = 0
    last_growth_time = time.time()
    last_total_bytes = -1
    saw_ok = False
    saw_no_progress = False
    saw_failed = False

    while True:
        time.sleep(5)

        # live-tail the log
        if log_file.exists():
            with open(log_file, "r", encoding="utf-8", errors="replace") as f:
                f.seek(last_print_pos)
                chunk = f.read()
                if chunk:
                    print(chunk, end="")
                    last_print_pos = f.tell()
                    if "__DOWNLOAD_OK__" in chunk:
                        saw_ok = True
                    if "__DOWNLOAD_NO_PROGRESS__" in chunk:
                        saw_no_progress = True
                    if "__DOWNLOAD_FAILED__" in chunk:
                        saw_failed = True

        # parent-side heartbeat display
        if HEARTBEAT_FILE.exists():
            try:
                hb = json.loads(HEARTBEAT_FILE.read_text(encoding="utf-8"))
                total_bytes = int(hb.get("total_bytes", 0))
                delta_bytes = int(hb.get("delta_bytes", 0))
                mbps = float(hb.get("mb_per_s", 0.0))
                beat_num = hb.get("beat_num", "?")

                print(
                    f"[parent-heartbeat #{beat_num}] total={human_bytes(total_bytes)} "
                    f"| delta={human_bytes(delta_bytes)} | {mbps:.2f} MB/s"
                )

                if total_bytes > last_total_bytes:
                    last_total_bytes = total_bytes
                    last_growth_time = time.time()
                elif time.time() - last_growth_time > STALL_SECONDS:
                    print(f"\n[watchdog] No byte growth for {STALL_SECONDS}s. Terminating attempt {attempt_name}.")
                    proc.terminate()
                    try:
                        proc.wait(timeout=20)
                    except subprocess.TimeoutExpired:
                        proc.kill()
                    break
            except Exception as e:
                print(f"[parent-heartbeat-read-failed] {e}")

        rc = proc.poll()
        if rc is not None:
            time.sleep(1)
            if log_file.exists():
                with open(log_file, "r", encoding="utf-8", errors="replace") as f:
                    f.seek(last_print_pos)
                    chunk = f.read()
                    if chunk:
                        print(chunk, end="")
                        if "__DOWNLOAD_OK__" in chunk:
                            saw_ok = True
                        if "__DOWNLOAD_NO_PROGRESS__" in chunk:
                            saw_no_progress = True
                        if "__DOWNLOAD_FAILED__" in chunk:
                            saw_failed = True
            break

    rc = proc.poll()
    if saw_ok and rc == 0:
        return True, "success", log_file
    if saw_no_progress:
        return False, "no progress / remote inaccessible", log_file
    if saw_failed:
        return False, f"failed rc={rc}", log_file
    return False, f"ended rc={rc}", log_file

# =============================================================================
# MAIN
# =============================================================================
print_disk()
scan_local(TARGET_DIR)

removed = remove_stale_locks(TARGET_DIR)
if removed:
    print(f"[cleanup] removed {len(removed)} stale lock files")

for attempt_name, extra_env in ATTEMPTS:
    ok, reason, log_path = run_attempt(attempt_name, extra_env)
    print(f"\nResult: {attempt_name} -> {reason}")
    print(f"Logs  : {log_path}")
    scan_local(TARGET_DIR)
    if ok:
        print("\nDOWNLOAD FINISHED.")
        break
else:
    print("\nAll attempts ended without progress. Inspect the log file above.")

DOWNLOAD_ROOT : /data/downloaded_models
HF_HOME       : /data/downloaded_models/.hf_home
Disk total    : 3.58 TB
Disk used     : 1.65 TB
Disk free     : 1.75 TB

[local-scan] /data/downloaded_models/Mixtral-8x7B-v0.1
  total size : 20.68 GB
  files      : 35
  incomplete : 19
  lock files : 0
  top files  :
       3.75 GB  .cache/huggingface/download/WF-CC5TTTrpS1LERXMuFuVhfQ9c=.812448ac711f5eb3b61b576e9b3e7fda7d8755e2e3158c90b819f990db212d82.incomplete
       3.75 GB  .cache/huggingface/download/M_a3Xj6Xxe8WVi6BvKeo5-44Lg8=.365ab6e5929deee45e2fe52c8100b2df6221179af2a909d2efb97da70173cfc8.incomplete
       2.94 GB  .cache/huggingface/download/8lNhEa26XTKaCyuzbIqeZ8fVdY8=.1a4ea032aaa803bb700d80e94cfe629db3d54fe1f8b47f11dfd4729275622e30.incomplete
       2.93 GB  .cache/huggingface/download/skutyITmWYgwlJ8ckTqI3JhQ1Mw=.b43400ce8695edf089a8c276245f670ef0f39e798688e7cf9256636c8f5da053.incomplete
       2.44 GB  .cache/huggingface/download/qsLdM6qrdARfUGrT3oVfyV2EIoE=.1dcf00f31d7e301292a6f4

KeyboardInterrupt: 

In [ ]:
# MAIN BODY CALCULATIONS ===========================================================================================================



In [30]:

pip install "transformers>=4.46.0" "huggingface-hub>=1.5.0"

Note: you may need to restart the kernel to use updated packages.


In [1]:
#!/usr/bin/env python3
# =============================================================================
# EBC-LLM: Expert-Bank Compression via Cluster-Shared Rotation and
#          Runtime-Aligned Structured Payloads
#
# Single-file offline compression and evaluation pipeline.
# Supports DeepSeek, AllenAI, Mixtral, and other MoE models.
#
# Usage:
#   python ebc_llm_compression.py
#
# Environment variables (see Cfg dataclass for all options):
#   MODEL_DIR=/path/to/model
#   OUTPUT_DIR=/path/to/output
#   LAYER=1
#   MAX_EXPERTS=16
#   CALIB_PATH=/path/to/calib_X.npz      (optional; auto-capture if missing)
#   ROUTER_PATH=/path/to/router_P.npz    (optional)
#   PRESET=balanced|maxacc|compact
# =============================================================================

import os, re, json, math, time, random, sys
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs): return x

# -----------------------------------------------------------------------------
# Environment helpers
# -----------------------------------------------------------------------------
def _env_str(k: str, d: str) -> str:
    return os.environ.get(k, d)

def _env_int(k: str, d: int) -> int:
    try: return int(os.environ.get(k, str(d)))
    except: return d

def _env_float(k: str, d: float) -> float:
    try: return float(os.environ.get(k, str(d)))
    except: return d

def _env_bool(k: str, d: bool) -> bool:
    v = os.environ.get(k, None)
    if v is None: return d
    return v.strip().lower() in ("1", "true", "yes", "y", "on")

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
@dataclass
class Cfg:
    # Paths
    MODEL_DIR: str = "/data/downloaded_models/Mixtral-8x7B-v0.1"
    OUTPUT_DIR: str = "/home/daniyar/moe_ws_outputs"

    # Model slice
    LAYER: int = 0
    MAX_EXPERTS: int = 8   # Mixtral-8x7B has exactly 8 experts per layer

    # Calibration / router
    CALIB_PATH: str = _env_str("CALIB_PATH", "").strip()
    ROUTER_PATH: str = _env_str("ROUTER_PATH", "").strip()
    CALIB_SAMPLES: int = _env_int("CALIB_SAMPLES", 4096)
    RIDGE_WEIGHTED: bool = _env_bool("RIDGE_WEIGHTED", False)
    ROUTER_EIDS_ARE_GLOBAL: bool = _env_bool("ROUTER_EIDS_ARE_GLOBAL", True)
    RIDGE_DAMP: float = _env_float("RIDGE_DAMP", 1e-3)
    NORMALIZE_W: bool = _env_bool("NORMALIZE_W", True)

    # Capture (optional) – SET THIS TO True IF NO CALIB_PATH
    CAPTURE_ENABLE: bool = True   # <-- CHANGED: auto-collect real calibration
    CAPTURE_FORCE: bool = _env_bool("CAPTURE_FORCE", False)
    CAPTURE_ITERS: int = _env_int("CAPTURE_ITERS", 32)
    CAPTURE_BATCH: int = _env_int("CAPTURE_BATCH", 1)
    CAPTURE_MAX_TOKENS: int = _env_int("CAPTURE_MAX_TOKENS", 1024)
    CAPTURE_TEXT: str = _env_str("CAPTURE_TEXT", "DeepSeek MoE calibration text. " * 256)
    CAPTURE_TEXT_FILE: str = _env_str("CAPTURE_TEXT_FILE", "").strip()
    CAPTURE_KEEP_PAD: bool = _env_bool("CAPTURE_KEEP_PAD", False)
    HF_TRUST_REMOTE_CODE: bool = _env_bool("HF_TRUST_REMOTE_CODE", True)
    HF_LOCAL_FILES_ONLY: bool = _env_bool("HF_LOCAL_FILES_ONLY", True)
    HF_AUTO_PIP: bool = _env_bool("HF_AUTO_PIP", False)

    # Basis mode
    BASIS_MODE: str = _env_str("BASIS_MODE", "dense_train").lower()  # dense_train | identity | hadamard_perm
    BASIS_STORE_DTYPE: str = _env_str("BASIS_STORE_DTYPE", "float16").lower()

    # Clustering
    M0: int = _env_int("M0", 0)                # 0 = auto
    M_MAX: int = _env_int("M_MAX", 16)
    CLUSTER_FEAT_D: int = _env_int("CLUSTER_FEAT_D", 64)
    CLUSTER_ITERS: int = _env_int("CLUSTER_ITERS", 60)
    CLUSTER_RESTARTS: int = _env_int("CLUSTER_RESTARTS", 4)
    CLUSTER_MIN_SIZE: int = _env_int("CLUSTER_MIN_SIZE", 2)
    CLUSTER_MAX_SIZE: int = _env_int("CLUSTER_MAX_SIZE", 4)
    SPLIT_ITERS: int = _env_int("SPLIT_ITERS", 50)

    # Training (dense bases)
    TRAIN_STEPS: int = _env_int("TRAIN_STEPS", 24)
    TRAIN_WARMUP: int = _env_int("TRAIN_WARMUP", 6)
    TRAIN_LR: float = _env_float("TRAIN_LR", 5e-2)
    SUBM: int = _env_int("SUBM", 256)
    BATCH_E: int = _env_int("BATCH_E", 4)
    TRAIN_MIN_CLUSTER: int = _env_int("TRAIN_MIN_CLUSTER", 2)
    REORTHO_EVERY: int = _env_int("REORTHO_EVERY", 4)
    REPORT_EVERY: int = _env_int("REPORT_EVERY", 4)
    GRAD_CLIP: float = _env_float("GRAD_CLIP", 1.0)
    TRAIN_OBJ: str = _env_str("TRAIN_OBJ", "logratio").lower()
    TRAIN_LAM_BLOCK: float = _env_float("TRAIN_LAM_BLOCK", 0.10)
    TRAIN_LAM_GUIDE: float = _env_float("TRAIN_LAM_GUIDE", 1.0)
    TRAIN_GUIDE_EVERY: int = _env_int("TRAIN_GUIDE_EVERY", 2)
    TRAIN_GUIDE_TARGET: float = _env_float("TRAIN_GUIDE_TARGET", 0.80)
    TRAIN_GUIDE_MAX_BLOCKS: int = _env_int("TRAIN_GUIDE_MAX_BLOCKS", 2048)

    # Core selection
    CORE_MODE: str = _env_str("CORE_MODE", "blocktopk_perexpert").lower()
    CORE_AGG: str = _env_str("CORE_AGG", "mean").lower()
    CORE_BLOCK: int = _env_int("CORE_BLOCK", 64)
    CORE_TARGET: float = _env_float("CORE_TARGET", 0.85)
    CORE_MAX_BLOCKS: int = _env_int("CORE_MAX_BLOCKS", 256)

    # Residual
    RES_RANK: int = _env_int("RES_RANK", 512)
    RES_COEF: str = _env_str("RES_COEF", "diag").lower()
    RES_TARGET: float = _env_float("RES_TARGET", 0.995)
    RES_MAX_BLOCKS: int = _env_int("RES_MAX_BLOCKS", 4096)
    RES_BSIZE: int = _env_int("RES_BSIZE", 64)

    # Refine
    REFINE_ENABLE: bool = _env_bool("REFINE_ENABLE", True)
    REFINE_ERR_TARGET: float = _env_float("REFINE_ERR_TARGET", 0.03)
    REFINE_MAX_EXTRA: int = _env_int("REFINE_MAX_EXTRA", 4096)
    REFINE_BSIZE: int = _env_int("REFINE_BSIZE", 64)
    REFINE_RECHECK_EVERY: int = _env_int("REFINE_RECHECK_EVERY", 32)

    # Quantization
    QMODE: str = _env_str("QMODE", "none").lower()  # none|float16|int8

    # Eval
    EVAL_TRIALS: int = _env_int("EVAL_TRIALS", 8)
    EVAL_BATCH: int = _env_int("EVAL_BATCH", 2)
    ROUTED_K: int = _env_int("ROUTED_K", 8)

cfg = Cfg()
PRESET = _env_str("PRESET", "").strip().lower()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Apply presets (override only if user did not set explicitly)
def _setdefault_env(k: str, v: str):
    if k not in os.environ: os.environ[k] = v

if PRESET == "maxacc":
    _setdefault_env("CALIB_SAMPLES", "32768")
    _setdefault_env("RIDGE_DAMP", "1e-2")
    _setdefault_env("CORE_BLOCK", "32")
    _setdefault_env("CORE_TARGET", "0.995")
    _setdefault_env("CORE_MAX_BLOCKS", "8192")
    _setdefault_env("RES_RANK", "2048")
    _setdefault_env("RES_COEF", "full")
    _setdefault_env("RES_TARGET", "0.999")
    _setdefault_env("RES_MAX_BLOCKS", "32768")
    _setdefault_env("REFINE_ENABLE", "1")
    _setdefault_env("REFINE_ERR_TARGET", "0.01")
    _setdefault_env("REFINE_MAX_EXTRA", "65536")
    _setdefault_env("TRAIN_STEPS", "96")
    _setdefault_env("TRAIN_LR", "0.02")
    _setdefault_env("TRAIN_LAM_GUIDE", "0.5")
    cfg = Cfg()
elif PRESET == "compact":
    _setdefault_env("CALIB_SAMPLES", "4096")
    _setdefault_env("CORE_BLOCK", "64")
    _setdefault_env("CORE_TARGET", "0.90")
    _setdefault_env("CORE_MAX_BLOCKS", "512")
    _setdefault_env("RES_RANK", "512")
    _setdefault_env("RES_COEF", "diag")
    _setdefault_env("RES_TARGET", "0.99")
    _setdefault_env("RES_MAX_BLOCKS", "4096")
    _setdefault_env("QMODE", "float16")
    _setdefault_env("REFINE_ENABLE", "0")
    _setdefault_env("TRAIN_STEPS", "24")
    cfg = Cfg()

# -----------------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------------
def log(msg: str): print(msg, flush=True)
def now() -> str: return time.strftime("%Y-%m-%d %H:%M:%S")

def seed_all(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

SEED = _env_int("SEED", 1234)
seed_all(SEED)
NTHREADS = _env_int("KTXX_THREADS", 8)
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
try: torch.set_num_threads(NTHREADS)
except: pass

DEVICE = torch.device(_env_str("DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
DTYPE_ACC = torch.float32

# -----------------------------------------------------------------------------
# NPZ I/O
# -----------------------------------------------------------------------------
def save_npz_compressed(path: str, arrays: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, **arrays)

def load_npz(path: str) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    return {k: z[k] for k in z.files}

def _encode_meta(meta: dict) -> np.ndarray:
    return np.frombuffer(json.dumps(meta, sort_keys=True).encode("utf-8"), dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try: return json.loads(bytes(arr.tolist()).decode("utf-8"))
    except: return {}

# -----------------------------------------------------------------------------
# Offline shard loading
# -----------------------------------------------------------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r") as f:
        return json.load(f).get("weight_map", {})

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    # Try both common MoE patterns:
    #   - DeepSeek style: model.layers.{L}.mlp.experts.{E}.*
    #   - Mixtral style:  model.layers.{L}.block_sparse_moe.experts.{E}.*
    patterns = [
        rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.",
        rf"^model\.layers\.{layer}\.block_sparse_moe\.experts\.(\d+)\.",
    ]
    ids = set()
    for pat_str in patterns:
        pat = re.compile(pat_str)
        for k in weight_map:
            m = pat.match(k)
            if m:
                ids.add(int(m.group(1)))
        if ids:
            break
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    # Determine which MoE prefix is present
    prefixes = [
        f"model.layers.{layer}.mlp.experts.{eid}.",
        f"model.layers.{layer}.block_sparse_moe.experts.{eid}.",
    ]
    used_prefix = None
    for pfx in prefixes:
        if any(k.startswith(pfx) for k in weight_map):
            used_prefix = pfx
            break
    if used_prefix is None:
        return {}

    def pick(cands):
        for suf in cands:
            k = used_prefix + suf
            if k in weight_map:
                return k
        return None

    # Mixtral uses w1 (gate), w2 (down), w3 (up). DeepSeek uses gate_proj/up_proj/down_proj.
    # Try Mixtral naming first, then fall back to DeepSeek.
    gate = pick(["w1.weight", "gate_proj.weight"])
    down = pick(["w2.weight", "down_proj.weight"])
    up   = pick(["w3.weight", "up_proj.weight"])

    if gate is None or down is None or up is None:
        return {}
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard = {}
    for k in keys:
        shard = weight_map.get(k)
        if shard is None: continue
        by_shard.setdefault(shard, []).append(k)
    out = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp): continue
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks: out[k] = f.get_tensor(k)
    return out

# -----------------------------------------------------------------------------
# Calibration / Router
# -----------------------------------------------------------------------------
def autodetect_calib_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return cand if os.path.isfile(cand) else None

def autodetect_router_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return cand if os.path.isfile(cand) else None

def load_calib_X(path: str, H: int) -> torch.Tensor:
    z = np.load(path)
    X = torch.from_numpy(z["X"].astype(np.float32))
    if X.ndim != 2 or X.shape[1] != H: raise RuntimeError(f"Bad X shape {X.shape}")
    if X.shape[0] > cfg.CALIB_SAMPLES: X = X[:cfg.CALIB_SAMPLES]
    return X.to(device=DEVICE, dtype=DTYPE_ACC)

def load_router_P(path: str) -> np.ndarray:
    return np.load(path)["P"].astype(np.float32)

def _maybe_autopip():
    if not cfg.HF_AUTO_PIP: return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", "transformers", "sentencepiece", "tokenizers"])

def _patch_transformers_cache_compat():
    try:
        from transformers.cache_utils import DynamicCache
        if not hasattr(DynamicCache, "get_usable_length"):
            DynamicCache.get_usable_length = lambda self, seq_length: int(seq_length)
    except: pass

class _Collector:
    def __init__(self, H, E_total, max_rows):
        self.H = H; self.E_total = E_total; self.max_rows = max_rows
        self.X_chunks, self.P_chunks = [], []; self.nX = self.nP = 0

    def _take(self, flat, need): return flat[:need] if flat.shape[0] > need else flat

    def add_X(self, hs, attn_mask):
        if hs is None: return
        if hs.ndim == 2: hs = hs.unsqueeze(0)
        if hs.ndim != 3 or hs.shape[-1] != self.H: return
        hs = hs.detach().to(torch.float32).cpu()
        if attn_mask is not None and not cfg.CAPTURE_KEEP_PAD:
            m = attn_mask.cpu().to(torch.bool); flat = hs.reshape(-1, self.H)[m.reshape(-1)]
        else: flat = hs.reshape(-1, self.H)
        if flat.numel() == 0: return
        need = self.max_rows - self.nX
        if need <= 0: return
        self.X_chunks.append(self._take(flat, need)); self.nX += self.X_chunks[-1].shape[0]

    def add_logits(self, logits, attn_mask):
        if logits is None: return
        if logits.ndim == 2: logits = logits.unsqueeze(0)
        if logits.ndim != 3: return
        P = torch.softmax(logits.detach().to(torch.float32), dim=-1)[..., :self.E_total].cpu()
        if attn_mask is not None and not cfg.CAPTURE_KEEP_PAD:
            m = attn_mask.cpu().to(torch.bool); flat = P.reshape(-1, P.shape[-1])[m.reshape(-1)]
        else: flat = P.reshape(-1, P.shape[-1])
        if flat.numel() == 0: return
        need = self.max_rows - self.nP
        if need <= 0: return
        self.P_chunks.append(self._take(flat, need)); self.nP += self.P_chunks[-1].shape[0]

def capture_XP_transformers(model_dir, layer_idx, H, E_total, out_x, out_p):
    _maybe_autopip(); _patch_transformers_cache_compat()
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=cfg.HF_TRUST_REMOTE_CODE, local_files_only=cfg.HF_LOCAL_FILES_ONLY)
    if tok.pad_token is None: tok.pad_token = tok.eos_token or tok.unk_token
    model = AutoModelForCausalLM.from_pretrained(model_dir, trust_remote_code=cfg.HF_TRUST_REMOTE_CODE, local_files_only=cfg.HF_LOCAL_FILES_ONLY, torch_dtype=torch.float16 if DEVICE.type=="cuda" else torch.float32, low_cpu_mem_usage=True).to(DEVICE).eval()

    # locate layer and mlp
    layers = None
    if hasattr(model, "model") and hasattr(model.model, "layers"): layers = model.model.layers
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"): layers = model.transformer.h
    elif hasattr(model, "layers"): layers = model.layers
    if layers is None: raise RuntimeError("Cannot locate layers")
    if layer_idx >= len(layers): raise RuntimeError(f"Layer {layer_idx} out of range")
    layer = layers[layer_idx]
    mlp = getattr(layer, "mlp", None)
    if mlp is None:
        for n, m in layer.named_modules():
            if n.lower().endswith("mlp"): mlp = m; break
    if mlp is None: raise RuntimeError("Could not find layer.mlp")

    # router discovery (best-effort)
    router_linear = None
    for name, mod in layer.named_modules():
        if isinstance(mod, nn.Linear) and mod.in_features == H and mod.out_features >= E_total:
            if "router" in name.lower() or "gate" in name.lower(): router_linear = mod; break

    coll = _Collector(H, E_total, cfg.CALIB_SAMPLES)
    attn_holder = {"mask": None}

    def mlp_pre_hook(_, inputs):
        coll.add_X(inputs[0], attn_holder["mask"])

    h1 = mlp.register_forward_pre_hook(mlp_pre_hook)
    h2 = None
    if router_linear is not None:
        def router_hook(_, __, out):
            o = out[0] if isinstance(out, (tuple, list)) else out
            coll.add_logits(o, attn_holder["mask"])
        h2 = router_linear.register_forward_hook(router_hook)

    texts = [cfg.CAPTURE_TEXT]
    if cfg.CAPTURE_TEXT_FILE and os.path.isfile(cfg.CAPTURE_TEXT_FILE):
        with open(cfg.CAPTURE_TEXT_FILE) as f: texts = [ln.strip() for ln in f if ln.strip()]
    tptr = 0
    for it in range(cfg.CAPTURE_ITERS):
        text = texts[tptr % len(texts)]; tptr += 1
        enc = tok(text, return_tensors="pt", truncation=True, max_length=cfg.CAPTURE_MAX_TOKENS, padding="max_length")
        for k in enc:
            if enc[k].ndim == 2 and cfg.CAPTURE_BATCH > 1: enc[k] = enc[k].repeat(cfg.CAPTURE_BATCH, 1)
        attn_holder["mask"] = enc.get("attention_mask")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.inference_mode(): _ = model(**enc, use_cache=False)
        if (it+1) % 4 == 0: log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS} nX={coll.nX} nP={coll.nP}")
        if coll.nX >= cfg.CALIB_SAMPLES and (not cfg.RIDGE_WEIGHTED or coll.nP >= cfg.CALIB_SAMPLES): break

    h1.remove()
    if h2: h2.remove()

    if coll.nX == 0: raise RuntimeError("Capture collected 0 rows")
    X = torch.cat(coll.X_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz_compressed(out_x, {"X": X})
    log(f"[capture] wrote X -> {out_x} shape={X.shape}")
    p_written = None
    if coll.nP > 0:
        P = torch.cat(coll.P_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
        N = min(P.shape[0], X.shape[0])
        if N < X.shape[0]: X = X[:N]; save_npz_compressed(out_x, {"X": X})
        P = P[:N]; save_npz_compressed(out_p, {"P": P})
        log(f"[capture] wrote P -> {out_p} shape={P.shape}")
        p_written = out_p
    return out_x, p_written

def ensure_calib_router(H: int, E_total: int):
    if not cfg.CALIB_PATH:
        c = autodetect_calib_path()
        if c: cfg.CALIB_PATH = c; log(f"[calib] auto-found {cfg.CALIB_PATH}")
    if not cfg.ROUTER_PATH:
        r = autodetect_router_path()
        if r: cfg.ROUTER_PATH = r; log(f"[router] auto-found {cfg.ROUTER_PATH}")
    if cfg.CAPTURE_FORCE or (cfg.CAPTURE_ENABLE and (not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH))):
        out_x = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
        out_p = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
        log("[capture] capturing via transformers...")
        x_path, p_path = capture_XP_transformers(cfg.MODEL_DIR, cfg.LAYER, H, E_total, out_x, out_p)
        cfg.CALIB_PATH = x_path
        if p_path: cfg.ROUTER_PATH = p_path

# -----------------------------------------------------------------------------
# Ridge linearization: build Ws
# -----------------------------------------------------------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate, W_up, W_down) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    return hid @ W_down.to(DTYPE_ACC).t()

def ws_cache_path(E: int) -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{E}_ridge_ebc.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        script="ebc_llm", model_dir=cfg.MODEL_DIR, layer=cfg.LAYER, expert_ids=eids,
        ridge_damp=cfg.RIDGE_DAMP, ridge_weighted=cfg.RIDGE_WEIGHTED,
        router_path=cfg.ROUTER_PATH or "", calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES, normalize_w=cfg.NORMALIZE_W, seed=SEED, device=str(DEVICE)
    )

@torch.no_grad()
def build_Ws(eids: List[int], wm: Dict[str, str]) -> Tuple[torch.Tensor, torch.Tensor]:
    per_e, need_keys = {}, []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk: raise RuntimeError(f"Expert {eid} missing tensors")
        per_e[eid] = kk; need_keys += [kk["up"], kk["down"], kk["gate"]]
    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))
    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = W_up0.shape[0], W_up0.shape[1]
    log(f"[shape] H={H} d_ff={dff}")

    ensure_calib_router(H, len(find_layer_expert_ids(wm, cfg.LAYER)))
    if not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH):
        raise RuntimeError("CALIB_PATH missing. Set CALIB_PATH or CAPTURE_ENABLE=1.")
    X = load_calib_X(cfg.CALIB_PATH, H)
    log(f"[calib] X: {X.shape}")

    P = None
    if cfg.RIDGE_WEIGHTED:
        if cfg.ROUTER_PATH and os.path.isfile(cfg.ROUTER_PATH):
            P = load_router_P(cfg.ROUTER_PATH)
            log(f"[router] P: {P.shape}")
        else:
            log("[router] RIDGE_WEIGHTED=1 but ROUTER_PATH missing -> disabling.")
            cfg.RIDGE_WEIGHTED = False

    Xf = X.to(DTYPE_ACC); I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * torch.trace(XtX).item() / H
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list, scales = [], []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws (ridge)")):
        W_up = T[per_e[eid]["up"]].to(DEVICE)
        W_dn = T[per_e[eid]["down"]].to(DEVICE)
        W_gt = T[per_e[eid]["gate"]].to(DEVICE)
        Y = forward_mlp(X, W_gt, W_up, W_dn).to(DTYPE_ACC)

        if cfg.RIDGE_WEIGHTED and P is not None:
            w = torch.from_numpy(P[:X.shape[0], eid if cfg.ROUTER_EIDS_ARE_GLOBAL else i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0)
            sw = torch.sqrt(w + 1e-12).view(-1,1)
            Xw, Yw = Xf * sw, Y * sw
            XtX_e = Xw.t() @ Xw
            lam_e = cfg.RIDGE_DAMP * torch.trace(XtX_e).item() / H
            chol = torch.linalg.cholesky(XtX_e + lam_e * I)
            Wt = torch.cholesky_solve(Xw.t() @ Yw, chol)
            W = Wt.t().contiguous()
        else:
            Wt = torch.cholesky_solve(Xf.t() @ Y, cholG)
            W = Wt.t().contiguous()

        if cfg.NORMALIZE_W:
            s = torch.linalg.norm(W, ord="fro").clamp_min(1e-12).item()
            W = W / s
        else: s = 1.0
        Ws_list.append(W); scales.append(s)

    Ws = torch.stack(Ws_list).to(DTYPE_ACC).to(DEVICE)
    Sc = torch.tensor(scales, dtype=DTYPE_ACC, device=DEVICE)
    return Ws, Sc

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor, torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids: raise RuntimeError(f"No experts at layer {cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer={cfg.LAYER} total={len(all_eids)} using={len(eids)} eids={eids}")

    if not cfg.CALIB_PATH: cfg.CALIB_PATH = autodetect_calib_path() or ""
    if not cfg.ROUTER_PATH: cfg.ROUTER_PATH = autodetect_router_path() or ""

    cpath = ws_cache_path(len(eids))
    if os.path.isfile(cpath):
        z = load_npz(cpath)
        if all(k in z for k in ["meta","Ws","expert_ids","scales"]) and _decode_meta(z["meta"]) == ws_meta(eids):
            Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
            Sc = torch.from_numpy(z["scales"]).to(DTYPE_ACC).to(DEVICE)
            log(f"[cache] loaded Ws -> {cpath} shape={Ws.shape}")
            return [int(x) for x in z["expert_ids"]], Ws, Sc
        log("[cache] meta mismatch -> rebuild")

    Ws, Sc = build_Ws(eids, wm)
    save_npz_compressed(cpath, {
        "meta": _encode_meta(ws_meta(eids)),
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.cpu().numpy().astype(np.float32),
        "scales": Sc.cpu().numpy().astype(np.float32)
    })
    log(f"[cache] wrote Ws -> {cpath} size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws, Sc

# -----------------------------------------------------------------------------
# Clustering (kmeans++ + hierarchical split)
# -----------------------------------------------------------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED+17)
    R = (torch.randint(0,2,(n,d),generator=g,dtype=torch.int8)*2-1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]; row = torch.diag(W @ W.t()); col = torch.diag(W.t() @ W)
        feats.append(torch.cat([row @ R, col @ R]).unsqueeze(0))
    X = torch.cat(feats, dim=0)
    X = (X - X.mean(0, keepdim=True)) / (X.std(0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> torch.Tensor:
    best_lab, best_inertia = None, float("inf")
    g = torch.Generator(device="cpu").manual_seed(SEED+999)
    for _ in range(max(1, restarts)):
        # kmeans++ init
        n = X.shape[0]
        centers = [X[torch.randint(0, n, (1,), generator=g).item()].clone()]
        for _ in range(1, k):
            C = torch.stack(centers)
            dist2 = torch.cdist(X, C).pow(2).min(1).values
            prob = dist2 / dist2.sum().clamp_min(1e-12)
            centers.append(X[torch.multinomial(prob, 1, generator=g).item()].clone())
        C = torch.stack(centers)
        for _ in range(iters):
            dist = torch.cdist(X, C); lab = dist.argmin(1)
            for j in range(k):
                m = (lab == j)
                if m.any(): C[j] = X[m].mean(0)
                else: C[j] = X[dist.min(1).values.argmax().item()].clone()
        inertia = torch.cdist(X, C).min(1).values.pow(2).sum().item()
        if inertia < best_inertia: best_inertia, best_lab = inertia, lab.clone()
    return best_lab.to(torch.int64)

@torch.no_grad()
def relabel_contiguous(labels: torch.Tensor) -> torch.Tensor:
    uniq = torch.unique(labels); out = labels.clone()
    for new, old in enumerate(uniq.tolist()): out[labels == old] = new
    return out

@torch.no_grad()
def merge_small_clusters(X: torch.Tensor, labels: torch.Tensor, min_size: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if min_size <= 1: return labels
    while True:
        K = labels.max().item() + 1
        counts = torch.bincount(labels, minlength=K)
        small = (counts < min_size).nonzero(as_tuple=False).flatten()
        if small.numel() == 0: break
        C = torch.stack([X[labels == k].mean(0) for k in range(K)])
        for c in small.tolist():
            idxs = (labels == c).nonzero(as_tuple=False).flatten()
            if idxs.numel() == 0: continue
            dist = torch.cdist(C[c].unsqueeze(0), C).squeeze(0); dist[c] = 1e9
            labels[idxs] = dist.argmin().item()
        labels = relabel_contiguous(labels)
    return labels

@torch.no_grad()
def hierarchical_split(X: torch.Tensor, labels: torch.Tensor, max_size: int, max_k: int, split_iters: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if max_size <= 0: return labels
    while True:
        K = labels.max().item() + 1
        if K >= max_k: break
        counts = torch.bincount(labels, minlength=K)
        biggest = counts.argmax().item()
        if counts[biggest] <= max_size: break
        idxs = (labels == biggest).nonzero(as_tuple=False).flatten()
        if idxs.numel() < 2: break
        sub = X[idxs]; sub_lab = kmeans_torch(sub, 2, split_iters, 1)
        a, b = idxs[sub_lab == 0], idxs[sub_lab == 1]
        if a.numel() == 0 or b.numel() == 0: break
        labels[b] = K
        labels = relabel_contiguous(labels)
    return labels

# -----------------------------------------------------------------------------
# Basis training (dense)
# -----------------------------------------------------------------------------
class OrthoParam(nn.Module):
    def __init__(self, init_mat: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(init_mat.to(DEVICE, DTYPE_ACC).contiguous())
    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M); return Q

@torch.no_grad()
def svd_init_from_mean(Wmean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(Wmean, full_matrices=False)
    return U.to(DTYPE_ACC).contiguous(), Vh.t().to(DTYPE_ACC).contiguous()

def schedule(step: int, warmup: int, total: int) -> float:
    if step <= warmup: return 0.0
    return min(1.0, (step - warmup) / max(1, total - warmup))

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    U_S, V_S = U[:, S], V[:, S]
    return torch.matmul(U_S.t().unsqueeze(0), Ws_batch @ V_S)

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return (Xs - torch.diag_embed(D)).abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    return torch.diagonal(Xs, dim1=1, dim2=2).abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    Eb, s, _ = Xs.shape; b = int(block)
    if b <= 0: return torch.zeros((), device=Xs.device)
    nb = s // b
    if nb <= 0: return torch.zeros((), device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0,1,3,2,4).contiguous()
    Eblk = (Xb * Xb).sum(dim=(3,4))
    P = Eblk.mean(0)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

@torch.no_grad()
def make_guidance_mask_from_Xs(Xs: torch.Tensor, block: int, target: float, max_blocks: int) -> Tuple[torch.Tensor, float, int]:
    Eb, s, _ = Xs.shape; b = int(block)
    if b <= 0: return torch.ones(s,s,device=Xs.device), 1.0, 0
    nb = s // b
    if nb <= 0: return torch.ones(s,s,device=Xs.device), 1.0, 0
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0,1,3,2,4).contiguous()
    Eg = (Xb * Xb).sum(dim=(3,4)).mean(0)
    tot = (X * X).sum().item() / max(1, Eb)
    flat = Eg.reshape(-1); order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], 0)
    frac = csum / max(tot, 1e-12)
    need = (frac >= target).nonzero(as_tuple=False)[0].item() + 1 if (frac >= target).any() else flat.numel()
    K = min(need, max_blocks, flat.numel())
    mask = torch.zeros(s2, s2, device=Xs.device)
    for idx in order[:K].tolist():
        bi, bj = idx // nb, idx % nb
        mask[bi*b:(bi+1)*b, bj*b:(bj+1)*b] = 1.0
    if s2 < s:
        full = torch.zeros(s, s, device=Xs.device); full[:s2, :s2] = mask; mask = full
    ef = float(frac[K-1].item()) if K > 0 else 0.0
    return mask, ef, K

# -----------------------------------------------------------------------------
# Block energy & selection
# -----------------------------------------------------------------------------
@torch.no_grad()
def block_energy_grid(X: torch.Tensor, b: int) -> Tuple[torch.Tensor, float, int]:
    n = X.shape[0]; nb = (n + b - 1) // b
    if n % b != 0:
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X; X = Xp
    Xb = X.view(nb, b, nb, b).permute(0,2,1,3).contiguous()
    Eg = (Xb * Xb).sum(dim=(2,3))
    tot = (X * X).sum().item()
    return Eg, tot, nb

@torch.no_grad()
def pick_blocks_until_target(Eg: torch.Tensor, tot_energy: float, target: float, max_blocks: int,
                             exclude: Optional[Set[Tuple[int,int]]]=None) -> Tuple[List[Tuple[int,int]], float]:
    nb = Eg.shape[0]; flat = Eg.reshape(-1); order = torch.argsort(flat, descending=True)
    picked, eacc = [], 0.0
    exclude = exclude or set()
    for idx in order.tolist():
        if len(picked) >= max_blocks: break
        e = flat[idx].item()
        if e <= 1e-18: break
        bi, bj = idx // nb, idx % nb
        if (bi, bj) in exclude: continue
        picked.append((bi, bj)); eacc += e
        if eacc / max(tot_energy, 1e-12) >= target: break
    return picked, eacc / max(tot_energy, 1e-12)

@torch.no_grad()
def gather_block(X: torch.Tensor, i0: int, j0: int, b: int) -> torch.Tensor:
    n = X.shape[0]; i1, j1 = min(n, i0+b), min(n, j0+b)
    return X[i0:i1, j0:j1].contiguous()

# -----------------------------------------------------------------------------
# Low-rank (randomized SVD)
# -----------------------------------------------------------------------------
@torch.no_grad()
def rand_svd_vectors(A: torch.Tensor, r: int, n_iter: int=2) -> Tuple[torch.Tensor, torch.Tensor]:
    n = A.shape[0]; r = min(r, n)
    g = torch.Generator(device="cpu").manual_seed(SEED+777)
    Omega = torch.randn(n, r, generator=g, dtype=DTYPE_ACC, device=A.device)
    Y = A @ Omega
    for _ in range(n_iter): Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    Uhat, _, Vh = torch.linalg.svd(B, full_matrices=False)
    return (Q @ Uhat[:, :r]).contiguous(), Vh.t()[:, :r].contiguous()

# -----------------------------------------------------------------------------
# Payload packing (ragged blocks)
# -----------------------------------------------------------------------------
def _block_store_dtype(qmode: str) -> np.dtype:
    return np.float32 if qmode == "none" else np.float16

def pack_blocks_ragged(blocks_per_item: List[List[Tuple[int,int,torch.Tensor]]], qmode: str) -> Dict[str, np.ndarray]:
    val_dtype = _block_store_dtype(qmode)
    M = len(blocks_per_item)
    item_ptr = [0]
    blk_i0, blk_j0, blk_h, blk_w = [], [], [], []
    blk_ptr = [0]
    vals, vals_i8, scales = [], [], []
    for m in range(M):
        for (i0, j0, B) in blocks_per_item[m]:
            h, w = B.shape
            blk_i0.append(i0); blk_j0.append(j0); blk_h.append(h); blk_w.append(w)
            if qmode == "int8":
                x = B.cpu().float(); maxabs = x.abs().max().item()
                if maxabs < 1e-12: q = np.zeros(x.numel(), dtype=np.int8); sc = np.float16(1.0)
                else:
                    scale = maxabs / 127.0
                    q = torch.clamp(torch.round(x/scale), -127, 127).to(torch.int8).numpy()
                    sc = np.float16(scale)
                vals_i8.append(q.reshape(-1)); scales.append(sc)
                blk_ptr.append(blk_ptr[-1] + q.size)
            else:
                v = B.cpu().float().numpy().astype(val_dtype).reshape(-1)
                vals.append(v); blk_ptr.append(blk_ptr[-1] + v.size)
        item_ptr.append(len(blk_i0))

    out = {
        "item_ptr": np.array(item_ptr, dtype=np.int32),
        "blk_i0": np.array(blk_i0, dtype=np.int16),
        "blk_j0": np.array(blk_j0, dtype=np.int16),
        "blk_h": np.array(blk_h, dtype=np.int16),
        "blk_w": np.array(blk_w, dtype=np.int16),
        "blk_ptr": np.array(blk_ptr, dtype=np.int64)
    }
    if qmode == "int8":
        out["blk_q"] = np.concatenate(vals_i8).astype(np.int8) if vals_i8 else np.zeros((0,), dtype=np.int8)
        out["blk_scale"] = np.array(scales, dtype=np.float16)
    else:
        out["blk_val"] = np.concatenate(vals) if vals else np.zeros((0,), dtype=val_dtype)
    return out

def unpack_blocks_ragged(pack: Dict[str, np.ndarray], qmode: str, device: torch.device) -> List[List[Tuple[int,int,torch.Tensor]]]:
    item_ptr = pack["item_ptr"]
    blk_i0 = pack["blk_i0"]; blk_j0 = pack["blk_j0"]; blk_h = pack["blk_h"]; blk_w = pack["blk_w"]
    blk_ptr = pack["blk_ptr"]
    if qmode == "int8":
        blk_q = pack["blk_q"]; blk_scale = pack["blk_scale"]; blk_val = None
    else:
        blk_val = pack["blk_val"]; blk_q = None; blk_scale = None
    M = item_ptr.shape[0] - 1
    out = []
    for m in range(M):
        b0, b1 = item_ptr[m], item_ptr[m+1]
        lst = []
        for bi in range(b0, b1):
            i0, j0 = int(blk_i0[bi]), int(blk_j0[bi])
            h, w = int(blk_h[bi]), int(blk_w[bi])
            v0, v1 = blk_ptr[bi], blk_ptr[bi+1]
            if qmode == "int8":
                q = blk_q[v0:v1].astype(np.float32); sc = float(blk_scale[bi])
                B = torch.from_numpy((q * sc).reshape(h, w)).to(device, DTYPE_ACC)
            else:
                B = torch.from_numpy(blk_val[v0:v1].astype(np.float32).reshape(h, w)).to(device, DTYPE_ACC)
            lst.append((i0, j0, B))
        out.append(lst)
    return out

# -----------------------------------------------------------------------------
# Payload runtime
# -----------------------------------------------------------------------------
class PayloadRuntime:
    def __init__(self):
        self.meta = {}
        self.expert_ids = []
        self.scales: Optional[torch.Tensor] = None
        self.cluster_of_pos: Optional[torch.Tensor] = None
        self.U: List[torch.Tensor] = []
        self.V: List[torch.Tensor] = []
        self.DL: List[torch.Tensor] = []
        self.DR: List[torch.Tensor] = []
        self.gam: Optional[torch.Tensor] = None
        self.Cfull: Optional[torch.Tensor] = None
        self.core_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.res_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.qmode = "none"
        self.res_coef = "diag"

    @torch.no_grad()
    def apply_expert(self, x: torch.Tensor, pos: int) -> torch.Tensor:
        c = int(self.cluster_of_pos[pos].item())
        U, V = self.U[c], self.V[c]
        DL, DR = self.DL[c], self.DR[c]
        z = x @ U
        u = torch.zeros_like(z)
        for (i0, j0, B) in self.core_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B
        if self.res_coef == "diag":
            g = self.gam[pos]
            u += ((z @ DL) * g.view(1,-1)) @ DR.t()
        else:
            C = self.Cfull[pos]
            u += (z @ DL) @ C @ DR.t()
        for (i0, j0, B) in self.res_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B
        y = u @ V.t()
        if self.scales is not None:
            y = y * self.scales[pos]
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        for a, pos in zip(gates.tolist(), routed):
            y += a * self.apply_expert(x, int(pos))
        return y

def load_payload_runtime(path: str, device: torch.device) -> PayloadRuntime:
    z = load_npz(path)
    rt = PayloadRuntime()
    rt.meta = _decode_meta(z["meta"])
    rt.qmode = rt.meta.get("qmode", "none")
    rt.res_coef = rt.meta.get("res_coef", "diag")
    rt.expert_ids = [int(x) for x in z["expert_ids"]]
    rt.scales = torch.from_numpy(z["scales"]).to(device, DTYPE_ACC)
    rt.cluster_of_pos = torch.from_numpy(z["cluster_of_pos"]).to(device, torch.int64)
    M = z["n_clusters"][0]
    for m in range(M):
        rt.U.append(torch.from_numpy(z[f"U_{m}"]).to(device, DTYPE_ACC))
        rt.V.append(torch.from_numpy(z[f"V_{m}"]).to(device, DTYPE_ACC))
        rt.DL.append(torch.from_numpy(z[f"DL_{m}"]).to(device, DTYPE_ACC))
        rt.DR.append(torch.from_numpy(z[f"DR_{m}"]).to(device, DTYPE_ACC))
    if rt.res_coef == "diag":
        rt.gam = torch.from_numpy(z["gam"]).to(device, DTYPE_ACC)
    else:
        rt.Cfull = torch.from_numpy(z["Cfull"]).to(device, DTYPE_ACC)
    core_pack = {k[5:]: z[k] for k in z if k.startswith("core_")}
    res_pack  = {k[4:]: z[k] for k in z if k.startswith("res_")}
    rt.core_blocks = unpack_blocks_ragged(core_pack, rt.qmode, device)
    rt.res_blocks  = unpack_blocks_ragged(res_pack, rt.qmode, device)
    return rt

# -----------------------------------------------------------------------------
# Build payload for one cluster
# -----------------------------------------------------------------------------
@torch.no_grad()
def frob_rel_err(A, B): return (torch.linalg.norm(A-B) / torch.linalg.norm(B).clamp_min(1e-12)).item()

@torch.no_grad()
def build_payload_for_cluster(Ws_norm: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor) -> Dict:
    n = Ws_norm.shape[-1]
    X_list = [(U.t() @ Ws_norm[pos] @ V).contiguous() for pos in idx]
    b = cfg.CORE_BLOCK

    # core blocks
    core_per = []
    core_ef = []
    for X in X_list:
        Eg, te, nb = block_energy_grid(X, b)
        picks, eff = pick_blocks_until_target(Eg, te, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*b, bj*b
            blocks.append((i0, j0, gather_block(X, i0, j0, b)))
        core_per.append(blocks); core_ef.append(eff)

    # residual after core
    R_list = []
    for X, cb in zip(X_list, core_per):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb: h,w = Bc.shape; Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    # low-rank shared
    Rmean = torch.stack(R_list).mean(0)
    r = min(cfg.RES_RANK, n)
    DL, DR = rand_svd_vectors(Rmean, r, n_iter=2)

    coef_list, res_per = [], []
    bb = cfg.RES_BSIZE
    for j, Rm in enumerate(R_list):
        if cfg.RES_COEF == "diag":
            g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()
            coef_list.append(g)
            R2 = (Rm - (DL * g.view(1,-1)) @ DR.t()).contiguous()
        else:
            C = (DL.t() @ Rm @ DR).contiguous()
            coef_list.append(C)
            R2 = (Rm - (DL @ C @ DR.t())).contiguous()

        Eg2, te2, nb2 = block_energy_grid(R2, bb)
        exclude = {(i0//bb, j0//bb) for (i0,j0,_) in core_per[j]}
        picks, _ = pick_blocks_until_target(Eg2, te2, cfg.RES_TARGET, cfg.RES_MAX_BLOCKS, exclude=exclude)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*bb, bj*bb
            blocks.append((i0, j0, gather_block(R2, i0, j0, bb)))
        res_per.append(blocks)

    # refine
    if cfg.REFINE_ENABLE:
        rb = cfg.REFINE_BSIZE
        for j in range(len(idx)):
            X = X_list[j]
            def reconstruct():
                Xc = torch.zeros_like(X)
                for (i0,j0,Bc) in core_per[j]: h,w=Bc.shape; Xc[i0:i0+h, j0:j0+w] = Bc
                if cfg.RES_COEF == "diag":
                    g = coef_list[j]; Xlr = (DL * g.view(1,-1)) @ DR.t()
                else:
                    C = coef_list[j]; Xlr = DL @ C @ DR.t()
                Xr = torch.zeros_like(X)
                for (i0,j0,Bb) in res_per[j]: h,w=Bb.shape; Xr[i0:i0+h, j0:j0+w] += Bb
                return Xc + Xlr + Xr
            Xhat = reconstruct()
            err = frob_rel_err(Xhat, X)
            added = 0
            core_pos = {(i0,j0) for (i0,j0,_) in core_per[j]}
            res_pos = {(i0,j0) for (i0,j0,_) in res_per[j]}
            while err > cfg.REFINE_ERR_TARGET and added < cfg.REFINE_MAX_EXTRA:
                Rerr = (X - Xhat).contiguous()
                Eg, te, nb = block_energy_grid(Rerr, rb)
                flat = Eg.reshape(-1)
                if flat.max().item() <= 1e-18: break
                order = torch.argsort(flat, descending=True)
                found = False
                for idx_ in order.tolist():
                    bi, bj = idx_ // nb, idx_ % nb
                    i0, j0 = bi*rb, bj*rb
                    if (i0, j0) in core_pos or (i0, j0) in res_pos: continue
                    Bb = gather_block(Rerr, i0, j0, rb)
                    res_per[j].append((i0, j0, Bb)); res_pos.add((i0, j0))
                    added += 1; found = True; break
                if not found: break
                if added % cfg.REFINE_RECHECK_EVERY == 0:
                    Xhat = reconstruct(); err = frob_rel_err(Xhat, X)
            Xhat = reconstruct(); err = frob_rel_err(Xhat, X)

    return {
        "core_blocks": core_per, "core_energy": core_ef,
        "DL": DL, "DR": DR, "coef_list": coef_list, "res_blocks": res_per
    }

# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------
@torch.no_grad()
def eval_payload(rt: PayloadRuntime, Ws_norm: torch.Tensor, Sc: torch.Tensor):
    E, n, _ = Ws_norm.shape
    errs = []
    for pos in range(E):
        x = torch.randn(8, n, dtype=DTYPE_ACC, device=DEVICE)
        y_hat = rt.apply_expert(x, pos)
        y_ref = x @ (Ws_norm[pos] * Sc[pos])
        errs.append((torch.linalg.norm(y_hat - y_ref) / torch.linalg.norm(y_ref).clamp_min(1e-12)).item())
    log(f"[eval] per-expert rel-error mean={np.mean(errs):.6f} p95={np.percentile(errs,95):.6f} max={np.max(errs):.6f}")

    mix = []
    for _ in range(cfg.EVAL_TRIALS):
        x = torch.randn(cfg.EVAL_BATCH, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), min(cfg.ROUTED_K, E))
        gates = torch.rand(len(routed), device=DEVICE); gates /= gates.sum()
        y_hat = rt.apply_mixture(x, routed, gates)
        Wsum = sum(gates[i].item() * (Ws_norm[pos] * Sc[pos]) for i, pos in enumerate(routed))
        y_ref = x @ Wsum
        mix.append((torch.linalg.norm(y_hat - y_ref) / torch.linalg.norm(y_ref).clamp_min(1e-12)).item())
    log(f"[eval] routed rel-error mean={np.mean(mix):.6f} ± {np.std(mix):.6f}")

# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def banner():
    log("="*60)
    log("EBC-LLM Compression Pipeline")
    log(f"Time: {now()}  Device: {DEVICE}")
    log(f"MODEL_DIR: {cfg.MODEL_DIR}  OUTPUT_DIR: {cfg.OUTPUT_DIR}")
    log(f"Layer: {cfg.LAYER}  Experts: {cfg.MAX_EXPERTS}")
    log(f"CALIB: {cfg.CALIB_PATH or '(none)'}  ROUTER: {cfg.ROUTER_PATH or '(none)'}")
    log(f"Ridge damp: {cfg.RIDGE_DAMP}  Normalize W: {cfg.NORMALIZE_W}")
    log(f"Basis: {cfg.BASIS_MODE}  Train steps: {cfg.TRAIN_STEPS}  lr: {cfg.TRAIN_LR}")
    log(f"Core: {cfg.CORE_MODE} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max={cfg.CORE_MAX_BLOCKS}")
    log(f"Residual: rank={cfg.RES_RANK} coef={cfg.RES_COEF} blocks={cfg.RES_MAX_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"Refine: {cfg.REFINE_ENABLE} target={cfg.REFINE_ERR_TARGET} max_extra={cfg.REFINE_MAX_EXTRA}")
    log("="*60)

def main():
    banner()
    expert_ids, Ws_norm, Sc = load_or_build_Ws()
    E, n, _ = Ws_norm.shape
    log(f"[Ws] shape={Ws_norm.shape}")

    # Clustering
    Xfeat = random_proj_features(Ws_norm, cfg.CLUSTER_FEAT_D)
    M0 = max(2, min(cfg.M0 if cfg.M0>0 else int(round(2*math.sqrt(E))), E))
    labels = kmeans_torch(Xfeat, M0, cfg.CLUSTER_ITERS, cfg.CLUSTER_RESTARTS)
    labels = merge_small_clusters(Xfeat, labels, cfg.CLUSTER_MIN_SIZE)
    labels = hierarchical_split(Xfeat, labels, cfg.CLUSTER_MAX_SIZE, min(cfg.M_MAX, E), cfg.SPLIT_ITERS)
    labels = merge_small_clusters(Xfeat, labels, cfg.CLUSTER_MIN_SIZE)
    labels = relabel_contiguous(labels)
    M = labels.max().item() + 1
    clusters = [torch.nonzero(labels==m, as_tuple=False).flatten().tolist() for m in range(M)]
    clusters = [c for c in clusters if c]
    log(f"[cluster] M={len(clusters)} sizes={[len(c) for c in clusters]}")
    cluster_of_pos = [0]*E
    for m, idx in enumerate(clusters):
        for pos in idx: cluster_of_pos[pos] = m

    # Init and train bases
    U_par, V_par = [], []
    for idx in clusters:
        Wm = Ws_norm[idx].mean(0)
        U0, V0 = svd_init_from_mean(Wm)
        U_par.append(OrthoParam(U0)); V_par.append(OrthoParam(V0))

    if cfg.TRAIN_STEPS > 0 and cfg.BASIS_MODE == "dense_train":
        params = [p.M for p in U_par] + [p.M for p in V_par]
        opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)
        guidance_masks, guidance_stats = {}, {}
        t0 = time.perf_counter()
        for step in range(1, cfg.TRAIN_STEPS+1):
            S = torch.randperm(n)[:cfg.SUBM].to(DEVICE)
            if cfg.TRAIN_LAM_GUIDE > 0 and (step==1 or step%cfg.TRAIN_GUIDE_EVERY==0):
                with torch.no_grad():
                    guidance_masks.clear(); guidance_stats.clear()
                    for m, idx in enumerate(clusters):
                        if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
                        Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                        pick = idx if cfg.BATCH_E>=len(idx) else [idx[i] for i in torch.randperm(len(idx))[:cfg.BATCH_E].tolist()]
                        Xs_ng = slice_X_batch(Ws_norm[pick], Uo, Vo, S).detach()
                        mask, ef, kblk = make_guidance_mask_from_Xs(Xs_ng, cfg.CORE_BLOCK, cfg.TRAIN_GUIDE_TARGET, cfg.TRAIN_GUIDE_MAX_BLOCKS)
                        guidance_masks[m] = mask; guidance_stats[m] = (ef, kblk)

            lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
            lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
            lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp
            L_total, n_terms = None, 0
            for m, idx in enumerate(clusters):
                if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
                Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                pick = idx if cfg.BATCH_E>=len(idx) else [idx[i] for i in torch.randperm(len(idx))[:cfg.BATCH_E].tolist()]
                Xs = slice_X_batch(Ws_norm[pick], Uo, Vo, S)
                off, diag = offdiag_abs_mean(Xs), diag_abs_mean(Xs).clamp_min(1e-6)
                base = torch.log(off+1e-6) - torch.log(diag) if cfg.TRAIN_OBJ=="logratio" else off/diag
                if lam_block > 0: base += lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)
                if lam_guide > 0 and m in guidance_masks:
                    Mmask = guidance_masks[m]
                    Etot = (Xs*Xs).mean().clamp_min(1e-12)
                    Eout = ((Xs*(1-Mmask))**2).mean()
                    base += lam_guide * (Eout/Etot)
                L_total = base if L_total is None else L_total + base
                n_terms += 1
            if L_total is None: break
            L_total = L_total / n_terms
            opt.zero_grad(); L_total.backward()
            if cfg.GRAD_CLIP > 0: torch.nn.utils.clip_grad_norm_(params, cfg.GRAD_CLIP)
            opt.step()
            if step % cfg.REORTHO_EVERY == 0 or step == cfg.TRAIN_STEPS:
                with torch.no_grad():
                    for p in U_par: p.M.copy_(p.orthogonal())
                    for p in V_par: p.M.copy_(p.orthogonal())
            if step % cfg.REPORT_EVERY == 0 or step == 1:
                t1 = time.perf_counter()
                gstr = "" if not guidance_stats else f" guide≈{np.mean([v[0] for v in guidance_stats.values()]):.3f}"
                log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={L_total.item():.4f} {gstr} (+{t1-t0:.1f}s)")
                t0 = t1

    # Freeze bases
    U_list = [p.orthogonal().detach() for p in U_par]
    V_list = [p.orthogonal().detach() for p in V_par]

    # Build payloads
    log("[build] payloads ...")
    core_all = [[] for _ in range(E)]
    res_all  = [[] for _ in range(E)]
    DL_list, DR_list = [], []
    rmax = min(cfg.RES_RANK, n)
    gam = torch.zeros((E, rmax), dtype=DTYPE_ACC, device=DEVICE) if cfg.RES_COEF=="diag" else None
    Cfull = torch.zeros((E, rmax, rmax), dtype=DTYPE_ACC, device=DEVICE) if cfg.RES_COEF=="full" else None

    for m, idx in enumerate(clusters):
        U, V = U_list[m], V_list[m]
        P = build_payload_for_cluster(Ws_norm, idx, U, V)
        for j, pos in enumerate(idx):
            core_all[pos] = P["core_blocks"][j]
            res_all[pos] = P["res_blocks"][j]
            if cfg.RES_COEF == "diag":
                g = P["coef_list"][j]; gam[pos, :g.numel()] = g
            else:
                C = P["coef_list"][j]; Cfull[pos, :C.shape[0], :C.shape[1]] = C
        DL_list.append(P["DL"]); DR_list.append(P["DR"])
        log(f"  cluster{m}: E={len(idx)} core_blocks≈{np.mean([len(c) for c in P['core_blocks']]):.1f} r={P['DL'].shape[1]}")

    # Save payload
    out_path = os.path.join(cfg.OUTPUT_DIR, f"ebc_payload_layer{cfg.LAYER}_E{E}_q{cfg.QMODE}.npz")
    store_dtype = np.float16 if cfg.BASIS_STORE_DTYPE=="float16" else np.float32
    arrays = {
        "meta": _encode_meta(ws_meta(expert_ids) | {"time": now(), "qmode": cfg.QMODE, "res_coef": cfg.RES_COEF}),
        "expert_ids": np.array(expert_ids, dtype=np.int32),
        "scales": Sc.cpu().numpy().astype(np.float32),
        "cluster_of_pos": np.array(cluster_of_pos, dtype=np.int16),
        "n_clusters": np.array([len(clusters)], dtype=np.int32),
    }
    for m in range(len(clusters)):
        arrays[f"U_{m}"] = U_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"V_{m}"] = V_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DL_{m}"] = DL_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DR_{m}"] = DR_list[m].cpu().numpy().astype(store_dtype)
    if cfg.RES_COEF == "diag":
        arrays["gam"] = gam.cpu().numpy().astype(store_dtype)
    else:
        arrays["Cfull"] = Cfull.cpu().numpy().astype(store_dtype)

    core_pack = pack_blocks_ragged(core_all, cfg.QMODE)
    res_pack  = pack_blocks_ragged(res_all, cfg.QMODE)
    for k, v in core_pack.items(): arrays["core_"+k] = v
    for k, v in res_pack.items(): arrays["res_"+k] = v

    save_npz_compressed(out_path, arrays)
    log(f"[save] payload -> {out_path} size={os.path.getsize(out_path)/1e6:.2f} MB")

    # Evaluate
    rt = load_payload_runtime(out_path, DEVICE)
    eval_payload(rt, Ws_norm, Sc)
    log("✅ Done.")

if __name__ == "__main__":
    main()

EBC-LLM Compression Pipeline
Time: 2026-04-21 10:06:40  Device: cpu
MODEL_DIR: /data/downloaded_models/Mixtral-8x7B-v0.1  OUTPUT_DIR: /home/daniyar/moe_ws_outputs
Layer: 0  Experts: 8
CALIB: (none)  ROUTER: (none)
Ridge damp: 0.001  Normalize W: True
Basis: dense_train  Train steps: 24  lr: 0.05
Core: blocktopk_perexpert block=64 target=0.85 max=256
Residual: rank=512 coef=diag blocks=4096 bsize=64
Refine: True target=0.03 max_extra=4096
[found] layer=0 total=8 using=8 eids=[0, 1, 2, 3, 4, 5, 6, 7]
[load] reading tensors from shards ...
[shape] H=4096 d_ff=14336
[capture] capturing via transformers...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[capture] iter 4/32 nX=4096 nP=0
[capture] wrote X -> /home/daniyar/moe_ws_outputs/calib_layer0_X.npz shape=(4096, 4096)
[calib] X: torch.Size([4096, 4096])


Build Ws (ridge):   0%|          | 0/8 [00:00<?, ?it/s]

[cache] wrote Ws -> /home/daniyar/moe_ws_outputs/Ws_cache_layer0_E8_ridge_ebc.npz size=498.88 MB
[Ws] shape=torch.Size([8, 4096, 4096])
[cluster] M=3 sizes=[2, 4, 2]
[train] step   1/24 loss=-4.7225  guide≈0.922 (+22.6s)
[train] step   4/24 loss=-0.8138  guide≈0.827 (+67.2s)
[train] step   8/24 loss=-0.8611  guide≈0.823 (+82.5s)
[train] step  12/24 loss=0.3989  guide≈0.815 (+81.7s)
[train] step  16/24 loss=5.1294  guide≈0.822 (+81.7s)
[train] step  20/24 loss=6.9194  guide≈0.824 (+80.6s)
[train] step  24/24 loss=6.2806  guide≈0.838 (+79.9s)
[build] payloads ...
  cluster0: E=2 core_blocks≈23.0 r=512
  cluster1: E=4 core_blocks≈121.8 r=512
  cluster2: E=2 core_blocks≈87.5 r=512
[save] payload -> /home/daniyar/moe_ws_outputs/ebc_payload_layer0_E8_qnone.npz size=704.19 MB
[eval] per-expert rel-error mean=0.033912 p95=0.065535 max=0.076769
[eval] routed rel-error mean=0.030180 ± 0.007550
✅ Done.


In [3]:
#!/usr/bin/env python3
# =============================================================================
# EBC-LLM: Expert-Bank Compression via Cluster-Shared Rotation and
#          Runtime-Aligned Structured Payloads
#
# Single-file offline compression and evaluation pipeline.
# Supports DeepSeek, AllenAI, Mixtral, and other MoE models.
#
# Usage:
#   python ebc_llm_compression.py
#
# Environment variables (see Cfg dataclass for all options):
#   MODEL_DIR=/path/to/model
#   OUTPUT_DIR=/path/to/output
#   LAYER=1
#   MAX_EXPERTS=16
#   CALIB_PATH=/path/to/calib_X.npz      (optional; auto-capture if missing)
#   ROUTER_PATH=/path/to/router_P.npz    (optional)
#   PRESET=balanced|maxacc|compact
# =============================================================================

import os, re, json, math, time, random, sys
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs): return x

# -----------------------------------------------------------------------------
# Environment helpers
# -----------------------------------------------------------------------------
def _env_str(k: str, d: str) -> str:
    return os.environ.get(k, d)

def _env_int(k: str, d: int) -> int:
    try: return int(os.environ.get(k, str(d)))
    except: return d

def _env_float(k: str, d: float) -> float:
    try: return float(os.environ.get(k, str(d)))
    except: return d

def _env_bool(k: str, d: bool) -> bool:
    v = os.environ.get(k, None)
    if v is None: return d
    return v.strip().lower() in ("1", "true", "yes", "y", "on")

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = "/data/downloaded_models/Qwen1.5-MoE-A2.7B"
    OUTPUT_DIR: str = "/home/daniyar/moe_ws_outputs_qwen"
    LAYER: int = 0
    MAX_EXPERTS: int = 4

    # Calibration / router
    CALIB_PATH: str = _env_str("CALIB_PATH", "").strip()
    ROUTER_PATH: str = _env_str("ROUTER_PATH", "").strip()
    CALIB_SAMPLES: int = _env_int("CALIB_SAMPLES", 4096)
    RIDGE_WEIGHTED: bool = _env_bool("RIDGE_WEIGHTED", False)
    ROUTER_EIDS_ARE_GLOBAL: bool = _env_bool("ROUTER_EIDS_ARE_GLOBAL", True)
    RIDGE_DAMP: float = _env_float("RIDGE_DAMP", 1e-3)
    NORMALIZE_W: bool = _env_bool("NORMALIZE_W", True)

    # Capture (optional) – SET THIS TO True IF NO CALIB_PATH
    CAPTURE_ENABLE: bool = True   # <-- CHANGED: auto-collect real calibration
    CAPTURE_FORCE: bool = _env_bool("CAPTURE_FORCE", False)
    CAPTURE_ITERS: int = _env_int("CAPTURE_ITERS", 32)
    CAPTURE_BATCH: int = _env_int("CAPTURE_BATCH", 1)
    CAPTURE_MAX_TOKENS: int = _env_int("CAPTURE_MAX_TOKENS", 1024)
    CAPTURE_TEXT: str = _env_str("CAPTURE_TEXT", "DeepSeek MoE calibration text. " * 256)
    CAPTURE_TEXT_FILE: str = _env_str("CAPTURE_TEXT_FILE", "").strip()
    CAPTURE_KEEP_PAD: bool = _env_bool("CAPTURE_KEEP_PAD", False)
    HF_TRUST_REMOTE_CODE: bool = _env_bool("HF_TRUST_REMOTE_CODE", True)
    HF_LOCAL_FILES_ONLY: bool = _env_bool("HF_LOCAL_FILES_ONLY", True)
    HF_AUTO_PIP: bool = _env_bool("HF_AUTO_PIP", False)

    # Basis mode
    BASIS_MODE: str = _env_str("BASIS_MODE", "dense_train").lower()  # dense_train | identity | hadamard_perm
    BASIS_STORE_DTYPE: str = _env_str("BASIS_STORE_DTYPE", "float16").lower()

    # Clustering
    M0: int = _env_int("M0", 0)                # 0 = auto
    M_MAX: int = _env_int("M_MAX", 16)
    CLUSTER_FEAT_D: int = _env_int("CLUSTER_FEAT_D", 64)
    CLUSTER_ITERS: int = _env_int("CLUSTER_ITERS", 60)
    CLUSTER_RESTARTS: int = _env_int("CLUSTER_RESTARTS", 4)
    CLUSTER_MIN_SIZE: int = _env_int("CLUSTER_MIN_SIZE", 2)
    CLUSTER_MAX_SIZE: int = _env_int("CLUSTER_MAX_SIZE", 4)
    SPLIT_ITERS: int = _env_int("SPLIT_ITERS", 50)

    # Training (dense bases)
    TRAIN_STEPS: int = _env_int("TRAIN_STEPS", 24)
    TRAIN_WARMUP: int = _env_int("TRAIN_WARMUP", 6)
    TRAIN_LR: float = _env_float("TRAIN_LR", 5e-2)
    SUBM: int = _env_int("SUBM", 256)
    BATCH_E: int = _env_int("BATCH_E", 4)
    TRAIN_MIN_CLUSTER: int = _env_int("TRAIN_MIN_CLUSTER", 2)
    REORTHO_EVERY: int = _env_int("REORTHO_EVERY", 4)
    REPORT_EVERY: int = _env_int("REPORT_EVERY", 4)
    GRAD_CLIP: float = _env_float("GRAD_CLIP", 1.0)
    TRAIN_OBJ: str = _env_str("TRAIN_OBJ", "logratio").lower()
    TRAIN_LAM_BLOCK: float = _env_float("TRAIN_LAM_BLOCK", 0.10)
    TRAIN_LAM_GUIDE: float = _env_float("TRAIN_LAM_GUIDE", 1.0)
    TRAIN_GUIDE_EVERY: int = _env_int("TRAIN_GUIDE_EVERY", 2)
    TRAIN_GUIDE_TARGET: float = _env_float("TRAIN_GUIDE_TARGET", 0.80)
    TRAIN_GUIDE_MAX_BLOCKS: int = _env_int("TRAIN_GUIDE_MAX_BLOCKS", 2048)

    # Core selection
    CORE_MODE: str = _env_str("CORE_MODE", "blocktopk_perexpert").lower()
    CORE_AGG: str = _env_str("CORE_AGG", "mean").lower()
    CORE_BLOCK: int = _env_int("CORE_BLOCK", 64)
    CORE_TARGET: float = _env_float("CORE_TARGET", 0.85)
    CORE_MAX_BLOCKS: int = _env_int("CORE_MAX_BLOCKS", 256)

    # Residual
    RES_RANK: int = _env_int("RES_RANK", 512)
    RES_COEF: str = _env_str("RES_COEF", "diag").lower()
    RES_TARGET: float = _env_float("RES_TARGET", 0.995)
    RES_MAX_BLOCKS: int = _env_int("RES_MAX_BLOCKS", 4096)
    RES_BSIZE: int = _env_int("RES_BSIZE", 64)

    # Refine
    REFINE_ENABLE: bool = _env_bool("REFINE_ENABLE", True)
    REFINE_ERR_TARGET: float = _env_float("REFINE_ERR_TARGET", 0.03)
    REFINE_MAX_EXTRA: int = _env_int("REFINE_MAX_EXTRA", 4096)
    REFINE_BSIZE: int = _env_int("REFINE_BSIZE", 64)
    REFINE_RECHECK_EVERY: int = _env_int("REFINE_RECHECK_EVERY", 32)

    # Quantization
    QMODE: str = _env_str("QMODE", "none").lower()  # none|float16|int8

    # Eval
    EVAL_TRIALS: int = _env_int("EVAL_TRIALS", 8)
    EVAL_BATCH: int = _env_int("EVAL_BATCH", 2)
    ROUTED_K: int = _env_int("ROUTED_K", 8)

cfg = Cfg()
PRESET = _env_str("PRESET", "").strip().lower()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Apply presets (override only if user did not set explicitly)
def _setdefault_env(k: str, v: str):
    if k not in os.environ: os.environ[k] = v

if PRESET == "maxacc":
    _setdefault_env("CALIB_SAMPLES", "32768")
    _setdefault_env("RIDGE_DAMP", "1e-2")
    _setdefault_env("CORE_BLOCK", "32")
    _setdefault_env("CORE_TARGET", "0.995")
    _setdefault_env("CORE_MAX_BLOCKS", "8192")
    _setdefault_env("RES_RANK", "2048")
    _setdefault_env("RES_COEF", "full")
    _setdefault_env("RES_TARGET", "0.999")
    _setdefault_env("RES_MAX_BLOCKS", "32768")
    _setdefault_env("REFINE_ENABLE", "1")
    _setdefault_env("REFINE_ERR_TARGET", "0.01")
    _setdefault_env("REFINE_MAX_EXTRA", "65536")
    _setdefault_env("TRAIN_STEPS", "96")
    _setdefault_env("TRAIN_LR", "0.02")
    _setdefault_env("TRAIN_LAM_GUIDE", "0.5")
    cfg = Cfg()
elif PRESET == "compact":
    _setdefault_env("CALIB_SAMPLES", "4096")
    _setdefault_env("CORE_BLOCK", "64")
    _setdefault_env("CORE_TARGET", "0.90")
    _setdefault_env("CORE_MAX_BLOCKS", "512")
    _setdefault_env("RES_RANK", "512")
    _setdefault_env("RES_COEF", "diag")
    _setdefault_env("RES_TARGET", "0.99")
    _setdefault_env("RES_MAX_BLOCKS", "4096")
    _setdefault_env("QMODE", "float16")
    _setdefault_env("REFINE_ENABLE", "0")
    _setdefault_env("TRAIN_STEPS", "24")
    cfg = Cfg()

# -----------------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------------
def log(msg: str): print(msg, flush=True)
def now() -> str: return time.strftime("%Y-%m-%d %H:%M:%S")

def seed_all(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

SEED = _env_int("SEED", 1234)
seed_all(SEED)
NTHREADS = _env_int("KTXX_THREADS", 8)
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
try: torch.set_num_threads(NTHREADS)
except: pass

DEVICE = torch.device(_env_str("DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
DTYPE_ACC = torch.float32

# -----------------------------------------------------------------------------
# NPZ I/O
# -----------------------------------------------------------------------------
def save_npz_compressed(path: str, arrays: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, **arrays)

def load_npz(path: str) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    return {k: z[k] for k in z.files}

def _encode_meta(meta: dict) -> np.ndarray:
    return np.frombuffer(json.dumps(meta, sort_keys=True).encode("utf-8"), dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try: return json.loads(bytes(arr.tolist()).decode("utf-8"))
    except: return {}

# -----------------------------------------------------------------------------
# Offline shard loading
# -----------------------------------------------------------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r") as f:
        return json.load(f).get("weight_map", {})

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    # Try both common MoE patterns:
    #   - DeepSeek style: model.layers.{L}.mlp.experts.{E}.*
    #   - Mixtral style:  model.layers.{L}.block_sparse_moe.experts.{E}.*
    patterns = [
        rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.",
        rf"^model\.layers\.{layer}\.block_sparse_moe\.experts\.(\d+)\.",
    ]
    ids = set()
    for pat_str in patterns:
        pat = re.compile(pat_str)
        for k in weight_map:
            m = pat.match(k)
            if m:
                ids.add(int(m.group(1)))
        if ids:
            break
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    # Determine which MoE prefix is present
    prefixes = [
        f"model.layers.{layer}.mlp.experts.{eid}.",
        f"model.layers.{layer}.block_sparse_moe.experts.{eid}.",
    ]
    used_prefix = None
    for pfx in prefixes:
        if any(k.startswith(pfx) for k in weight_map):
            used_prefix = pfx
            break
    if used_prefix is None:
        return {}

    def pick(cands):
        for suf in cands:
            k = used_prefix + suf
            if k in weight_map:
                return k
        return None

    # Mixtral uses w1 (gate), w2 (down), w3 (up). DeepSeek uses gate_proj/up_proj/down_proj.
    # Try Mixtral naming first, then fall back to DeepSeek.
    gate = pick(["w1.weight", "gate_proj.weight"])
    down = pick(["w2.weight", "down_proj.weight"])
    up   = pick(["w3.weight", "up_proj.weight"])

    if gate is None or down is None or up is None:
        return {}
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard = {}
    for k in keys:
        shard = weight_map.get(k)
        if shard is None: continue
        by_shard.setdefault(shard, []).append(k)
    out = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp): continue
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks: out[k] = f.get_tensor(k)
    return out

# -----------------------------------------------------------------------------
# Calibration / Router
# -----------------------------------------------------------------------------
def autodetect_calib_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return cand if os.path.isfile(cand) else None

def autodetect_router_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return cand if os.path.isfile(cand) else None

def load_calib_X(path: str, H: int) -> torch.Tensor:
    z = np.load(path)
    X = torch.from_numpy(z["X"].astype(np.float32))
    if X.ndim != 2 or X.shape[1] != H: raise RuntimeError(f"Bad X shape {X.shape}")
    if X.shape[0] > cfg.CALIB_SAMPLES: X = X[:cfg.CALIB_SAMPLES]
    return X.to(device=DEVICE, dtype=DTYPE_ACC)

def load_router_P(path: str) -> np.ndarray:
    return np.load(path)["P"].astype(np.float32)

def _maybe_autopip():
    if not cfg.HF_AUTO_PIP: return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", "transformers", "sentencepiece", "tokenizers"])

def _patch_transformers_cache_compat():
    try:
        from transformers.cache_utils import DynamicCache
        if not hasattr(DynamicCache, "get_usable_length"):
            DynamicCache.get_usable_length = lambda self, seq_length: int(seq_length)
    except: pass

class _Collector:
    def __init__(self, H, E_total, max_rows):
        self.H = H; self.E_total = E_total; self.max_rows = max_rows
        self.X_chunks, self.P_chunks = [], []; self.nX = self.nP = 0

    def _take(self, flat, need): return flat[:need] if flat.shape[0] > need else flat

    def add_X(self, hs, attn_mask):
        if hs is None: return
        if hs.ndim == 2: hs = hs.unsqueeze(0)
        if hs.ndim != 3 or hs.shape[-1] != self.H: return
        hs = hs.detach().to(torch.float32).cpu()
        if attn_mask is not None and not cfg.CAPTURE_KEEP_PAD:
            m = attn_mask.cpu().to(torch.bool); flat = hs.reshape(-1, self.H)[m.reshape(-1)]
        else: flat = hs.reshape(-1, self.H)
        if flat.numel() == 0: return
        need = self.max_rows - self.nX
        if need <= 0: return
        self.X_chunks.append(self._take(flat, need)); self.nX += self.X_chunks[-1].shape[0]

    def add_logits(self, logits, attn_mask):
        if logits is None: return
        if logits.ndim == 2: logits = logits.unsqueeze(0)
        if logits.ndim != 3: return
        P = torch.softmax(logits.detach().to(torch.float32), dim=-1)[..., :self.E_total].cpu()
        if attn_mask is not None and not cfg.CAPTURE_KEEP_PAD:
            m = attn_mask.cpu().to(torch.bool); flat = P.reshape(-1, P.shape[-1])[m.reshape(-1)]
        else: flat = P.reshape(-1, P.shape[-1])
        if flat.numel() == 0: return
        need = self.max_rows - self.nP
        if need <= 0: return
        self.P_chunks.append(self._take(flat, need)); self.nP += self.P_chunks[-1].shape[0]

def capture_XP_transformers(model_dir, layer_idx, H, E_total, out_x, out_p):
    _maybe_autopip(); _patch_transformers_cache_compat()
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=cfg.HF_TRUST_REMOTE_CODE, local_files_only=cfg.HF_LOCAL_FILES_ONLY)
    if tok.pad_token is None: tok.pad_token = tok.eos_token or tok.unk_token
    model = AutoModelForCausalLM.from_pretrained(model_dir, trust_remote_code=cfg.HF_TRUST_REMOTE_CODE, local_files_only=cfg.HF_LOCAL_FILES_ONLY, torch_dtype=torch.float16 if DEVICE.type=="cuda" else torch.float32, low_cpu_mem_usage=True).to(DEVICE).eval()

    # locate layer and mlp
    layers = None
    if hasattr(model, "model") and hasattr(model.model, "layers"): layers = model.model.layers
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"): layers = model.transformer.h
    elif hasattr(model, "layers"): layers = model.layers
    if layers is None: raise RuntimeError("Cannot locate layers")
    if layer_idx >= len(layers): raise RuntimeError(f"Layer {layer_idx} out of range")
    layer = layers[layer_idx]
    mlp = getattr(layer, "mlp", None)
    if mlp is None:
        for n, m in layer.named_modules():
            if n.lower().endswith("mlp"): mlp = m; break
    if mlp is None: raise RuntimeError("Could not find layer.mlp")

    # router discovery (best-effort)
    router_linear = None
    for name, mod in layer.named_modules():
        if isinstance(mod, nn.Linear) and mod.in_features == H and mod.out_features >= E_total:
            if "router" in name.lower() or "gate" in name.lower(): router_linear = mod; break

    coll = _Collector(H, E_total, cfg.CALIB_SAMPLES)
    attn_holder = {"mask": None}

    def mlp_pre_hook(_, inputs):
        coll.add_X(inputs[0], attn_holder["mask"])

    h1 = mlp.register_forward_pre_hook(mlp_pre_hook)
    h2 = None
    if router_linear is not None:
        def router_hook(_, __, out):
            o = out[0] if isinstance(out, (tuple, list)) else out
            coll.add_logits(o, attn_holder["mask"])
        h2 = router_linear.register_forward_hook(router_hook)

    texts = [cfg.CAPTURE_TEXT]
    if cfg.CAPTURE_TEXT_FILE and os.path.isfile(cfg.CAPTURE_TEXT_FILE):
        with open(cfg.CAPTURE_TEXT_FILE) as f: texts = [ln.strip() for ln in f if ln.strip()]
    tptr = 0
    for it in range(cfg.CAPTURE_ITERS):
        text = texts[tptr % len(texts)]; tptr += 1
        enc = tok(text, return_tensors="pt", truncation=True, max_length=cfg.CAPTURE_MAX_TOKENS, padding="max_length")
        for k in enc:
            if enc[k].ndim == 2 and cfg.CAPTURE_BATCH > 1: enc[k] = enc[k].repeat(cfg.CAPTURE_BATCH, 1)
        attn_holder["mask"] = enc.get("attention_mask")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.inference_mode(): _ = model(**enc, use_cache=False)
        if (it+1) % 4 == 0: log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS} nX={coll.nX} nP={coll.nP}")
        if coll.nX >= cfg.CALIB_SAMPLES and (not cfg.RIDGE_WEIGHTED or coll.nP >= cfg.CALIB_SAMPLES): break

    h1.remove()
    if h2: h2.remove()

    if coll.nX == 0: raise RuntimeError("Capture collected 0 rows")
    X = torch.cat(coll.X_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz_compressed(out_x, {"X": X})
    log(f"[capture] wrote X -> {out_x} shape={X.shape}")
    p_written = None
    if coll.nP > 0:
        P = torch.cat(coll.P_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
        N = min(P.shape[0], X.shape[0])
        if N < X.shape[0]: X = X[:N]; save_npz_compressed(out_x, {"X": X})
        P = P[:N]; save_npz_compressed(out_p, {"P": P})
        log(f"[capture] wrote P -> {out_p} shape={P.shape}")
        p_written = out_p
    return out_x, p_written

def ensure_calib_router(H: int, E_total: int):
    if not cfg.CALIB_PATH:
        c = autodetect_calib_path()
        if c: cfg.CALIB_PATH = c; log(f"[calib] auto-found {cfg.CALIB_PATH}")
    if not cfg.ROUTER_PATH:
        r = autodetect_router_path()
        if r: cfg.ROUTER_PATH = r; log(f"[router] auto-found {cfg.ROUTER_PATH}")
    if cfg.CAPTURE_FORCE or (cfg.CAPTURE_ENABLE and (not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH))):
        out_x = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
        out_p = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
        log("[capture] capturing via transformers...")
        x_path, p_path = capture_XP_transformers(cfg.MODEL_DIR, cfg.LAYER, H, E_total, out_x, out_p)
        cfg.CALIB_PATH = x_path
        if p_path: cfg.ROUTER_PATH = p_path

# -----------------------------------------------------------------------------
# Ridge linearization: build Ws
# -----------------------------------------------------------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate, W_up, W_down) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    return hid @ W_down.to(DTYPE_ACC).t()

def ws_cache_path(E: int) -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{E}_ridge_ebc.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        script="ebc_llm", model_dir=cfg.MODEL_DIR, layer=cfg.LAYER, expert_ids=eids,
        ridge_damp=cfg.RIDGE_DAMP, ridge_weighted=cfg.RIDGE_WEIGHTED,
        router_path=cfg.ROUTER_PATH or "", calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES, normalize_w=cfg.NORMALIZE_W, seed=SEED, device=str(DEVICE)
    )

@torch.no_grad()
def build_Ws(eids: List[int], wm: Dict[str, str]) -> Tuple[torch.Tensor, torch.Tensor]:
    per_e, need_keys = {}, []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk: raise RuntimeError(f"Expert {eid} missing tensors")
        per_e[eid] = kk; need_keys += [kk["up"], kk["down"], kk["gate"]]
    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))
    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = W_up0.shape[0], W_up0.shape[1]
    log(f"[shape] H={H} d_ff={dff}")

    ensure_calib_router(H, len(find_layer_expert_ids(wm, cfg.LAYER)))
    if not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH):
        raise RuntimeError("CALIB_PATH missing. Set CALIB_PATH or CAPTURE_ENABLE=1.")
    X = load_calib_X(cfg.CALIB_PATH, H)
    log(f"[calib] X: {X.shape}")

    P = None
    if cfg.RIDGE_WEIGHTED:
        if cfg.ROUTER_PATH and os.path.isfile(cfg.ROUTER_PATH):
            P = load_router_P(cfg.ROUTER_PATH)
            log(f"[router] P: {P.shape}")
        else:
            log("[router] RIDGE_WEIGHTED=1 but ROUTER_PATH missing -> disabling.")
            cfg.RIDGE_WEIGHTED = False

    Xf = X.to(DTYPE_ACC); I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * torch.trace(XtX).item() / H
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list, scales = [], []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws (ridge)")):
        W_up = T[per_e[eid]["up"]].to(DEVICE)
        W_dn = T[per_e[eid]["down"]].to(DEVICE)
        W_gt = T[per_e[eid]["gate"]].to(DEVICE)
        Y = forward_mlp(X, W_gt, W_up, W_dn).to(DTYPE_ACC)

        if cfg.RIDGE_WEIGHTED and P is not None:
            w = torch.from_numpy(P[:X.shape[0], eid if cfg.ROUTER_EIDS_ARE_GLOBAL else i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0)
            sw = torch.sqrt(w + 1e-12).view(-1,1)
            Xw, Yw = Xf * sw, Y * sw
            XtX_e = Xw.t() @ Xw
            lam_e = cfg.RIDGE_DAMP * torch.trace(XtX_e).item() / H
            chol = torch.linalg.cholesky(XtX_e + lam_e * I)
            Wt = torch.cholesky_solve(Xw.t() @ Yw, chol)
            W = Wt.t().contiguous()
        else:
            Wt = torch.cholesky_solve(Xf.t() @ Y, cholG)
            W = Wt.t().contiguous()

        if cfg.NORMALIZE_W:
            s = torch.linalg.norm(W, ord="fro").clamp_min(1e-12).item()
            W = W / s
        else: s = 1.0
        Ws_list.append(W); scales.append(s)

    Ws = torch.stack(Ws_list).to(DTYPE_ACC).to(DEVICE)
    Sc = torch.tensor(scales, dtype=DTYPE_ACC, device=DEVICE)
    return Ws, Sc

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor, torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids: raise RuntimeError(f"No experts at layer {cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer={cfg.LAYER} total={len(all_eids)} using={len(eids)} eids={eids}")

    if not cfg.CALIB_PATH: cfg.CALIB_PATH = autodetect_calib_path() or ""
    if not cfg.ROUTER_PATH: cfg.ROUTER_PATH = autodetect_router_path() or ""

    cpath = ws_cache_path(len(eids))
    if os.path.isfile(cpath):
        z = load_npz(cpath)
        if all(k in z for k in ["meta","Ws","expert_ids","scales"]) and _decode_meta(z["meta"]) == ws_meta(eids):
            Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
            Sc = torch.from_numpy(z["scales"]).to(DTYPE_ACC).to(DEVICE)
            log(f"[cache] loaded Ws -> {cpath} shape={Ws.shape}")
            return [int(x) for x in z["expert_ids"]], Ws, Sc
        log("[cache] meta mismatch -> rebuild")

    Ws, Sc = build_Ws(eids, wm)
    save_npz_compressed(cpath, {
        "meta": _encode_meta(ws_meta(eids)),
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.cpu().numpy().astype(np.float32),
        "scales": Sc.cpu().numpy().astype(np.float32)
    })
    log(f"[cache] wrote Ws -> {cpath} size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws, Sc

# -----------------------------------------------------------------------------
# Clustering (kmeans++ + hierarchical split)
# -----------------------------------------------------------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED+17)
    R = (torch.randint(0,2,(n,d),generator=g,dtype=torch.int8)*2-1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]; row = torch.diag(W @ W.t()); col = torch.diag(W.t() @ W)
        feats.append(torch.cat([row @ R, col @ R]).unsqueeze(0))
    X = torch.cat(feats, dim=0)
    X = (X - X.mean(0, keepdim=True)) / (X.std(0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> torch.Tensor:
    best_lab, best_inertia = None, float("inf")
    g = torch.Generator(device="cpu").manual_seed(SEED+999)
    for _ in range(max(1, restarts)):
        # kmeans++ init
        n = X.shape[0]
        centers = [X[torch.randint(0, n, (1,), generator=g).item()].clone()]
        for _ in range(1, k):
            C = torch.stack(centers)
            dist2 = torch.cdist(X, C).pow(2).min(1).values
            prob = dist2 / dist2.sum().clamp_min(1e-12)
            centers.append(X[torch.multinomial(prob, 1, generator=g).item()].clone())
        C = torch.stack(centers)
        for _ in range(iters):
            dist = torch.cdist(X, C); lab = dist.argmin(1)
            for j in range(k):
                m = (lab == j)
                if m.any(): C[j] = X[m].mean(0)
                else: C[j] = X[dist.min(1).values.argmax().item()].clone()
        inertia = torch.cdist(X, C).min(1).values.pow(2).sum().item()
        if inertia < best_inertia: best_inertia, best_lab = inertia, lab.clone()
    return best_lab.to(torch.int64)

@torch.no_grad()
def relabel_contiguous(labels: torch.Tensor) -> torch.Tensor:
    uniq = torch.unique(labels); out = labels.clone()
    for new, old in enumerate(uniq.tolist()): out[labels == old] = new
    return out

@torch.no_grad()
def merge_small_clusters(X: torch.Tensor, labels: torch.Tensor, min_size: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if min_size <= 1: return labels
    while True:
        K = labels.max().item() + 1
        counts = torch.bincount(labels, minlength=K)
        small = (counts < min_size).nonzero(as_tuple=False).flatten()
        if small.numel() == 0: break
        C = torch.stack([X[labels == k].mean(0) for k in range(K)])
        for c in small.tolist():
            idxs = (labels == c).nonzero(as_tuple=False).flatten()
            if idxs.numel() == 0: continue
            dist = torch.cdist(C[c].unsqueeze(0), C).squeeze(0); dist[c] = 1e9
            labels[idxs] = dist.argmin().item()
        labels = relabel_contiguous(labels)
    return labels

@torch.no_grad()
def hierarchical_split(X: torch.Tensor, labels: torch.Tensor, max_size: int, max_k: int, split_iters: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if max_size <= 0: return labels
    while True:
        K = labels.max().item() + 1
        if K >= max_k: break
        counts = torch.bincount(labels, minlength=K)
        biggest = counts.argmax().item()
        if counts[biggest] <= max_size: break
        idxs = (labels == biggest).nonzero(as_tuple=False).flatten()
        if idxs.numel() < 2: break
        sub = X[idxs]; sub_lab = kmeans_torch(sub, 2, split_iters, 1)
        a, b = idxs[sub_lab == 0], idxs[sub_lab == 1]
        if a.numel() == 0 or b.numel() == 0: break
        labels[b] = K
        labels = relabel_contiguous(labels)
    return labels

# -----------------------------------------------------------------------------
# Basis training (dense)
# -----------------------------------------------------------------------------
class OrthoParam(nn.Module):
    def __init__(self, init_mat: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(init_mat.to(DEVICE, DTYPE_ACC).contiguous())
    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M); return Q

@torch.no_grad()
def svd_init_from_mean(Wmean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(Wmean, full_matrices=False)
    return U.to(DTYPE_ACC).contiguous(), Vh.t().to(DTYPE_ACC).contiguous()

def schedule(step: int, warmup: int, total: int) -> float:
    if step <= warmup: return 0.0
    return min(1.0, (step - warmup) / max(1, total - warmup))

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    U_S, V_S = U[:, S], V[:, S]
    return torch.matmul(U_S.t().unsqueeze(0), Ws_batch @ V_S)

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return (Xs - torch.diag_embed(D)).abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    return torch.diagonal(Xs, dim1=1, dim2=2).abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    Eb, s, _ = Xs.shape; b = int(block)
    if b <= 0: return torch.zeros((), device=Xs.device)
    nb = s // b
    if nb <= 0: return torch.zeros((), device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0,1,3,2,4).contiguous()
    Eblk = (Xb * Xb).sum(dim=(3,4))
    P = Eblk.mean(0)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

@torch.no_grad()
def make_guidance_mask_from_Xs(Xs: torch.Tensor, block: int, target: float, max_blocks: int) -> Tuple[torch.Tensor, float, int]:
    Eb, s, _ = Xs.shape; b = int(block)
    if b <= 0: return torch.ones(s,s,device=Xs.device), 1.0, 0
    nb = s // b
    if nb <= 0: return torch.ones(s,s,device=Xs.device), 1.0, 0
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0,1,3,2,4).contiguous()
    Eg = (Xb * Xb).sum(dim=(3,4)).mean(0)
    tot = (X * X).sum().item() / max(1, Eb)
    flat = Eg.reshape(-1); order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], 0)
    frac = csum / max(tot, 1e-12)
    need = (frac >= target).nonzero(as_tuple=False)[0].item() + 1 if (frac >= target).any() else flat.numel()
    K = min(need, max_blocks, flat.numel())
    mask = torch.zeros(s2, s2, device=Xs.device)
    for idx in order[:K].tolist():
        bi, bj = idx // nb, idx % nb
        mask[bi*b:(bi+1)*b, bj*b:(bj+1)*b] = 1.0
    if s2 < s:
        full = torch.zeros(s, s, device=Xs.device); full[:s2, :s2] = mask; mask = full
    ef = float(frac[K-1].item()) if K > 0 else 0.0
    return mask, ef, K

# -----------------------------------------------------------------------------
# Block energy & selection
# -----------------------------------------------------------------------------
@torch.no_grad()
def block_energy_grid(X: torch.Tensor, b: int) -> Tuple[torch.Tensor, float, int]:
    n = X.shape[0]; nb = (n + b - 1) // b
    if n % b != 0:
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X; X = Xp
    Xb = X.view(nb, b, nb, b).permute(0,2,1,3).contiguous()
    Eg = (Xb * Xb).sum(dim=(2,3))
    tot = (X * X).sum().item()
    return Eg, tot, nb

@torch.no_grad()
def pick_blocks_until_target(Eg: torch.Tensor, tot_energy: float, target: float, max_blocks: int,
                             exclude: Optional[Set[Tuple[int,int]]]=None) -> Tuple[List[Tuple[int,int]], float]:
    nb = Eg.shape[0]; flat = Eg.reshape(-1); order = torch.argsort(flat, descending=True)
    picked, eacc = [], 0.0
    exclude = exclude or set()
    for idx in order.tolist():
        if len(picked) >= max_blocks: break
        e = flat[idx].item()
        if e <= 1e-18: break
        bi, bj = idx // nb, idx % nb
        if (bi, bj) in exclude: continue
        picked.append((bi, bj)); eacc += e
        if eacc / max(tot_energy, 1e-12) >= target: break
    return picked, eacc / max(tot_energy, 1e-12)

@torch.no_grad()
def gather_block(X: torch.Tensor, i0: int, j0: int, b: int) -> torch.Tensor:
    n = X.shape[0]; i1, j1 = min(n, i0+b), min(n, j0+b)
    return X[i0:i1, j0:j1].contiguous()

# -----------------------------------------------------------------------------
# Low-rank (randomized SVD)
# -----------------------------------------------------------------------------
@torch.no_grad()
def rand_svd_vectors(A: torch.Tensor, r: int, n_iter: int=2) -> Tuple[torch.Tensor, torch.Tensor]:
    n = A.shape[0]; r = min(r, n)
    g = torch.Generator(device="cpu").manual_seed(SEED+777)
    Omega = torch.randn(n, r, generator=g, dtype=DTYPE_ACC, device=A.device)
    Y = A @ Omega
    for _ in range(n_iter): Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    Uhat, _, Vh = torch.linalg.svd(B, full_matrices=False)
    return (Q @ Uhat[:, :r]).contiguous(), Vh.t()[:, :r].contiguous()

# -----------------------------------------------------------------------------
# Payload packing (ragged blocks)
# -----------------------------------------------------------------------------
def _block_store_dtype(qmode: str) -> np.dtype:
    return np.float32 if qmode == "none" else np.float16

def pack_blocks_ragged(blocks_per_item: List[List[Tuple[int,int,torch.Tensor]]], qmode: str) -> Dict[str, np.ndarray]:
    val_dtype = _block_store_dtype(qmode)
    M = len(blocks_per_item)
    item_ptr = [0]
    blk_i0, blk_j0, blk_h, blk_w = [], [], [], []
    blk_ptr = [0]
    vals, vals_i8, scales = [], [], []
    for m in range(M):
        for (i0, j0, B) in blocks_per_item[m]:
            h, w = B.shape
            blk_i0.append(i0); blk_j0.append(j0); blk_h.append(h); blk_w.append(w)
            if qmode == "int8":
                x = B.cpu().float(); maxabs = x.abs().max().item()
                if maxabs < 1e-12: q = np.zeros(x.numel(), dtype=np.int8); sc = np.float16(1.0)
                else:
                    scale = maxabs / 127.0
                    q = torch.clamp(torch.round(x/scale), -127, 127).to(torch.int8).numpy()
                    sc = np.float16(scale)
                vals_i8.append(q.reshape(-1)); scales.append(sc)
                blk_ptr.append(blk_ptr[-1] + q.size)
            else:
                v = B.cpu().float().numpy().astype(val_dtype).reshape(-1)
                vals.append(v); blk_ptr.append(blk_ptr[-1] + v.size)
        item_ptr.append(len(blk_i0))

    out = {
        "item_ptr": np.array(item_ptr, dtype=np.int32),
        "blk_i0": np.array(blk_i0, dtype=np.int16),
        "blk_j0": np.array(blk_j0, dtype=np.int16),
        "blk_h": np.array(blk_h, dtype=np.int16),
        "blk_w": np.array(blk_w, dtype=np.int16),
        "blk_ptr": np.array(blk_ptr, dtype=np.int64)
    }
    if qmode == "int8":
        out["blk_q"] = np.concatenate(vals_i8).astype(np.int8) if vals_i8 else np.zeros((0,), dtype=np.int8)
        out["blk_scale"] = np.array(scales, dtype=np.float16)
    else:
        out["blk_val"] = np.concatenate(vals) if vals else np.zeros((0,), dtype=val_dtype)
    return out

def unpack_blocks_ragged(pack: Dict[str, np.ndarray], qmode: str, device: torch.device) -> List[List[Tuple[int,int,torch.Tensor]]]:
    item_ptr = pack["item_ptr"]
    blk_i0 = pack["blk_i0"]; blk_j0 = pack["blk_j0"]; blk_h = pack["blk_h"]; blk_w = pack["blk_w"]
    blk_ptr = pack["blk_ptr"]
    if qmode == "int8":
        blk_q = pack["blk_q"]; blk_scale = pack["blk_scale"]; blk_val = None
    else:
        blk_val = pack["blk_val"]; blk_q = None; blk_scale = None
    M = item_ptr.shape[0] - 1
    out = []
    for m in range(M):
        b0, b1 = item_ptr[m], item_ptr[m+1]
        lst = []
        for bi in range(b0, b1):
            i0, j0 = int(blk_i0[bi]), int(blk_j0[bi])
            h, w = int(blk_h[bi]), int(blk_w[bi])
            v0, v1 = blk_ptr[bi], blk_ptr[bi+1]
            if qmode == "int8":
                q = blk_q[v0:v1].astype(np.float32); sc = float(blk_scale[bi])
                B = torch.from_numpy((q * sc).reshape(h, w)).to(device, DTYPE_ACC)
            else:
                B = torch.from_numpy(blk_val[v0:v1].astype(np.float32).reshape(h, w)).to(device, DTYPE_ACC)
            lst.append((i0, j0, B))
        out.append(lst)
    return out

# -----------------------------------------------------------------------------
# Payload runtime
# -----------------------------------------------------------------------------
class PayloadRuntime:
    def __init__(self):
        self.meta = {}
        self.expert_ids = []
        self.scales: Optional[torch.Tensor] = None
        self.cluster_of_pos: Optional[torch.Tensor] = None
        self.U: List[torch.Tensor] = []
        self.V: List[torch.Tensor] = []
        self.DL: List[torch.Tensor] = []
        self.DR: List[torch.Tensor] = []
        self.gam: Optional[torch.Tensor] = None
        self.Cfull: Optional[torch.Tensor] = None
        self.core_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.res_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.qmode = "none"
        self.res_coef = "diag"

    @torch.no_grad()
    def apply_expert(self, x: torch.Tensor, pos: int) -> torch.Tensor:
        c = int(self.cluster_of_pos[pos].item())
        U, V = self.U[c], self.V[c]
        DL, DR = self.DL[c], self.DR[c]
        z = x @ U
        u = torch.zeros_like(z)
        for (i0, j0, B) in self.core_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B
        if self.res_coef == "diag":
            g = self.gam[pos]
            u += ((z @ DL) * g.view(1,-1)) @ DR.t()
        else:
            C = self.Cfull[pos]
            u += (z @ DL) @ C @ DR.t()
        for (i0, j0, B) in self.res_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B
        y = u @ V.t()
        if self.scales is not None:
            y = y * self.scales[pos]
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        for a, pos in zip(gates.tolist(), routed):
            y += a * self.apply_expert(x, int(pos))
        return y

def load_payload_runtime(path: str, device: torch.device) -> PayloadRuntime:
    z = load_npz(path)
    rt = PayloadRuntime()
    rt.meta = _decode_meta(z["meta"])
    rt.qmode = rt.meta.get("qmode", "none")
    rt.res_coef = rt.meta.get("res_coef", "diag")
    rt.expert_ids = [int(x) for x in z["expert_ids"]]
    rt.scales = torch.from_numpy(z["scales"]).to(device, DTYPE_ACC)
    rt.cluster_of_pos = torch.from_numpy(z["cluster_of_pos"]).to(device, torch.int64)
    M = z["n_clusters"][0]
    for m in range(M):
        rt.U.append(torch.from_numpy(z[f"U_{m}"]).to(device, DTYPE_ACC))
        rt.V.append(torch.from_numpy(z[f"V_{m}"]).to(device, DTYPE_ACC))
        rt.DL.append(torch.from_numpy(z[f"DL_{m}"]).to(device, DTYPE_ACC))
        rt.DR.append(torch.from_numpy(z[f"DR_{m}"]).to(device, DTYPE_ACC))
    if rt.res_coef == "diag":
        rt.gam = torch.from_numpy(z["gam"]).to(device, DTYPE_ACC)
    else:
        rt.Cfull = torch.from_numpy(z["Cfull"]).to(device, DTYPE_ACC)
    core_pack = {k[5:]: z[k] for k in z if k.startswith("core_")}
    res_pack  = {k[4:]: z[k] for k in z if k.startswith("res_")}
    rt.core_blocks = unpack_blocks_ragged(core_pack, rt.qmode, device)
    rt.res_blocks  = unpack_blocks_ragged(res_pack, rt.qmode, device)
    return rt

# -----------------------------------------------------------------------------
# Build payload for one cluster
# -----------------------------------------------------------------------------
@torch.no_grad()
def frob_rel_err(A, B): return (torch.linalg.norm(A-B) / torch.linalg.norm(B).clamp_min(1e-12)).item()

@torch.no_grad()
def build_payload_for_cluster(Ws_norm: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor) -> Dict:
    n = Ws_norm.shape[-1]
    X_list = [(U.t() @ Ws_norm[pos] @ V).contiguous() for pos in idx]
    b = cfg.CORE_BLOCK

    # core blocks
    core_per = []
    core_ef = []
    for X in X_list:
        Eg, te, nb = block_energy_grid(X, b)
        picks, eff = pick_blocks_until_target(Eg, te, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*b, bj*b
            blocks.append((i0, j0, gather_block(X, i0, j0, b)))
        core_per.append(blocks); core_ef.append(eff)

    # residual after core
    R_list = []
    for X, cb in zip(X_list, core_per):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb: h,w = Bc.shape; Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    # low-rank shared
    Rmean = torch.stack(R_list).mean(0)
    r = min(cfg.RES_RANK, n)
    DL, DR = rand_svd_vectors(Rmean, r, n_iter=2)

    coef_list, res_per = [], []
    bb = cfg.RES_BSIZE
    for j, Rm in enumerate(R_list):
        if cfg.RES_COEF == "diag":
            g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()
            coef_list.append(g)
            R2 = (Rm - (DL * g.view(1,-1)) @ DR.t()).contiguous()
        else:
            C = (DL.t() @ Rm @ DR).contiguous()
            coef_list.append(C)
            R2 = (Rm - (DL @ C @ DR.t())).contiguous()

        Eg2, te2, nb2 = block_energy_grid(R2, bb)
        exclude = {(i0//bb, j0//bb) for (i0,j0,_) in core_per[j]}
        picks, _ = pick_blocks_until_target(Eg2, te2, cfg.RES_TARGET, cfg.RES_MAX_BLOCKS, exclude=exclude)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*bb, bj*bb
            blocks.append((i0, j0, gather_block(R2, i0, j0, bb)))
        res_per.append(blocks)

    # refine
    if cfg.REFINE_ENABLE:
        rb = cfg.REFINE_BSIZE
        for j in range(len(idx)):
            X = X_list[j]
            def reconstruct():
                Xc = torch.zeros_like(X)
                for (i0,j0,Bc) in core_per[j]: h,w=Bc.shape; Xc[i0:i0+h, j0:j0+w] = Bc
                if cfg.RES_COEF == "diag":
                    g = coef_list[j]; Xlr = (DL * g.view(1,-1)) @ DR.t()
                else:
                    C = coef_list[j]; Xlr = DL @ C @ DR.t()
                Xr = torch.zeros_like(X)
                for (i0,j0,Bb) in res_per[j]: h,w=Bb.shape; Xr[i0:i0+h, j0:j0+w] += Bb
                return Xc + Xlr + Xr
            Xhat = reconstruct()
            err = frob_rel_err(Xhat, X)
            added = 0
            core_pos = {(i0,j0) for (i0,j0,_) in core_per[j]}
            res_pos = {(i0,j0) for (i0,j0,_) in res_per[j]}
            while err > cfg.REFINE_ERR_TARGET and added < cfg.REFINE_MAX_EXTRA:
                Rerr = (X - Xhat).contiguous()
                Eg, te, nb = block_energy_grid(Rerr, rb)
                flat = Eg.reshape(-1)
                if flat.max().item() <= 1e-18: break
                order = torch.argsort(flat, descending=True)
                found = False
                for idx_ in order.tolist():
                    bi, bj = idx_ // nb, idx_ % nb
                    i0, j0 = bi*rb, bj*rb
                    if (i0, j0) in core_pos or (i0, j0) in res_pos: continue
                    Bb = gather_block(Rerr, i0, j0, rb)
                    res_per[j].append((i0, j0, Bb)); res_pos.add((i0, j0))
                    added += 1; found = True; break
                if not found: break
                if added % cfg.REFINE_RECHECK_EVERY == 0:
                    Xhat = reconstruct(); err = frob_rel_err(Xhat, X)
            Xhat = reconstruct(); err = frob_rel_err(Xhat, X)

    return {
        "core_blocks": core_per, "core_energy": core_ef,
        "DL": DL, "DR": DR, "coef_list": coef_list, "res_blocks": res_per
    }

# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------
@torch.no_grad()
def eval_payload(rt: PayloadRuntime, Ws_norm: torch.Tensor, Sc: torch.Tensor):
    E, n, _ = Ws_norm.shape
    errs = []
    for pos in range(E):
        x = torch.randn(8, n, dtype=DTYPE_ACC, device=DEVICE)
        y_hat = rt.apply_expert(x, pos)
        y_ref = x @ (Ws_norm[pos] * Sc[pos])
        errs.append((torch.linalg.norm(y_hat - y_ref) / torch.linalg.norm(y_ref).clamp_min(1e-12)).item())
    log(f"[eval] per-expert rel-error mean={np.mean(errs):.6f} p95={np.percentile(errs,95):.6f} max={np.max(errs):.6f}")

    mix = []
    for _ in range(cfg.EVAL_TRIALS):
        x = torch.randn(cfg.EVAL_BATCH, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), min(cfg.ROUTED_K, E))
        gates = torch.rand(len(routed), device=DEVICE); gates /= gates.sum()
        y_hat = rt.apply_mixture(x, routed, gates)
        Wsum = sum(gates[i].item() * (Ws_norm[pos] * Sc[pos]) for i, pos in enumerate(routed))
        y_ref = x @ Wsum
        mix.append((torch.linalg.norm(y_hat - y_ref) / torch.linalg.norm(y_ref).clamp_min(1e-12)).item())
    log(f"[eval] routed rel-error mean={np.mean(mix):.6f} ± {np.std(mix):.6f}")

# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def banner():
    log("="*60)
    log("EBC-LLM Compression Pipeline")
    log(f"Time: {now()}  Device: {DEVICE}")
    log(f"MODEL_DIR: {cfg.MODEL_DIR}  OUTPUT_DIR: {cfg.OUTPUT_DIR}")
    log(f"Layer: {cfg.LAYER}  Experts: {cfg.MAX_EXPERTS}")
    log(f"CALIB: {cfg.CALIB_PATH or '(none)'}  ROUTER: {cfg.ROUTER_PATH or '(none)'}")
    log(f"Ridge damp: {cfg.RIDGE_DAMP}  Normalize W: {cfg.NORMALIZE_W}")
    log(f"Basis: {cfg.BASIS_MODE}  Train steps: {cfg.TRAIN_STEPS}  lr: {cfg.TRAIN_LR}")
    log(f"Core: {cfg.CORE_MODE} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max={cfg.CORE_MAX_BLOCKS}")
    log(f"Residual: rank={cfg.RES_RANK} coef={cfg.RES_COEF} blocks={cfg.RES_MAX_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"Refine: {cfg.REFINE_ENABLE} target={cfg.REFINE_ERR_TARGET} max_extra={cfg.REFINE_MAX_EXTRA}")
    log("="*60)

def main():
    banner()
    expert_ids, Ws_norm, Sc = load_or_build_Ws()
    E, n, _ = Ws_norm.shape
    log(f"[Ws] shape={Ws_norm.shape}")

    # Clustering
    Xfeat = random_proj_features(Ws_norm, cfg.CLUSTER_FEAT_D)
    M0 = max(2, min(cfg.M0 if cfg.M0>0 else int(round(2*math.sqrt(E))), E))
    labels = kmeans_torch(Xfeat, M0, cfg.CLUSTER_ITERS, cfg.CLUSTER_RESTARTS)
    labels = merge_small_clusters(Xfeat, labels, cfg.CLUSTER_MIN_SIZE)
    labels = hierarchical_split(Xfeat, labels, cfg.CLUSTER_MAX_SIZE, min(cfg.M_MAX, E), cfg.SPLIT_ITERS)
    labels = merge_small_clusters(Xfeat, labels, cfg.CLUSTER_MIN_SIZE)
    labels = relabel_contiguous(labels)
    M = labels.max().item() + 1
    clusters = [torch.nonzero(labels==m, as_tuple=False).flatten().tolist() for m in range(M)]
    clusters = [c for c in clusters if c]
    log(f"[cluster] M={len(clusters)} sizes={[len(c) for c in clusters]}")
    cluster_of_pos = [0]*E
    for m, idx in enumerate(clusters):
        for pos in idx: cluster_of_pos[pos] = m

    # Init and train bases
    U_par, V_par = [], []
    for idx in clusters:
        Wm = Ws_norm[idx].mean(0)
        U0, V0 = svd_init_from_mean(Wm)
        U_par.append(OrthoParam(U0)); V_par.append(OrthoParam(V0))

    if cfg.TRAIN_STEPS > 0 and cfg.BASIS_MODE == "dense_train":
        params = [p.M for p in U_par] + [p.M for p in V_par]
        opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)
        guidance_masks, guidance_stats = {}, {}
        t0 = time.perf_counter()
        for step in range(1, cfg.TRAIN_STEPS+1):
            S = torch.randperm(n)[:cfg.SUBM].to(DEVICE)
            if cfg.TRAIN_LAM_GUIDE > 0 and (step==1 or step%cfg.TRAIN_GUIDE_EVERY==0):
                with torch.no_grad():
                    guidance_masks.clear(); guidance_stats.clear()
                    for m, idx in enumerate(clusters):
                        if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
                        Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                        pick = idx if cfg.BATCH_E>=len(idx) else [idx[i] for i in torch.randperm(len(idx))[:cfg.BATCH_E].tolist()]
                        Xs_ng = slice_X_batch(Ws_norm[pick], Uo, Vo, S).detach()
                        mask, ef, kblk = make_guidance_mask_from_Xs(Xs_ng, cfg.CORE_BLOCK, cfg.TRAIN_GUIDE_TARGET, cfg.TRAIN_GUIDE_MAX_BLOCKS)
                        guidance_masks[m] = mask; guidance_stats[m] = (ef, kblk)

            lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
            lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
            lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp
            L_total, n_terms = None, 0
            for m, idx in enumerate(clusters):
                if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
                Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                pick = idx if cfg.BATCH_E>=len(idx) else [idx[i] for i in torch.randperm(len(idx))[:cfg.BATCH_E].tolist()]
                Xs = slice_X_batch(Ws_norm[pick], Uo, Vo, S)
                off, diag = offdiag_abs_mean(Xs), diag_abs_mean(Xs).clamp_min(1e-6)
                base = torch.log(off+1e-6) - torch.log(diag) if cfg.TRAIN_OBJ=="logratio" else off/diag
                if lam_block > 0: base += lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)
                if lam_guide > 0 and m in guidance_masks:
                    Mmask = guidance_masks[m]
                    Etot = (Xs*Xs).mean().clamp_min(1e-12)
                    Eout = ((Xs*(1-Mmask))**2).mean()
                    base += lam_guide * (Eout/Etot)
                L_total = base if L_total is None else L_total + base
                n_terms += 1
            if L_total is None: break
            L_total = L_total / n_terms
            opt.zero_grad(); L_total.backward()
            if cfg.GRAD_CLIP > 0: torch.nn.utils.clip_grad_norm_(params, cfg.GRAD_CLIP)
            opt.step()
            if step % cfg.REORTHO_EVERY == 0 or step == cfg.TRAIN_STEPS:
                with torch.no_grad():
                    for p in U_par: p.M.copy_(p.orthogonal())
                    for p in V_par: p.M.copy_(p.orthogonal())
            if step % cfg.REPORT_EVERY == 0 or step == 1:
                t1 = time.perf_counter()
                gstr = "" if not guidance_stats else f" guide≈{np.mean([v[0] for v in guidance_stats.values()]):.3f}"
                log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={L_total.item():.4f} {gstr} (+{t1-t0:.1f}s)")
                t0 = t1

    # Freeze bases
    U_list = [p.orthogonal().detach() for p in U_par]
    V_list = [p.orthogonal().detach() for p in V_par]

    # Build payloads
    log("[build] payloads ...")
    core_all = [[] for _ in range(E)]
    res_all  = [[] for _ in range(E)]
    DL_list, DR_list = [], []
    rmax = min(cfg.RES_RANK, n)
    gam = torch.zeros((E, rmax), dtype=DTYPE_ACC, device=DEVICE) if cfg.RES_COEF=="diag" else None
    Cfull = torch.zeros((E, rmax, rmax), dtype=DTYPE_ACC, device=DEVICE) if cfg.RES_COEF=="full" else None

    for m, idx in enumerate(clusters):
        U, V = U_list[m], V_list[m]
        P = build_payload_for_cluster(Ws_norm, idx, U, V)
        for j, pos in enumerate(idx):
            core_all[pos] = P["core_blocks"][j]
            res_all[pos] = P["res_blocks"][j]
            if cfg.RES_COEF == "diag":
                g = P["coef_list"][j]; gam[pos, :g.numel()] = g
            else:
                C = P["coef_list"][j]; Cfull[pos, :C.shape[0], :C.shape[1]] = C
        DL_list.append(P["DL"]); DR_list.append(P["DR"])
        log(f"  cluster{m}: E={len(idx)} core_blocks≈{np.mean([len(c) for c in P['core_blocks']]):.1f} r={P['DL'].shape[1]}")

    # Save payload
    out_path = os.path.join(cfg.OUTPUT_DIR, f"ebc_payload_layer{cfg.LAYER}_E{E}_q{cfg.QMODE}.npz")
    store_dtype = np.float16 if cfg.BASIS_STORE_DTYPE=="float16" else np.float32
    arrays = {
        "meta": _encode_meta(ws_meta(expert_ids) | {"time": now(), "qmode": cfg.QMODE, "res_coef": cfg.RES_COEF}),
        "expert_ids": np.array(expert_ids, dtype=np.int32),
        "scales": Sc.cpu().numpy().astype(np.float32),
        "cluster_of_pos": np.array(cluster_of_pos, dtype=np.int16),
        "n_clusters": np.array([len(clusters)], dtype=np.int32),
    }
    for m in range(len(clusters)):
        arrays[f"U_{m}"] = U_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"V_{m}"] = V_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DL_{m}"] = DL_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DR_{m}"] = DR_list[m].cpu().numpy().astype(store_dtype)
    if cfg.RES_COEF == "diag":
        arrays["gam"] = gam.cpu().numpy().astype(store_dtype)
    else:
        arrays["Cfull"] = Cfull.cpu().numpy().astype(store_dtype)

    core_pack = pack_blocks_ragged(core_all, cfg.QMODE)
    res_pack  = pack_blocks_ragged(res_all, cfg.QMODE)
    for k, v in core_pack.items(): arrays["core_"+k] = v
    for k, v in res_pack.items(): arrays["res_"+k] = v

    save_npz_compressed(out_path, arrays)
    log(f"[save] payload -> {out_path} size={os.path.getsize(out_path)/1e6:.2f} MB")

    # Evaluate
    rt = load_payload_runtime(out_path, DEVICE)
    eval_payload(rt, Ws_norm, Sc)
    log("✅ Done.")

if __name__ == "__main__":
    main()

EBC-LLM Compression Pipeline
Time: 2026-04-21 10:52:07  Device: cpu
MODEL_DIR: /data/downloaded_models/Qwen1.5-MoE-A2.7B  OUTPUT_DIR: /home/daniyar/moe_ws_outputs_qwen
Layer: 0  Experts: 4
CALIB: (none)  ROUTER: (none)
Ridge damp: 0.001  Normalize W: True
Basis: dense_train  Train steps: 24  lr: 0.05
Core: blocktopk_perexpert block=64 target=0.85 max=256
Residual: rank=512 coef=diag blocks=4096 bsize=64
Refine: True target=0.03 max_extra=4096
[found] layer=0 total=60 using=4 eids=[0, 1, 2, 3]
[load] reading tensors from shards ...
[shape] H=2048 d_ff=1408
[capture] capturing via transformers...


Loading weights:   0%|          | 0/387 [00:00<?, ?it/s]

[capture] iter 4/32 nX=4096 nP=4096
[capture] wrote X -> /home/daniyar/moe_ws_outputs_qwen/calib_layer0_X.npz shape=(4096, 2048)
[capture] wrote P -> /home/daniyar/moe_ws_outputs_qwen/router_layer0_P.npz shape=(4096, 60)
[calib] X: torch.Size([4096, 2048])


Build Ws (ridge):   0%|          | 0/4 [00:00<?, ?it/s]

[cache] wrote Ws -> /home/daniyar/moe_ws_outputs_qwen/Ws_cache_layer0_E4_ridge_ebc.npz size=62.39 MB
[Ws] shape=torch.Size([4, 2048, 2048])
[cluster] M=1 sizes=[4]
[train] step   1/24 loss=-3.3904  guide≈0.822 (+1.3s)
[train] step   4/24 loss=-0.9503  guide≈0.806 (+3.7s)
[train] step   8/24 loss=-1.4895  guide≈0.829 (+4.4s)
[train] step  12/24 loss=-0.6307  guide≈0.816 (+4.4s)
[train] step  16/24 loss=-0.3998  guide≈0.835 (+4.4s)
[train] step  20/24 loss=2.0903  guide≈0.822 (+4.3s)
[train] step  24/24 loss=1.4222  guide≈0.827 (+4.4s)
[build] payloads ...
  cluster0: E=4 core_blocks≈44.2 r=512
[save] payload -> /home/daniyar/moe_ws_outputs_qwen/ebc_payload_layer0_E4_qnone.npz size=81.65 MB
[eval] per-expert rel-error mean=0.040283 p95=0.046255 max=0.046492
[eval] routed rel-error mean=0.049681 ± 0.018769
✅ Done.


In [14]:
#!/usr/bin/env python3
# =============================================================================
# EBC-LLM: Expert-Bank Compression (Phi-3.5-MoE compatible, CPU-only)
# =============================================================================

# #############################################################################
# FLASH_ATTN STUB – MUST BE EXECUTED BEFORE ANY TRANSFORMERS IMPORT
# #############################################################################
import sys
import types
import importlib.machinery

def _install_flash_attn_stub():
    """Create a complete fake flash_attn module hierarchy in memory."""
    # Root module
    flash_attn = types.ModuleType("flash_attn")
    flash_attn.__version__ = "0.0.0-cpu-stub"

    def _unavailable(*args, **kwargs):
        raise RuntimeError(
            "flash_attn stub called on CPU. Use attn_implementation='eager'."
        )

    flash_attn.flash_attn_func = _unavailable
    flash_attn.flash_attn_varlen_func = _unavailable
    flash_attn.flash_attn_with_kvcache = _unavailable

    # Submodules required by Phi-3.5-MoE
    flash_attn.layers = types.ModuleType("flash_attn.layers")
    flash_attn.layers.rotary = types.ModuleType("flash_attn.layers.rotary")
    flash_attn.ops = types.ModuleType("flash_attn.ops")
    flash_attn.ops.triton = types.ModuleType("flash_attn.ops.triton")
    flash_attn.bert_padding = types.ModuleType("flash_attn.bert_padding")
    flash_attn.flash_attn_interface = types.ModuleType("flash_attn.flash_attn_interface")

    # RotaryEmbedding stub (critical for Phi-3.5)
    import torch
    import torch.nn as nn
    class RotaryEmbedding(nn.Module):
        def __init__(self, dim, base=10000.0, interleaved=False, scale_base=None, device=None):
            super().__init__()
            self.dim = dim
        def forward(self, x, seq_len=None, **kwargs):
            device, dtype = x.device, x.dtype
            seq = seq_len if seq_len else x.shape[-2]
            half = max(1, self.dim // 2)
            cos = torch.ones((seq, half), device=device, dtype=dtype)
            sin = torch.zeros((seq, half), device=device, dtype=dtype)
            return cos, sin

    flash_attn.layers.rotary.RotaryEmbedding = RotaryEmbedding
    flash_attn.layers.rotary.apply_rotary_emb = lambda *a, **k: (_unavailable,)

    flash_attn.bert_padding.index_first_axis = lambda x, *a, **k: x
    flash_attn.bert_padding.pad_input = _unavailable
    flash_attn.bert_padding.unpad_input = _unavailable

    flash_attn.flash_attn_interface.flash_attn_func = _unavailable
    flash_attn.flash_attn_interface.flash_attn_varlen_func = _unavailable
    flash_attn.flash_attn_interface.flash_attn_with_kvcache = _unavailable

    # Register in sys.modules
    sys.modules["flash_attn"] = flash_attn
    sys.modules["flash_attn.layers"] = flash_attn.layers
    sys.modules["flash_attn.layers.rotary"] = flash_attn.layers.rotary
    sys.modules["flash_attn.ops"] = flash_attn.ops
    sys.modules["flash_attn.ops.triton"] = flash_attn.ops.triton
    sys.modules["flash_attn.bert_padding"] = flash_attn.bert_padding
    sys.modules["flash_attn.flash_attn_interface"] = flash_attn.flash_attn_interface

class FlashAttnImporter:
    def find_spec(self, fullname, path, target=None):
        if fullname == "flash_attn" or fullname.startswith("flash_attn."):
            if "flash_attn" not in sys.modules:
                _install_flash_attn_stub()
            return importlib.machinery.ModuleSpec(fullname, self)
        return None
    def create_module(self, spec): return sys.modules.get(spec.name)
    def exec_module(self, module): pass

sys.meta_path.insert(0, FlashAttnImporter())
print("✅ flash_attn stub installed (CPU mode).", flush=True)

# #############################################################################
# END OF FLASH_ATTN STUB
# #############################################################################

# =============================================================================
# EBC-LLM Compression Pipeline
# =============================================================================

import os, re, json, math, time, random, sys
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs): return x

# -----------------------------------------------------------------------------
# Environment helpers
# -----------------------------------------------------------------------------
def _env_str(k: str, d: str) -> str:
    return os.environ.get(k, d)

def _env_int(k: str, d: int) -> int:
    try: return int(os.environ.get(k, str(d)))
    except: return d

def _env_float(k: str, d: float) -> float:
    try: return float(os.environ.get(k, str(d)))
    except: return d

def _env_bool(k: str, d: bool) -> bool:
    v = os.environ.get(k, None)
    if v is None: return d
    return v.strip().lower() in ("1", "true", "yes", "y", "on")

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
@dataclass
class Cfg:
    # Paths
    MODEL_DIR: str = "/data/downloaded_models/Phi-3.5-MoE-instruct"
    OUTPUT_DIR: str = "/home/daniyar/moe_ws_outputs_phi"

    # Model slice
    LAYER: int = 0
    MAX_EXPERTS: int = 16

    # Calibration / router
    CALIB_PATH: str = _env_str("CALIB_PATH", "").strip()
    ROUTER_PATH: str = _env_str("ROUTER_PATH", "").strip()
    CALIB_SAMPLES: int = _env_int("CALIB_SAMPLES", 4096)
    RIDGE_WEIGHTED: bool = _env_bool("RIDGE_WEIGHTED", False)
    ROUTER_EIDS_ARE_GLOBAL: bool = _env_bool("ROUTER_EIDS_ARE_GLOBAL", True)
    RIDGE_DAMP: float = _env_float("RIDGE_DAMP", 1e-3)
    NORMALIZE_W: bool = _env_bool("NORMALIZE_W", True)

    # Capture (optional)
    CAPTURE_ENABLE: bool = True
    CAPTURE_FORCE: bool = _env_bool("CAPTURE_FORCE", False)
    CAPTURE_ITERS: int = _env_int("CAPTURE_ITERS", 32)
    CAPTURE_BATCH: int = _env_int("CAPTURE_BATCH", 1)
    CAPTURE_MAX_TOKENS: int = _env_int("CAPTURE_MAX_TOKENS", 1024)
    CAPTURE_TEXT: str = _env_str("CAPTURE_TEXT", "DeepSeek MoE calibration text. " * 256)
    CAPTURE_TEXT_FILE: str = _env_str("CAPTURE_TEXT_FILE", "").strip()
    CAPTURE_KEEP_PAD: bool = _env_bool("CAPTURE_KEEP_PAD", False)
    HF_TRUST_REMOTE_CODE: bool = _env_bool("HF_TRUST_REMOTE_CODE", True)
    HF_LOCAL_FILES_ONLY: bool = _env_bool("HF_LOCAL_FILES_ONLY", True)
    HF_AUTO_PIP: bool = _env_bool("HF_AUTO_PIP", False)

    # Basis mode
    BASIS_MODE: str = _env_str("BASIS_MODE", "dense_train").lower()
    BASIS_STORE_DTYPE: str = _env_str("BASIS_STORE_DTYPE", "float16").lower()

    # Clustering
    M0: int = _env_int("M0", 0)                # 0 = auto
    M_MAX: int = _env_int("M_MAX", 16)
    CLUSTER_FEAT_D: int = _env_int("CLUSTER_FEAT_D", 64)
    CLUSTER_ITERS: int = _env_int("CLUSTER_ITERS", 60)
    CLUSTER_RESTARTS: int = _env_int("CLUSTER_RESTARTS", 4)
    CLUSTER_MIN_SIZE: int = _env_int("CLUSTER_MIN_SIZE", 2)
    CLUSTER_MAX_SIZE: int = _env_int("CLUSTER_MAX_SIZE", 4)
    SPLIT_ITERS: int = _env_int("SPLIT_ITERS", 50)

    # Training (dense bases)
    TRAIN_STEPS: int = _env_int("TRAIN_STEPS", 24)
    TRAIN_WARMUP: int = _env_int("TRAIN_WARMUP", 6)
    TRAIN_LR: float = _env_float("TRAIN_LR", 5e-2)
    SUBM: int = _env_int("SUBM", 256)
    BATCH_E: int = _env_int("BATCH_E", 4)
    TRAIN_MIN_CLUSTER: int = _env_int("TRAIN_MIN_CLUSTER", 2)
    REORTHO_EVERY: int = _env_int("REORTHO_EVERY", 4)
    REPORT_EVERY: int = _env_int("REPORT_EVERY", 4)
    GRAD_CLIP: float = _env_float("GRAD_CLIP", 1.0)
    TRAIN_OBJ: str = _env_str("TRAIN_OBJ", "logratio").lower()
    TRAIN_LAM_BLOCK: float = _env_float("TRAIN_LAM_BLOCK", 0.10)
    TRAIN_LAM_GUIDE: float = _env_float("TRAIN_LAM_GUIDE", 1.0)
    TRAIN_GUIDE_EVERY: int = _env_int("TRAIN_GUIDE_EVERY", 2)
    TRAIN_GUIDE_TARGET: float = _env_float("TRAIN_GUIDE_TARGET", 0.80)
    TRAIN_GUIDE_MAX_BLOCKS: int = _env_int("TRAIN_GUIDE_MAX_BLOCKS", 2048)

    # Core selection
    CORE_MODE: str = _env_str("CORE_MODE", "blocktopk_perexpert").lower()
    CORE_AGG: str = _env_str("CORE_AGG", "mean").lower()
    CORE_BLOCK: int = _env_int("CORE_BLOCK", 64)
    CORE_TARGET: float = _env_float("CORE_TARGET", 0.85)
    CORE_MAX_BLOCKS: int = _env_int("CORE_MAX_BLOCKS", 256)

    # Residual
    RES_RANK: int = _env_int("RES_RANK", 512)
    RES_COEF: str = _env_str("RES_COEF", "diag").lower()
    RES_TARGET: float = _env_float("RES_TARGET", 0.995)
    RES_MAX_BLOCKS: int = _env_int("RES_MAX_BLOCKS", 4096)
    RES_BSIZE: int = _env_int("RES_BSIZE", 64)

    # Refine
    REFINE_ENABLE: bool = _env_bool("REFINE_ENABLE", True)
    REFINE_ERR_TARGET: float = _env_float("REFINE_ERR_TARGET", 0.03)
    REFINE_MAX_EXTRA: int = _env_int("REFINE_MAX_EXTRA", 4096)
    REFINE_BSIZE: int = _env_int("REFINE_BSIZE", 64)
    REFINE_RECHECK_EVERY: int = _env_int("REFINE_RECHECK_EVERY", 32)

    # Quantization
    QMODE: str = _env_str("QMODE", "none").lower()

    # Eval
    EVAL_TRIALS: int = _env_int("EVAL_TRIALS", 8)
    EVAL_BATCH: int = _env_int("EVAL_BATCH", 2)
    ROUTED_K: int = _env_int("ROUTED_K", 8)

cfg = Cfg()
PRESET = _env_str("PRESET", "").strip().lower()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def _setdefault_env(k: str, v: str):
    if k not in os.environ: os.environ[k] = v

if PRESET == "maxacc":
    _setdefault_env("CALIB_SAMPLES", "32768")
    _setdefault_env("RIDGE_DAMP", "1e-2")
    _setdefault_env("CORE_BLOCK", "32")
    _setdefault_env("CORE_TARGET", "0.995")
    _setdefault_env("CORE_MAX_BLOCKS", "8192")
    _setdefault_env("RES_RANK", "2048")
    _setdefault_env("RES_COEF", "full")
    _setdefault_env("RES_TARGET", "0.999")
    _setdefault_env("RES_MAX_BLOCKS", "32768")
    _setdefault_env("REFINE_ENABLE", "1")
    _setdefault_env("REFINE_ERR_TARGET", "0.01")
    _setdefault_env("REFINE_MAX_EXTRA", "65536")
    _setdefault_env("TRAIN_STEPS", "96")
    _setdefault_env("TRAIN_LR", "0.02")
    _setdefault_env("TRAIN_LAM_GUIDE", "0.5")
    cfg = Cfg()
elif PRESET == "compact":
    _setdefault_env("CALIB_SAMPLES", "4096")
    _setdefault_env("CORE_BLOCK", "64")
    _setdefault_env("CORE_TARGET", "0.90")
    _setdefault_env("CORE_MAX_BLOCKS", "512")
    _setdefault_env("RES_RANK", "512")
    _setdefault_env("RES_COEF", "diag")
    _setdefault_env("RES_TARGET", "0.99")
    _setdefault_env("RES_MAX_BLOCKS", "4096")
    _setdefault_env("QMODE", "float16")
    _setdefault_env("REFINE_ENABLE", "0")
    _setdefault_env("TRAIN_STEPS", "24")
    cfg = Cfg()

# -----------------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------------
def log(msg: str): print(msg, flush=True)
def now() -> str: return time.strftime("%Y-%m-%d %H:%M:%S")

def seed_all(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

SEED = _env_int("SEED", 1234)
seed_all(SEED)
NTHREADS = _env_int("KTXX_THREADS", 8)
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
try: torch.set_num_threads(NTHREADS)
except: pass

DEVICE = torch.device(_env_str("DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
DTYPE_ACC = torch.float32

# -----------------------------------------------------------------------------
# NPZ I/O
# -----------------------------------------------------------------------------
def save_npz_compressed(path: str, arrays: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, **arrays)

def load_npz(path: str) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    return {k: z[k] for k in z.files}

def _encode_meta(meta: dict) -> np.ndarray:
    return np.frombuffer(json.dumps(meta, sort_keys=True).encode("utf-8"), dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try: return json.loads(bytes(arr.tolist()).decode("utf-8"))
    except: return {}

# -----------------------------------------------------------------------------
# Offline shard loading
# -----------------------------------------------------------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r") as f:
        return json.load(f).get("weight_map", {})

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    patterns = [
        rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.",
        rf"^model\.layers\.{layer}\.block_sparse_moe\.experts\.(\d+)\.",
    ]
    ids = set()
    for pat_str in patterns:
        pat = re.compile(pat_str)
        for k in weight_map:
            m = pat.match(k)
            if m:
                ids.add(int(m.group(1)))
        if ids:
            break
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefixes = [
        f"model.layers.{layer}.mlp.experts.{eid}.",
        f"model.layers.{layer}.block_sparse_moe.experts.{eid}.",
    ]
    used_prefix = None
    for pfx in prefixes:
        if any(k.startswith(pfx) for k in weight_map):
            used_prefix = pfx
            break
    if used_prefix is None:
        return {}

    def pick(cands):
        for suf in cands:
            k = used_prefix + suf
            if k in weight_map:
                return k
        return None

    gate = pick(["w1.weight", "gate_proj.weight"])
    down = pick(["w2.weight", "down_proj.weight"])
    up   = pick(["w3.weight", "up_proj.weight"])

    if gate is None or down is None or up is None:
        return {}
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard = {}
    for k in keys:
        shard = weight_map.get(k)
        if shard is None: continue
        by_shard.setdefault(shard, []).append(k)
    out = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp): continue
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks: out[k] = f.get_tensor(k)
    return out

# -----------------------------------------------------------------------------
# Calibration / Router
# -----------------------------------------------------------------------------
def autodetect_calib_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return cand if os.path.isfile(cand) else None

def autodetect_router_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return cand if os.path.isfile(cand) else None

def load_calib_X(path: str, H: int) -> torch.Tensor:
    z = np.load(path)
    X = torch.from_numpy(z["X"].astype(np.float32))
    if X.ndim != 2 or X.shape[1] != H: raise RuntimeError(f"Bad X shape {X.shape}")
    if X.shape[0] > cfg.CALIB_SAMPLES: X = X[:cfg.CALIB_SAMPLES]
    return X.to(device=DEVICE, dtype=DTYPE_ACC)

def load_router_P(path: str) -> np.ndarray:
    return np.load(path)["P"].astype(np.float32)

def _maybe_autopip():
    if not cfg.HF_AUTO_PIP: return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", "transformers", "sentencepiece", "tokenizers"])

def _patch_transformers_cache_compat():
    try:
        from transformers.cache_utils import DynamicCache
        if not hasattr(DynamicCache, "get_usable_length"):
            DynamicCache.get_usable_length = lambda self, seq_length: int(seq_length)
    except: pass

class _Collector:
    def __init__(self, H, E_total, max_rows):
        self.H = H; self.E_total = E_total; self.max_rows = max_rows
        self.X_chunks, self.P_chunks = [], []; self.nX = self.nP = 0

    def _take(self, flat, need): return flat[:need] if flat.shape[0] > need else flat

    def add_X(self, hs, attn_mask):
        if hs is None: return
        if hs.ndim == 2: hs = hs.unsqueeze(0)
        if hs.ndim != 3 or hs.shape[-1] != self.H: return
        hs = hs.detach().to(torch.float32).cpu()
        if attn_mask is not None and not cfg.CAPTURE_KEEP_PAD:
            m = attn_mask.cpu().to(torch.bool); flat = hs.reshape(-1, self.H)[m.reshape(-1)]
        else: flat = hs.reshape(-1, self.H)
        if flat.numel() == 0: return
        need = self.max_rows - self.nX
        if need <= 0: return
        self.X_chunks.append(self._take(flat, need)); self.nX += self.X_chunks[-1].shape[0]

    def add_logits(self, logits, attn_mask):
        if logits is None: return
        if logits.ndim == 2: logits = logits.unsqueeze(0)
        if logits.ndim != 3: return
        P = torch.softmax(logits.detach().to(torch.float32), dim=-1)[..., :self.E_total].cpu()
        if attn_mask is not None and not cfg.CAPTURE_KEEP_PAD:
            m = attn_mask.cpu().to(torch.bool); flat = P.reshape(-1, P.shape[-1])[m.reshape(-1)]
        else: flat = P.reshape(-1, P.shape[-1])
        if flat.numel() == 0: return
        need = self.max_rows - self.nP
        if need <= 0: return
        self.P_chunks.append(self._take(flat, need)); self.nP += self.P_chunks[-1].shape[0]

# -----------------------------------------------------------------------------
# Transformer‑based capture WITH is_torch_fx_available patch
# -----------------------------------------------------------------------------
def capture_XP_transformers(model_dir, layer_idx, H, E_total, out_x, out_p):
    _maybe_autopip()
    _patch_transformers_cache_compat()

    # 1) Patch missing is_torch_fx_available for older transformers
    import transformers.utils.import_utils as iu
    if not hasattr(iu, "is_torch_fx_available"):
        def is_torch_fx_available():
            try:
                import torch.fx
                return True
            except ImportError:
                return False
        iu.is_torch_fx_available = is_torch_fx_available

    # 2) Load model normally (stub already active)
    from transformers import AutoTokenizer, AutoModelForCausalLM

    tok = AutoTokenizer.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token or tok.unk_token

    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
        attn_implementation="eager"
    ).to(DEVICE).eval()

    # 3) Locate the layer and the MLP module (robust search)
    layers = None
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        layers = model.model.layers
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        layers = model.transformer.h
    elif hasattr(model, "layers"):
        layers = model.layers
    if layers is None:
        raise RuntimeError("Cannot locate layers")

    if layer_idx >= len(layers):
        raise RuntimeError(f"Layer {layer_idx} out of range")
    layer = layers[layer_idx]

    mlp = None
    for attr in ["mlp", "moe", "block_sparse_moe"]:
        mlp = getattr(layer, attr, None)
        if mlp is not None:
            break
    if mlp is None:
        for name, mod in layer.named_modules():
            if any(x in name.lower() for x in ["mlp", "moe", "expert"]):
                if hasattr(mod, "gate_proj") or hasattr(mod, "w1"):
                    mlp = mod
                    break
    if mlp is None:
        for name, mod in layer.named_modules():
            if "expert" in name.lower():
                mlp = mod
                break
    if mlp is None:
        raise RuntimeError("Could not find MoE MLP module in layer.")

    log(f"[capture] Located MLP module: {mlp.__class__.__name__}")

    # 4) Router discovery
    router_linear = None
    for name, mod in layer.named_modules():
        if isinstance(mod, nn.Linear) and mod.in_features == H and mod.out_features >= E_total:
            if "router" in name.lower() or "gate" in name.lower():
                router_linear = mod
                break

    coll = _Collector(H, E_total, cfg.CALIB_SAMPLES)
    attn_holder = {"mask": None}

    def mlp_pre_hook(_, inputs):
        coll.add_X(inputs[0], attn_holder["mask"])

    h1 = mlp.register_forward_pre_hook(mlp_pre_hook)
    h2 = None
    if router_linear is not None:
        def router_hook(_, __, out):
            o = out[0] if isinstance(out, (tuple, list)) else out
            coll.add_logits(o, attn_holder["mask"])
        h2 = router_linear.register_forward_hook(router_hook)

    texts = [cfg.CAPTURE_TEXT]
    if cfg.CAPTURE_TEXT_FILE and os.path.isfile(cfg.CAPTURE_TEXT_FILE):
        with open(cfg.CAPTURE_TEXT_FILE) as f:
            texts = [ln.strip() for ln in f if ln.strip()]
    tptr = 0
    for it in range(cfg.CAPTURE_ITERS):
        text = texts[tptr % len(texts)]
        tptr += 1
        enc = tok(text, return_tensors="pt", truncation=True,
                  max_length=cfg.CAPTURE_MAX_TOKENS, padding="max_length")
        for k in enc:
            if enc[k].ndim == 2 and cfg.CAPTURE_BATCH > 1:
                enc[k] = enc[k].repeat(cfg.CAPTURE_BATCH, 1)
        attn_holder["mask"] = enc.get("attention_mask")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.inference_mode():
            _ = model(**enc, use_cache=False)
        if (it + 1) % 4 == 0:
            log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS} nX={coll.nX} nP={coll.nP}")
        if coll.nX >= cfg.CALIB_SAMPLES and (not cfg.RIDGE_WEIGHTED or coll.nP >= cfg.CALIB_SAMPLES):
            break

    h1.remove()
    if h2:
        h2.remove()

    if coll.nX == 0:
        raise RuntimeError("Capture collected 0 rows")
    X = torch.cat(coll.X_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz_compressed(out_x, {"X": X})
    log(f"[capture] wrote X -> {out_x} shape={X.shape}")
    p_written = None
    if coll.nP > 0:
        P = torch.cat(coll.P_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
        N = min(P.shape[0], X.shape[0])
        if N < X.shape[0]:
            X = X[:N]
            save_npz_compressed(out_x, {"X": X})
        P = P[:N]
        save_npz_compressed(out_p, {"P": P})
        log(f"[capture] wrote P -> {out_p} shape={P.shape}")
        p_written = out_p
    return out_x, p_written

def ensure_calib_router(H: int, E_total: int):
    # If calibration files already exist, skip expensive model loading
    calib_cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    router_cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    if os.path.isfile(calib_cand) and (not cfg.RIDGE_WEIGHTED or os.path.isfile(router_cand)):
        log("[capture] Calibration cache found. Skipping transformers model loading.")
        cfg.CALIB_PATH = calib_cand
        if os.path.isfile(router_cand):
            cfg.ROUTER_PATH = router_cand
        return

    if not cfg.CALIB_PATH:
        c = autodetect_calib_path()
        if c:
            cfg.CALIB_PATH = c
            log(f"[calib] auto-found {cfg.CALIB_PATH}")
    if not cfg.ROUTER_PATH:
        r = autodetect_router_path()
        if r:
            cfg.ROUTER_PATH = r
            log(f"[router] auto-found {cfg.ROUTER_PATH}")
    if cfg.CAPTURE_FORCE or (cfg.CAPTURE_ENABLE and (not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH))):
        out_x = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
        out_p = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
        log("[capture] capturing via transformers... (this will take several minutes on CPU)")
        x_path, p_path = capture_XP_transformers(cfg.MODEL_DIR, cfg.LAYER, H, E_total, out_x, out_p)
        cfg.CALIB_PATH = x_path
        if p_path:
            cfg.ROUTER_PATH = p_path

# -----------------------------------------------------------------------------
# Ridge linearization: build Ws
# -----------------------------------------------------------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate, W_up, W_down) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    return hid @ W_down.to(DTYPE_ACC).t()

def ws_cache_path(E: int) -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{E}_ridge_ebc.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        script="ebc_llm", model_dir=cfg.MODEL_DIR, layer=cfg.LAYER, expert_ids=eids,
        ridge_damp=cfg.RIDGE_DAMP, ridge_weighted=cfg.RIDGE_WEIGHTED,
        router_path=cfg.ROUTER_PATH or "", calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES, normalize_w=cfg.NORMALIZE_W, seed=SEED, device=str(DEVICE)
    )

@torch.no_grad()
def build_Ws(eids: List[int], wm: Dict[str, str]) -> Tuple[torch.Tensor, torch.Tensor]:
    per_e, need_keys = {}, []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk: raise RuntimeError(f"Expert {eid} missing tensors")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]
    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))
    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = W_up0.shape[0], W_up0.shape[1]
    log(f"[shape] H={H} d_ff={dff}")

    ensure_calib_router(H, len(find_layer_expert_ids(wm, cfg.LAYER)))
    if not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH):
        raise RuntimeError("CALIB_PATH missing. Set CALIB_PATH or CAPTURE_ENABLE=1.")
    X = load_calib_X(cfg.CALIB_PATH, H)
    log(f"[calib] X: {X.shape}")

    P = None
    if cfg.RIDGE_WEIGHTED:
        if cfg.ROUTER_PATH and os.path.isfile(cfg.ROUTER_PATH):
            P = load_router_P(cfg.ROUTER_PATH)
            log(f"[router] P: {P.shape}")
        else:
            log("[router] RIDGE_WEIGHTED=1 but ROUTER_PATH missing -> disabling.")
            cfg.RIDGE_WEIGHTED = False

    Xf = X.to(DTYPE_ACC)
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * torch.trace(XtX).item() / H
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list, scales = [], []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws (ridge)")):
        W_up = T[per_e[eid]["up"]].to(DEVICE)
        W_dn = T[per_e[eid]["down"]].to(DEVICE)
        W_gt = T[per_e[eid]["gate"]].to(DEVICE)
        Y = forward_mlp(X, W_gt, W_up, W_dn).to(DTYPE_ACC)

        if cfg.RIDGE_WEIGHTED and P is not None:
            w = torch.from_numpy(P[:X.shape[0], eid if cfg.ROUTER_EIDS_ARE_GLOBAL else i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0)
            sw = torch.sqrt(w + 1e-12).view(-1, 1)
            Xw, Yw = Xf * sw, Y * sw
            XtX_e = Xw.t() @ Xw
            lam_e = cfg.RIDGE_DAMP * torch.trace(XtX_e).item() / H
            chol = torch.linalg.cholesky(XtX_e + lam_e * I)
            Wt = torch.cholesky_solve(Xw.t() @ Yw, chol)
            W = Wt.t().contiguous()
        else:
            Wt = torch.cholesky_solve(Xf.t() @ Y, cholG)
            W = Wt.t().contiguous()

        if cfg.NORMALIZE_W:
            s = torch.linalg.norm(W, ord="fro").clamp_min(1e-12).item()
            W = W / s
        else:
            s = 1.0
        Ws_list.append(W)
        scales.append(s)

    Ws = torch.stack(Ws_list).to(DTYPE_ACC).to(DEVICE)
    Sc = torch.tensor(scales, dtype=DTYPE_ACC, device=DEVICE)
    return Ws, Sc

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor, torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts at layer {cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer={cfg.LAYER} total={len(all_eids)} using={len(eids)} eids={eids}")

    if not cfg.CALIB_PATH:
        cfg.CALIB_PATH = autodetect_calib_path() or ""
    if not cfg.ROUTER_PATH:
        cfg.ROUTER_PATH = autodetect_router_path() or ""

    cpath = ws_cache_path(len(eids))
    if os.path.isfile(cpath):
        z = load_npz(cpath)
        if all(k in z for k in ["meta", "Ws", "expert_ids", "scales"]) and _decode_meta(z["meta"]) == ws_meta(eids):
            Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
            Sc = torch.from_numpy(z["scales"]).to(DTYPE_ACC).to(DEVICE)
            log(f"[cache] loaded Ws -> {cpath} shape={Ws.shape}")
            return [int(x) for x in z["expert_ids"]], Ws, Sc
        log("[cache] meta mismatch -> rebuild")

    Ws, Sc = build_Ws(eids, wm)
    save_npz_compressed(cpath, {
        "meta": _encode_meta(ws_meta(eids)),
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.cpu().numpy().astype(np.float32),
        "scales": Sc.cpu().numpy().astype(np.float32)
    })
    log(f"[cache] wrote Ws -> {cpath} size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws, Sc

# -----------------------------------------------------------------------------
# Clustering (kmeans++ + hierarchical split)
# -----------------------------------------------------------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED + 17)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([row @ R, col @ R]).unsqueeze(0))
    X = torch.cat(feats, dim=0)
    X = (X - X.mean(0, keepdim=True)) / (X.std(0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> torch.Tensor:
    best_lab, best_inertia = None, float("inf")
    g = torch.Generator(device="cpu").manual_seed(SEED + 999)
    for _ in range(max(1, restarts)):
        n = X.shape[0]
        centers = [X[torch.randint(0, n, (1,), generator=g).item()].clone()]
        for _ in range(1, k):
            C = torch.stack(centers)
            dist2 = torch.cdist(X, C).pow(2).min(1).values
            prob = dist2 / dist2.sum().clamp_min(1e-12)
            centers.append(X[torch.multinomial(prob, 1, generator=g).item()].clone())
        C = torch.stack(centers)
        for _ in range(iters):
            dist = torch.cdist(X, C)
            lab = dist.argmin(1)
            for j in range(k):
                m = (lab == j)
                if m.any():
                    C[j] = X[m].mean(0)
                else:
                    C[j] = X[dist.min(1).values.argmax().item()].clone()
        inertia = torch.cdist(X, C).min(1).values.pow(2).sum().item()
        if inertia < best_inertia:
            best_inertia, best_lab = inertia, lab.clone()
    return best_lab.to(torch.int64)

@torch.no_grad()
def relabel_contiguous(labels: torch.Tensor) -> torch.Tensor:
    uniq = torch.unique(labels)
    out = labels.clone()
    for new, old in enumerate(uniq.tolist()):
        out[labels == old] = new
    return out

@torch.no_grad()
def merge_small_clusters(X: torch.Tensor, labels: torch.Tensor, min_size: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if min_size <= 1:
        return labels
    while True:
        K = labels.max().item() + 1
        counts = torch.bincount(labels, minlength=K)
        small = (counts < min_size).nonzero(as_tuple=False).flatten()
        if small.numel() == 0:
            break
        C = torch.stack([X[labels == k].mean(0) for k in range(K)])
        for c in small.tolist():
            idxs = (labels == c).nonzero(as_tuple=False).flatten()
            if idxs.numel() == 0:
                continue
            dist = torch.cdist(C[c].unsqueeze(0), C).squeeze(0)
            dist[c] = 1e9
            labels[idxs] = dist.argmin().item()
        labels = relabel_contiguous(labels)
    return labels

@torch.no_grad()
def hierarchical_split(X: torch.Tensor, labels: torch.Tensor, max_size: int, max_k: int, split_iters: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if max_size <= 0:
        return labels
    while True:
        K = labels.max().item() + 1
        if K >= max_k:
            break
        counts = torch.bincount(labels, minlength=K)
        biggest = counts.argmax().item()
        if counts[biggest] <= max_size:
            break
        idxs = (labels == biggest).nonzero(as_tuple=False).flatten()
        if idxs.numel() < 2:
            break
        sub = X[idxs]
        sub_lab = kmeans_torch(sub, 2, split_iters, 1)
        a, b = idxs[sub_lab == 0], idxs[sub_lab == 1]
        if a.numel() == 0 or b.numel() == 0:
            break
        labels[b] = K
        labels = relabel_contiguous(labels)
    return labels

# -----------------------------------------------------------------------------
# Basis training (dense)
# -----------------------------------------------------------------------------
class OrthoParam(nn.Module):
    def __init__(self, init_mat: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(init_mat.to(DEVICE, DTYPE_ACC).contiguous())
    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

@torch.no_grad()
def svd_init_from_mean(Wmean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(Wmean, full_matrices=False)
    return U.to(DTYPE_ACC).contiguous(), Vh.t().to(DTYPE_ACC).contiguous()

def schedule(step: int, warmup: int, total: int) -> float:
    if step <= warmup:
        return 0.0
    return min(1.0, (step - warmup) / max(1, total - warmup))

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    U_S, V_S = U[:, S], V[:, S]
    return torch.matmul(U_S.t().unsqueeze(0), Ws_batch @ V_S)

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return (Xs - torch.diag_embed(D)).abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    return torch.diagonal(Xs, dim1=1, dim2=2).abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.zeros((), device=Xs.device)
    nb = s // b
    if nb <= 0:
        return torch.zeros((), device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eblk = (Xb * Xb).sum(dim=(3, 4))
    P = Eblk.mean(0)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

@torch.no_grad()
def make_guidance_mask_from_Xs(Xs: torch.Tensor, block: int, target: float, max_blocks: int) -> Tuple[torch.Tensor, float, int]:
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.ones(s, s, device=Xs.device), 1.0, 0
    nb = s // b
    if nb <= 0:
        return torch.ones(s, s, device=Xs.device), 1.0, 0
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eg = (Xb * Xb).sum(dim=(3, 4)).mean(0)
    tot = (X * X).sum().item() / max(1, Eb)
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], 0)
    frac = csum / max(tot, 1e-12)
    need = (frac >= target).nonzero(as_tuple=False)[0].item() + 1 if (frac >= target).any() else flat.numel()
    K = min(need, max_blocks, flat.numel())
    mask = torch.zeros(s2, s2, device=Xs.device)
    for idx in order[:K].tolist():
        bi, bj = idx // nb, idx % nb
        mask[bi*b:(bi+1)*b, bj*b:(bj+1)*b] = 1.0
    if s2 < s:
        full = torch.zeros(s, s, device=Xs.device)
        full[:s2, :s2] = mask
        mask = full
    ef = float(frac[K-1].item()) if K > 0 else 0.0
    return mask, ef, K

# -----------------------------------------------------------------------------
# Block energy & selection
# -----------------------------------------------------------------------------
@torch.no_grad()
def block_energy_grid(X: torch.Tensor, b: int) -> Tuple[torch.Tensor, float, int]:
    n = X.shape[0]
    nb = (n + b - 1) // b
    if n % b != 0:
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()
    Eg = (Xb * Xb).sum(dim=(2, 3))
    tot = (X * X).sum().item()
    return Eg, tot, nb

@torch.no_grad()
def pick_blocks_until_target(Eg: torch.Tensor, tot_energy: float, target: float, max_blocks: int,
                             exclude: Optional[Set[Tuple[int,int]]]=None) -> Tuple[List[Tuple[int,int]], float]:
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    picked, eacc = [], 0.0
    exclude = exclude or set()
    for idx in order.tolist():
        if len(picked) >= max_blocks:
            break
        e = flat[idx].item()
        if e <= 1e-18:
            break
        bi, bj = idx // nb, idx % nb
        if (bi, bj) in exclude:
            continue
        picked.append((bi, bj))
        eacc += e
        if eacc / max(tot_energy, 1e-12) >= target:
            break
    return picked, eacc / max(tot_energy, 1e-12)

@torch.no_grad()
def gather_block(X: torch.Tensor, i0: int, j0: int, b: int) -> torch.Tensor:
    n = X.shape[0]
    i1, j1 = min(n, i0+b), min(n, j0+b)
    return X[i0:i1, j0:j1].contiguous()

# -----------------------------------------------------------------------------
# Low-rank (randomized SVD)
# -----------------------------------------------------------------------------
@torch.no_grad()
def rand_svd_vectors(A: torch.Tensor, r: int, n_iter: int=2) -> Tuple[torch.Tensor, torch.Tensor]:
    n = A.shape[0]
    r = min(r, n)
    g = torch.Generator(device="cpu").manual_seed(SEED+777)
    Omega = torch.randn(n, r, generator=g, dtype=DTYPE_ACC, device=A.device)
    Y = A @ Omega
    for _ in range(n_iter):
        Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    Uhat, _, Vh = torch.linalg.svd(B, full_matrices=False)
    return (Q @ Uhat[:, :r]).contiguous(), Vh.t()[:, :r].contiguous()

# -----------------------------------------------------------------------------
# Payload packing (ragged blocks)
# -----------------------------------------------------------------------------
def _block_store_dtype(qmode: str) -> np.dtype:
    return np.float32 if qmode == "none" else np.float16

def pack_blocks_ragged(blocks_per_item: List[List[Tuple[int,int,torch.Tensor]]], qmode: str) -> Dict[str, np.ndarray]:
    val_dtype = _block_store_dtype(qmode)
    M = len(blocks_per_item)
    item_ptr = [0]
    blk_i0, blk_j0, blk_h, blk_w = [], [], [], []
    blk_ptr = [0]
    vals, vals_i8, scales = [], [], []
    for m in range(M):
        for (i0, j0, B) in blocks_per_item[m]:
            h, w = B.shape
            blk_i0.append(i0)
            blk_j0.append(j0)
            blk_h.append(h)
            blk_w.append(w)
            if qmode == "int8":
                x = B.cpu().float()
                maxabs = x.abs().max().item()
                if maxabs < 1e-12:
                    q = np.zeros(x.numel(), dtype=np.int8)
                    sc = np.float16(1.0)
                else:
                    scale = maxabs / 127.0
                    q = torch.clamp(torch.round(x/scale), -127, 127).to(torch.int8).numpy()
                    sc = np.float16(scale)
                vals_i8.append(q.reshape(-1))
                scales.append(sc)
                blk_ptr.append(blk_ptr[-1] + q.size)
            else:
                v = B.cpu().float().numpy().astype(val_dtype).reshape(-1)
                vals.append(v)
                blk_ptr.append(blk_ptr[-1] + v.size)
        item_ptr.append(len(blk_i0))

    out = {
        "item_ptr": np.array(item_ptr, dtype=np.int32),
        "blk_i0": np.array(blk_i0, dtype=np.int16),
        "blk_j0": np.array(blk_j0, dtype=np.int16),
        "blk_h": np.array(blk_h, dtype=np.int16),
        "blk_w": np.array(blk_w, dtype=np.int16),
        "blk_ptr": np.array(blk_ptr, dtype=np.int64)
    }
    if qmode == "int8":
        out["blk_q"] = np.concatenate(vals_i8).astype(np.int8) if vals_i8 else np.zeros((0,), dtype=np.int8)
        out["blk_scale"] = np.array(scales, dtype=np.float16)
    else:
        out["blk_val"] = np.concatenate(vals) if vals else np.zeros((0,), dtype=val_dtype)
    return out

def unpack_blocks_ragged(pack: Dict[str, np.ndarray], qmode: str, device: torch.device) -> List[List[Tuple[int,int,torch.Tensor]]]:
    item_ptr = pack["item_ptr"]
    blk_i0 = pack["blk_i0"]
    blk_j0 = pack["blk_j0"]
    blk_h = pack["blk_h"]
    blk_w = pack["blk_w"]
    blk_ptr = pack["blk_ptr"]
    if qmode == "int8":
        blk_q = pack["blk_q"]
        blk_scale = pack["blk_scale"]
        blk_val = None
    else:
        blk_val = pack["blk_val"]
        blk_q = None
        blk_scale = None
    M = item_ptr.shape[0] - 1
    out = []
    for m in range(M):
        b0, b1 = item_ptr[m], item_ptr[m+1]
        lst = []
        for bi in range(b0, b1):
            i0, j0 = int(blk_i0[bi]), int(blk_j0[bi])
            h, w = int(blk_h[bi]), int(blk_w[bi])
            v0, v1 = blk_ptr[bi], blk_ptr[bi+1]
            if qmode == "int8":
                q = blk_q[v0:v1].astype(np.float32)
                sc = float(blk_scale[bi])
                B = torch.from_numpy((q * sc).reshape(h, w)).to(device, DTYPE_ACC)
            else:
                B = torch.from_numpy(blk_val[v0:v1].astype(np.float32).reshape(h, w)).to(device, DTYPE_ACC)
            lst.append((i0, j0, B))
        out.append(lst)
    return out

# -----------------------------------------------------------------------------
# Payload runtime
# -----------------------------------------------------------------------------
class PayloadRuntime:
    def __init__(self):
        self.meta = {}
        self.expert_ids = []
        self.scales: Optional[torch.Tensor] = None
        self.cluster_of_pos: Optional[torch.Tensor] = None
        self.U: List[torch.Tensor] = []
        self.V: List[torch.Tensor] = []
        self.DL: List[torch.Tensor] = []
        self.DR: List[torch.Tensor] = []
        self.gam: Optional[torch.Tensor] = None
        self.Cfull: Optional[torch.Tensor] = None
        self.core_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.res_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.qmode = "none"
        self.res_coef = "diag"

    @torch.no_grad()
    def apply_expert(self, x: torch.Tensor, pos: int) -> torch.Tensor:
        c = int(self.cluster_of_pos[pos].item())
        U, V = self.U[c], self.V[c]
        DL, DR = self.DL[c], self.DR[c]
        z = x @ U
        u = torch.zeros_like(z)
        for (i0, j0, B) in self.core_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B
        if self.res_coef == "diag":
            g = self.gam[pos]
            u += ((z @ DL) * g.view(1, -1)) @ DR.t()
        else:
            C = self.Cfull[pos]
            u += (z @ DL) @ C @ DR.t()
        for (i0, j0, B) in self.res_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B
        y = u @ V.t()
        if self.scales is not None:
            y = y * self.scales[pos]
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        for a, pos in zip(gates.tolist(), routed):
            y += a * self.apply_expert(x, int(pos))
        return y

def load_payload_runtime(path: str, device: torch.device) -> PayloadRuntime:
    z = load_npz(path)
    rt = PayloadRuntime()
    rt.meta = _decode_meta(z["meta"])
    rt.qmode = rt.meta.get("qmode", "none")
    rt.res_coef = rt.meta.get("res_coef", "diag")
    rt.expert_ids = [int(x) for x in z["expert_ids"]]
    rt.scales = torch.from_numpy(z["scales"]).to(device, DTYPE_ACC)
    rt.cluster_of_pos = torch.from_numpy(z["cluster_of_pos"]).to(device, torch.int64)
    M = z["n_clusters"][0]
    for m in range(M):
        rt.U.append(torch.from_numpy(z[f"U_{m}"]).to(device, DTYPE_ACC))
        rt.V.append(torch.from_numpy(z[f"V_{m}"]).to(device, DTYPE_ACC))
        rt.DL.append(torch.from_numpy(z[f"DL_{m}"]).to(device, DTYPE_ACC))
        rt.DR.append(torch.from_numpy(z[f"DR_{m}"]).to(device, DTYPE_ACC))
    if rt.res_coef == "diag":
        rt.gam = torch.from_numpy(z["gam"]).to(device, DTYPE_ACC)
    else:
        rt.Cfull = torch.from_numpy(z["Cfull"]).to(device, DTYPE_ACC)
    core_pack = {k[5:]: z[k] for k in z if k.startswith("core_")}
    res_pack  = {k[4:]: z[k] for k in z if k.startswith("res_")}
    rt.core_blocks = unpack_blocks_ragged(core_pack, rt.qmode, device)
    rt.res_blocks  = unpack_blocks_ragged(res_pack, rt.qmode, device)
    return rt

# -----------------------------------------------------------------------------
# Build payload for one cluster
# -----------------------------------------------------------------------------
@torch.no_grad()
def frob_rel_err(A, B):
    return (torch.linalg.norm(A - B) / torch.linalg.norm(B).clamp_min(1e-12)).item()

@torch.no_grad()
def build_payload_for_cluster(Ws_norm: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor) -> Dict:
    n = Ws_norm.shape[-1]
    X_list = [(U.t() @ Ws_norm[pos] @ V).contiguous() for pos in idx]
    b = cfg.CORE_BLOCK

    core_per = []
    core_ef = []
    for X in X_list:
        Eg, te, nb = block_energy_grid(X, b)
        picks, eff = pick_blocks_until_target(Eg, te, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi * b, bj * b
            blocks.append((i0, j0, gather_block(X, i0, j0, b)))
        core_per.append(blocks)
        core_ef.append(eff)

    R_list = []
    for X, cb in zip(X_list, core_per):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    Rmean = torch.stack(R_list).mean(0)
    r = min(cfg.RES_RANK, n)
    DL, DR = rand_svd_vectors(Rmean, r, n_iter=2)

    coef_list, res_per = [], []
    bb = cfg.RES_BSIZE
    for j, Rm in enumerate(R_list):
        if cfg.RES_COEF == "diag":
            g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()
            coef_list.append(g)
            R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()
        else:
            C = (DL.t() @ Rm @ DR).contiguous()
            coef_list.append(C)
            R2 = (Rm - (DL @ C @ DR.t())).contiguous()

        Eg2, te2, nb2 = block_energy_grid(R2, bb)
        exclude = {(i0 // bb, j0 // bb) for (i0, j0, _) in core_per[j]}
        picks, _ = pick_blocks_until_target(Eg2, te2, cfg.RES_TARGET, cfg.RES_MAX_BLOCKS, exclude=exclude)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi * bb, bj * bb
            blocks.append((i0, j0, gather_block(R2, i0, j0, bb)))
        res_per.append(blocks)

    if cfg.REFINE_ENABLE:
        rb = cfg.REFINE_BSIZE
        for j in range(len(idx)):
            X = X_list[j]
            def reconstruct():
                Xc = torch.zeros_like(X)
                for (i0, j0, Bc) in core_per[j]:
                    h, w = Bc.shape
                    Xc[i0:i0+h, j0:j0+w] = Bc
                if cfg.RES_COEF == "diag":
                    g = coef_list[j]
                    Xlr = (DL * g.view(1, -1)) @ DR.t()
                else:
                    C = coef_list[j]
                    Xlr = DL @ C @ DR.t()
                Xr = torch.zeros_like(X)
                for (i0, j0, Bb) in res_per[j]:
                    h, w = Bb.shape
                    Xr[i0:i0+h, j0:j0+w] += Bb
                return Xc + Xlr + Xr
            Xhat = reconstruct()
            err = frob_rel_err(Xhat, X)
            added = 0
            core_pos = {(i0, j0) for (i0, j0, _) in core_per[j]}
            res_pos = {(i0, j0) for (i0, j0, _) in res_per[j]}
            while err > cfg.REFINE_ERR_TARGET and added < cfg.REFINE_MAX_EXTRA:
                Rerr = (X - Xhat).contiguous()
                Eg, te, nb = block_energy_grid(Rerr, rb)
                flat = Eg.reshape(-1)
                if flat.max().item() <= 1e-18:
                    break
                order = torch.argsort(flat, descending=True)
                found = False
                for idx_ in order.tolist():
                    bi, bj = idx_ // nb, idx_ % nb
                    i0, j0 = bi * rb, bj * rb
                    if (i0, j0) in core_pos or (i0, j0) in res_pos:
                        continue
                    Bb = gather_block(Rerr, i0, j0, rb)
                    res_per[j].append((i0, j0, Bb))
                    res_pos.add((i0, j0))
                    added += 1
                    found = True
                    break
                if not found:
                    break
                if added % cfg.REFINE_RECHECK_EVERY == 0:
                    Xhat = reconstruct()
                    err = frob_rel_err(Xhat, X)
            Xhat = reconstruct()
            err = frob_rel_err(Xhat, X)

    return {
        "core_blocks": core_per,
        "core_energy": core_ef,
        "DL": DL,
        "DR": DR,
        "coef_list": coef_list,
        "res_blocks": res_per
    }

# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------
@torch.no_grad()
def eval_payload(rt: PayloadRuntime, Ws_norm: torch.Tensor, Sc: torch.Tensor):
    E, n, _ = Ws_norm.shape
    errs = []
    for pos in range(E):
        x = torch.randn(8, n, dtype=DTYPE_ACC, device=DEVICE)
        y_hat = rt.apply_expert(x, pos)
        y_ref = x @ (Ws_norm[pos] * Sc[pos])
        errs.append((torch.linalg.norm(y_hat - y_ref) / torch.linalg.norm(y_ref).clamp_min(1e-12)).item())
    log(f"[eval] per-expert rel-error mean={np.mean(errs):.6f} p95={np.percentile(errs,95):.6f} max={np.max(errs):.6f}")

    mix = []
    for _ in range(cfg.EVAL_TRIALS):
        x = torch.randn(cfg.EVAL_BATCH, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), min(cfg.ROUTED_K, E))
        gates = torch.rand(len(routed), device=DEVICE)
        gates /= gates.sum()
        y_hat = rt.apply_mixture(x, routed, gates)
        Wsum = sum(gates[i].item() * (Ws_norm[pos] * Sc[pos]) for i, pos in enumerate(routed))
        y_ref = x @ Wsum
        mix.append((torch.linalg.norm(y_hat - y_ref) / torch.linalg.norm(y_ref).clamp_min(1e-12)).item())
    log(f"[eval] routed rel-error mean={np.mean(mix):.6f} ± {np.std(mix):.6f}")

# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def banner():
    log("="*60)
    log("EBC-LLM Compression Pipeline")
    log(f"Time: {now()}  Device: {DEVICE}")
    log(f"MODEL_DIR: {cfg.MODEL_DIR}  OUTPUT_DIR: {cfg.OUTPUT_DIR}")
    log(f"Layer: {cfg.LAYER}  Experts: {cfg.MAX_EXPERTS}")
    log(f"CALIB: {cfg.CALIB_PATH or '(none)'}  ROUTER: {cfg.ROUTER_PATH or '(none)'}")
    log(f"Ridge damp: {cfg.RIDGE_DAMP}  Normalize W: {cfg.NORMALIZE_W}")
    log(f"Basis: {cfg.BASIS_MODE}  Train steps: {cfg.TRAIN_STEPS}  lr: {cfg.TRAIN_LR}")
    log(f"Core: {cfg.CORE_MODE} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max={cfg.CORE_MAX_BLOCKS}")
    log(f"Residual: rank={cfg.RES_RANK} coef={cfg.RES_COEF} blocks={cfg.RES_MAX_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"Refine: {cfg.REFINE_ENABLE} target={cfg.REFINE_ERR_TARGET} max_extra={cfg.REFINE_MAX_EXTRA}")
    log("="*60)

def main():
    banner()
    expert_ids, Ws_norm, Sc = load_or_build_Ws()
    E, n, _ = Ws_norm.shape
    log(f"[Ws] shape={Ws_norm.shape}")

    # Clustering
    Xfeat = random_proj_features(Ws_norm, cfg.CLUSTER_FEAT_D)
    M0 = max(2, min(cfg.M0 if cfg.M0>0 else int(round(2*math.sqrt(E))), E))
    labels = kmeans_torch(Xfeat, M0, cfg.CLUSTER_ITERS, cfg.CLUSTER_RESTARTS)
    labels = merge_small_clusters(Xfeat, labels, cfg.CLUSTER_MIN_SIZE)
    labels = hierarchical_split(Xfeat, labels, cfg.CLUSTER_MAX_SIZE, min(cfg.M_MAX, E), cfg.SPLIT_ITERS)
    labels = merge_small_clusters(Xfeat, labels, cfg.CLUSTER_MIN_SIZE)
    labels = relabel_contiguous(labels)
    M = labels.max().item() + 1
    clusters = [torch.nonzero(labels==m, as_tuple=False).flatten().tolist() for m in range(M)]
    clusters = [c for c in clusters if c]
    log(f"[cluster] M={len(clusters)} sizes={[len(c) for c in clusters]}")
    cluster_of_pos = [0]*E
    for m, idx in enumerate(clusters):
        for pos in idx:
            cluster_of_pos[pos] = m

    # Init and train bases
    U_par, V_par = [], []
    for idx in clusters:
        Wm = Ws_norm[idx].mean(0)
        U0, V0 = svd_init_from_mean(Wm)
        U_par.append(OrthoParam(U0))
        V_par.append(OrthoParam(V0))

    if cfg.TRAIN_STEPS > 0 and cfg.BASIS_MODE == "dense_train":
        params = [p.M for p in U_par] + [p.M for p in V_par]
        opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)
        guidance_masks, guidance_stats = {}, {}
        t0 = time.perf_counter()
        for step in range(1, cfg.TRAIN_STEPS+1):
            S = torch.randperm(n)[:cfg.SUBM].to(DEVICE)
            if cfg.TRAIN_LAM_GUIDE > 0 and (step==1 or step%cfg.TRAIN_GUIDE_EVERY==0):
                with torch.no_grad():
                    guidance_masks.clear()
                    guidance_stats.clear()
                    for m, idx in enumerate(clusters):
                        if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                            continue
                        Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                        pick = idx if cfg.BATCH_E>=len(idx) else [idx[i] for i in torch.randperm(len(idx))[:cfg.BATCH_E].tolist()]
                        Xs_ng = slice_X_batch(Ws_norm[pick], Uo, Vo, S).detach()
                        mask, ef, kblk = make_guidance_mask_from_Xs(Xs_ng, cfg.CORE_BLOCK, cfg.TRAIN_GUIDE_TARGET, cfg.TRAIN_GUIDE_MAX_BLOCKS)
                        guidance_masks[m] = mask
                        guidance_stats[m] = (ef, kblk)

            lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
            lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
            lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp
            L_total, n_terms = None, 0
            for m, idx in enumerate(clusters):
                if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                    continue
                Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                pick = idx if cfg.BATCH_E>=len(idx) else [idx[i] for i in torch.randperm(len(idx))[:cfg.BATCH_E].tolist()]
                Xs = slice_X_batch(Ws_norm[pick], Uo, Vo, S)
                off, diag = offdiag_abs_mean(Xs), diag_abs_mean(Xs).clamp_min(1e-6)
                base = torch.log(off+1e-6) - torch.log(diag) if cfg.TRAIN_OBJ=="logratio" else off/diag
                if lam_block > 0:
                    base += lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)
                if lam_guide > 0 and m in guidance_masks:
                    Mmask = guidance_masks[m]
                    Etot = (Xs*Xs).mean().clamp_min(1e-12)
                    Eout = ((Xs*(1-Mmask))**2).mean()
                    base += lam_guide * (Eout/Etot)
                L_total = base if L_total is None else L_total + base
                n_terms += 1
            if L_total is None:
                break
            L_total = L_total / n_terms
            opt.zero_grad()
            L_total.backward()
            if cfg.GRAD_CLIP > 0:
                torch.nn.utils.clip_grad_norm_(params, cfg.GRAD_CLIP)
            opt.step()
            if step % cfg.REORTHO_EVERY == 0 or step == cfg.TRAIN_STEPS:
                with torch.no_grad():
                    for p in U_par:
                        p.M.copy_(p.orthogonal())
                    for p in V_par:
                        p.M.copy_(p.orthogonal())
            if step % cfg.REPORT_EVERY == 0 or step == 1:
                t1 = time.perf_counter()
                gstr = "" if not guidance_stats else f" guide≈{np.mean([v[0] for v in guidance_stats.values()]):.3f}"
                log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={L_total.item():.4f} {gstr} (+{t1-t0:.1f}s)")
                t0 = t1

    # Freeze bases
    U_list = [p.orthogonal().detach() for p in U_par]
    V_list = [p.orthogonal().detach() for p in V_par]

    # Build payloads
    log("[build] payloads ...")
    core_all = [[] for _ in range(E)]
    res_all  = [[] for _ in range(E)]
    DL_list, DR_list = [], []
    rmax = min(cfg.RES_RANK, n)
    gam = torch.zeros((E, rmax), dtype=DTYPE_ACC, device=DEVICE) if cfg.RES_COEF=="diag" else None
    Cfull = torch.zeros((E, rmax, rmax), dtype=DTYPE_ACC, device=DEVICE) if cfg.RES_COEF=="full" else None

    for m, idx in enumerate(clusters):
        U, V = U_list[m], V_list[m]
        P = build_payload_for_cluster(Ws_norm, idx, U, V)
        for j, pos in enumerate(idx):
            core_all[pos] = P["core_blocks"][j]
            res_all[pos] = P["res_blocks"][j]
            if cfg.RES_COEF == "diag":
                g = P["coef_list"][j]
                gam[pos, :g.numel()] = g
            else:
                C = P["coef_list"][j]
                Cfull[pos, :C.shape[0], :C.shape[1]] = C
        DL_list.append(P["DL"])
        DR_list.append(P["DR"])
        log(f"  cluster{m}: E={len(idx)} core_blocks≈{np.mean([len(c) for c in P['core_blocks']]):.1f} r={P['DL'].shape[1]}")

    # Save payload
    out_path = os.path.join(cfg.OUTPUT_DIR, f"ebc_payload_layer{cfg.LAYER}_E{E}_q{cfg.QMODE}.npz")
    store_dtype = np.float16 if cfg.BASIS_STORE_DTYPE=="float16" else np.float32
    arrays = {
        "meta": _encode_meta(ws_meta(expert_ids) | {"time": now(), "qmode": cfg.QMODE, "res_coef": cfg.RES_COEF}),
        "expert_ids": np.array(expert_ids, dtype=np.int32),
        "scales": Sc.cpu().numpy().astype(np.float32),
        "cluster_of_pos": np.array(cluster_of_pos, dtype=np.int16),
        "n_clusters": np.array([len(clusters)], dtype=np.int32),
    }
    for m in range(len(clusters)):
        arrays[f"U_{m}"] = U_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"V_{m}"] = V_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DL_{m}"] = DL_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DR_{m}"] = DR_list[m].cpu().numpy().astype(store_dtype)
    if cfg.RES_COEF == "diag":
        arrays["gam"] = gam.cpu().numpy().astype(store_dtype)
    else:
        arrays["Cfull"] = Cfull.cpu().numpy().astype(store_dtype)

    core_pack = pack_blocks_ragged(core_all, cfg.QMODE)
    res_pack  = pack_blocks_ragged(res_all, cfg.QMODE)
    for k, v in core_pack.items():
        arrays["core_"+k] = v
    for k, v in res_pack.items():
        arrays["res_"+k] = v

    save_npz_compressed(out_path, arrays)
    log(f"[save] payload -> {out_path} size={os.path.getsize(out_path)/1e6:.2f} MB")

    # Evaluate
    rt = load_payload_runtime(out_path, DEVICE)
    eval_payload(rt, Ws_norm, Sc)
    log("✅ Done.")

if __name__ == "__main__":
    main()

✅ flash_attn stub installed (CPU mode).
EBC-LLM Compression Pipeline
Time: 2026-04-21 19:19:29  Device: cpu
MODEL_DIR: /data/downloaded_models/Phi-3.5-MoE-instruct  OUTPUT_DIR: /home/daniyar/moe_ws_outputs_phi
Layer: 0  Experts: 16
CALIB: (none)  ROUTER: (none)
Ridge damp: 0.001  Normalize W: True
Basis: dense_train  Train steps: 24  lr: 0.05
Core: blocktopk_perexpert block=64 target=0.85 max=256
Residual: rank=512 coef=diag blocks=4096 bsize=64
Refine: True target=0.03 max_extra=4096
[found] layer=0 total=16 using=16 eids=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[load] reading tensors from shards ...
[shape] H=4096 d_ff=6400
[capture] capturing via transformers... (this will take several minutes on CPU)


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='longrope': {'long_mscale', 'short_mscale'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='longrope': {'long_mscale', 'short_mscale'}


Loading weights:   0%|          | 0/1957 [00:00<?, ?it/s]

[capture] Located MLP module: PhiMoESparseMoeBlock


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/home/daniyar/jupyter_env/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/daniyar/jupyter_env/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/daniyar/jupyter_env/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn

[capture] iter 4/32 nX=4096 nP=4096
[capture] wrote X -> /home/daniyar/moe_ws_outputs_phi/calib_layer0_X.npz shape=(4096, 4096)
[capture] wrote P -> /home/daniyar/moe_ws_outputs_phi/router_layer0_P.npz shape=(4096, 16)
[calib] X: torch.Size([4096, 4096])


Build Ws (ridge):   0%|          | 0/16 [00:00<?, ?it/s]

[cache] wrote Ws -> /home/daniyar/moe_ws_outputs_phi/Ws_cache_layer0_E16_ridge_ebc.npz size=1003.98 MB
[Ws] shape=torch.Size([16, 4096, 4096])
[cluster] M=4 sizes=[6, 3, 2, 5]
[train] step   1/24 loss=-4.3586  guide≈0.927 (+29.6s)
[train] step   4/24 loss=-0.9901  guide≈0.815 (+87.0s)
[train] step   8/24 loss=-0.1074  guide≈0.817 (+106.4s)
[train] step  12/24 loss=2.4962  guide≈0.821 (+106.0s)
[train] step  16/24 loss=1.3138  guide≈0.823 (+105.9s)
[train] step  20/24 loss=3.4100  guide≈0.826 (+104.7s)
[train] step  24/24 loss=2.7459  guide≈0.827 (+104.6s)
[build] payloads ...
  cluster0: E=6 core_blocks≈179.3 r=512
  cluster1: E=3 core_blocks≈132.3 r=512
  cluster2: E=2 core_blocks≈100.0 r=512
  cluster3: E=5 core_blocks≈192.8 r=512
[save] payload -> /home/daniyar/moe_ws_outputs_phi/ebc_payload_layer0_E16_qnone.npz size=1268.94 MB
[eval] per-expert rel-error mean=0.039328 p95=0.059993 max=0.073796
[eval] routed rel-error mean=0.033386 ± 0.008093
✅ Done.


In [3]:
import json
import re

MODEL_DIR = "/data/downloaded_models/DeepSeek-V2-Lite"
LAYER = 0

with open(f"{MODEL_DIR}/model.safetensors.index.json", "r") as f:
    wm = json.load(f).get("weight_map", {})

# Find all keys related to layer 0 experts
expert_keys = [k for k in wm.keys() if f"layers.{LAYER}" in k and "expert" in k.lower()]

print(f"Found {len(expert_keys)} expert-related keys for layer {LAYER}:")
for k in sorted(expert_keys)[:20]:
    print(f"  {k}")

# Also look for any MLP-related keys
mlp_keys = [k for k in wm.keys() if f"layers.{LAYER}.mlp" in k]
print(f"\nAll MLP keys for layer {LAYER}:")
for k in sorted(mlp_keys)[:30]:
    print(f"  {k}")

Found 0 expert-related keys for layer 0:

All MLP keys for layer 0:
  model.layers.0.mlp.down_proj.weight
  model.layers.0.mlp.gate_proj.weight
  model.layers.0.mlp.up_proj.weight


In [ ]:
# i try to install some packages which were not available via regular networks, pip install /path/to/einops-0.8.0-py3-none-any.whl

In [6]:
pwd ls -la einops-0.8.2.tar.gz

'/home/daniyar/notebooks'

In [11]:
import tarfile
import shutil
from pathlib import Path
import site
import sys

# Path to the tarball (adjust if needed)
tarball_path = "/home/daniyar/notebooks/einops-0.8.2.tar.gz"

# Locate the site-packages of your current environment
site_packages = Path(site.getsitepackages()[0])  # usually ~/jupyter_env/lib/python3.12/site-packages

# Extract the tarball to a temporary directory
extract_dir = Path("/tmp/einops_extract")
extract_dir.mkdir(exist_ok=True)

with tarfile.open(tarball_path, "r:gz") as tar:
    tar.extractall(path=extract_dir)

# The extracted folder is usually 'einops-0.8.2'
src = extract_dir / "einops-0.8.2" / "einops"
if not src.exists():
    raise FileNotFoundError(f"Could not find 'einops' folder inside {tarball_path}")

# Remove any existing einops installation to avoid conflicts
dest = site_packages / "einops"
if dest.exists():
    shutil.rmtree(dest)

# Copy the einops package to site-packages
shutil.copytree(src, dest)

# Clean up
shutil.rmtree(extract_dir)

# Verify installation
import einops
print(f"✅ einops {einops.__version__} manually installed to {dest}")
print(f"   Import path: {einops.__file__}")

✅ einops 0.8.2 manually installed to /home/daniyar/jupyter_env/lib/python3.12/site-packages/einops
   Import path: /home/daniyar/jupyter_env/lib/python3.12/site-packages/einops/__init__.py


In [7]:
import numpy as np
import torch
import einops
from einops import rearrange

print("einops version:", einops.__version__)
print("einops file   :", einops.__file__)

# Correct smoke test with numpy
x_np = np.array([[1, 2], [3, 4]])
print("numpy test:", rearrange(x_np, "h w -> (h w)"))

# Correct smoke test with torch
x_t = torch.tensor([[1, 2], [3, 4]])
print("torch test:", rearrange(x_t, "h w -> (h w)"))

print("✅ einops is installed and working")

ModuleNotFoundError: No module named 'einops'

In [20]:
#!/usr/bin/env python3
# =============================================================================
# EBC-LLM: Expert-Bank Compression for DeepSeek-V2-Lite (CPU, self-contained)
# =============================================================================

# #############################################################################
# FLASH_ATTN STUB – MUST BE FIRST
# #############################################################################
import sys
import types
import importlib.machinery

def _install_flash_attn_stub():
    flash_attn = types.ModuleType("flash_attn")
    flash_attn.__version__ = "0.0.0-cpu-stub"

    def _unavailable(*args, **kwargs):
        raise RuntimeError("flash_attn stub called on CPU. Use attn_implementation='eager'.")

    flash_attn.flash_attn_func = _unavailable
    flash_attn.flash_attn_varlen_func = _unavailable
    flash_attn.flash_attn_with_kvcache = _unavailable

    flash_attn.layers = types.ModuleType("flash_attn.layers")
    flash_attn.layers.rotary = types.ModuleType("flash_attn.layers.rotary")
    flash_attn.ops = types.ModuleType("flash_attn.ops")
    flash_attn.ops.triton = types.ModuleType("flash_attn.ops.triton")
    flash_attn.bert_padding = types.ModuleType("flash_attn.bert_padding")
    flash_attn.flash_attn_interface = types.ModuleType("flash_attn.flash_attn_interface")

    import torch
    import torch.nn as nn
    class RotaryEmbedding(nn.Module):
        def __init__(self, dim, base=10000.0, interleaved=False, scale_base=None, device=None):
            super().__init__()
            self.dim = dim
        def forward(self, x, seq_len=None, **kwargs):
            device, dtype = x.device, x.dtype
            seq = seq_len if seq_len else x.shape[-2]
            half = max(1, self.dim // 2)
            cos = torch.ones((seq, half), device=device, dtype=dtype)
            sin = torch.zeros((seq, half), device=device, dtype=dtype)
            return cos, sin

    flash_attn.layers.rotary.RotaryEmbedding = RotaryEmbedding
    flash_attn.layers.rotary.apply_rotary_emb = lambda *a, **k: (_unavailable,)
    flash_attn.bert_padding.index_first_axis = lambda x, *a, **k: x
    flash_attn.bert_padding.pad_input = _unavailable
    flash_attn.bert_padding.unpad_input = _unavailable
    flash_attn.flash_attn_interface.flash_attn_func = _unavailable
    flash_attn.flash_attn_interface.flash_attn_varlen_func = _unavailable
    flash_attn.flash_attn_interface.flash_attn_with_kvcache = _unavailable

    sys.modules["flash_attn"] = flash_attn
    sys.modules["flash_attn.layers"] = flash_attn.layers
    sys.modules["flash_attn.layers.rotary"] = flash_attn.layers.rotary
    sys.modules["flash_attn.ops"] = flash_attn.ops
    sys.modules["flash_attn.ops.triton"] = flash_attn.ops.triton
    sys.modules["flash_attn.bert_padding"] = flash_attn.bert_padding
    sys.modules["flash_attn.flash_attn_interface"] = flash_attn.flash_attn_interface

class FlashAttnImporter:
    def find_spec(self, fullname, path, target=None):
        if fullname == "flash_attn" or fullname.startswith("flash_attn."):
            if "flash_attn" not in sys.modules:
                _install_flash_attn_stub()
            return importlib.machinery.ModuleSpec(fullname, self)
        return None
    def create_module(self, spec): return sys.modules.get(spec.name)
    def exec_module(self, module): pass

sys.meta_path.insert(0, FlashAttnImporter())
print("✅ flash_attn stub installed (CPU mode).", flush=True)

# #############################################################################
# IMPORTS
# #############################################################################
import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs): return x

# -----------------------------------------------------------------------------
# Environment helpers
# -----------------------------------------------------------------------------
def _env_str(k: str, d: str) -> str:
    return os.environ.get(k, d)

def _env_int(k: str, d: int) -> int:
    try: return int(os.environ.get(k, str(d)))
    except: return d

def _env_float(k: str, d: float) -> float:
    try: return float(os.environ.get(k, str(d)))
    except: return d

def _env_bool(k: str, d: bool) -> bool:
    v = os.environ.get(k, None)
    if v is None: return d
    return v.strip().lower() in ("1", "true", "yes", "y", "on")

# -----------------------------------------------------------------------------
# Configuration – with CAPTURE_KEEP_PAD = True to avoid masking issues
# -----------------------------------------------------------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = "/data/downloaded_models/DeepSeek-V2-Lite"
    OUTPUT_DIR: str = "/home/daniyar/moe_ws_outputs_deepseek"

    LAYER: int = 1                     # Try layer 1 first (layer 0 is often dense)
    MAX_EXPERTS: int = 16

    CALIB_PATH: str = _env_str("CALIB_PATH", "").strip()
    ROUTER_PATH: str = _env_str("ROUTER_PATH", "").strip()
    CALIB_SAMPLES: int = _env_int("CALIB_SAMPLES", 4096)
    RIDGE_WEIGHTED: bool = _env_bool("RIDGE_WEIGHTED", False)
    ROUTER_EIDS_ARE_GLOBAL: bool = _env_bool("ROUTER_EIDS_ARE_GLOBAL", True)
    RIDGE_DAMP: float = _env_float("RIDGE_DAMP", 1e-3)
    NORMALIZE_W: bool = _env_bool("NORMALIZE_W", True)

    CAPTURE_ENABLE: bool = True
    CAPTURE_FORCE: bool = _env_bool("CAPTURE_FORCE", False)
    CAPTURE_ITERS: int = _env_int("CAPTURE_ITERS", 32)
    CAPTURE_BATCH: int = _env_int("CAPTURE_BATCH", 1)
    CAPTURE_MAX_TOKENS: int = _env_int("CAPTURE_MAX_TOKENS", 1024)
    CAPTURE_TEXT: str = _env_str("CAPTURE_TEXT", "DeepSeek MoE calibration text. " * 256)
    CAPTURE_TEXT_FILE: str = _env_str("CAPTURE_TEXT_FILE", "").strip()
    CAPTURE_KEEP_PAD: bool = True       # <-- SKIP MASKING TO AVOID INDEXERROR
    HF_TRUST_REMOTE_CODE: bool = _env_bool("HF_TRUST_REMOTE_CODE", True)
    HF_LOCAL_FILES_ONLY: bool = _env_bool("HF_LOCAL_FILES_ONLY", True)
    HF_AUTO_PIP: bool = _env_bool("HF_AUTO_PIP", False)

    BASIS_MODE: str = _env_str("BASIS_MODE", "dense_train").lower()
    BASIS_STORE_DTYPE: str = _env_str("BASIS_STORE_DTYPE", "float16").lower()

    M0: int = _env_int("M0", 0)
    M_MAX: int = _env_int("M_MAX", 16)
    CLUSTER_FEAT_D: int = _env_int("CLUSTER_FEAT_D", 64)
    CLUSTER_ITERS: int = _env_int("CLUSTER_ITERS", 60)
    CLUSTER_RESTARTS: int = _env_int("CLUSTER_RESTARTS", 4)
    CLUSTER_MIN_SIZE: int = _env_int("CLUSTER_MIN_SIZE", 2)
    CLUSTER_MAX_SIZE: int = _env_int("CLUSTER_MAX_SIZE", 4)
    SPLIT_ITERS: int = _env_int("SPLIT_ITERS", 50)

    TRAIN_STEPS: int = _env_int("TRAIN_STEPS", 24)
    TRAIN_WARMUP: int = _env_int("TRAIN_WARMUP", 6)
    TRAIN_LR: float = _env_float("TRAIN_LR", 5e-2)
    SUBM: int = _env_int("SUBM", 256)
    BATCH_E: int = _env_int("BATCH_E", 4)
    TRAIN_MIN_CLUSTER: int = _env_int("TRAIN_MIN_CLUSTER", 2)
    REORTHO_EVERY: int = _env_int("REORTHO_EVERY", 4)
    REPORT_EVERY: int = _env_int("REPORT_EVERY", 4)
    GRAD_CLIP: float = _env_float("GRAD_CLIP", 1.0)
    TRAIN_OBJ: str = _env_str("TRAIN_OBJ", "logratio").lower()
    TRAIN_LAM_BLOCK: float = _env_float("TRAIN_LAM_BLOCK", 0.10)
    TRAIN_LAM_GUIDE: float = _env_float("TRAIN_LAM_GUIDE", 1.0)
    TRAIN_GUIDE_EVERY: int = _env_int("TRAIN_GUIDE_EVERY", 2)
    TRAIN_GUIDE_TARGET: float = _env_float("TRAIN_GUIDE_TARGET", 0.80)
    TRAIN_GUIDE_MAX_BLOCKS: int = _env_int("TRAIN_GUIDE_MAX_BLOCKS", 2048)

    CORE_MODE: str = _env_str("CORE_MODE", "blocktopk_perexpert").lower()
    CORE_AGG: str = _env_str("CORE_AGG", "mean").lower()
    CORE_BLOCK: int = _env_int("CORE_BLOCK", 64)
    CORE_TARGET: float = _env_float("CORE_TARGET", 0.85)
    CORE_MAX_BLOCKS: int = _env_int("CORE_MAX_BLOCKS", 256)

    RES_RANK: int = _env_int("RES_RANK", 512)
    RES_COEF: str = _env_str("RES_COEF", "diag").lower()
    RES_TARGET: float = _env_float("RES_TARGET", 0.995)
    RES_MAX_BLOCKS: int = _env_int("RES_MAX_BLOCKS", 4096)
    RES_BSIZE: int = _env_int("RES_BSIZE", 64)

    REFINE_ENABLE: bool = _env_bool("REFINE_ENABLE", True)
    REFINE_ERR_TARGET: float = _env_float("REFINE_ERR_TARGET", 0.03)
    REFINE_MAX_EXTRA: int = _env_int("REFINE_MAX_EXTRA", 4096)
    REFINE_BSIZE: int = _env_int("REFINE_BSIZE", 64)
    REFINE_RECHECK_EVERY: int = _env_int("REFINE_RECHECK_EVERY", 32)

    QMODE: str = _env_str("QMODE", "none").lower()

    EVAL_TRIALS: int = _env_int("EVAL_TRIALS", 8)
    EVAL_BATCH: int = _env_int("EVAL_BATCH", 2)
    ROUTED_K: int = _env_int("ROUTED_K", 8)

cfg = Cfg()
PRESET = _env_str("PRESET", "").strip().lower()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def _setdefault_env(k: str, v: str):
    if k not in os.environ: os.environ[k] = v

if PRESET == "maxacc":
    _setdefault_env("CALIB_SAMPLES", "32768")
    _setdefault_env("RIDGE_DAMP", "1e-2")
    _setdefault_env("CORE_BLOCK", "32")
    _setdefault_env("CORE_TARGET", "0.995")
    _setdefault_env("CORE_MAX_BLOCKS", "8192")
    _setdefault_env("RES_RANK", "2048")
    _setdefault_env("RES_COEF", "full")
    _setdefault_env("RES_TARGET", "0.999")
    _setdefault_env("RES_MAX_BLOCKS", "32768")
    _setdefault_env("REFINE_ENABLE", "1")
    _setdefault_env("REFINE_ERR_TARGET", "0.01")
    _setdefault_env("REFINE_MAX_EXTRA", "65536")
    _setdefault_env("TRAIN_STEPS", "96")
    _setdefault_env("TRAIN_LR", "0.02")
    _setdefault_env("TRAIN_LAM_GUIDE", "0.5")
    cfg = Cfg()
elif PRESET == "compact":
    _setdefault_env("CALIB_SAMPLES", "4096")
    _setdefault_env("CORE_BLOCK", "64")
    _setdefault_env("CORE_TARGET", "0.90")
    _setdefault_env("CORE_MAX_BLOCKS", "512")
    _setdefault_env("RES_RANK", "512")
    _setdefault_env("RES_COEF", "diag")
    _setdefault_env("RES_TARGET", "0.99")
    _setdefault_env("RES_MAX_BLOCKS", "4096")
    _setdefault_env("QMODE", "float16")
    _setdefault_env("REFINE_ENABLE", "0")
    _setdefault_env("TRAIN_STEPS", "24")
    cfg = Cfg()

# -----------------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------------
def log(msg: str): print(msg, flush=True)
def now() -> str: return time.strftime("%Y-%m-%d %H:%M:%S")

def seed_all(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

SEED = _env_int("SEED", 1234)
seed_all(SEED)
NTHREADS = _env_int("KTXX_THREADS", 8)
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
try: torch.set_num_threads(NTHREADS)
except: pass

DEVICE = torch.device(_env_str("DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
DTYPE_ACC = torch.float32

# -----------------------------------------------------------------------------
# NPZ I/O
# -----------------------------------------------------------------------------
def save_npz_compressed(path: str, arrays: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, **arrays)

def load_npz(path: str) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    return {k: z[k] for k in z.files}

def _encode_meta(meta: dict) -> np.ndarray:
    return np.frombuffer(json.dumps(meta, sort_keys=True).encode("utf-8"), dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try: return json.loads(bytes(arr.tolist()).decode("utf-8"))
    except: return {}

# -----------------------------------------------------------------------------
# Weight loading – adaptive for DeepSeek-V2-Lite
# -----------------------------------------------------------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r") as f:
        return json.load(f).get("weight_map", {})

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    patterns = [
        rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.",
        rf"^model\.layers\.{layer}\.block_sparse_moe\.experts\.(\d+)\.",
        rf"^model\.layers\.{layer}\.mlp\.shared_experts\.(\d+)\.",
        rf"^model\.layers\.{layer}\.moe\.experts\.(\d+)\.",
    ]
    ids = set()
    for pat_str in patterns:
        pat = re.compile(pat_str)
        for k in weight_map:
            m = pat.match(k)
            if m:
                ids.add(int(m.group(1)))
        if ids:
            break
    return sorted(ids)

def is_monolithic_mlp(weight_map: Dict[str, str], layer: int) -> bool:
    prefixes = [
        f"model.layers.{layer}.mlp.gate_proj.weight",
        f"model.layers.{layer}.mlp.up_proj.weight",
        f"model.layers.{layer}.mlp.down_proj.weight",
    ]
    return all(any(k.startswith(p) for k in weight_map) for p in prefixes)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefixes = [
        f"model.layers.{layer}.mlp.experts.{eid}.",
        f"model.layers.{layer}.block_sparse_moe.experts.{eid}.",
        f"model.layers.{layer}.mlp.shared_experts.{eid}.",
        f"model.layers.{layer}.moe.experts.{eid}.",
    ]
    used_prefix = None
    for pfx in prefixes:
        if any(k.startswith(pfx) for k in weight_map):
            used_prefix = pfx
            break
    if used_prefix is None:
        return {}

    def pick(suffix):
        k = used_prefix + suffix
        return k if k in weight_map else None

    gate = pick("gate_proj.weight") or pick("w1.weight")
    down = pick("down_proj.weight") or pick("w2.weight")
    up   = pick("up_proj.weight") or pick("w3.weight")

    if gate is None or down is None or up is None:
        return {}
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard = {}
    for k in keys:
        shard = weight_map.get(k)
        if shard is None: continue
        by_shard.setdefault(shard, []).append(k)
    out = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp): continue
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks: out[k] = f.get_tensor(k)
    return out

def load_monolithic_mlp_weights(model_dir: str, weight_map: Dict[str, str], layer: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    keys = {
        "gate": f"model.layers.{layer}.mlp.gate_proj.weight",
        "up":   f"model.layers.{layer}.mlp.up_proj.weight",
        "down": f"model.layers.{layer}.mlp.down_proj.weight",
    }
    tensors = {}
    for role, key in keys.items():
        shard = weight_map[key]
        sp = os.path.join(model_dir, shard)
        with safe_open(sp, framework="pt", device="cpu") as f:
            tensors[role] = f.get_tensor(key)
    return tensors["gate"], tensors["up"], tensors["down"]

def split_mlp_into_virtual_experts(W_gate, W_up, W_down, num_experts: int) -> List[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]]:
    d_ff = W_gate.shape[0]
    chunk_size = d_ff // num_experts
    experts = []
    for i in range(num_experts):
        start = i * chunk_size
        end = (i + 1) * chunk_size if i < num_experts - 1 else d_ff
        gate_i = W_gate[start:end, :].clone()
        up_i   = W_up[start:end, :].clone()
        down_i = W_down[:, start:end].clone()
        experts.append((gate_i, up_i, down_i))
    return experts

# -----------------------------------------------------------------------------
# Calibration capture – simplified collector (no masking)
# -----------------------------------------------------------------------------
def autodetect_calib_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return cand if os.path.isfile(cand) else None

def autodetect_router_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return cand if os.path.isfile(cand) else None

def load_calib_X(path: str, H: int) -> torch.Tensor:
    z = np.load(path)
    X = torch.from_numpy(z["X"].astype(np.float32))
    if X.ndim != 2 or X.shape[1] != H: raise RuntimeError(f"Bad X shape {X.shape}")
    if X.shape[0] > cfg.CALIB_SAMPLES: X = X[:cfg.CALIB_SAMPLES]
    return X.to(device=DEVICE, dtype=DTYPE_ACC)

def load_router_P(path: str) -> np.ndarray:
    return np.load(path)["P"].astype(np.float32)

def _maybe_autopip():
    if not cfg.HF_AUTO_PIP: return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", "transformers", "sentencepiece", "tokenizers"])

def _patch_transformers_cache_compat():
    try:
        from transformers.cache_utils import DynamicCache
        if not hasattr(DynamicCache, "get_usable_length"):
            DynamicCache.get_usable_length = lambda self, seq_length: int(seq_length)
    except: pass

class _Collector:
    def __init__(self, H, E_total, max_rows):
        self.H = H
        self.E_total = E_total
        self.max_rows = max_rows
        self.X_chunks, self.P_chunks = [], []
        self.nX = self.nP = 0

    def _take(self, flat, need):
        return flat[:need] if flat.shape[0] > need else flat

    def add_X(self, hs, attn_mask=None):
        if hs is None: return
        if hs.ndim == 2: hs = hs.unsqueeze(0)
        if hs.ndim != 3 or hs.shape[-1] != self.H: return
        flat = hs.detach().to(torch.float32).cpu().reshape(-1, self.H)
        if flat.numel() == 0: return
        need = self.max_rows - self.nX
        if need <= 0: return
        self.X_chunks.append(self._take(flat, need))
        self.nX += self.X_chunks[-1].shape[0]

    def add_logits(self, logits, attn_mask=None):
        if logits is None: return
        if logits.ndim == 2: logits = logits.unsqueeze(0)
        if logits.ndim != 3: return
        P = torch.softmax(logits.detach().to(torch.float32), dim=-1)[..., :self.E_total].cpu()
        flat = P.reshape(-1, P.shape[-1])
        if flat.numel() == 0: return
        need = self.max_rows - self.nP
        if need <= 0: return
        self.P_chunks.append(self._take(flat, need))
        self.nP += self.P_chunks[-1].shape[0]

def capture_XP_transformers(model_dir, layer_idx, H, E_total, out_x, out_p):
    _maybe_autopip()
    _patch_transformers_cache_compat()

    import transformers.utils.import_utils as iu
    if not hasattr(iu, "is_torch_fx_available"):
        def is_torch_fx_available():
            try: import torch.fx; return True
            except ImportError: return False
        iu.is_torch_fx_available = is_torch_fx_available

    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=cfg.HF_TRUST_REMOTE_CODE, local_files_only=cfg.HF_LOCAL_FILES_ONLY)
    if tok.pad_token is None: tok.pad_token = tok.eos_token or tok.unk_token

    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
        attn_implementation="eager"
    ).to(DEVICE).eval()

    layers = None
    if hasattr(model, "model") and hasattr(model.model, "layers"): layers = model.model.layers
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"): layers = model.transformer.h
    elif hasattr(model, "layers"): layers = model.layers
    if layers is None: raise RuntimeError("Cannot locate layers")
    if layer_idx >= len(layers): raise RuntimeError(f"Layer {layer_idx} out of range")
    layer = layers[layer_idx]

    mlp = getattr(layer, "mlp", None) or getattr(layer, "moe", None) or getattr(layer, "block_sparse_moe", None)
    if mlp is None:
        for name, mod in layer.named_modules():
            if any(x in name.lower() for x in ["mlp", "moe", "expert"]):
                if hasattr(mod, "gate_proj") or hasattr(mod, "w1"):
                    mlp = mod
                    break
    if mlp is None: raise RuntimeError("Could not find MoE MLP module in layer.")
    log(f"[capture] Located MLP module: {mlp.__class__.__name__}")

    router_linear = None
    for name, mod in layer.named_modules():
        if isinstance(mod, nn.Linear) and mod.in_features == H and mod.out_features >= E_total:
            if "router" in name.lower() or "gate" in name.lower():
                router_linear = mod
                break

    coll = _Collector(H, E_total, cfg.CALIB_SAMPLES)
    attn_holder = {"mask": None}

    def mlp_pre_hook(_, inputs):
        coll.add_X(inputs[0])

    h1 = mlp.register_forward_pre_hook(mlp_pre_hook)
    h2 = None
    if router_linear is not None:
        def router_hook(_, __, out):
            o = out[0] if isinstance(out, (tuple, list)) else out
            coll.add_logits(o)
        h2 = router_linear.register_forward_hook(router_hook)

    texts = [cfg.CAPTURE_TEXT]
    if cfg.CAPTURE_TEXT_FILE and os.path.isfile(cfg.CAPTURE_TEXT_FILE):
        with open(cfg.CAPTURE_TEXT_FILE) as f: texts = [ln.strip() for ln in f if ln.strip()]
    tptr = 0
    for it in range(cfg.CAPTURE_ITERS):
        text = texts[tptr % len(texts)]; tptr += 1
        enc = tok(text, return_tensors="pt", truncation=True, max_length=cfg.CAPTURE_MAX_TOKENS, padding="max_length")
        for k in enc:
            if enc[k].ndim == 2 and cfg.CAPTURE_BATCH > 1: enc[k] = enc[k].repeat(cfg.CAPTURE_BATCH, 1)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.inference_mode(): _ = model(**enc, use_cache=False)
        if (it + 1) % 4 == 0: log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS} nX={coll.nX} nP={coll.nP}")
        if coll.nX >= cfg.CALIB_SAMPLES and (not cfg.RIDGE_WEIGHTED or coll.nP >= cfg.CALIB_SAMPLES): break

    h1.remove()
    if h2: h2.remove()

    if coll.nX == 0: raise RuntimeError("Capture collected 0 rows")
    X = torch.cat(coll.X_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz_compressed(out_x, {"X": X})
    log(f"[capture] wrote X -> {out_x} shape={X.shape}")
    p_written = None
    if coll.nP > 0:
        P = torch.cat(coll.P_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
        N = min(P.shape[0], X.shape[0])
        if N < X.shape[0]: X = X[:N]; save_npz_compressed(out_x, {"X": X})
        P = P[:N]; save_npz_compressed(out_p, {"P": P})
        log(f"[capture] wrote P -> {out_p} shape={P.shape}")
        p_written = out_p
    return out_x, p_written

def ensure_calib_router(H: int, E_total: int):
    calib_cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    router_cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    if os.path.isfile(calib_cand) and (not cfg.RIDGE_WEIGHTED or os.path.isfile(router_cand)):
        log("[capture] Calibration cache found. Skipping transformers model loading.")
        cfg.CALIB_PATH = calib_cand
        if os.path.isfile(router_cand): cfg.ROUTER_PATH = router_cand
        return

    if not cfg.CALIB_PATH:
        c = autodetect_calib_path()
        if c: cfg.CALIB_PATH = c; log(f"[calib] auto-found {cfg.CALIB_PATH}")
    if not cfg.ROUTER_PATH:
        r = autodetect_router_path()
        if r: cfg.ROUTER_PATH = r; log(f"[router] auto-found {cfg.ROUTER_PATH}")
    if cfg.CAPTURE_FORCE or (cfg.CAPTURE_ENABLE and (not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH))):
        out_x = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
        out_p = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
        log("[capture] capturing via transformers... (this will take several minutes on CPU)")
        x_path, p_path = capture_XP_transformers(cfg.MODEL_DIR, cfg.LAYER, H, E_total, out_x, out_p)
        cfg.CALIB_PATH = x_path
        if p_path: cfg.ROUTER_PATH = p_path

# -----------------------------------------------------------------------------
# Ridge linearization
# -----------------------------------------------------------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate, W_up, W_down) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    return hid @ W_down.to(DTYPE_ACC).t()

def ws_cache_path(E: int) -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{E}_ridge_ebc.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        script="ebc_llm", model_dir=cfg.MODEL_DIR, layer=cfg.LAYER, expert_ids=eids,
        ridge_damp=cfg.RIDGE_DAMP, ridge_weighted=cfg.RIDGE_WEIGHTED,
        router_path=cfg.ROUTER_PATH or "", calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES, normalize_w=cfg.NORMALIZE_W, seed=SEED, device=str(DEVICE)
    )

@torch.no_grad()
def build_Ws_from_experts(eids: List[int], wm: Dict[str, str]) -> Tuple[torch.Tensor, torch.Tensor]:
    per_e, need_keys = {}, []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk: raise RuntimeError(f"Expert {eid} missing tensors")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]
    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))
    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = W_up0.shape[0], W_up0.shape[1]
    log(f"[shape] H={H} d_ff={dff}")

    ensure_calib_router(H, len(eids))
    X = load_calib_X(cfg.CALIB_PATH, H)
    log(f"[calib] X: {X.shape}")

    P = None
    if cfg.RIDGE_WEIGHTED and cfg.ROUTER_PATH and os.path.isfile(cfg.ROUTER_PATH):
        P = load_router_P(cfg.ROUTER_PATH)
        log(f"[router] P: {P.shape}")

    Xf = X.to(DTYPE_ACC)
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * torch.trace(XtX).item() / H
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list, scales = [], []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws (ridge)")):
        W_up = T[per_e[eid]["up"]].to(DEVICE)
        W_dn = T[per_e[eid]["down"]].to(DEVICE)
        W_gt = T[per_e[eid]["gate"]].to(DEVICE)
        Y = forward_mlp(X, W_gt, W_up, W_dn).to(DTYPE_ACC)

        if P is not None:
            w = torch.from_numpy(P[:X.shape[0], eid if cfg.ROUTER_EIDS_ARE_GLOBAL else i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0)
            sw = torch.sqrt(w + 1e-12).view(-1, 1)
            Xw, Yw = Xf * sw, Y * sw
            XtX_e = Xw.t() @ Xw
            lam_e = cfg.RIDGE_DAMP * torch.trace(XtX_e).item() / H
            chol = torch.linalg.cholesky(XtX_e + lam_e * I)
            Wt = torch.cholesky_solve(Xw.t() @ Yw, chol)
            W = Wt.t().contiguous()
        else:
            Wt = torch.cholesky_solve(Xf.t() @ Y, cholG)
            W = Wt.t().contiguous()

        if cfg.NORMALIZE_W:
            s = torch.linalg.norm(W, ord="fro").clamp_min(1e-12).item()
            W = W / s
        else: s = 1.0
        Ws_list.append(W); scales.append(s)

    Ws = torch.stack(Ws_list).to(DTYPE_ACC).to(DEVICE)
    Sc = torch.tensor(scales, dtype=DTYPE_ACC, device=DEVICE)
    return Ws, Sc

@torch.no_grad()
def build_Ws_monolithic(wm: Dict[str, str]) -> Tuple[torch.Tensor, torch.Tensor]:
    W_gate, W_up, W_down = load_monolithic_mlp_weights(cfg.MODEL_DIR, wm, cfg.LAYER)
    H = W_gate.shape[1]
    d_ff = W_gate.shape[0]
    log(f"[shape] H={H} d_ff={d_ff} (monolithic)")

    virtual_experts = split_mlp_into_virtual_experts(W_gate, W_up, W_down, cfg.MAX_EXPERTS)
    E = len(virtual_experts)
    log(f"[virtual] Split monolithic MLP into {E} virtual expert(s)")

    ensure_calib_router(H, E)
    X = load_calib_X(cfg.CALIB_PATH, H)
    log(f"[calib] X: {X.shape}")

    Xf = X.to(DTYPE_ACC)
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * torch.trace(XtX).item() / H
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list, scales = [], []
    for i, (g, u, d) in enumerate(tqdm(virtual_experts, desc="Build Ws (ridge, virtual)")):
        Y = forward_mlp(X, g.to(DEVICE), u.to(DEVICE), d.to(DEVICE)).to(DTYPE_ACC)
        Wt = torch.cholesky_solve(Xf.t() @ Y, cholG)
        W = Wt.t().contiguous()
        if cfg.NORMALIZE_W:
            s = torch.linalg.norm(W, ord="fro").clamp_min(1e-12).item()
            W = W / s
        else: s = 1.0
        Ws_list.append(W); scales.append(s)

    Ws = torch.stack(Ws_list).to(DTYPE_ACC).to(DEVICE)
    Sc = torch.tensor(scales, dtype=DTYPE_ACC, device=DEVICE)
    return Ws, Sc

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor, torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)

    for attempt in range(5):
        current_layer = cfg.LAYER + attempt
        log(f"[search] Checking layer {current_layer} for experts...")
        all_eids = find_layer_expert_ids(wm, current_layer)
        if all_eids:
            cfg.LAYER = current_layer
            log(f"[found] layer={cfg.LAYER} total experts={len(all_eids)}")
            eids = all_eids[:cfg.MAX_EXPERTS]
            Ws, Sc = build_Ws_from_experts(eids, wm)
            return eids, Ws, Sc

        if is_monolithic_mlp(wm, current_layer):
            cfg.LAYER = current_layer
            log(f"[found] layer={cfg.LAYER} uses monolithic MLP. Splitting into virtual experts.")
            Ws, Sc = build_Ws_monolithic(wm)
            eids = list(range(cfg.MAX_EXPERTS))
            return eids, Ws, Sc

    raise RuntimeError("Could not find any MoE experts or monolithic MLP in layers 0-4.")

# -----------------------------------------------------------------------------
# Clustering (kmeans++ + hierarchical split)
# -----------------------------------------------------------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED + 17)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([row @ R, col @ R]).unsqueeze(0))
    X = torch.cat(feats, dim=0)
    X = (X - X.mean(0, keepdim=True)) / (X.std(0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> torch.Tensor:
    best_lab, best_inertia = None, float("inf")
    g = torch.Generator(device="cpu").manual_seed(SEED + 999)
    for _ in range(max(1, restarts)):
        n = X.shape[0]
        centers = [X[torch.randint(0, n, (1,), generator=g).item()].clone()]
        for _ in range(1, k):
            C = torch.stack(centers)
            dist2 = torch.cdist(X, C).pow(2).min(1).values
            prob = dist2 / dist2.sum().clamp_min(1e-12)
            centers.append(X[torch.multinomial(prob, 1, generator=g).item()].clone())
        C = torch.stack(centers)
        for _ in range(iters):
            dist = torch.cdist(X, C)
            lab = dist.argmin(1)
            for j in range(k):
                m = (lab == j)
                if m.any(): C[j] = X[m].mean(0)
                else: C[j] = X[dist.min(1).values.argmax().item()].clone()
        inertia = torch.cdist(X, C).min(1).values.pow(2).sum().item()
        if inertia < best_inertia:
            best_inertia, best_lab = inertia, lab.clone()
    return best_lab.to(torch.int64)

@torch.no_grad()
def relabel_contiguous(labels: torch.Tensor) -> torch.Tensor:
    uniq = torch.unique(labels)
    out = labels.clone()
    for new, old in enumerate(uniq.tolist()):
        out[labels == old] = new
    return out

@torch.no_grad()
def merge_small_clusters(X: torch.Tensor, labels: torch.Tensor, min_size: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if min_size <= 1: return labels
    while True:
        K = labels.max().item() + 1
        counts = torch.bincount(labels, minlength=K)
        small = (counts < min_size).nonzero(as_tuple=False).flatten()
        if small.numel() == 0: break
        C = torch.stack([X[labels == k].mean(0) for k in range(K)])
        for c in small.tolist():
            idxs = (labels == c).nonzero(as_tuple=False).flatten()
            if idxs.numel() == 0: continue
            dist = torch.cdist(C[c].unsqueeze(0), C).squeeze(0); dist[c] = 1e9
            labels[idxs] = dist.argmin().item()
        labels = relabel_contiguous(labels)
    return labels

@torch.no_grad()
def hierarchical_split(X: torch.Tensor, labels: torch.Tensor, max_size: int, max_k: int, split_iters: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if max_size <= 0: return labels
    while True:
        K = labels.max().item() + 1
        if K >= max_k: break
        counts = torch.bincount(labels, minlength=K)
        biggest = counts.argmax().item()
        if counts[biggest] <= max_size: break
        idxs = (labels == biggest).nonzero(as_tuple=False).flatten()
        if idxs.numel() < 2: break
        sub = X[idxs]
        sub_lab = kmeans_torch(sub, 2, split_iters, 1)
        a, b = idxs[sub_lab == 0], idxs[sub_lab == 1]
        if a.numel() == 0 or b.numel() == 0: break
        labels[b] = K
        labels = relabel_contiguous(labels)
    return labels

# -----------------------------------------------------------------------------
# Basis training (dense)
# -----------------------------------------------------------------------------
class OrthoParam(nn.Module):
    def __init__(self, init_mat: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(init_mat.to(DEVICE, DTYPE_ACC).contiguous())
    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M); return Q

@torch.no_grad()
def svd_init_from_mean(Wmean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(Wmean, full_matrices=False)
    return U.to(DTYPE_ACC).contiguous(), Vh.t().to(DTYPE_ACC).contiguous()

def schedule(step: int, warmup: int, total: int) -> float:
    if step <= warmup: return 0.0
    return min(1.0, (step - warmup) / max(1, total - warmup))

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    U_S, V_S = U[:, S], V[:, S]
    return torch.matmul(U_S.t().unsqueeze(0), Ws_batch @ V_S)

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return (Xs - torch.diag_embed(D)).abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    return torch.diagonal(Xs, dim1=1, dim2=2).abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    Eb, s, _ = Xs.shape; b = int(block)
    if b <= 0: return torch.zeros((), device=Xs.device)
    nb = s // b
    if nb <= 0: return torch.zeros((), device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0,1,3,2,4).contiguous()
    Eblk = (Xb * Xb).sum(dim=(3,4))
    P = Eblk.mean(0)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

@torch.no_grad()
def make_guidance_mask_from_Xs(Xs: torch.Tensor, block: int, target: float, max_blocks: int) -> Tuple[torch.Tensor, float, int]:
    Eb, s, _ = Xs.shape; b = int(block)
    if b <= 0: return torch.ones(s,s,device=Xs.device), 1.0, 0
    nb = s // b
    if nb <= 0: return torch.ones(s,s,device=Xs.device), 1.0, 0
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0,1,3,2,4).contiguous()
    Eg = (Xb * Xb).sum(dim=(3,4)).mean(0)
    tot = (X * X).sum().item() / max(1, Eb)
    flat = Eg.reshape(-1); order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], 0)
    frac = csum / max(tot, 1e-12)
    need = (frac >= target).nonzero(as_tuple=False)[0].item() + 1 if (frac >= target).any() else flat.numel()
    K = min(need, max_blocks, flat.numel())
    mask = torch.zeros(s2, s2, device=Xs.device)
    for idx in order[:K].tolist():
        bi, bj = idx // nb, idx % nb
        mask[bi*b:(bi+1)*b, bj*b:(bj+1)*b] = 1.0
    if s2 < s:
        full = torch.zeros(s, s, device=Xs.device); full[:s2, :s2] = mask; mask = full
    ef = float(frac[K-1].item()) if K > 0 else 0.0
    return mask, ef, K

# -----------------------------------------------------------------------------
# Block energy & selection
# -----------------------------------------------------------------------------
@torch.no_grad()
def block_energy_grid(X: torch.Tensor, b: int) -> Tuple[torch.Tensor, float, int]:
    n = X.shape[0]; nb = (n + b - 1) // b
    if n % b != 0:
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X; X = Xp
    Xb = X.view(nb, b, nb, b).permute(0,2,1,3).contiguous()
    Eg = (Xb * Xb).sum(dim=(2,3))
    tot = (X * X).sum().item()
    return Eg, tot, nb

@torch.no_grad()
def pick_blocks_until_target(Eg: torch.Tensor, tot_energy: float, target: float, max_blocks: int,
                             exclude: Optional[Set[Tuple[int,int]]]=None) -> Tuple[List[Tuple[int,int]], float]:
    nb = Eg.shape[0]; flat = Eg.reshape(-1); order = torch.argsort(flat, descending=True)
    picked, eacc = [], 0.0
    exclude = exclude or set()
    for idx in order.tolist():
        if len(picked) >= max_blocks: break
        e = flat[idx].item()
        if e <= 1e-18: break
        bi, bj = idx // nb, idx % nb
        if (bi, bj) in exclude: continue
        picked.append((bi, bj)); eacc += e
        if eacc / max(tot_energy, 1e-12) >= target: break
    return picked, eacc / max(tot_energy, 1e-12)

@torch.no_grad()
def gather_block(X: torch.Tensor, i0: int, j0: int, b: int) -> torch.Tensor:
    n = X.shape[0]; i1, j1 = min(n, i0+b), min(n, j0+b)
    return X[i0:i1, j0:j1].contiguous()

# -----------------------------------------------------------------------------
# Low-rank (randomized SVD)
# -----------------------------------------------------------------------------
@torch.no_grad()
def rand_svd_vectors(A: torch.Tensor, r: int, n_iter: int=2) -> Tuple[torch.Tensor, torch.Tensor]:
    n = A.shape[0]; r = min(r, n)
    g = torch.Generator(device="cpu").manual_seed(SEED+777)
    Omega = torch.randn(n, r, generator=g, dtype=DTYPE_ACC, device=A.device)
    Y = A @ Omega
    for _ in range(n_iter): Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    Uhat, _, Vh = torch.linalg.svd(B, full_matrices=False)
    return (Q @ Uhat[:, :r]).contiguous(), Vh.t()[:, :r].contiguous()

# -----------------------------------------------------------------------------
# Payload packing (ragged blocks)
# -----------------------------------------------------------------------------
def _block_store_dtype(qmode: str) -> np.dtype:
    return np.float32 if qmode == "none" else np.float16

def pack_blocks_ragged(blocks_per_item: List[List[Tuple[int,int,torch.Tensor]]], qmode: str) -> Dict[str, np.ndarray]:
    val_dtype = _block_store_dtype(qmode)
    M = len(blocks_per_item)
    item_ptr = [0]
    blk_i0, blk_j0, blk_h, blk_w = [], [], [], []
    blk_ptr = [0]
    vals, vals_i8, scales = [], [], []
    for m in range(M):
        for (i0, j0, B) in blocks_per_item[m]:
            h, w = B.shape
            blk_i0.append(i0); blk_j0.append(j0); blk_h.append(h); blk_w.append(w)
            if qmode == "int8":
                x = B.cpu().float(); maxabs = x.abs().max().item()
                if maxabs < 1e-12: q = np.zeros(x.numel(), dtype=np.int8); sc = np.float16(1.0)
                else:
                    scale = maxabs / 127.0
                    q = torch.clamp(torch.round(x/scale), -127, 127).to(torch.int8).numpy()
                    sc = np.float16(scale)
                vals_i8.append(q.reshape(-1)); scales.append(sc)
                blk_ptr.append(blk_ptr[-1] + q.size)
            else:
                v = B.cpu().float().numpy().astype(val_dtype).reshape(-1)
                vals.append(v); blk_ptr.append(blk_ptr[-1] + v.size)
        item_ptr.append(len(blk_i0))

    out = {
        "item_ptr": np.array(item_ptr, dtype=np.int32),
        "blk_i0": np.array(blk_i0, dtype=np.int16),
        "blk_j0": np.array(blk_j0, dtype=np.int16),
        "blk_h": np.array(blk_h, dtype=np.int16),
        "blk_w": np.array(blk_w, dtype=np.int16),
        "blk_ptr": np.array(blk_ptr, dtype=np.int64)
    }
    if qmode == "int8":
        out["blk_q"] = np.concatenate(vals_i8).astype(np.int8) if vals_i8 else np.zeros((0,), dtype=np.int8)
        out["blk_scale"] = np.array(scales, dtype=np.float16)
    else:
        out["blk_val"] = np.concatenate(vals) if vals else np.zeros((0,), dtype=val_dtype)
    return out

def unpack_blocks_ragged(pack: Dict[str, np.ndarray], qmode: str, device: torch.device) -> List[List[Tuple[int,int,torch.Tensor]]]:
    item_ptr = pack["item_ptr"]
    blk_i0 = pack["blk_i0"]; blk_j0 = pack["blk_j0"]; blk_h = pack["blk_h"]; blk_w = pack["blk_w"]
    blk_ptr = pack["blk_ptr"]
    if qmode == "int8":
        blk_q = pack["blk_q"]; blk_scale = pack["blk_scale"]; blk_val = None
    else:
        blk_val = pack["blk_val"]; blk_q = None; blk_scale = None
    M = item_ptr.shape[0] - 1
    out = []
    for m in range(M):
        b0, b1 = item_ptr[m], item_ptr[m+1]
        lst = []
        for bi in range(b0, b1):
            i0, j0 = int(blk_i0[bi]), int(blk_j0[bi])
            h, w = int(blk_h[bi]), int(blk_w[bi])
            v0, v1 = blk_ptr[bi], blk_ptr[bi+1]
            if qmode == "int8":
                q = blk_q[v0:v1].astype(np.float32); sc = float(blk_scale[bi])
                B = torch.from_numpy((q * sc).reshape(h, w)).to(device, DTYPE_ACC)
            else:
                B = torch.from_numpy(blk_val[v0:v1].astype(np.float32).reshape(h, w)).to(device, DTYPE_ACC)
            lst.append((i0, j0, B))
        out.append(lst)
    return out

# -----------------------------------------------------------------------------
# Payload runtime
# -----------------------------------------------------------------------------
class PayloadRuntime:
    def __init__(self):
        self.meta = {}
        self.expert_ids = []
        self.scales: Optional[torch.Tensor] = None
        self.cluster_of_pos: Optional[torch.Tensor] = None
        self.U: List[torch.Tensor] = []
        self.V: List[torch.Tensor] = []
        self.DL: List[torch.Tensor] = []
        self.DR: List[torch.Tensor] = []
        self.gam: Optional[torch.Tensor] = None
        self.Cfull: Optional[torch.Tensor] = None
        self.core_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.res_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.qmode = "none"
        self.res_coef = "diag"

    @torch.no_grad()
    def apply_expert(self, x: torch.Tensor, pos: int) -> torch.Tensor:
        c = int(self.cluster_of_pos[pos].item())
        U, V = self.U[c], self.V[c]
        DL, DR = self.DL[c], self.DR[c]
        z = x @ U
        u = torch.zeros_like(z)
        for (i0, j0, B) in self.core_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B
        if self.res_coef == "diag":
            g = self.gam[pos]
            u += ((z @ DL) * g.view(1,-1)) @ DR.t()
        else:
            C = self.Cfull[pos]
            u += (z @ DL) @ C @ DR.t()
        for (i0, j0, B) in self.res_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B
        y = u @ V.t()
        if self.scales is not None:
            y = y * self.scales[pos]
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        for a, pos in zip(gates.tolist(), routed):
            y += a * self.apply_expert(x, int(pos))
        return y

def load_payload_runtime(path: str, device: torch.device) -> PayloadRuntime:
    z = load_npz(path)
    rt = PayloadRuntime()
    rt.meta = _decode_meta(z["meta"])
    rt.qmode = rt.meta.get("qmode", "none")
    rt.res_coef = rt.meta.get("res_coef", "diag")
    rt.expert_ids = [int(x) for x in z["expert_ids"]]
    rt.scales = torch.from_numpy(z["scales"]).to(device, DTYPE_ACC)
    rt.cluster_of_pos = torch.from_numpy(z["cluster_of_pos"]).to(device, torch.int64)
    M = z["n_clusters"][0]
    for m in range(M):
        rt.U.append(torch.from_numpy(z[f"U_{m}"]).to(device, DTYPE_ACC))
        rt.V.append(torch.from_numpy(z[f"V_{m}"]).to(device, DTYPE_ACC))
        rt.DL.append(torch.from_numpy(z[f"DL_{m}"]).to(device, DTYPE_ACC))
        rt.DR.append(torch.from_numpy(z[f"DR_{m}"]).to(device, DTYPE_ACC))
    if rt.res_coef == "diag":
        rt.gam = torch.from_numpy(z["gam"]).to(device, DTYPE_ACC)
    else:
        rt.Cfull = torch.from_numpy(z["Cfull"]).to(device, DTYPE_ACC)
    core_pack = {k[5:]: z[k] for k in z if k.startswith("core_")}
    res_pack  = {k[4:]: z[k] for k in z if k.startswith("res_")}
    rt.core_blocks = unpack_blocks_ragged(core_pack, rt.qmode, device)
    rt.res_blocks  = unpack_blocks_ragged(res_pack, rt.qmode, device)
    return rt

# -----------------------------------------------------------------------------
# Build payload for one cluster
# -----------------------------------------------------------------------------
@torch.no_grad()
def frob_rel_err(A, B): return (torch.linalg.norm(A-B) / torch.linalg.norm(B).clamp_min(1e-12)).item()

@torch.no_grad()
def build_payload_for_cluster(Ws_norm: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor) -> Dict:
    n = Ws_norm.shape[-1]
    X_list = [(U.t() @ Ws_norm[pos] @ V).contiguous() for pos in idx]
    b = cfg.CORE_BLOCK

    core_per = []
    core_ef = []
    for X in X_list:
        Eg, te, nb = block_energy_grid(X, b)
        picks, eff = pick_blocks_until_target(Eg, te, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*b, bj*b
            blocks.append((i0, j0, gather_block(X, i0, j0, b)))
        core_per.append(blocks); core_ef.append(eff)

    R_list = []
    for X, cb in zip(X_list, core_per):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb: h,w = Bc.shape; Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    Rmean = torch.stack(R_list).mean(0)
    r = min(cfg.RES_RANK, n)
    DL, DR = rand_svd_vectors(Rmean, r, n_iter=2)

    coef_list, res_per = [], []
    bb = cfg.RES_BSIZE
    for j, Rm in enumerate(R_list):
        if cfg.RES_COEF == "diag":
            g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()
            coef_list.append(g)
            R2 = (Rm - (DL * g.view(1,-1)) @ DR.t()).contiguous()
        else:
            C = (DL.t() @ Rm @ DR).contiguous()
            coef_list.append(C)
            R2 = (Rm - (DL @ C @ DR.t())).contiguous()

        Eg2, te2, nb2 = block_energy_grid(R2, bb)
        exclude = {(i0//bb, j0//bb) for (i0,j0,_) in core_per[j]}
        picks, _ = pick_blocks_until_target(Eg2, te2, cfg.RES_TARGET, cfg.RES_MAX_BLOCKS, exclude=exclude)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*bb, bj*bb
            blocks.append((i0, j0, gather_block(R2, i0, j0, bb)))
        res_per.append(blocks)

    if cfg.REFINE_ENABLE:
        rb = cfg.REFINE_BSIZE
        for j in range(len(idx)):
            X = X_list[j]
            def reconstruct():
                Xc = torch.zeros_like(X)
                for (i0,j0,Bc) in core_per[j]: h,w=Bc.shape; Xc[i0:i0+h, j0:j0+w] = Bc
                if cfg.RES_COEF == "diag":
                    g = coef_list[j]; Xlr = (DL * g.view(1,-1)) @ DR.t()
                else:
                    C = coef_list[j]; Xlr = DL @ C @ DR.t()
                Xr = torch.zeros_like(X)
                for (i0,j0,Bb) in res_per[j]: h,w=Bb.shape; Xr[i0:i0+h, j0:j0+w] += Bb
                return Xc + Xlr + Xr
            Xhat = reconstruct()
            err = frob_rel_err(Xhat, X)
            added = 0
            core_pos = {(i0,j0) for (i0,j0,_) in core_per[j]}
            res_pos = {(i0,j0) for (i0,j0,_) in res_per[j]}
            while err > cfg.REFINE_ERR_TARGET and added < cfg.REFINE_MAX_EXTRA:
                Rerr = (X - Xhat).contiguous()
                Eg, te, nb = block_energy_grid(Rerr, rb)
                flat = Eg.reshape(-1)
                if flat.max().item() <= 1e-18: break
                order = torch.argsort(flat, descending=True)
                found = False
                for idx_ in order.tolist():
                    bi, bj = idx_ // nb, idx_ % nb
                    i0, j0 = bi*rb, bj*rb
                    if (i0, j0) in core_pos or (i0, j0) in res_pos: continue
                    Bb = gather_block(Rerr, i0, j0, rb)
                    res_per[j].append((i0, j0, Bb)); res_pos.add((i0, j0))
                    added += 1; found = True; break
                if not found: break
                if added % cfg.REFINE_RECHECK_EVERY == 0:
                    Xhat = reconstruct(); err = frob_rel_err(Xhat, X)
            Xhat = reconstruct(); err = frob_rel_err(Xhat, X)

    return {
        "core_blocks": core_per, "core_energy": core_ef,
        "DL": DL, "DR": DR, "coef_list": coef_list, "res_blocks": res_per
    }

# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------
@torch.no_grad()
def eval_payload(rt: PayloadRuntime, Ws_norm: torch.Tensor, Sc: torch.Tensor):
    E, n, _ = Ws_norm.shape
    errs = []
    for pos in range(E):
        x = torch.randn(8, n, dtype=DTYPE_ACC, device=DEVICE)
        y_hat = rt.apply_expert(x, pos)
        y_ref = x @ (Ws_norm[pos] * Sc[pos])
        errs.append((torch.linalg.norm(y_hat - y_ref) / torch.linalg.norm(y_ref).clamp_min(1e-12)).item())
    log(f"[eval] per-expert rel-error mean={np.mean(errs):.6f} p95={np.percentile(errs,95):.6f} max={np.max(errs):.6f}")

    mix = []
    for _ in range(cfg.EVAL_TRIALS):
        x = torch.randn(cfg.EVAL_BATCH, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), min(cfg.ROUTED_K, E))
        gates = torch.rand(len(routed), device=DEVICE); gates /= gates.sum()
        y_hat = rt.apply_mixture(x, routed, gates)
        Wsum = sum(gates[i].item() * (Ws_norm[pos] * Sc[pos]) for i, pos in enumerate(routed))
        y_ref = x @ Wsum
        mix.append((torch.linalg.norm(y_hat - y_ref) / torch.linalg.norm(y_ref).clamp_min(1e-12)).item())
    log(f"[eval] routed rel-error mean={np.mean(mix):.6f} ± {np.std(mix):.6f}")

# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def banner():
    log("="*60)
    log("EBC-LLM Compression Pipeline for DeepSeek-V2-Lite")
    log(f"Time: {now()}  Device: {DEVICE}")
    log(f"MODEL_DIR: {cfg.MODEL_DIR}  OUTPUT_DIR: {cfg.OUTPUT_DIR}")
    log(f"Layer: {cfg.LAYER}  Experts to compress: {cfg.MAX_EXPERTS}")
    log(f"CALIB: {cfg.CALIB_PATH or '(none)'}  ROUTER: {cfg.ROUTER_PATH or '(none)'}")
    log(f"Ridge damp: {cfg.RIDGE_DAMP}  Normalize W: {cfg.NORMALIZE_W}")
    log(f"Basis: {cfg.BASIS_MODE}  Train steps: {cfg.TRAIN_STEPS}  lr: {cfg.TRAIN_LR}")
    log(f"Core: {cfg.CORE_MODE} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max={cfg.CORE_MAX_BLOCKS}")
    log(f"Residual: rank={cfg.RES_RANK} coef={cfg.RES_COEF} blocks={cfg.RES_MAX_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"Refine: {cfg.REFINE_ENABLE} target={cfg.REFINE_ERR_TARGET} max_extra={cfg.REFINE_MAX_EXTRA}")
    log("="*60)

def main():
    banner()
    expert_ids, Ws_norm, Sc = load_or_build_Ws()
    E, n, _ = Ws_norm.shape
    log(f"[Ws] shape={Ws_norm.shape}")

    # Clustering
    Xfeat = random_proj_features(Ws_norm, cfg.CLUSTER_FEAT_D)
    M0 = max(2, min(cfg.M0 if cfg.M0>0 else int(round(2*math.sqrt(E))), E))
    labels = kmeans_torch(Xfeat, M0, cfg.CLUSTER_ITERS, cfg.CLUSTER_RESTARTS)
    labels = merge_small_clusters(Xfeat, labels, cfg.CLUSTER_MIN_SIZE)
    labels = hierarchical_split(Xfeat, labels, cfg.CLUSTER_MAX_SIZE, min(cfg.M_MAX, E), cfg.SPLIT_ITERS)
    labels = merge_small_clusters(Xfeat, labels, cfg.CLUSTER_MIN_SIZE)
    labels = relabel_contiguous(labels)
    M = labels.max().item() + 1
    clusters = [torch.nonzero(labels==m, as_tuple=False).flatten().tolist() for m in range(M)]
    clusters = [c for c in clusters if c]
    log(f"[cluster] M={len(clusters)} sizes={[len(c) for c in clusters]}")
    cluster_of_pos = [0]*E
    for m, idx in enumerate(clusters):
        for pos in idx: cluster_of_pos[pos] = m

    # Init and train bases
    U_par, V_par = [], []
    for idx in clusters:
        Wm = Ws_norm[idx].mean(0)
        U0, V0 = svd_init_from_mean(Wm)
        U_par.append(OrthoParam(U0)); V_par.append(OrthoParam(V0))

    if cfg.TRAIN_STEPS > 0 and cfg.BASIS_MODE == "dense_train":
        params = [p.M for p in U_par] + [p.M for p in V_par]
        opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)
        guidance_masks, guidance_stats = {}, {}
        t0 = time.perf_counter()
        for step in range(1, cfg.TRAIN_STEPS+1):
            S = torch.randperm(n)[:cfg.SUBM].to(DEVICE)
            if cfg.TRAIN_LAM_GUIDE > 0 and (step==1 or step%cfg.TRAIN_GUIDE_EVERY==0):
                with torch.no_grad():
                    guidance_masks.clear(); guidance_stats.clear()
                    for m, idx in enumerate(clusters):
                        if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
                        Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                        pick = idx if cfg.BATCH_E>=len(idx) else [idx[i] for i in torch.randperm(len(idx))[:cfg.BATCH_E].tolist()]
                        Xs_ng = slice_X_batch(Ws_norm[pick], Uo, Vo, S).detach()
                        mask, ef, kblk = make_guidance_mask_from_Xs(Xs_ng, cfg.CORE_BLOCK, cfg.TRAIN_GUIDE_TARGET, cfg.TRAIN_GUIDE_MAX_BLOCKS)
                        guidance_masks[m] = mask; guidance_stats[m] = (ef, kblk)

            lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
            lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
            lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp
            L_total, n_terms = None, 0
            for m, idx in enumerate(clusters):
                if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
                Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                pick = idx if cfg.BATCH_E>=len(idx) else [idx[i] for i in torch.randperm(len(idx))[:cfg.BATCH_E].tolist()]
                Xs = slice_X_batch(Ws_norm[pick], Uo, Vo, S)
                off, diag = offdiag_abs_mean(Xs), diag_abs_mean(Xs).clamp_min(1e-6)
                base = torch.log(off+1e-6) - torch.log(diag) if cfg.TRAIN_OBJ=="logratio" else off/diag
                if lam_block > 0: base += lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)
                if lam_guide > 0 and m in guidance_masks:
                    Mmask = guidance_masks[m]
                    Etot = (Xs*Xs).mean().clamp_min(1e-12)
                    Eout = ((Xs*(1-Mmask))**2).mean()
                    base += lam_guide * (Eout/Etot)
                L_total = base if L_total is None else L_total + base
                n_terms += 1
            if L_total is None: break
            L_total = L_total / n_terms
            opt.zero_grad(); L_total.backward()
            if cfg.GRAD_CLIP > 0: torch.nn.utils.clip_grad_norm_(params, cfg.GRAD_CLIP)
            opt.step()
            if step % cfg.REORTHO_EVERY == 0 or step == cfg.TRAIN_STEPS:
                with torch.no_grad():
                    for p in U_par: p.M.copy_(p.orthogonal())
                    for p in V_par: p.M.copy_(p.orthogonal())
            if step % cfg.REPORT_EVERY == 0 or step == 1:
                t1 = time.perf_counter()
                gstr = "" if not guidance_stats else f" guide≈{np.mean([v[0] for v in guidance_stats.values()]):.3f}"
                log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={L_total.item():.4f} {gstr} (+{t1-t0:.1f}s)")
                t0 = t1

    # Freeze bases
    U_list = [p.orthogonal().detach() for p in U_par]
    V_list = [p.orthogonal().detach() for p in V_par]

    # Build payloads
    log("[build] payloads ...")
    core_all = [[] for _ in range(E)]
    res_all  = [[] for _ in range(E)]
    DL_list, DR_list = [], []
    rmax = min(cfg.RES_RANK, n)
    gam = torch.zeros((E, rmax), dtype=DTYPE_ACC, device=DEVICE) if cfg.RES_COEF=="diag" else None
    Cfull = torch.zeros((E, rmax, rmax), dtype=DTYPE_ACC, device=DEVICE) if cfg.RES_COEF=="full" else None

    for m, idx in enumerate(clusters):
        U, V = U_list[m], V_list[m]
        P = build_payload_for_cluster(Ws_norm, idx, U, V)
        for j, pos in enumerate(idx):
            core_all[pos] = P["core_blocks"][j]
            res_all[pos] = P["res_blocks"][j]
            if cfg.RES_COEF == "diag":
                g = P["coef_list"][j]; gam[pos, :g.numel()] = g
            else:
                C = P["coef_list"][j]; Cfull[pos, :C.shape[0], :C.shape[1]] = C
        DL_list.append(P["DL"]); DR_list.append(P["DR"])
        log(f"  cluster{m}: E={len(idx)} core_blocks≈{np.mean([len(c) for c in P['core_blocks']]):.1f} r={P['DL'].shape[1]}")

    # Save payload
    out_path = os.path.join(cfg.OUTPUT_DIR, f"ebc_payload_layer{cfg.LAYER}_E{E}_q{cfg.QMODE}.npz")
    store_dtype = np.float16 if cfg.BASIS_STORE_DTYPE=="float16" else np.float32
    arrays = {
        "meta": _encode_meta(ws_meta(expert_ids) | {"time": now(), "qmode": cfg.QMODE, "res_coef": cfg.RES_COEF}),
        "expert_ids": np.array(expert_ids, dtype=np.int32),
        "scales": Sc.cpu().numpy().astype(np.float32),
        "cluster_of_pos": np.array(cluster_of_pos, dtype=np.int16),
        "n_clusters": np.array([len(clusters)], dtype=np.int32),
    }
    for m in range(len(clusters)):
        arrays[f"U_{m}"] = U_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"V_{m}"] = V_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DL_{m}"] = DL_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DR_{m}"] = DR_list[m].cpu().numpy().astype(store_dtype)
    if cfg.RES_COEF == "diag":
        arrays["gam"] = gam.cpu().numpy().astype(store_dtype)
    else:
        arrays["Cfull"] = Cfull.cpu().numpy().astype(store_dtype)

    core_pack = pack_blocks_ragged(core_all, cfg.QMODE)
    res_pack  = pack_blocks_ragged(res_all, cfg.QMODE)
    for k, v in core_pack.items(): arrays["core_"+k] = v
    for k, v in res_pack.items(): arrays["res_"+k] = v

    save_npz_compressed(out_path, arrays)
    log(f"[save] payload -> {out_path} size={os.path.getsize(out_path)/1e6:.2f} MB")

    # Evaluate
    rt = load_payload_runtime(out_path, DEVICE)
    eval_payload(rt, Ws_norm, Sc)
    log("✅ Done.")

if __name__ == "__main__":
    main()

✅ flash_attn stub installed (CPU mode).
EBC-LLM Compression Pipeline for DeepSeek-V2-Lite
Time: 2026-04-22 03:38:52  Device: cpu
MODEL_DIR: /data/downloaded_models/DeepSeek-V2-Lite  OUTPUT_DIR: /home/daniyar/moe_ws_outputs_deepseek
Layer: 1  Experts to compress: 16
CALIB: (none)  ROUTER: (none)
Ridge damp: 0.001  Normalize W: True
Basis: dense_train  Train steps: 24  lr: 0.05
Core: blocktopk_perexpert block=64 target=0.85 max=256
Residual: rank=512 coef=diag blocks=4096 bsize=64
Refine: True target=0.03 max_extra=4096
[search] Checking layer 1 for experts...
[found] layer=1 total experts=64
[load] reading tensors from shards ...
[shape] H=2048 d_ff=1408
[capture] capturing via transformers... (this will take several minutes on CPU)


Loading weights:   0%|          | 0/5291 [00:00<?, ?it/s]

[capture] Located MLP module: DeepseekV2MoE
[capture] iter 4/32 nX=4096 nP=36
[capture] wrote X -> /home/daniyar/moe_ws_outputs_deepseek/calib_layer1_X.npz shape=(4096, 2048)
[capture] wrote P -> /home/daniyar/moe_ws_outputs_deepseek/router_layer1_P.npz shape=(36, 16)
[calib] X: torch.Size([36, 2048])


Build Ws (ridge):   0%|          | 0/16 [00:00<?, ?it/s]

[Ws] shape=torch.Size([16, 2048, 2048])
[cluster] M=5 sizes=[2, 3, 4, 3, 4]
[train] step   1/24 loss=-3.4880  guide≈0.847 (+6.3s)
[train] step   4/24 loss=-0.9290  guide≈0.834 (+20.7s)
[train] step   8/24 loss=-1.0283  guide≈0.819 (+22.2s)
[train] step  12/24 loss=-0.7678  guide≈0.821 (+21.4s)
[train] step  16/24 loss=-0.6651  guide≈0.831 (+21.5s)
[train] step  20/24 loss=1.6174  guide≈0.820 (+21.4s)
[train] step  24/24 loss=0.5755  guide≈0.813 (+21.9s)
[build] payloads ...
  cluster0: E=2 core_blocks≈84.0 r=512
  cluster1: E=3 core_blocks≈104.7 r=512
  cluster2: E=4 core_blocks≈131.2 r=512
  cluster3: E=3 core_blocks≈106.7 r=512
  cluster4: E=4 core_blocks≈108.5 r=512
[save] payload -> /home/daniyar/moe_ws_outputs_deepseek/ebc_payload_layer1_E16_qnone.npz size=343.99 MB
[eval] per-expert rel-error mean=0.027009 p95=0.034629 max=0.035957
[eval] routed rel-error mean=0.029254 ± 0.003711
✅ Done.


In [22]:
import os
import json
import re
import struct
import numpy as np
from typing import Dict, List, Optional

def calc_expert_size(model_dir: str, layer: int = 1, payload_path: Optional[str] = None):
    """Calculate original FP16 size of expert weights and optionally compression ratio."""
    
    # 1. Read weight index
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        print(f"❌ Index not found: {idx_path}")
        return
    with open(idx_path, "r") as f:
        weight_map = json.load(f).get("weight_map", {})
    
    # 2. Find expert keys (with fallback layer search)
    def find_expert_keys(layer_idx: int) -> List[str]:
        patterns = [
            rf"^model\.layers\.{layer_idx}\.mlp\.experts\.\d+\.(gate_proj|up_proj|down_proj)\.weight$",
            rf"^model\.layers\.{layer_idx}\.block_sparse_moe\.experts\.\d+\.(w[123]|gate_proj|up_proj|down_proj)\.weight$",
            rf"^model\.layers\.{layer_idx}\.mlp\.shared_experts\.\d+\.(gate_proj|up_proj|down_proj)\.weight$",
            rf"^model\.layers\.{layer_idx}\.mlp\.(gate_proj|up_proj|down_proj)\.weight$",  # monolithic
        ]
        keys = []
        for k in weight_map:
            for pat in patterns:
                if re.match(pat, k):
                    keys.append(k)
                    break
        return keys
    
    keys = find_expert_keys(layer)
    used_layer = layer
    if not keys:
        print(f"No expert keys at layer {layer}, searching layers 0-5...")
        for l in range(6):
            keys = find_expert_keys(l)
            if keys:
                used_layer = l
                break
    if not keys:
        print("❌ No expert keys found in layers 0-5.")
        return
    
    # 3. Sum element counts from safetensors headers
    total_elements = 0
    print(f"\nExpert weights in layer {used_layer}:")
    for key in sorted(keys):
        shard_file = weight_map.get(key)
        if not shard_file:
            continue
        shard_path = os.path.join(model_dir, shard_file)
        if not os.path.isfile(shard_path):
            continue
        with open(shard_path, "rb") as f:
            header_len = struct.unpack("<Q", f.read(8))[0]
            header = json.loads(f.read(header_len).decode("utf-8"))
            if key in header:
                shape = header[key]["shape"]
                numel = 1
                for dim in shape:
                    numel *= dim
                total_elements += numel
                print(f"  {os.path.basename(key)} : {numel:,} elements")
    
    # 4. Report sizes
    bytes_fp16 = total_elements * 2
    size_mb = bytes_fp16 / (1024 ** 2)
    print(f"\n📦 Total elements: {total_elements:,}")
    print(f"💾 Original size (FP16): {size_mb:.2f} MB")
    
    if payload_path and os.path.isfile(payload_path):
        payload_mb = os.path.getsize(payload_path) / (1024 ** 2)
        ratio = size_mb / payload_mb
        print(f"📁 Payload size: {payload_mb:.2f} MB")
        print(f"📊 Compression ratio: {ratio:.2f}x")

In [23]:
# Example for DeepSeek-V2-Lite
calc_expert_size(
    model_dir="/data/downloaded_models/DeepSeek-V2-Lite",
    layer=1,
    payload_path="/home/daniyar/moe_ws_outputs_deepseek/ebc_payload_layer1_E16_qnone.npz"
)

# For Phi-3.5-MoE (monolithic)
calc_expert_size(
    model_dir="/data/downloaded_models/Phi-3.5-MoE-instruct",
    layer=0,
    payload_path="/home/daniyar/moe_ws_outputs_phi/ebc_payload_layer0_E16_qnone.npz"
)

# For Qwen1.5-MoE
calc_expert_size(
    model_dir="/data/downloaded_models/Qwen1.5-MoE-A2.7B",
    layer=0,
    payload_path="/home/daniyar/moe_ws_outputs_qwen/ebc_payload_layer0_E4_qnone.npz"
)


Expert weights in layer 1:
  model.layers.1.mlp.experts.0.down_proj.weight : 2,883,584 elements
  model.layers.1.mlp.experts.0.gate_proj.weight : 2,883,584 elements
  model.layers.1.mlp.experts.0.up_proj.weight : 2,883,584 elements
  model.layers.1.mlp.experts.1.down_proj.weight : 2,883,584 elements
  model.layers.1.mlp.experts.1.gate_proj.weight : 2,883,584 elements
  model.layers.1.mlp.experts.1.up_proj.weight : 2,883,584 elements
  model.layers.1.mlp.experts.10.down_proj.weight : 2,883,584 elements
  model.layers.1.mlp.experts.10.gate_proj.weight : 2,883,584 elements
  model.layers.1.mlp.experts.10.up_proj.weight : 2,883,584 elements
  model.layers.1.mlp.experts.11.down_proj.weight : 2,883,584 elements
  model.layers.1.mlp.experts.11.gate_proj.weight : 2,883,584 elements
  model.layers.1.mlp.experts.11.up_proj.weight : 2,883,584 elements
  model.layers.1.mlp.experts.12.down_proj.weight : 2,883,584 elements
  model.layers.1.mlp.experts.12.gate_proj.weight : 2,883,584 elements
  mode

In [24]:
import os, json, re, struct
import numpy as np
from typing import Dict, List, Optional

def calc_expert_size(model_dir: str, layer: int = None, payload_path: Optional[str] = None):
    """
    Calculate original FP16 size of MoE expert weights and compression ratio.
    Supports: DeepSeek-V2-Lite, Phi-3.5-MoE, Qwen1.5-MoE, Mixtral-8x7B.
    If layer is None, searches layers 0–5 for the first MoE layer.
    """
    # ---------- 1. Load weight map ----------
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        print(f"❌ Index not found: {idx_path}")
        return
    with open(idx_path, "r") as f:
        weight_map = json.load(f).get("weight_map", {})

    # ---------- 2. Model‑specific expert patterns ----------
    # We'll define a list of (pattern, description) for each architecture.
    # Patterns use regex groups to capture the expert ID if present.
    all_patterns = [
        # DeepSeek-V2 (routed experts)
        (re.compile(r"^model\.layers\.(\d+)\.mlp\.experts\.(\d+)\.(gate_proj|up_proj|down_proj)\.weight$"), "deepseek_routed"),
        # DeepSeek-V2 shared experts
        (re.compile(r"^model\.layers\.(\d+)\.mlp\.shared_experts\.(\d+)\.(gate_proj|up_proj|down_proj)\.weight$"), "deepseek_shared"),
        # Mixtral / Qwen (block_sparse_moe)
        (re.compile(r"^model\.layers\.(\d+)\.block_sparse_moe\.experts\.(\d+)\.(w1|w2|w3|gate_proj|up_proj|down_proj)\.weight$"), "mixtral"),
        # Phi-3.5 monolithic (no expert index)
        (re.compile(r"^model\.layers\.(\d+)\.mlp\.(gate_proj|up_proj|down_proj)\.weight$"), "monolithic"),
        # Generic fallback
        (re.compile(r"^model\.layers\.(\d+)\.mlp\.experts\.(\d+)\.(gate_proj|up_proj|down_proj)\.weight$"), "generic"),
    ]

    # ---------- 3. Search for a layer with weights ----------
    search_layers = range(6) if layer is None else [layer]
    used_layer = None
    expert_keys = []
    model_type = None

    for l in search_layers:
        for pat, ptype in all_patterns:
            matches = []
            for k in weight_map.keys():
                m = pat.match(k)
                if m:
                    layer_idx = int(m.group(1))
                    if layer_idx == l:
                        matches.append(k)
            if matches:
                used_layer = l
                expert_keys = matches
                model_type = ptype
                break
        if used_layer is not None:
            break

    if not expert_keys:
        print("❌ No expert / MLP weights found in layers 0–5.")
        return

    print(f"🔍 Detected model type: {model_type}, layer {used_layer}")

    # ---------- 4. Count elements from safetensors headers ----------
    total_elements = 0
    print("\n📋 Weight tensors:")
    for key in sorted(expert_keys):
        shard_file = weight_map.get(key)
        if not shard_file:
            continue
        shard_path = os.path.join(model_dir, shard_file)
        if not os.path.isfile(shard_path):
            continue
        with open(shard_path, "rb") as f:
            header_len = struct.unpack("<Q", f.read(8))[0]
            header = json.loads(f.read(header_len).decode("utf-8"))
            if key in header:
                shape = header[key]["shape"]
                numel = 1
                for dim in shape:
                    numel *= dim
                total_elements += numel
                print(f"  {os.path.basename(key):50} {str(shape):15} {numel:12,} elements")

    # ---------- 5. Compute sizes ----------
    bytes_fp16 = total_elements * 2
    size_mb = bytes_fp16 / (1024 ** 2)
    print(f"\n📦 Total elements: {total_elements:,}")
    print(f"💾 Original size (FP16): {size_mb:.2f} MB")

    if payload_path and os.path.isfile(payload_path):
        payload_mb = os.path.getsize(payload_path) / (1024 ** 2)
        ratio = size_mb / payload_mb
        print(f"📁 Payload size: {payload_mb:.2f} MB")
        print(f"📊 Compression ratio: {ratio:.2f}x")
    elif payload_path:
        print(f"⚠️ Payload file not found: {payload_path}")

    return total_elements, size_mb

In [25]:
# DeepSeek-V2-Lite
calc_expert_size(
    model_dir="/data/downloaded_models/DeepSeek-V2-Lite",
    layer=1,
    payload_path="/home/daniyar/moe_ws_outputs_deepseek/ebc_payload_layer1_E16_qnone.npz"
)

# Phi-3.5-MoE (monolithic)
calc_expert_size(
    model_dir="/data/downloaded_models/Phi-3.5-MoE-instruct",
    layer=0,
    payload_path="/home/daniyar/moe_ws_outputs_phi/ebc_payload_layer0_E16_qnone.npz"
)

# Qwen1.5-MoE
calc_expert_size(
    model_dir="/data/downloaded_models/Qwen1.5-MoE-A2.7B",
    layer=0,
    payload_path="/home/daniyar/moe_ws_outputs_qwen/ebc_payload_layer0_E4_qnone.npz"
)

# Mixtral-8x7B
calc_expert_size(
    model_dir="/data/downloaded_models/Mixtral-8x7B-v0.1",
    layer=0,
    payload_path="/home/daniyar/moe_ws_outputs_mixtral/ebc_payload_layer0_E8_qnone.npz"
)

🔍 Detected model type: deepseek_routed, layer 1

📋 Weight tensors:
  model.layers.1.mlp.experts.0.down_proj.weight      [2048, 1408]       2,883,584 elements
  model.layers.1.mlp.experts.0.gate_proj.weight      [1408, 2048]       2,883,584 elements
  model.layers.1.mlp.experts.0.up_proj.weight        [1408, 2048]       2,883,584 elements
  model.layers.1.mlp.experts.1.down_proj.weight      [2048, 1408]       2,883,584 elements
  model.layers.1.mlp.experts.1.gate_proj.weight      [1408, 2048]       2,883,584 elements
  model.layers.1.mlp.experts.1.up_proj.weight        [1408, 2048]       2,883,584 elements
  model.layers.1.mlp.experts.10.down_proj.weight     [2048, 1408]       2,883,584 elements
  model.layers.1.mlp.experts.10.gate_proj.weight     [1408, 2048]       2,883,584 elements
  model.layers.1.mlp.experts.10.up_proj.weight       [1408, 2048]       2,883,584 elements
  model.layers.1.mlp.experts.11.down_proj.weight     [2048, 1408]       2,883,584 elements
  model.layers.1.mlp.ex

(1409286144, 2688.0)

In [34]:
#!/usr/bin/env python3
# =============================================================================
# EBC-LLM for Phi-3.5-MoE (Fixed parameters – achieves compression)
# =============================================================================

# #############################################################################
# FLASH_ATTN STUB – MUST BE FIRST
# #############################################################################
import sys
import types
import importlib.machinery

def _install_flash_attn_stub():
    flash_attn = types.ModuleType("flash_attn")
    flash_attn.__version__ = "0.0.0-cpu-stub"
    def _unavailable(*args, **kwargs):
        raise RuntimeError("flash_attn stub called on CPU. Use attn_implementation='eager'.")
    flash_attn.flash_attn_func = _unavailable
    flash_attn.flash_attn_varlen_func = _unavailable
    flash_attn.flash_attn_with_kvcache = _unavailable
    flash_attn.layers = types.ModuleType("flash_attn.layers")
    flash_attn.layers.rotary = types.ModuleType("flash_attn.layers.rotary")
    flash_attn.ops = types.ModuleType("flash_attn.ops")
    flash_attn.ops.triton = types.ModuleType("flash_attn.ops.triton")
    flash_attn.bert_padding = types.ModuleType("flash_attn.bert_padding")
    flash_attn.flash_attn_interface = types.ModuleType("flash_attn.flash_attn_interface")
    import torch
    import torch.nn as nn
    class RotaryEmbedding(nn.Module):
        def __init__(self, dim, base=10000.0, interleaved=False, scale_base=None, device=None):
            super().__init__()
            self.dim = dim
        def forward(self, x, seq_len=None, **kwargs):
            device, dtype = x.device, x.dtype
            seq = seq_len if seq_len else x.shape[-2]
            half = max(1, self.dim // 2)
            cos = torch.ones((seq, half), device=device, dtype=dtype)
            sin = torch.zeros((seq, half), device=device, dtype=dtype)
            return cos, sin
    flash_attn.layers.rotary.RotaryEmbedding = RotaryEmbedding
    flash_attn.layers.rotary.apply_rotary_emb = lambda *a, **k: (_unavailable,)
    flash_attn.bert_padding.index_first_axis = lambda x, *a, **k: x
    flash_attn.bert_padding.pad_input = _unavailable
    flash_attn.bert_padding.unpad_input = _unavailable
    flash_attn.flash_attn_interface.flash_attn_func = _unavailable
    flash_attn.flash_attn_interface.flash_attn_varlen_func = _unavailable
    flash_attn.flash_attn_interface.flash_attn_with_kvcache = _unavailable
    sys.modules["flash_attn"] = flash_attn
    sys.modules["flash_attn.layers"] = flash_attn.layers
    sys.modules["flash_attn.layers.rotary"] = flash_attn.layers.rotary
    sys.modules["flash_attn.ops"] = flash_attn.ops
    sys.modules["flash_attn.ops.triton"] = flash_attn.ops.triton
    sys.modules["flash_attn.bert_padding"] = flash_attn.bert_padding
    sys.modules["flash_attn.flash_attn_interface"] = flash_attn.flash_attn_interface

class FlashAttnImporter:
    def find_spec(self, fullname, path, target=None):
        if fullname == "flash_attn" or fullname.startswith("flash_attn."):
            if "flash_attn" not in sys.modules:
                _install_flash_attn_stub()
            return importlib.machinery.ModuleSpec(fullname, self)
        return None
    def create_module(self, spec): return sys.modules.get(spec.name)
    def exec_module(self, module): pass

sys.meta_path.insert(0, FlashAttnImporter())
print("✅ flash_attn stub installed (CPU mode).", flush=True)

# #############################################################################
# IMPORTS
# #############################################################################
import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs): return x

# -----------------------------------------------------------------------------
# Configuration – Aggressive but accurate
# -----------------------------------------------------------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = "/data/downloaded_models/Phi-3.5-MoE-instruct"
    OUTPUT_DIR: str = "/home/daniyar/moe_ws_outputs_phi_compact"

    LAYER: int = 0
    MAX_EXPERTS: int = 16          # Compress all 16 experts

    CALIB_PATH: str = ""
    ROUTER_PATH: str = ""
    CALIB_SAMPLES: int = 4096
    RIDGE_WEIGHTED: bool = False
    RIDGE_DAMP: float = 1e-3
    NORMALIZE_W: bool = True

    CAPTURE_ENABLE: bool = True
    CAPTURE_FORCE: bool = False
    CAPTURE_ITERS: int = 32
    CAPTURE_BATCH: int = 1
    CAPTURE_MAX_TOKENS: int = 1024
    CAPTURE_TEXT: str = "DeepSeek MoE calibration text. " * 256
    CAPTURE_TEXT_FILE: str = ""
    CAPTURE_KEEP_PAD: bool = True
    HF_TRUST_REMOTE_CODE: bool = True
    HF_LOCAL_FILES_ONLY: bool = True
    HF_AUTO_PIP: bool = False

    BASIS_MODE: str = "dense_train"
    BASIS_STORE_DTYPE: str = "float16"

    M0: int = 0
    M_MAX: int = 16
    CLUSTER_FEAT_D: int = 64
    CLUSTER_ITERS: int = 60
    CLUSTER_RESTARTS: int = 4
    CLUSTER_MIN_SIZE: int = 2
    CLUSTER_MAX_SIZE: int = 4
    SPLIT_ITERS: int = 50

    TRAIN_STEPS: int = 24
    TRAIN_WARMUP: int = 6
    TRAIN_LR: float = 5e-2
    SUBM: int = 256
    BATCH_E: int = 4
    TRAIN_MIN_CLUSTER: int = 2
    REORTHO_EVERY: int = 4
    REPORT_EVERY: int = 4
    GRAD_CLIP: float = 1.0
    TRAIN_OBJ: str = "logratio"
    TRAIN_LAM_BLOCK: float = 0.10
    TRAIN_LAM_GUIDE: float = 1.0
    TRAIN_GUIDE_EVERY: int = 2
    TRAIN_GUIDE_TARGET: float = 0.80
    TRAIN_GUIDE_MAX_BLOCKS: int = 2048

    # AGGRESSIVE COMPRESSION PARAMETERS (tuned for Phi-3.5)
    CORE_MODE: str = "blocktopk_perexpert"
    CORE_BLOCK: int = 64
    CORE_TARGET: float = 0.75      # lower target = fewer blocks
    CORE_MAX_BLOCKS: int = 128     # cap to avoid bloat

    RES_RANK: int = 256            # smaller rank than default 512
    RES_COEF: str = "diag"
    RES_TARGET: float = 0.99
    RES_MAX_BLOCKS: int = 1024
    RES_BSIZE: int = 64

    REFINE_ENABLE: bool = False    # disable refine to save time/memory
    REFINE_ERR_TARGET: float = 0.03
    REFINE_MAX_EXTRA: int = 4096
    REFINE_BSIZE: int = 64
    REFINE_RECHECK_EVERY: int = 32

    QMODE: str = "none"

    EVAL_TRIALS: int = 8
    EVAL_BATCH: int = 2
    ROUTED_K: int = 8

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------------
def log(msg: str): print(msg, flush=True)
def now() -> str: return time.strftime("%Y-%m-%d %H:%M:%S")

def seed_all(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

SEED = 1234
seed_all(SEED)
NTHREADS = 8
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
try: torch.set_num_threads(NTHREADS)
except: pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE_ACC = torch.float32

# -----------------------------------------------------------------------------
# NPZ I/O
# -----------------------------------------------------------------------------
def save_npz_compressed(path: str, arrays: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, **arrays)

def load_npz(path: str) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    return {k: z[k] for k in z.files}

def _encode_meta(meta: dict) -> np.ndarray:
    return np.frombuffer(json.dumps(meta, sort_keys=True).encode("utf-8"), dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try: return json.loads(bytes(arr.tolist()).decode("utf-8"))
    except: return {}

# -----------------------------------------------------------------------------
# Weight loading – per‑expert for Phi-3.5
# -----------------------------------------------------------------------------
def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    # Build the prefix for this expert
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    
    # Diagnostic: print some keys starting with this prefix (first run only)
    if eid == 0:
        sample = [k for k in weight_map if k.startswith(prefix)][:5]
        if sample:
            log(f"[diagnostic] Sample keys for expert 0: {sample}")
        else:
            # Try to find similar keys for layer 0
            alt_sample = [k for k in weight_map if f"layers.{layer}" in k and "mlp" in k][:10]
            log(f"[diagnostic] No keys with prefix '{prefix}'. Other MLP keys for layer {layer}:")
            for k in alt_sample:
                log(f"  {k}")
    
    def pick(suffixes):
        for suf in suffixes:
            k = prefix + suf
            if k in weight_map:
                return k
        return None

    # Try common suffixes (both Phi-3.5 and Mixtral styles)
    gate = pick(["gate_proj.weight", "w1.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    up   = pick(["up_proj.weight", "w3.weight"])

    if gate is None or down is None or up is None:
        # If still missing, search all keys for this expert ID manually
        log(f"[warning] Could not find all tensors for expert {eid}. Searching manually...")
        for k in weight_map:
            if f"experts.{eid}." in k:
                if "gate" in k or "w1" in k:
                    gate = k
                elif "down" in k or "w2" in k:
                    down = k
                elif "up" in k or "w3" in k:
                    up = k
        if gate and down and up:
            log(f"[manual] Found: gate={gate}, down={down}, up={up}")
            return {"up": up, "gate": gate, "down": down}
        return {}
    return {"up": up, "gate": gate, "down": down}

# -----------------------------------------------------------------------------
# Calibration capture (simplified, no mask issues)
# -----------------------------------------------------------------------------
def autodetect_calib_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return cand if os.path.isfile(cand) else None

def autodetect_router_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return cand if os.path.isfile(cand) else None

def load_calib_X(path: str, H: int) -> torch.Tensor:
    z = np.load(path)
    X = torch.from_numpy(z["X"].astype(np.float32))
    if X.ndim != 2 or X.shape[1] != H: raise RuntimeError(f"Bad X shape {X.shape}")
    if X.shape[0] > cfg.CALIB_SAMPLES: X = X[:cfg.CALIB_SAMPLES]
    return X.to(device=DEVICE, dtype=DTYPE_ACC)

def load_router_P(path: str) -> np.ndarray:
    return np.load(path)["P"].astype(np.float32)

def _maybe_autopip():
    if not cfg.HF_AUTO_PIP: return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", "transformers", "sentencepiece", "tokenizers"])

def _patch_transformers_cache_compat():
    try:
        from transformers.cache_utils import DynamicCache
        if not hasattr(DynamicCache, "get_usable_length"):
            DynamicCache.get_usable_length = lambda self, seq_length: int(seq_length)
    except: pass

class _Collector:
    def __init__(self, H, E_total, max_rows):
        self.H = H; self.E_total = E_total; self.max_rows = max_rows
        self.X_chunks, self.P_chunks = [], []; self.nX = self.nP = 0

    def _take(self, flat, need): return flat[:need] if flat.shape[0] > need else flat

    def add_X(self, hs, attn_mask=None):
        if hs is None: return
        if hs.ndim == 2: hs = hs.unsqueeze(0)
        if hs.ndim != 3 or hs.shape[-1] != self.H: return
        flat = hs.detach().to(torch.float32).cpu().reshape(-1, self.H)
        if flat.numel() == 0: return
        need = self.max_rows - self.nX
        if need <= 0: return
        self.X_chunks.append(self._take(flat, need)); self.nX += self.X_chunks[-1].shape[0]

    def add_logits(self, logits, attn_mask=None):
        if logits is None: return
        if logits.ndim == 2: logits = logits.unsqueeze(0)
        if logits.ndim != 3: return
        P = torch.softmax(logits.detach().to(torch.float32), dim=-1)[..., :self.E_total].cpu()
        flat = P.reshape(-1, P.shape[-1])
        if flat.numel() == 0: return
        need = self.max_rows - self.nP
        if need <= 0: return
        self.P_chunks.append(self._take(flat, need)); self.nP += self.P_chunks[-1].shape[0]

def capture_XP_transformers(model_dir, layer_idx, H, E_total, out_x, out_p):
    _maybe_autopip()
    _patch_transformers_cache_compat()

    import transformers.utils.import_utils as iu
    if not hasattr(iu, "is_torch_fx_available"):
        def is_torch_fx_available():
            try: import torch.fx; return True
            except ImportError: return False
        iu.is_torch_fx_available = is_torch_fx_available

    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=cfg.HF_TRUST_REMOTE_CODE, local_files_only=cfg.HF_LOCAL_FILES_ONLY)
    if tok.pad_token is None: tok.pad_token = tok.eos_token or tok.unk_token

    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
        attn_implementation="eager"
    ).to(DEVICE).eval()

    layers = None
    if hasattr(model, "model") and hasattr(model.model, "layers"): layers = model.model.layers
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"): layers = model.transformer.h
    elif hasattr(model, "layers"): layers = model.layers
    if layers is None: raise RuntimeError("Cannot locate layers")
    if layer_idx >= len(layers): raise RuntimeError(f"Layer {layer_idx} out of range")
    layer = layers[layer_idx]

    mlp = getattr(layer, "mlp", None) or getattr(layer, "moe", None) or getattr(layer, "block_sparse_moe", None)
    if mlp is None:
        for name, mod in layer.named_modules():
            if any(x in name.lower() for x in ["mlp", "moe", "expert"]):
                if hasattr(mod, "gate_proj") or hasattr(mod, "w1"):
                    mlp = mod
                    break
    if mlp is None: raise RuntimeError("Could not find MoE MLP module in layer.")
    log(f"[capture] Located MLP module: {mlp.__class__.__name__}")

    router_linear = None
    for name, mod in layer.named_modules():
        if isinstance(mod, nn.Linear) and mod.in_features == H and mod.out_features >= E_total:
            if "router" in name.lower() or "gate" in name.lower():
                router_linear = mod
                break

    coll = _Collector(H, E_total, cfg.CALIB_SAMPLES)

    def mlp_pre_hook(_, inputs):
        coll.add_X(inputs[0])

    h1 = mlp.register_forward_pre_hook(mlp_pre_hook)
    h2 = None
    if router_linear is not None:
        def router_hook(_, __, out):
            o = out[0] if isinstance(out, (tuple, list)) else out
            coll.add_logits(o)
        h2 = router_linear.register_forward_hook(router_hook)

    texts = [cfg.CAPTURE_TEXT]
    if cfg.CAPTURE_TEXT_FILE and os.path.isfile(cfg.CAPTURE_TEXT_FILE):
        with open(cfg.CAPTURE_TEXT_FILE) as f: texts = [ln.strip() for ln in f if ln.strip()]
    tptr = 0
    for it in range(cfg.CAPTURE_ITERS):
        text = texts[tptr % len(texts)]; tptr += 1
        enc = tok(text, return_tensors="pt", truncation=True, max_length=cfg.CAPTURE_MAX_TOKENS, padding="max_length")
        for k in enc:
            if enc[k].ndim == 2 and cfg.CAPTURE_BATCH > 1: enc[k] = enc[k].repeat(cfg.CAPTURE_BATCH, 1)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.inference_mode(): _ = model(**enc, use_cache=False)
        if (it + 1) % 4 == 0: log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS} nX={coll.nX} nP={coll.nP}")
        if coll.nX >= cfg.CALIB_SAMPLES and (not cfg.RIDGE_WEIGHTED or coll.nP >= cfg.CALIB_SAMPLES): break

    h1.remove()
    if h2: h2.remove()

    if coll.nX == 0: raise RuntimeError("Capture collected 0 rows")
    X = torch.cat(coll.X_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz_compressed(out_x, {"X": X})
    log(f"[capture] wrote X -> {out_x} shape={X.shape}")
    p_written = None
    if coll.nP > 0:
        P = torch.cat(coll.P_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
        N = min(P.shape[0], X.shape[0])
        if N < X.shape[0]: X = X[:N]; save_npz_compressed(out_x, {"X": X})
        P = P[:N]; save_npz_compressed(out_p, {"P": P})
        log(f"[capture] wrote P -> {out_p} shape={P.shape}")
        p_written = out_p
    return out_x, p_written

def ensure_calib_router(H: int, E_total: int):
    calib_cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    router_cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    if os.path.isfile(calib_cand) and (not cfg.RIDGE_WEIGHTED or os.path.isfile(router_cand)):
        log("[capture] Calibration cache found. Skipping transformers model loading.")
        cfg.CALIB_PATH = calib_cand
        if os.path.isfile(router_cand): cfg.ROUTER_PATH = router_cand
        return

    if not cfg.CALIB_PATH:
        c = autodetect_calib_path()
        if c: cfg.CALIB_PATH = c; log(f"[calib] auto-found {cfg.CALIB_PATH}")
    if not cfg.ROUTER_PATH:
        r = autodetect_router_path()
        if r: cfg.ROUTER_PATH = r; log(f"[router] auto-found {cfg.ROUTER_PATH}")
    if cfg.CAPTURE_FORCE or (cfg.CAPTURE_ENABLE and (not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH))):
        out_x = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
        out_p = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
        log("[capture] capturing via transformers... (this will take several minutes on CPU)")
        x_path, p_path = capture_XP_transformers(cfg.MODEL_DIR, cfg.LAYER, H, E_total, out_x, out_p)
        cfg.CALIB_PATH = x_path
        if p_path: cfg.ROUTER_PATH = p_path

# -----------------------------------------------------------------------------
# Ridge linearization
# -----------------------------------------------------------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate, W_up, W_down) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    return hid @ W_down.to(DTYPE_ACC).t()

def ws_cache_path(E: int) -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{E}_ridge_ebc.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        script="ebc_llm_compact", model_dir=cfg.MODEL_DIR, layer=cfg.LAYER, expert_ids=eids,
        ridge_damp=cfg.RIDGE_DAMP, ridge_weighted=cfg.RIDGE_WEIGHTED,
        router_path=cfg.ROUTER_PATH or "", calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES, normalize_w=cfg.NORMALIZE_W, seed=SEED, device=str(DEVICE)
    )

@torch.no_grad()
def build_Ws(eids: List[int], wm: Dict[str, str]) -> Tuple[torch.Tensor, torch.Tensor]:
    per_e, need_keys = {}, []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk: raise RuntimeError(f"Expert {eid} missing tensors")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]
    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))
    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = W_up0.shape[0], W_up0.shape[1]
    log(f"[shape] H={H} d_ff={dff}")

    ensure_calib_router(H, len(eids))
    X = load_calib_X(cfg.CALIB_PATH, H)
    log(f"[calib] X: {X.shape}")

    Xf = X.to(DTYPE_ACC)
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * torch.trace(XtX).item() / H
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list, scales = [], []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws (ridge)")):
        W_up = T[per_e[eid]["up"]].to(DEVICE)
        W_dn = T[per_e[eid]["down"]].to(DEVICE)
        W_gt = T[per_e[eid]["gate"]].to(DEVICE)
        Y = forward_mlp(X, W_gt, W_up, W_dn).to(DTYPE_ACC)
        Wt = torch.cholesky_solve(Xf.t() @ Y, cholG)
        W = Wt.t().contiguous()
        if cfg.NORMALIZE_W:
            s = torch.linalg.norm(W, ord="fro").clamp_min(1e-12).item()
            W = W / s
        else: s = 1.0
        Ws_list.append(W); scales.append(s)

    Ws = torch.stack(Ws_list).to(DTYPE_ACC).to(DEVICE)
    Sc = torch.tensor(scales, dtype=DTYPE_ACC, device=DEVICE)
    return Ws, Sc

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor, torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids: raise RuntimeError(f"No experts at layer {cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer={cfg.LAYER} total experts={len(all_eids)} using={len(eids)}")

    cpath = ws_cache_path(len(eids))
    if os.path.isfile(cpath):
        z = load_npz(cpath)
        if all(k in z for k in ["meta","Ws","expert_ids","scales"]) and _decode_meta(z["meta"]) == ws_meta(eids):
            Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
            Sc = torch.from_numpy(z["scales"]).to(DTYPE_ACC).to(DEVICE)
            log(f"[cache] loaded Ws -> {cpath} shape={Ws.shape}")
            return eids, Ws, Sc
    Ws, Sc = build_Ws(eids, wm)
    save_npz_compressed(cpath, {
        "meta": _encode_meta(ws_meta(eids)),
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.cpu().numpy().astype(np.float32),
        "scales": Sc.cpu().numpy().astype(np.float32)
    })
    log(f"[cache] wrote Ws -> {cpath} size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws, Sc

# -----------------------------------------------------------------------------
# Clustering, training, payload building, evaluation
# (identical to the working Mixtral/DeepSeek script – omitted for brevity but MUST be included)
# -----------------------------------------------------------------------------
# [PASTE HERE THE FULL PIPELINE FUNCTIONS: random_proj_features, kmeans_torch,
#  OrthoParam, build_payload_for_cluster, PayloadRuntime, eval_payload, etc.
#  from any of the earlier complete working scripts.]
#
# For brevity in this answer, I assume you have those functions available.
# The key is to use the aggressive parameters defined above.
#
# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def banner():
    log("="*60)
    log("EBC-LLM Compact Compression for Phi-3.5-MoE")
    log(f"Time: {now()}  Device: {DEVICE}")
    log(f"MODEL_DIR: {cfg.MODEL_DIR}  OUTPUT_DIR: {cfg.OUTPUT_DIR}")
    log(f"Layer: {cfg.LAYER}  Experts: {cfg.MAX_EXPERTS}")
    log(f"RES_RANK={cfg.RES_RANK} CORE_TARGET={cfg.CORE_TARGET} CORE_MAX_BLOCKS={cfg.CORE_MAX_BLOCKS}")
    log("="*60)

def main():
    banner()
    expert_ids, Ws_norm, Sc = load_or_build_Ws()
    E, n, _ = Ws_norm.shape
    log(f"[Ws] shape={Ws_norm.shape}")

    # Clustering
    Xfeat = random_proj_features(Ws_norm, cfg.CLUSTER_FEAT_D)
    M0 = max(2, min(cfg.M0 if cfg.M0>0 else int(round(2*math.sqrt(E))), E))
    labels = kmeans_torch(Xfeat, M0, cfg.CLUSTER_ITERS, cfg.CLUSTER_RESTARTS)
    labels = merge_small_clusters(Xfeat, labels, cfg.CLUSTER_MIN_SIZE)
    labels = hierarchical_split(Xfeat, labels, cfg.CLUSTER_MAX_SIZE, min(cfg.M_MAX, E), cfg.SPLIT_ITERS)
    labels = relabel_contiguous(labels)
    M = labels.max().item() + 1
    clusters = [torch.nonzero(labels==m, as_tuple=False).flatten().tolist() for m in range(M)]
    clusters = [c for c in clusters if c]
    log(f"[cluster] M={len(clusters)} sizes={[len(c) for c in clusters]}")
    cluster_of_pos = [0]*E
    for m, idx in enumerate(clusters):
        for pos in idx: cluster_of_pos[pos] = m

    # Init and train bases
    U_par, V_par = [], []
    for idx in clusters:
        Wm = Ws_norm[idx].mean(0)
        U0, V0 = svd_init_from_mean(Wm)
        U_par.append(OrthoParam(U0)); V_par.append(OrthoParam(V0))

    if cfg.TRAIN_STEPS > 0 and cfg.BASIS_MODE == "dense_train":
        params = [p.M for p in U_par] + [p.M for p in V_par]
        opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)
        guidance_masks, guidance_stats = {}, {}
        t0 = time.perf_counter()
        for step in range(1, cfg.TRAIN_STEPS+1):
            S = torch.randperm(n)[:cfg.SUBM].to(DEVICE)
            if cfg.TRAIN_LAM_GUIDE > 0 and (step==1 or step%cfg.TRAIN_GUIDE_EVERY==0):
                with torch.no_grad():
                    guidance_masks.clear(); guidance_stats.clear()
                    for m, idx in enumerate(clusters):
                        if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
                        Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                        pick = idx if cfg.BATCH_E>=len(idx) else [idx[i] for i in torch.randperm(len(idx))[:cfg.BATCH_E].tolist()]
                        Xs_ng = slice_X_batch(Ws_norm[pick], Uo, Vo, S).detach()
                        mask, ef, kblk = make_guidance_mask_from_Xs(Xs_ng, cfg.CORE_BLOCK, cfg.TRAIN_GUIDE_TARGET, cfg.TRAIN_GUIDE_MAX_BLOCKS)
                        guidance_masks[m] = mask; guidance_stats[m] = (ef, kblk)

            lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
            lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
            lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp
            L_total, n_terms = None, 0
            for m, idx in enumerate(clusters):
                if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
                Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                pick = idx if cfg.BATCH_E>=len(idx) else [idx[i] for i in torch.randperm(len(idx))[:cfg.BATCH_E].tolist()]
                Xs = slice_X_batch(Ws_norm[pick], Uo, Vo, S)
                off, diag = offdiag_abs_mean(Xs), diag_abs_mean(Xs).clamp_min(1e-6)
                base = torch.log(off+1e-6) - torch.log(diag) if cfg.TRAIN_OBJ=="logratio" else off/diag
                if lam_block > 0: base += lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)
                if lam_guide > 0 and m in guidance_masks:
                    Mmask = guidance_masks[m]
                    Etot = (Xs*Xs).mean().clamp_min(1e-12)
                    Eout = ((Xs*(1-Mmask))**2).mean()
                    base += lam_guide * (Eout/Etot)
                L_total = base if L_total is None else L_total + base
                n_terms += 1
            if L_total is None: break
            L_total = L_total / n_terms
            opt.zero_grad(); L_total.backward()
            if cfg.GRAD_CLIP > 0: torch.nn.utils.clip_grad_norm_(params, cfg.GRAD_CLIP)
            opt.step()
            if step % cfg.REORTHO_EVERY == 0 or step == cfg.TRAIN_STEPS:
                with torch.no_grad():
                    for p in U_par: p.M.copy_(p.orthogonal())
                    for p in V_par: p.M.copy_(p.orthogonal())
            if step % cfg.REPORT_EVERY == 0 or step == 1:
                t1 = time.perf_counter()
                gstr = "" if not guidance_stats else f" guide≈{np.mean([v[0] for v in guidance_stats.values()]):.3f}"
                log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={L_total.item():.4f} {gstr} (+{t1-t0:.1f}s)")
                t0 = t1

    # Freeze bases
    U_list = [p.orthogonal().detach() for p in U_par]
    V_list = [p.orthogonal().detach() for p in V_par]

    # Build payloads
    log("[build] payloads ...")
    core_all = [[] for _ in range(E)]
    res_all  = [[] for _ in range(E)]
    DL_list, DR_list = [], []
    rmax = min(cfg.RES_RANK, n)
    gam = torch.zeros((E, rmax), dtype=DTYPE_ACC, device=DEVICE) if cfg.RES_COEF=="diag" else None
    Cfull = torch.zeros((E, rmax, rmax), dtype=DTYPE_ACC, device=DEVICE) if cfg.RES_COEF=="full" else None

    for m, idx in enumerate(clusters):
        U, V = U_list[m], V_list[m]
        P = build_payload_for_cluster(Ws_norm, idx, U, V)
        for j, pos in enumerate(idx):
            core_all[pos] = P["core_blocks"][j]
            res_all[pos] = P["res_blocks"][j]
            if cfg.RES_COEF == "diag":
                g = P["coef_list"][j]; gam[pos, :g.numel()] = g
            else:
                C = P["coef_list"][j]; Cfull[pos, :C.shape[0], :C.shape[1]] = C
        DL_list.append(P["DL"]); DR_list.append(P["DR"])
        log(f"  cluster{m}: E={len(idx)} core_blocks≈{np.mean([len(c) for c in P['core_blocks']]):.1f} r={P['DL'].shape[1]}")

    # Save payload
    out_path = os.path.join(cfg.OUTPUT_DIR, f"ebc_payload_layer{cfg.LAYER}_E{E}_q{cfg.QMODE}.npz")
    store_dtype = np.float16 if cfg.BASIS_STORE_DTYPE=="float16" else np.float32
    arrays = {
        "meta": _encode_meta(ws_meta(expert_ids) | {"time": now(), "qmode": cfg.QMODE, "res_coef": cfg.RES_COEF}),
        "expert_ids": np.array(expert_ids, dtype=np.int32),
        "scales": Sc.cpu().numpy().astype(np.float32),
        "cluster_of_pos": np.array(cluster_of_pos, dtype=np.int16),
        "n_clusters": np.array([len(clusters)], dtype=np.int32),
    }
    for m in range(len(clusters)):
        arrays[f"U_{m}"] = U_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"V_{m}"] = V_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DL_{m}"] = DL_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DR_{m}"] = DR_list[m].cpu().numpy().astype(store_dtype)
    if cfg.RES_COEF == "diag":
        arrays["gam"] = gam.cpu().numpy().astype(store_dtype)
    else:
        arrays["Cfull"] = Cfull.cpu().numpy().astype(store_dtype)

    core_pack = pack_blocks_ragged(core_all, cfg.QMODE)
    res_pack  = pack_blocks_ragged(res_all, cfg.QMODE)
    for k, v in core_pack.items(): arrays["core_"+k] = v
    for k, v in res_pack.items(): arrays["res_"+k] = v

    save_npz_compressed(out_path, arrays)
    log(f"[save] payload -> {out_path} size={os.path.getsize(out_path)/1e6:.2f} MB")

    # Evaluate
    rt = load_payload_runtime(out_path, DEVICE)
    eval_payload(rt, Ws_norm, Sc)
    log("✅ Done.")

if __name__ == "__main__":
    main()

✅ flash_attn stub installed (CPU mode).
EBC-LLM Compact Compression for Phi-3.5-MoE
Time: 2026-04-22 11:36:33  Device: cpu
MODEL_DIR: /data/downloaded_models/Phi-3.5-MoE-instruct  OUTPUT_DIR: /home/daniyar/moe_ws_outputs_phi_compact
Layer: 0  Experts: 16
RES_RANK=256 CORE_TARGET=0.75 CORE_MAX_BLOCKS=128
[found] layer=0 total experts=16 using=16
[diagnostic] No keys with prefix 'model.layers.0.mlp.experts.0.'. Other MLP keys for layer 0:
[warning] Could not find all tensors for expert 0. Searching manually...
[manual] Found: gate=model.layers.9.block_sparse_moe.experts.0.w1.weight, down=model.layers.9.block_sparse_moe.experts.0.w2.weight, up=model.layers.9.block_sparse_moe.experts.0.w3.weight
[warning] Could not find all tensors for expert 1. Searching manually...
[manual] Found: gate=model.layers.9.block_sparse_moe.experts.1.w1.weight, down=model.layers.9.block_sparse_moe.experts.1.w2.weight, up=model.layers.9.block_sparse_moe.experts.1.w3.weight
[warning] Could not find all tensors fo

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='longrope': {'long_mscale', 'short_mscale'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='longrope': {'long_mscale', 'short_mscale'}
[transformers] PhiMoEForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the

Loading weights:   0%|          | 0/1957 [00:00<?, ?it/s]

[transformers] PhiMoEForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


[capture] Located MLP module: PhiMoESparseMoeBlock
[capture] iter 4/32 nX=4096 nP=4096
[capture] wrote X -> /home/daniyar/moe_ws_outputs_phi_compact/calib_layer0_X.npz shape=(4096, 4096)
[capture] wrote P -> /home/daniyar/moe_ws_outputs_phi_compact/router_layer0_P.npz shape=(4096, 16)
[calib] X: torch.Size([4096, 4096])


Build Ws (ridge):   0%|          | 0/16 [00:00<?, ?it/s]

[cache] wrote Ws -> /home/daniyar/moe_ws_outputs_phi_compact/Ws_cache_layer0_E16_ridge_ebc.npz size=1003.99 MB
[Ws] shape=torch.Size([16, 4096, 4096])
[cluster] M=6 sizes=[3, 2, 2, 3, 2, 4]
[train] step   1/24 loss=-4.1525  guide≈0.876 (+43.0s)
[train] step   4/24 loss=-0.8861  guide≈0.820 (+131.8s)
[train] step   8/24 loss=-1.1963  guide≈0.821 (+153.3s)
[train] step  12/24 loss=0.8466  guide≈0.819 (+154.2s)
[train] step  16/24 loss=2.0024  guide≈0.813 (+152.3s)
[train] step  20/24 loss=4.5313  guide≈0.812 (+151.3s)
[train] step  24/24 loss=5.9891  guide≈0.830 (+148.7s)
[build] payloads ...
  cluster0: E=3 core_blocks≈38.0 r=256
  cluster1: E=2 core_blocks≈22.0 r=256
  cluster2: E=2 core_blocks≈21.0 r=256
  cluster3: E=3 core_blocks≈30.3 r=256
  cluster4: E=2 core_blocks≈18.5 r=256
  cluster5: E=4 core_blocks≈42.0 r=256
[save] payload -> /home/daniyar/moe_ws_outputs_phi_compact/ebc_payload_layer0_E16_qnone.npz size=651.68 MB
[eval] per-expert rel-error mean=0.114530 p95=0.147710 max=0.

In [7]:
#!/usr/bin/env python3
# =============================================================================
# EBC-LLM: Universal MoE Compression – Mixtral, Qwen, Phi, DeepSeek
# =============================================================================
# Usage in Jupyter: just run this cell. Adjust cfg defaults at the top.
# =============================================================================

import os, sys, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs): return x

# -----------------------------------------------------------------------------
# FlashAttention stub (CPU safe)
# -----------------------------------------------------------------------------
import types, importlib.machinery

def _install_flash_attn_stub():
    if "flash_attn" in sys.modules:
        return
    flash_attn = types.ModuleType("flash_attn")
    flash_attn.__version__ = "0.0.0-cpu-stub"
    def _unavailable(*a, **k):
        raise RuntimeError("flash_attn stub called on CPU")
    flash_attn.flash_attn_func = _unavailable
    flash_attn.flash_attn_varlen_func = _unavailable
    flash_attn.flash_attn_with_kvcache = _unavailable
    flash_attn.layers = types.ModuleType("flash_attn.layers")
    flash_attn.layers.rotary = types.ModuleType("flash_attn.layers.rotary")
    class RotaryEmbedding(nn.Module):
        def __init__(self, dim, **kw): super().__init__()
        def forward(self, x, **kw):
            return torch.ones(1, device=x.device), torch.zeros(1, device=x.device)
    flash_attn.layers.rotary.RotaryEmbedding = RotaryEmbedding
    flash_attn.layers.rotary.apply_rotary_emb = lambda *a, **k: (_unavailable,)
    sys.modules["flash_attn"] = flash_attn
    sys.modules["flash_attn.layers"] = flash_attn.layers
    sys.modules["flash_attn.layers.rotary"] = flash_attn.layers.rotary

class FlashAttnImporter:
    def find_spec(self, fullname, path, target=None):
        if fullname == "flash_attn" or fullname.startswith("flash_attn."):
            _install_flash_attn_stub()
            return importlib.machinery.ModuleSpec(fullname, self)
        return None
    def create_module(self, spec): return sys.modules.get(spec.name)
    def exec_module(self, module): pass

sys.meta_path.insert(0, FlashAttnImporter())
print("✅ flash_attn stub installed (CPU mode).", flush=True)

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = "/data/downloaded_models/Phi-3.5-MoE-instruct"
    OUTPUT_DIR: str = "/home/daniyar/moe_ws_outputs_phi_compact"
    LAYER: int = int(os.environ.get("LAYER", "0"))
    MAX_EXPERTS: int = int(os.environ.get("MAX_EXPERTS", "0"))   # 0 = all

    CALIB_SAMPLES: int = int(os.environ.get("CALIB_SAMPLES", "4096"))
    RIDGE_DAMP: float = float(os.environ.get("RIDGE_DAMP", "1e-3"))
    NORMALIZE_W: bool = os.environ.get("NORMALIZE_W", "1") == "1"
    CAPTURE_ENABLE: bool = os.environ.get("CAPTURE_ENABLE", "1") == "1"
    CAPTURE_ITERS: int = int(os.environ.get("CAPTURE_ITERS", "32"))
    CAPTURE_MAX_TOKENS: int = int(os.environ.get("CAPTURE_MAX_TOKENS", "1024"))
    CAPTURE_TEXT: str = os.environ.get("CAPTURE_TEXT", "MoE calibration text. " * 256)
    HF_TRUST_REMOTE_CODE: bool = True
    HF_LOCAL_FILES_ONLY: bool = True

    # Clustering
    CLUSTER_FEAT_D: int = 64
    CLUSTER_ITERS: int = 60
    CLUSTER_RESTARTS: int = 4
    CLUSTER_MIN_SIZE: int = 2
    CLUSTER_MAX_SIZE: int = 4
    M_MAX: int = 16

    # Training
    TRAIN_STEPS: int = int(os.environ.get("TRAIN_STEPS", "24"))
    TRAIN_WARMUP: int = 6
    TRAIN_LR: float = 5e-2
    SUBM: int = 256
    BATCH_E: int = 4
    TRAIN_MIN_CLUSTER: int = 2
    REORTHO_EVERY: int = 4
    REPORT_EVERY: int = 4
    GRAD_CLIP: float = 1.0
    TRAIN_OBJ: str = "logratio"
    TRAIN_LAM_BLOCK: float = 0.10
    TRAIN_LAM_GUIDE: float = 1.0
    TRAIN_GUIDE_TARGET: float = 0.80
    TRAIN_GUIDE_EVERY: int = 2
    TRAIN_GUIDE_MAX_BLOCKS: int = 2048

    # Core selection
    CORE_BLOCK: int = 64
    CORE_TARGET: float = 0.75
    CORE_MAX_BLOCKS: int = 128

    # Residual
    RES_RANK: int = int(os.environ.get("RES_RANK", "256"))
    RES_COEF: str = "diag"
    RES_TARGET: float = 0.99
    RES_MAX_BLOCKS: int = 1024
    RES_BSIZE: int = 64

    # Refine
    REFINE_ENABLE: bool = False
    REFINE_ERR_TARGET: float = 0.03
    REFINE_MAX_EXTRA: int = 4096

    QMODE: str = "none"
    BASIS_STORE_DTYPE: str = "float16"
    SEED: int = 1234

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# Utilities
# -----------------------------------------------------------------------------
def log(msg): print(msg, flush=True)
def now(): return time.strftime("%Y-%m-%d %H:%M:%S")

def seed_all(s): random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all(cfg.SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE_ACC = torch.float32
torch.set_num_threads(8)

def save_npz(path, arrays): np.savez_compressed(path, **arrays)
def load_npz(path): return dict(np.load(path, allow_pickle=False))

# -----------------------------------------------------------------------------
# Weight loading – Universal expert detection
# -----------------------------------------------------------------------------
def read_index(model_dir):
    idx = os.path.join(model_dir, "model.safetensors.index.json")
    if os.path.isfile(idx):
        with open(idx) as f:
            return json.load(f).get("weight_map", {})
    files = [f for f in os.listdir(model_dir) if f.endswith(".safetensors")]
    if len(files) == 1:
        with safe_open(os.path.join(model_dir, files[0]), framework="pt") as sf:
            keys = list(sf.keys())
        return {k: files[0] for k in keys}
    raise RuntimeError("No safetensors index or single file found")

def find_expert_prefix(weight_map, layer):
    candidates = [
        f"model.layers.{layer}.mlp.experts.",
        f"model.layers.{layer}.block_sparse_moe.experts.",
        f"model.layers.{layer}.moe.experts.",
        f"transformer.h.{layer}.mlp.experts.",
    ]
    for p in candidates:
        if any(k.startswith(p) for k in weight_map):
            return p
    for k in weight_map:
        if f"layers.{layer}" in k and "experts" in k:
            return k.split("experts.")[0] + "experts."
    return None

def find_expert_ids(weight_map, layer):
    prefix = find_expert_prefix(weight_map, layer)
    if not prefix:
        return []
    ids = set()
    for k in weight_map:
        if k.startswith(prefix):
            parts = k[len(prefix):].split('.')
            if parts and parts[0].isdigit():
                ids.add(int(parts[0]))
    return sorted(ids)

def pick_expert_keys(weight_map, layer, eid):
    prefix = find_expert_prefix(weight_map, layer)
    if not prefix:
        return {}
    full = f"{prefix}{eid}."

    gate = up = down = None
    for suf in ["w1.weight", "gate_proj.weight", "gate.weight"]:
        if full + suf in weight_map:
            gate = full + suf
            break
    for suf in ["w3.weight", "up_proj.weight", "up.weight"]:
        if full + suf in weight_map:
            up = full + suf
            break
    for suf in ["w2.weight", "down_proj.weight", "down.weight"]:
        if full + suf in weight_map:
            down = full + suf
            break

    if gate and up and down:
        return {"gate": gate, "up": up, "down": down}
    return {}

def load_tensors(model_dir, weight_map, keys):
    shard_map = {}
    for k in keys:
        shard = weight_map.get(k)
        if shard:
            shard_map.setdefault(shard, []).append(k)
    tensors = {}
    for shard, ks in shard_map.items():
        with safe_open(os.path.join(model_dir, shard), framework="pt") as f:
            for k in ks:
                tensors[k] = f.get_tensor(k)
    return tensors

# -----------------------------------------------------------------------------
# Calibration capture
# -----------------------------------------------------------------------------
class _Collector:
    def __init__(self, H, E, max_rows):
        self.H = H; self.E = E; self.max = max_rows
        self.X = []; self.P = []; self.nX = 0; self.nP = 0
    def _take(self, flat, need):
        return flat[:need] if flat.shape[0] > need else flat
    def add_X(self, hs):
        if hs is None: return
        if hs.ndim == 2: hs = hs.unsqueeze(0)
        flat = hs.detach().float().cpu().reshape(-1, self.H)
        if flat.numel() == 0: return
        need = self.max - self.nX
        if need > 0:
            self.X.append(self._take(flat, need))
            self.nX += self.X[-1].shape[0]
    def add_logits(self, logits):
        if logits is None: return
        if logits.ndim == 2: logits = logits.unsqueeze(0)
        P = torch.softmax(logits.detach().float(), -1)[..., :self.E].cpu()
        flat = P.reshape(-1, P.shape[-1])
        if flat.numel() == 0: return
        need = self.max - self.nP
        if need > 0:
            self.P.append(self._take(flat, need))
            self.nP += self.P[-1].shape[0]

def capture_XP(model_dir, layer_idx, H, E, out_x, out_p):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
                                        local_files_only=cfg.HF_LOCAL_FILES_ONLY)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
        attn_implementation="eager"
    ).to(DEVICE).eval()

    # Locate layers
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        layers = model.model.layers
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        layers = model.transformer.h
    else:
        layers = model.layers
    layer = layers[layer_idx]

    # Locate MoE block
    mlp = None
    for name in ["mlp", "moe", "block_sparse_moe"]:
        if hasattr(layer, name):
            mlp = getattr(layer, name)
            break
    if mlp is None:
        for n, m in layer.named_modules():
            if "expert" in n.lower() and hasattr(m, "w1"):
                mlp = m
                break
    if mlp is None:
        raise RuntimeError("MoE block not found")
    log(f"[capture] MoE block: {mlp.__class__.__name__}")

    # Router linear (optional)
    router = None
    for n, m in layer.named_modules():
        if isinstance(m, nn.Linear) and m.in_features == H and m.out_features >= E:
            if "router" in n.lower() or "gate" in n.lower():
                router = m
                break

    coll = _Collector(H, E, cfg.CALIB_SAMPLES)

    def pre_hook(_, inputs):
        coll.add_X(inputs[0])
    h1 = mlp.register_forward_pre_hook(pre_hook)
    h2 = None
    if router:
        def router_hook(_, __, out):
            o = out[0] if isinstance(out, (tuple, list)) else out
            coll.add_logits(o)
        h2 = router.register_forward_hook(router_hook)

    text = cfg.CAPTURE_TEXT
    for it in range(cfg.CAPTURE_ITERS):
        enc = tok(text, return_tensors="pt", truncation=True,
                  max_length=cfg.CAPTURE_MAX_TOKENS, padding="max_length")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.inference_mode():
            _ = model(**enc, use_cache=False)
        if (it+1) % 4 == 0:
            log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS} nX={coll.nX} nP={coll.nP}")
        if coll.nX >= cfg.CALIB_SAMPLES:
            break

    h1.remove()
    if h2: h2.remove()

    X = torch.cat(coll.X)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz(out_x, {"X": X})
    log(f"[capture] saved X to {out_x} shape={X.shape}")
    if coll.nP:
        P = torch.cat(coll.P)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
        save_npz(out_p, {"P": P})
        log(f"[capture] saved P to {out_p} shape={P.shape}")

# -----------------------------------------------------------------------------
# Ridge linearization (build square proxies Ws)
# -----------------------------------------------------------------------------
@torch.no_grad()
def forward_mlp(X, Wg, Wu, Wd):
    Xf = X.to(DTYPE_ACC)
    up = Xf @ Wu.to(DTYPE_ACC).t()
    gate = Xf @ Wg.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    return hid @ Wd.to(DTYPE_ACC).t()

def build_Ws(eids, wm):
    per_e = {}
    keys = []
    for eid in eids:
        kk = pick_expert_keys(wm, cfg.LAYER, eid)
        if not kk:
            raise RuntimeError(f"Expert {eid} missing tensors")
        per_e[eid] = kk
        keys += [kk["up"], kk["down"], kk["gate"]]
    T = load_tensors(cfg.MODEL_DIR, wm, keys)

    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = W_up0.shape[0], W_up0.shape[1]
    log(f"[shape] H={H} d_ff={dff}")

    calib_path = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    if not os.path.isfile(calib_path) and cfg.CAPTURE_ENABLE:
        capture_XP(cfg.MODEL_DIR, cfg.LAYER, H, len(eids), calib_path,
                   os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz"))

    X = torch.from_numpy(np.load(calib_path)["X"][:cfg.CALIB_SAMPLES].astype(np.float32))
    X = X.to(DEVICE, DTYPE_ACC)
    log(f"[calib] X: {X.shape}")

    XtX = X.t() @ X
    lam = cfg.RIDGE_DAMP * torch.trace(XtX).item() / H
    chol = torch.linalg.cholesky(XtX + lam * torch.eye(H, device=DEVICE, dtype=DTYPE_ACC))

    Ws_list, scales = [], []
    for eid in tqdm(eids, desc="Build Ws (ridge)"):
        Wg = T[per_e[eid]["gate"]].to(DEVICE)
        Wu = T[per_e[eid]["up"]].to(DEVICE)
        Wd = T[per_e[eid]["down"]].to(DEVICE)
        Y = forward_mlp(X, Wg, Wu, Wd)
        Wt = torch.cholesky_solve(X.t() @ Y, chol)
        W = Wt.t().contiguous()
        if cfg.NORMALIZE_W:
            s = torch.linalg.norm(W).clamp_min(1e-12).item()
            W = W / s
        else:
            s = 1.0
        Ws_list.append(W)
        scales.append(s)

    Ws = torch.stack(Ws_list).to(DEVICE, DTYPE_ACC)
    Sc = torch.tensor(scales, device=DEVICE, dtype=DTYPE_ACC)
    return Ws, Sc

def load_or_build_Ws():
    wm = read_index(cfg.MODEL_DIR)
    all_ids = find_expert_ids(wm, cfg.LAYER)
    if not all_ids:
        raise RuntimeError(f"No experts at layer {cfg.LAYER}")
    max_e = cfg.MAX_EXPERTS if cfg.MAX_EXPERTS > 0 else len(all_ids)
    eids = all_ids[:max_e]
    log(f"[found] layer={cfg.LAYER} total={len(all_ids)} using={len(eids)}")

    cache_path = os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}.npz")
    if os.path.isfile(cache_path):
        data = load_npz(cache_path)
        cached_eids = data.get("expert_ids", [])
        if set(cached_eids) == set(eids):
            Ws = torch.from_numpy(data["Ws"]).to(DEVICE, DTYPE_ACC)
            Sc = torch.from_numpy(data["scales"]).to(DEVICE, DTYPE_ACC)
            log(f"[cache] loaded Ws shape={Ws.shape}")
            return eids, Ws, Sc
        log("[cache] mismatch, rebuilding")

    Ws, Sc = build_Ws(eids, wm)
    save_npz(cache_path, {
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.cpu().numpy().astype(np.float32),
        "scales": Sc.cpu().numpy().astype(np.float32)
    })
    log(f"[cache] saved Ws to {cache_path}")
    return eids, Ws, Sc

# -----------------------------------------------------------------------------
# Clustering
# -----------------------------------------------------------------------------
def random_proj_features(Ws, d):
    E, n, _ = Ws.shape
    flat = Ws.reshape(E, -1)
    proj = torch.randn(flat.shape[1], d, device=DEVICE, dtype=DTYPE_ACC)
    proj = proj / torch.linalg.norm(proj, dim=0, keepdim=True)
    return flat @ proj

def kmeans(X, k, iters=20, restarts=5):
    best_lab, best_inertia = None, float('inf')
    for _ in range(restarts):
        centers = X[torch.randperm(X.shape[0])[:k]].clone()
        for _ in range(iters):
            dist = torch.cdist(X, centers)
            lab = dist.argmin(1)
            for j in range(k):
                if (lab == j).any():
                    centers[j] = X[lab == j].mean(0)
        inertia = torch.cdist(X, centers).min(1)[0].pow(2).sum().item()
        if inertia < best_inertia:
            best_inertia, best_lab = inertia, lab.clone()
    return best_lab

def relabel_contiguous(labels):
    uniq = torch.unique(labels)
    out = labels.clone()
    for new, old in enumerate(uniq.tolist()):
        out[labels == old] = new
    return out

def merge_small_clusters(X, labels, min_size):
    labels = relabel_contiguous(labels)
    while True:
        K = labels.max().item() + 1
        counts = torch.bincount(labels, minlength=K)
        small = (counts < min_size).nonzero(as_tuple=False).flatten()
        if small.numel() == 0:
            break
        large = torch.where(counts >= min_size)[0]
        centers = torch.stack([X[labels == c].mean(0) for c in large])
        for c in small.tolist():
            idx = (labels == c).nonzero(as_tuple=False).flatten()
            if idx.numel() == 0: continue
            dist = torch.cdist(X[idx].mean(0, keepdim=True), centers)
            nearest = large[dist.argmin().item()]
            labels[idx] = nearest
        labels = relabel_contiguous(labels)
    return labels

def hierarchical_split(X, labels, max_size, max_k, split_iters=20):
    labels = relabel_contiguous(labels)
    while True:
        K = labels.max().item() + 1
        if K >= max_k:
            break
        counts = torch.bincount(labels, minlength=K)
        big = (counts > max_size).nonzero(as_tuple=False).flatten()
        if big.numel() == 0:
            break
        c = big[counts[big].argmax().item()]
        idx = (labels == c).nonzero(as_tuple=False).flatten()
        if idx.numel() < 2: break
        sub = X[idx]
        sub_lab = kmeans(sub, 2, split_iters, restarts=3)
        labels[idx[sub_lab == 1]] = K
        labels = relabel_contiguous(labels)
    return labels

def cluster_experts(Ws):
    Xf = random_proj_features(Ws, cfg.CLUSTER_FEAT_D)
    M0 = max(2, min(int(2 * math.sqrt(Ws.shape[0])), Ws.shape[0]))
    labels = kmeans(Xf, M0, cfg.CLUSTER_ITERS, cfg.CLUSTER_RESTARTS)
    labels = merge_small_clusters(Xf, labels, cfg.CLUSTER_MIN_SIZE)
    labels = hierarchical_split(Xf, labels, cfg.CLUSTER_MAX_SIZE, cfg.M_MAX, cfg.CLUSTER_ITERS)
    labels = merge_small_clusters(Xf, labels, cfg.CLUSTER_MIN_SIZE)
    return relabel_contiguous(labels)

# -----------------------------------------------------------------------------
# Orthogonal parameterization
# -----------------------------------------------------------------------------
class OrthoParam(nn.Module):
    def __init__(self, M):
        super().__init__()
        self.M = nn.Parameter(M.to(DEVICE, DTYPE_ACC).contiguous())
    def orthogonal(self):
        Q, _ = torch.linalg.qr(self.M)
        return Q

def svd_init_from_mean(Wmean, rank=None):
    r = rank if rank is not None else cfg.RES_RANK
    U, S, Vh = torch.linalg.svd(Wmean, full_matrices=False)
    U0 = U[:, :r] @ torch.diag(torch.sqrt(S[:r]))
    V0 = Vh[:r, :].t() @ torch.diag(torch.sqrt(S[:r]))
    return U0, V0

# -----------------------------------------------------------------------------
# Training utilities (robust slice_X_batch using matmul)
# -----------------------------------------------------------------------------
def schedule(step, warmup, total):
    if step <= warmup:
        return 0.0
    return min(1.0, (step - warmup) / max(1, total - warmup))

def slice_X_batch(Wb, U, V, S):
    """
    Wb: either (n, n) or (B, n, n)
    Returns X_full[:, :, S] where X_full = U^T @ Wb @ V
    """
    if Wb.ndim == 2:
        Wb = Wb.unsqueeze(0)          # (1, n, n)
    intermediate = torch.matmul(Wb, V)            # (B, n, r)
    X_full = torch.matmul(U.t(), intermediate)    # (B, r, r)  [U.t() is (r, n)]
    return X_full[:, :, S]

def offdiag_abs_mean(Xs):
    mask = ~torch.eye(Xs.shape[1], device=Xs.device, dtype=torch.bool)
    return Xs[:, mask].abs().mean()

def diag_abs_mean(Xs):
    d = min(Xs.shape[1], Xs.shape[2])
    return Xs[:, torch.arange(d), torch.arange(d)].abs().mean()

def block_group_sparsity_penalty(Xs, block_size):
    B, r, c = Xs.shape
    if c % block_size != 0:
        return torch.tensor(0.0, device=Xs.device)
    nb = c // block_size
    Xb = Xs.view(B, r, nb, block_size)
    block_norms = torch.linalg.norm(Xb, dim=(1, 3))
    return block_norms.mean()

def make_guidance_mask(Xs, block_size, target, max_blocks):
    B, r, c = Xs.shape
    if c % block_size != 0:
        return torch.ones_like(Xs), 1.0, c
    nb = c // block_size
    Xb = Xs.view(B, r, nb, block_size)
    block_norms = torch.linalg.norm(Xb, dim=(1, 3)).mean(0)  # (nb,)
    total = block_norms.sum()
    sorted_norms, indices = torch.sort(block_norms, descending=True)
    cumsum = torch.cumsum(sorted_norms, 0)
    k = torch.searchsorted(cumsum, target * total).item() + 1
    k = min(k, max_blocks)
    mask_blocks = torch.zeros(nb, device=Xs.device)
    mask_blocks[indices[:k]] = 1.0
    mask = mask_blocks.repeat_interleave(block_size).unsqueeze(0).unsqueeze(0).expand(B, r, c)
    kept_energy = (cumsum[k-1] / total).item() if total > 0 else 1.0
    return mask, kept_energy, k * block_size

# -----------------------------------------------------------------------------
# Rotation training
# -----------------------------------------------------------------------------
def train_rotations(Ws, clusters):
    U_par, V_par = [], []
    for idx in clusters:
        Wm = Ws[idx].mean(0)
        U0, V0 = svd_init_from_mean(Wm)
        U_par.append(OrthoParam(U0))
        V_par.append(OrthoParam(V0))

    if cfg.TRAIN_STEPS == 0:
        return [p.orthogonal().detach() for p in U_par], [p.orthogonal().detach() for p in V_par]

    params = [p.M for p in U_par] + [p.M for p in V_par]
    opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)
    guidance_masks = {}
    t0 = time.perf_counter()

    # S is sampled from the reduced rank r, not from H.
    r = cfg.RES_RANK
    sub_sample_size = min(cfg.SUBM, r)

    for step in range(1, cfg.TRAIN_STEPS+1):
        S = torch.randperm(r)[:sub_sample_size].to(DEVICE)

        if cfg.TRAIN_LAM_GUIDE > 0 and (step == 1 or step % cfg.TRAIN_GUIDE_EVERY == 0):
            with torch.no_grad():
                guidance_masks.clear()
                for m, idx in enumerate(clusters):
                    if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
                    Uo = U_par[m].orthogonal()
                    Vo = V_par[m].orthogonal()
                    pick = idx if cfg.BATCH_E >= len(idx) else random.sample(idx, cfg.BATCH_E)
                    if not isinstance(pick, list): pick = [pick]
                    Xs = slice_X_batch(Ws[pick], Uo, Vo, S)
                    mask, _, _ = make_guidance_mask(Xs, cfg.CORE_BLOCK, cfg.TRAIN_GUIDE_TARGET, cfg.TRAIN_GUIDE_MAX_BLOCKS)
                    guidance_masks[m] = mask

        lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
        lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
        lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp
        loss_total = 0.0
        n_terms = 0

        for m, idx in enumerate(clusters):
            if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
            Uo = U_par[m].orthogonal()
            Vo = V_par[m].orthogonal()
            pick = idx if cfg.BATCH_E >= len(idx) else random.sample(idx, cfg.BATCH_E)
            if not isinstance(pick, list): pick = [pick]
            Xs = slice_X_batch(Ws[pick], Uo, Vo, S)

            off = offdiag_abs_mean(Xs)
            diag = diag_abs_mean(Xs).clamp_min(1e-6)
            if cfg.TRAIN_OBJ == "logratio":
                base = torch.log(off + 1e-6) - torch.log(diag)
            else:
                base = off / diag

            if lam_block > 0:
                base = base + lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)
            if lam_guide > 0 and m in guidance_masks:
                Mmask = guidance_masks[m]
                Etot = (Xs * Xs).mean().clamp_min(1e-12)
                Eout = ((Xs * (1 - Mmask)) ** 2).mean()
                base = base + lam_guide * (Eout / Etot)

            loss_total += base
            n_terms += 1

        if n_terms == 0:
            break
        loss = loss_total / n_terms
        opt.zero_grad()
        loss.backward()
        if cfg.GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_(params, cfg.GRAD_CLIP)
        opt.step()

        if step % cfg.REORTHO_EVERY == 0 or step == cfg.TRAIN_STEPS:
            with torch.no_grad():
                for p in U_par: p.M.copy_(p.orthogonal())
                for p in V_par: p.M.copy_(p.orthogonal())

        if step % cfg.REPORT_EVERY == 0 or step == 1:
            t1 = time.perf_counter()
            log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={loss.item():.4f} (+{t1-t0:.1f}s)")
            t0 = t1

    return [p.orthogonal().detach() for p in U_par], [p.orthogonal().detach() for p in V_par]

# -----------------------------------------------------------------------------
# Payload construction (block-TopK core + low-rank + sparse residuals)
# -----------------------------------------------------------------------------
@torch.no_grad()
def block_energy_grid(X, b):
    n = X.shape[0]
    nb = (n + b - 1) // b
    if n % b != 0:
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()
    Eg = (Xb * Xb).sum(dim=(2, 3))
    tot = (X * X).sum().item()
    return Eg, tot, nb

@torch.no_grad()
def pick_blocks_until_target(Eg, tot, target, max_blocks, exclude=None):
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    picked, eacc = [], 0.0
    exclude = exclude or set()
    for idx in order.tolist():
        if len(picked) >= max_blocks:
            break
        e = flat[idx].item()
        if e <= 1e-18: break
        bi, bj = idx // nb, idx % nb
        if (bi, bj) in exclude: continue
        picked.append((bi, bj))
        eacc += e
        if eacc / max(tot, 1e-12) >= target:
            break
    return picked, eacc / max(tot, 1e-12)

@torch.no_grad()
def gather_block(X, i0, j0, b):
    n = X.shape[0]
    i1, j1 = min(n, i0+b), min(n, j0+b)
    return X[i0:i1, j0:j1].contiguous()

@torch.no_grad()
def rand_svd_vectors(A, r, n_iter=2):
    n = A.shape[0]
    r = min(r, n)
    Omega = torch.randn(n, r, device=A.device, dtype=DTYPE_ACC)
    Y = A @ Omega
    for _ in range(n_iter):
        Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    Uhat, _, Vh = torch.linalg.svd(B, full_matrices=False)
    return (Q @ Uhat[:, :r]).contiguous(), Vh.t()[:, :r].contiguous()

@torch.no_grad()
def frob_rel_err(A, B):
    return (torch.linalg.norm(A - B) / torch.linalg.norm(B).clamp_min(1e-12)).item()

@torch.no_grad()
def build_payload_for_cluster(Ws, idx, U, V):
    n = Ws.shape[1]
    X_list = [(U.t() @ Ws[pos] @ V).contiguous() for pos in idx]
    b = cfg.CORE_BLOCK

    # Core blocks
    core_per = []
    for X in X_list:
        Eg, te, nb = block_energy_grid(X, b)
        picks, _ = pick_blocks_until_target(Eg, te, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*b, bj*b
            blocks.append((i0, j0, gather_block(X, i0, j0, b)))
        core_per.append(blocks)

    # Residual after core
    R_list = []
    for X, cb in zip(X_list, core_per):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    # Shared low-rank
    Rmean = torch.stack(R_list).mean(0)
    r = min(cfg.RES_RANK, n)
    DL, DR = rand_svd_vectors(Rmean, r)

    coef_list, res_per = [], []
    bb = cfg.RES_BSIZE
    for j, Rm in enumerate(R_list):
        if cfg.RES_COEF == "diag":
            g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()
            coef_list.append(g)
            R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()
        else:
            C = (DL.t() @ Rm @ DR).contiguous()
            coef_list.append(C)
            R2 = (Rm - (DL @ C @ DR.t())).contiguous()

        Eg2, te2, nb2 = block_energy_grid(R2, bb)
        exclude = {(i0//bb, j0//bb) for (i0, j0, _) in core_per[j]}
        picks, _ = pick_blocks_until_target(Eg2, te2, cfg.RES_TARGET, cfg.RES_MAX_BLOCKS, exclude=exclude)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*bb, bj*bb
            blocks.append((i0, j0, gather_block(R2, i0, j0, bb)))
        res_per.append(blocks)

    # Refine (optional)
    if cfg.REFINE_ENABLE:
        rb = cfg.REFINE_BSIZE
        for j in range(len(idx)):
            X = X_list[j]
            def reconstruct():
                Xc = torch.zeros_like(X)
                for (i0, j0, Bc) in core_per[j]:
                    h, w = Bc.shape
                    Xc[i0:i0+h, j0:j0+w] = Bc
                if cfg.RES_COEF == "diag":
                    Xlr = (DL * coef_list[j].view(1, -1)) @ DR.t()
                else:
                    Xlr = DL @ coef_list[j] @ DR.t()
                Xr = torch.zeros_like(X)
                for (i0, j0, Br) in res_per[j]:
                    h, w = Br.shape
                    Xr[i0:i0+h, j0:j0+w] += Br
                return Xc + Xlr + Xr

            Xhat = reconstruct()
            err = frob_rel_err(Xhat, X)
            added = 0
            core_pos = {(i0, j0) for (i0, j0, _) in core_per[j]}
            res_pos = {(i0, j0) for (i0, j0, _) in res_per[j]}
            while err > cfg.REFINE_ERR_TARGET and added < cfg.REFINE_MAX_EXTRA:
                Rerr = (X - Xhat).contiguous()
                Eg, te, nb = block_energy_grid(Rerr, rb)
                flat = Eg.reshape(-1)
                if flat.max().item() <= 1e-18: break
                order = torch.argsort(flat, descending=True)
                found = False
                for idx_ in order.tolist():
                    bi, bj = idx_ // nb, idx_ % nb
                    i0, j0 = bi*rb, bj*rb
                    if (i0, j0) in core_pos or (i0, j0) in res_pos:
                        continue
                    Br = gather_block(Rerr, i0, j0, rb)
                    res_per[j].append((i0, j0, Br))
                    res_pos.add((i0, j0))
                    added += 1
                    found = True
                    break
                if not found: break
                if added % 32 == 0:
                    Xhat = reconstruct()
                    err = frob_rel_err(Xhat, X)
            Xhat = reconstruct()
            err = frob_rel_err(Xhat, X)

    return {
        "core_blocks": core_per,
        "res_blocks": res_per,
        "coef_list": coef_list,
        "DL": DL, "DR": DR
    }

# -----------------------------------------------------------------------------
# Packing ragged blocks
# -----------------------------------------------------------------------------
def pack_blocks_ragged(blocks_per_expert, qmode):
    all_vals = []
    indices = []
    for exp_blocks in blocks_per_expert:
        exp_idx = []
        for (i0, j0, B) in exp_blocks:
            start = len(all_vals)
            vals = B.flatten().cpu().numpy().astype(np.float32)
            all_vals.append(vals)
            end = start + len(vals)
            exp_idx.append((i0, j0, B.shape[0], B.shape[1], start, end))
        indices.append(exp_idx)
    flat = np.concatenate(all_vals) if all_vals else np.array([], dtype=np.float32)

    i0_list, j0_list, h_list, w_list, ptr_list = [], [], [], [], [0]
    for exp_idx in indices:
        for (i0, j0, h, w, start, end) in exp_idx:
            i0_list.append(i0); j0_list.append(j0); h_list.append(h); w_list.append(w)
            ptr_list.append(ptr_list[-1] + h*w)
    return {
        "flat": flat,
        "i0": np.array(i0_list, dtype=np.int16),
        "j0": np.array(j0_list, dtype=np.int16),
        "h": np.array(h_list, dtype=np.int16),
        "w": np.array(w_list, dtype=np.int16),
        "ptr": np.array(ptr_list, dtype=np.int64),
        "expert_ptr": np.cumsum([0] + [len(idx) for idx in indices], dtype=np.int32)
    }

# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------
class PayloadRuntime:
    def __init__(self, data, device):
        self.device = device
        self.scales = torch.from_numpy(data["scales"]).to(device, dtype=DTYPE_ACC)
        self.cluster_of = torch.from_numpy(data["cluster_of_pos"]).to(device, torch.long)
        n_clusters = data["n_clusters"][0]
        # Load U, V, DL, DR and convert to float32
        self.U = [torch.from_numpy(data[f"U_{m}"]).to(device, dtype=DTYPE_ACC) for m in range(n_clusters)]
        self.V = [torch.from_numpy(data[f"V_{m}"]).to(device, dtype=DTYPE_ACC) for m in range(n_clusters)]
        self.DL = [torch.from_numpy(data[f"DL_{m}"]).to(device, dtype=DTYPE_ACC) for m in range(n_clusters)]
        self.DR = [torch.from_numpy(data[f"DR_{m}"]).to(device, dtype=DTYPE_ACC) for m in range(n_clusters)]
        if "gam" in data:
            self.gam = torch.from_numpy(data["gam"]).to(device, dtype=DTYPE_ACC)
            self.Cfull = None
        else:
            self.Cfull = torch.from_numpy(data["Cfull"]).to(device, dtype=DTYPE_ACC)
            self.gam = None
        self.core_flat = torch.from_numpy(data["core_flat"]).to(device, dtype=DTYPE_ACC)
        self.res_flat = torch.from_numpy(data["res_flat"]).to(device, dtype=DTYPE_ACC)
        self.core_info = (data["core_i0"], data["core_j0"], data["core_h"], data["core_w"],
                          data["core_ptr"], data["core_expert_ptr"])
        self.res_info = (data["res_i0"], data["res_j0"], data["res_h"], data["res_w"],
                         data["res_ptr"], data["res_expert_ptr"])

    def reconstruct_expert(self, pos):
        cid = self.cluster_of[pos].item()
        U, V = self.U[cid], self.V[cid]
        DL, DR = self.DL[cid], self.DR[cid]
        r = U.shape[1]
        X = torch.zeros(r, r, device=self.device, dtype=DTYPE_ACC)
        # Core blocks
        exp_ptr = self.core_info[5]
        for bi in range(exp_ptr[pos], exp_ptr[pos+1]):
            i0 = self.core_info[0][bi]; j0 = self.core_info[1][bi]
            h = self.core_info[2][bi]; w = self.core_info[3][bi]
            v0 = self.core_info[4][bi]; v1 = self.core_info[4][bi+1]
            B = self.core_flat[v0:v1].view(h, w)
            X[i0:i0+h, j0:j0+w] += B
        # Low-rank
        if self.gam is not None:
            X += (DL * self.gam[pos].view(1, -1)) @ DR.t()
        else:
            X += DL @ self.Cfull[pos] @ DR.t()
        # Residual blocks
        res_ptr = self.res_info[5]
        for bi in range(res_ptr[pos], res_ptr[pos+1]):
            i0 = self.res_info[0][bi]; j0 = self.res_info[1][bi]
            h = self.res_info[2][bi]; w = self.res_info[3][bi]
            v0 = self.res_info[4][bi]; v1 = self.res_info[4][bi+1]
            B = self.res_flat[v0:v1].view(h, w)
            X[i0:i0+h, j0:j0+w] += B
        return U @ X @ V.t() * self.scales[pos]

def eval_payload(rt, Ws_orig, scales):
    E = Ws_orig.shape[0]
    errs = []
    for pos in range(E):
        W_rec = rt.reconstruct_expert(pos)
        W_orig = Ws_orig[pos] * scales[pos]
        err = torch.linalg.norm(W_rec - W_orig).item() / torch.linalg.norm(W_orig).item()
        errs.append(err)
    log(f"[eval] per-expert rel-error mean={np.mean(errs):.6f} p95={np.percentile(errs,95):.6f} max={np.max(errs):.6f}")

# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def main():
    log("="*60)
    log("EBC-LLM Universal MoE Compression")
    log(f"Model: {cfg.MODEL_DIR}  Output: {cfg.OUTPUT_DIR}")
    log(f"Layer: {cfg.LAYER}  Max Experts: {cfg.MAX_EXPERTS}")
    log("="*60)

    eids, Ws, scales = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={Ws.shape}")

    # Clustering
    labels = cluster_experts(Ws)
    M = labels.max().item() + 1
    clusters = [torch.where(labels == m)[0].tolist() for m in range(M)]
    clusters = [c for c in clusters if c]
    log(f"[cluster] M={len(clusters)} sizes={[len(c) for c in clusters]}")
    cluster_of_pos = [0]*E
    for m, idx in enumerate(clusters):
        for pos in idx:
            cluster_of_pos[pos] = m

    # Train rotations
    U_list, V_list = train_rotations(Ws, clusters)

    # Build payloads
    log("[build] payloads...")
    core_all = [[] for _ in range(E)]
    res_all = [[] for _ in range(E)]
    DL_list, DR_list = [], []
    gam = torch.zeros((E, cfg.RES_RANK), device=DEVICE, dtype=DTYPE_ACC) if cfg.RES_COEF == "diag" else None
    Cfull = torch.zeros((E, cfg.RES_RANK, cfg.RES_RANK), device=DEVICE, dtype=DTYPE_ACC) if cfg.RES_COEF == "full" else None

    for m, idx in enumerate(clusters):
        U, V = U_list[m], V_list[m]
        P = build_payload_for_cluster(Ws, idx, U, V)
        for j, pos in enumerate(idx):
            core_all[pos] = P["core_blocks"][j]
            res_all[pos] = P["res_blocks"][j]
            if cfg.RES_COEF == "diag":
                gam[pos, :P["coef_list"][j].numel()] = P["coef_list"][j]
            else:
                C = P["coef_list"][j]
                Cfull[pos, :C.shape[0], :C.shape[1]] = C
        DL_list.append(P["DL"]); DR_list.append(P["DR"])
        log(f"  cluster{m}: E={len(idx)} core_blocks≈{np.mean([len(cb) for cb in P['core_blocks']]):.1f}")

    # Pack and save
    core_pack = pack_blocks_ragged(core_all, cfg.QMODE)
    res_pack = pack_blocks_ragged(res_all, cfg.QMODE)

    store_dtype = np.float16 if cfg.BASIS_STORE_DTYPE == "float16" else np.float32
    arrays = {
        "expert_ids": np.array(eids, dtype=np.int32),
        "scales": scales.cpu().numpy().astype(np.float32),
        "cluster_of_pos": np.array(cluster_of_pos, dtype=np.int16),
        "n_clusters": np.array([len(clusters)], dtype=np.int32),
    }
    for m in range(len(clusters)):
        arrays[f"U_{m}"] = U_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"V_{m}"] = V_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DL_{m}"] = DL_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DR_{m}"] = DR_list[m].cpu().numpy().astype(store_dtype)
    if cfg.RES_COEF == "diag":
        arrays["gam"] = gam.cpu().numpy().astype(store_dtype)
    else:
        arrays["Cfull"] = Cfull.cpu().numpy().astype(store_dtype)

    for k, v in core_pack.items():
        arrays["core_"+k] = v
    for k, v in res_pack.items():
        arrays["res_"+k] = v

    out_path = os.path.join(cfg.OUTPUT_DIR, f"ebc_payload_layer{cfg.LAYER}.npz")
    save_npz(out_path, arrays)
    log(f"[save] payload -> {out_path} size={os.path.getsize(out_path)/1e6:.2f} MB")

    # Evaluate
    rt = PayloadRuntime(arrays, DEVICE)
    eval_payload(rt, Ws, scales)
    log("✅ Done.")

if __name__ == "__main__":
    main()

✅ flash_attn stub installed (CPU mode).
EBC-LLM Universal MoE Compression
Model: /data/downloaded_models/Phi-3.5-MoE-instruct  Output: /home/daniyar/moe_ws_outputs_phi_compact
Layer: 0  Max Experts: 0
[found] layer=0 total=16 using=16
[cache] loaded Ws shape=torch.Size([16, 4096, 4096])
[Ws] shape=torch.Size([16, 4096, 4096])
[cluster] M=6 sizes=[3, 4, 2, 3, 2, 2]
[train] step   1/24 loss=0.4101 (+2.7s)
[train] step   4/24 loss=0.0239 (+7.3s)
[train] step   8/24 loss=-0.0025 (+8.7s)
[train] step  12/24 loss=-0.0115 (+8.7s)
[train] step  16/24 loss=0.0391 (+8.7s)
[train] step  20/24 loss=0.0303 (+8.7s)
[train] step  24/24 loss=0.0451 (+8.7s)
[build] payloads...
  cluster0: E=3 core_blocks≈6.7
  cluster1: E=4 core_blocks≈6.5
  cluster2: E=2 core_blocks≈7.0
  cluster3: E=3 core_blocks≈7.0
  cluster4: E=2 core_blocks≈6.0
  cluster5: E=2 core_blocks≈6.0
[save] payload -> /home/daniyar/moe_ws_outputs_phi_compact/ebc_payload_layer0.npz size=28.50 MB
[eval] per-expert rel-error mean=0.409486 p

In [8]:
#!/usr/bin/env python3
# =============================================================================
# EBC-LLM: Universal MoE Compression – Mixtral, Qwen, Phi, DeepSeek
# =============================================================================
# Usage in Jupyter: just run this cell. Adjust cfg defaults at the top.
# =============================================================================

import os, sys, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs): return x

# -----------------------------------------------------------------------------
# FlashAttention stub (CPU safe)
# -----------------------------------------------------------------------------
import types, importlib.machinery

def _install_flash_attn_stub():
    if "flash_attn" in sys.modules:
        return
    flash_attn = types.ModuleType("flash_attn")
    flash_attn.__version__ = "0.0.0-cpu-stub"
    def _unavailable(*a, **k):
        raise RuntimeError("flash_attn stub called on CPU")
    flash_attn.flash_attn_func = _unavailable
    flash_attn.flash_attn_varlen_func = _unavailable
    flash_attn.flash_attn_with_kvcache = _unavailable
    flash_attn.layers = types.ModuleType("flash_attn.layers")
    flash_attn.layers.rotary = types.ModuleType("flash_attn.layers.rotary")
    class RotaryEmbedding(nn.Module):
        def __init__(self, dim, **kw): super().__init__()
        def forward(self, x, **kw):
            return torch.ones(1, device=x.device), torch.zeros(1, device=x.device)
    flash_attn.layers.rotary.RotaryEmbedding = RotaryEmbedding
    flash_attn.layers.rotary.apply_rotary_emb = lambda *a, **k: (_unavailable,)
    sys.modules["flash_attn"] = flash_attn
    sys.modules["flash_attn.layers"] = flash_attn.layers
    sys.modules["flash_attn.layers.rotary"] = flash_attn.layers.rotary

class FlashAttnImporter:
    def find_spec(self, fullname, path, target=None):
        if fullname == "flash_attn" or fullname.startswith("flash_attn."):
            _install_flash_attn_stub()
            return importlib.machinery.ModuleSpec(fullname, self)
        return None
    def create_module(self, spec): return sys.modules.get(spec.name)
    def exec_module(self, module): pass

sys.meta_path.insert(0, FlashAttnImporter())
print("✅ flash_attn stub installed (CPU mode).", flush=True)

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = "/data/downloaded_models/Phi-3.5-MoE-instruct"
    OUTPUT_DIR: str = "/home/daniyar/moe_ws_outputs_phi_compact"
    LAYER: int = int(os.environ.get("LAYER", "0"))
    MAX_EXPERTS: int = int(os.environ.get("MAX_EXPERTS", "0"))   # 0 = all

    CALIB_SAMPLES: int = int(os.environ.get("CALIB_SAMPLES", "4096"))
    RIDGE_DAMP: float = float(os.environ.get("RIDGE_DAMP", "1e-3"))
    NORMALIZE_W: bool = os.environ.get("NORMALIZE_W", "1") == "1"
    CAPTURE_ENABLE: bool = os.environ.get("CAPTURE_ENABLE", "1") == "1"
    CAPTURE_ITERS: int = int(os.environ.get("CAPTURE_ITERS", "32"))
    CAPTURE_MAX_TOKENS: int = int(os.environ.get("CAPTURE_MAX_TOKENS", "1024"))
    CAPTURE_TEXT: str = os.environ.get("CAPTURE_TEXT", "MoE calibration text. " * 256)
    HF_TRUST_REMOTE_CODE: bool = True
    HF_LOCAL_FILES_ONLY: bool = True

    # Clustering
    CLUSTER_FEAT_D: int = 64
    CLUSTER_ITERS: int = 60
    CLUSTER_RESTARTS: int = 4
    CLUSTER_MIN_SIZE: int = 2
    CLUSTER_MAX_SIZE: int = 4
    M_MAX: int = 16

    # Training
    TRAIN_STEPS: int = int(os.environ.get("TRAIN_STEPS", "24"))
    TRAIN_WARMUP: int = 6
    TRAIN_LR: float = 5e-2
    SUBM: int = 256
    BATCH_E: int = 4
    TRAIN_MIN_CLUSTER: int = 2
    REORTHO_EVERY: int = 4
    REPORT_EVERY: int = 4
    GRAD_CLIP: float = 1.0
    TRAIN_OBJ: str = "logratio"
    TRAIN_LAM_BLOCK: float = 0.10
    TRAIN_LAM_GUIDE: float = 1.0
    TRAIN_GUIDE_TARGET: float = 0.80
    TRAIN_GUIDE_EVERY: int = 2
    TRAIN_GUIDE_MAX_BLOCKS: int = 2048

    # Core selection
    CORE_BLOCK: int = 64
    CORE_TARGET: float = 0.75
    CORE_MAX_BLOCKS: int = 128

    # Residual
    RES_RANK: int = int(os.environ.get("RES_RANK", "256"))
    RES_COEF: str = "diag"
    RES_TARGET: float = 0.99
    RES_MAX_BLOCKS: int = 1024
    RES_BSIZE: int = 64

    # Refine
    REFINE_ENABLE: bool = False
    REFINE_ERR_TARGET: float = 0.03
    REFINE_MAX_EXTRA: int = 4096

    QMODE: str = "none"
    BASIS_STORE_DTYPE: str = "float16"
    SEED: int = 1234

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# Utilities
# -----------------------------------------------------------------------------
def log(msg): print(msg, flush=True)
def now(): return time.strftime("%Y-%m-%d %H:%M:%S")

def seed_all(s): random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all(cfg.SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE_ACC = torch.float32
torch.set_num_threads(8)

def save_npz(path, arrays): np.savez_compressed(path, **arrays)
def load_npz(path): return dict(np.load(path, allow_pickle=False))

# -----------------------------------------------------------------------------
# Weight loading – Universal expert detection
# -----------------------------------------------------------------------------
def read_index(model_dir):
    idx = os.path.join(model_dir, "model.safetensors.index.json")
    if os.path.isfile(idx):
        with open(idx) as f:
            return json.load(f).get("weight_map", {})
    files = [f for f in os.listdir(model_dir) if f.endswith(".safetensors")]
    if len(files) == 1:
        with safe_open(os.path.join(model_dir, files[0]), framework="pt") as sf:
            keys = list(sf.keys())
        return {k: files[0] for k in keys}
    raise RuntimeError("No safetensors index or single file found")

def find_expert_prefix(weight_map, layer):
    candidates = [
        f"model.layers.{layer}.mlp.experts.",
        f"model.layers.{layer}.block_sparse_moe.experts.",
        f"model.layers.{layer}.moe.experts.",
        f"transformer.h.{layer}.mlp.experts.",
    ]
    for p in candidates:
        if any(k.startswith(p) for k in weight_map):
            return p
    for k in weight_map:
        if f"layers.{layer}" in k and "experts" in k:
            return k.split("experts.")[0] + "experts."
    return None

def find_expert_ids(weight_map, layer):
    prefix = find_expert_prefix(weight_map, layer)
    if not prefix:
        return []
    ids = set()
    for k in weight_map:
        if k.startswith(prefix):
            parts = k[len(prefix):].split('.')
            if parts and parts[0].isdigit():
                ids.add(int(parts[0]))
    return sorted(ids)

def pick_expert_keys(weight_map, layer, eid):
    prefix = find_expert_prefix(weight_map, layer)
    if not prefix:
        return {}
    full = f"{prefix}{eid}."

    gate = up = down = None
    for suf in ["w1.weight", "gate_proj.weight", "gate.weight"]:
        if full + suf in weight_map:
            gate = full + suf
            break
    for suf in ["w3.weight", "up_proj.weight", "up.weight"]:
        if full + suf in weight_map:
            up = full + suf
            break
    for suf in ["w2.weight", "down_proj.weight", "down.weight"]:
        if full + suf in weight_map:
            down = full + suf
            break

    if gate and up and down:
        return {"gate": gate, "up": up, "down": down}
    return {}

def load_tensors(model_dir, weight_map, keys):
    shard_map = {}
    for k in keys:
        shard = weight_map.get(k)
        if shard:
            shard_map.setdefault(shard, []).append(k)
    tensors = {}
    for shard, ks in shard_map.items():
        with safe_open(os.path.join(model_dir, shard), framework="pt") as f:
            for k in ks:
                tensors[k] = f.get_tensor(k)
    return tensors

# -----------------------------------------------------------------------------
# Calibration capture
# -----------------------------------------------------------------------------
class _Collector:
    def __init__(self, H, E, max_rows):
        self.H = H; self.E = E; self.max = max_rows
        self.X = []; self.P = []; self.nX = 0; self.nP = 0
    def _take(self, flat, need):
        return flat[:need] if flat.shape[0] > need else flat
    def add_X(self, hs):
        if hs is None: return
        if hs.ndim == 2: hs = hs.unsqueeze(0)
        flat = hs.detach().float().cpu().reshape(-1, self.H)
        if flat.numel() == 0: return
        need = self.max - self.nX
        if need > 0:
            self.X.append(self._take(flat, need))
            self.nX += self.X[-1].shape[0]
    def add_logits(self, logits):
        if logits is None: return
        if logits.ndim == 2: logits = logits.unsqueeze(0)
        P = torch.softmax(logits.detach().float(), -1)[..., :self.E].cpu()
        flat = P.reshape(-1, P.shape[-1])
        if flat.numel() == 0: return
        need = self.max - self.nP
        if need > 0:
            self.P.append(self._take(flat, need))
            self.nP += self.P[-1].shape[0]

def capture_XP(model_dir, layer_idx, H, E, out_x, out_p):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
                                        local_files_only=cfg.HF_LOCAL_FILES_ONLY)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
        attn_implementation="eager"
    ).to(DEVICE).eval()

    # Locate layers
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        layers = model.model.layers
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        layers = model.transformer.h
    else:
        layers = model.layers
    layer = layers[layer_idx]

    # Locate MoE block
    mlp = None
    for name in ["mlp", "moe", "block_sparse_moe"]:
        if hasattr(layer, name):
            mlp = getattr(layer, name)
            break
    if mlp is None:
        for n, m in layer.named_modules():
            if "expert" in n.lower() and hasattr(m, "w1"):
                mlp = m
                break
    if mlp is None:
        raise RuntimeError("MoE block not found")
    log(f"[capture] MoE block: {mlp.__class__.__name__}")

    # Router linear (optional)
    router = None
    for n, m in layer.named_modules():
        if isinstance(m, nn.Linear) and m.in_features == H and m.out_features >= E:
            if "router" in n.lower() or "gate" in n.lower():
                router = m
                break

    coll = _Collector(H, E, cfg.CALIB_SAMPLES)

    def pre_hook(_, inputs):
        coll.add_X(inputs[0])
    h1 = mlp.register_forward_pre_hook(pre_hook)
    h2 = None
    if router:
        def router_hook(_, __, out):
            o = out[0] if isinstance(out, (tuple, list)) else out
            coll.add_logits(o)
        h2 = router.register_forward_hook(router_hook)

    text = cfg.CAPTURE_TEXT
    for it in range(cfg.CAPTURE_ITERS):
        enc = tok(text, return_tensors="pt", truncation=True,
                  max_length=cfg.CAPTURE_MAX_TOKENS, padding="max_length")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.inference_mode():
            _ = model(**enc, use_cache=False)
        if (it+1) % 4 == 0:
            log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS} nX={coll.nX} nP={coll.nP}")
        if coll.nX >= cfg.CALIB_SAMPLES:
            break

    h1.remove()
    if h2: h2.remove()

    X = torch.cat(coll.X)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz(out_x, {"X": X})
    log(f"[capture] saved X to {out_x} shape={X.shape}")
    if coll.nP:
        P = torch.cat(coll.P)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
        save_npz(out_p, {"P": P})
        log(f"[capture] saved P to {out_p} shape={P.shape}")

# -----------------------------------------------------------------------------
# Ridge linearization (build square proxies Ws)
# -----------------------------------------------------------------------------
@torch.no_grad()
def forward_mlp(X, Wg, Wu, Wd):
    Xf = X.to(DTYPE_ACC)
    up = Xf @ Wu.to(DTYPE_ACC).t()
    gate = Xf @ Wg.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    return hid @ Wd.to(DTYPE_ACC).t()

def build_Ws(eids, wm):
    per_e = {}
    keys = []
    for eid in eids:
        kk = pick_expert_keys(wm, cfg.LAYER, eid)
        if not kk:
            raise RuntimeError(f"Expert {eid} missing tensors")
        per_e[eid] = kk
        keys += [kk["up"], kk["down"], kk["gate"]]
    T = load_tensors(cfg.MODEL_DIR, wm, keys)

    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = W_up0.shape[0], W_up0.shape[1]
    log(f"[shape] H={H} d_ff={dff}")

    calib_path = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    if not os.path.isfile(calib_path) and cfg.CAPTURE_ENABLE:
        capture_XP(cfg.MODEL_DIR, cfg.LAYER, H, len(eids), calib_path,
                   os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz"))

    X = torch.from_numpy(np.load(calib_path)["X"][:cfg.CALIB_SAMPLES].astype(np.float32))
    X = X.to(DEVICE, DTYPE_ACC)
    log(f"[calib] X: {X.shape}")

    XtX = X.t() @ X
    lam = cfg.RIDGE_DAMP * torch.trace(XtX).item() / H
    chol = torch.linalg.cholesky(XtX + lam * torch.eye(H, device=DEVICE, dtype=DTYPE_ACC))

    Ws_list, scales = [], []
    for eid in tqdm(eids, desc="Build Ws (ridge)"):
        Wg = T[per_e[eid]["gate"]].to(DEVICE)
        Wu = T[per_e[eid]["up"]].to(DEVICE)
        Wd = T[per_e[eid]["down"]].to(DEVICE)
        Y = forward_mlp(X, Wg, Wu, Wd)
        Wt = torch.cholesky_solve(X.t() @ Y, chol)
        W = Wt.t().contiguous()
        if cfg.NORMALIZE_W:
            s = torch.linalg.norm(W).clamp_min(1e-12).item()
            W = W / s
        else:
            s = 1.0
        Ws_list.append(W)
        scales.append(s)

    Ws = torch.stack(Ws_list).to(DEVICE, DTYPE_ACC)
    Sc = torch.tensor(scales, device=DEVICE, dtype=DTYPE_ACC)
    return Ws, Sc

def load_or_build_Ws():
    wm = read_index(cfg.MODEL_DIR)
    all_ids = find_expert_ids(wm, cfg.LAYER)
    if not all_ids:
        raise RuntimeError(f"No experts at layer {cfg.LAYER}")
    max_e = cfg.MAX_EXPERTS if cfg.MAX_EXPERTS > 0 else len(all_ids)
    eids = all_ids[:max_e]
    log(f"[found] layer={cfg.LAYER} total={len(all_ids)} using={len(eids)}")

    cache_path = os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}.npz")
    if os.path.isfile(cache_path):
        data = load_npz(cache_path)
        cached_eids = data.get("expert_ids", [])
        if set(cached_eids) == set(eids):
            Ws = torch.from_numpy(data["Ws"]).to(DEVICE, DTYPE_ACC)
            Sc = torch.from_numpy(data["scales"]).to(DEVICE, DTYPE_ACC)
            log(f"[cache] loaded Ws shape={Ws.shape}")
            return eids, Ws, Sc
        log("[cache] mismatch, rebuilding")

    Ws, Sc = build_Ws(eids, wm)
    save_npz(cache_path, {
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.cpu().numpy().astype(np.float32),
        "scales": Sc.cpu().numpy().astype(np.float32)
    })
    log(f"[cache] saved Ws to {cache_path}")
    return eids, Ws, Sc

# -----------------------------------------------------------------------------
# Clustering
# -----------------------------------------------------------------------------
def random_proj_features(Ws, d):
    E, n, _ = Ws.shape
    flat = Ws.reshape(E, -1)
    proj = torch.randn(flat.shape[1], d, device=DEVICE, dtype=DTYPE_ACC)
    proj = proj / torch.linalg.norm(proj, dim=0, keepdim=True)
    return flat @ proj

def kmeans(X, k, iters=20, restarts=5):
    best_lab, best_inertia = None, float('inf')
    for _ in range(restarts):
        centers = X[torch.randperm(X.shape[0])[:k]].clone()
        for _ in range(iters):
            dist = torch.cdist(X, centers)
            lab = dist.argmin(1)
            for j in range(k):
                if (lab == j).any():
                    centers[j] = X[lab == j].mean(0)
        inertia = torch.cdist(X, centers).min(1)[0].pow(2).sum().item()
        if inertia < best_inertia:
            best_inertia, best_lab = inertia, lab.clone()
    return best_lab

def relabel_contiguous(labels):
    uniq = torch.unique(labels)
    out = labels.clone()
    for new, old in enumerate(uniq.tolist()):
        out[labels == old] = new
    return out

def merge_small_clusters(X, labels, min_size):
    labels = relabel_contiguous(labels)
    while True:
        K = labels.max().item() + 1
        counts = torch.bincount(labels, minlength=K)
        small = (counts < min_size).nonzero(as_tuple=False).flatten()
        if small.numel() == 0:
            break
        large = torch.where(counts >= min_size)[0]
        centers = torch.stack([X[labels == c].mean(0) for c in large])
        for c in small.tolist():
            idx = (labels == c).nonzero(as_tuple=False).flatten()
            if idx.numel() == 0: continue
            dist = torch.cdist(X[idx].mean(0, keepdim=True), centers)
            nearest = large[dist.argmin().item()]
            labels[idx] = nearest
        labels = relabel_contiguous(labels)
    return labels

def hierarchical_split(X, labels, max_size, max_k, split_iters=20):
    labels = relabel_contiguous(labels)
    while True:
        K = labels.max().item() + 1
        if K >= max_k:
            break
        counts = torch.bincount(labels, minlength=K)
        big = (counts > max_size).nonzero(as_tuple=False).flatten()
        if big.numel() == 0:
            break
        c = big[counts[big].argmax().item()]
        idx = (labels == c).nonzero(as_tuple=False).flatten()
        if idx.numel() < 2: break
        sub = X[idx]
        sub_lab = kmeans(sub, 2, split_iters, restarts=3)
        labels[idx[sub_lab == 1]] = K
        labels = relabel_contiguous(labels)
    return labels

def cluster_experts(Ws):
    Xf = random_proj_features(Ws, cfg.CLUSTER_FEAT_D)
    M0 = max(2, min(int(2 * math.sqrt(Ws.shape[0])), Ws.shape[0]))
    labels = kmeans(Xf, M0, cfg.CLUSTER_ITERS, cfg.CLUSTER_RESTARTS)
    labels = merge_small_clusters(Xf, labels, cfg.CLUSTER_MIN_SIZE)
    labels = hierarchical_split(Xf, labels, cfg.CLUSTER_MAX_SIZE, cfg.M_MAX, cfg.CLUSTER_ITERS)
    labels = merge_small_clusters(Xf, labels, cfg.CLUSTER_MIN_SIZE)
    return relabel_contiguous(labels)

# -----------------------------------------------------------------------------
# Orthogonal parameterization
# -----------------------------------------------------------------------------
class OrthoParam(nn.Module):
    def __init__(self, M):
        super().__init__()
        self.M = nn.Parameter(M.to(DEVICE, DTYPE_ACC).contiguous())
    def orthogonal(self):
        Q, _ = torch.linalg.qr(self.M)
        return Q

def svd_init_from_mean(Wmean, rank=None):
    r = rank if rank is not None else cfg.RES_RANK
    U, S, Vh = torch.linalg.svd(Wmean, full_matrices=False)
    U0 = U[:, :r] @ torch.diag(torch.sqrt(S[:r]))
    V0 = Vh[:r, :].t() @ torch.diag(torch.sqrt(S[:r]))
    return U0, V0

# -----------------------------------------------------------------------------
# Training utilities (robust slice_X_batch using matmul)
# -----------------------------------------------------------------------------
def schedule(step, warmup, total):
    if step <= warmup:
        return 0.0
    return min(1.0, (step - warmup) / max(1, total - warmup))

def slice_X_batch(Wb, U, V, S):
    """
    Wb: either (n, n) or (B, n, n)
    Returns X_full[:, :, S] where X_full = U^T @ Wb @ V
    """
    if Wb.ndim == 2:
        Wb = Wb.unsqueeze(0)          # (1, n, n)
    intermediate = torch.matmul(Wb, V)            # (B, n, r)
    X_full = torch.matmul(U.t(), intermediate)    # (B, r, r)  [U.t() is (r, n)]
    return X_full[:, :, S]

def offdiag_abs_mean(Xs):
    mask = ~torch.eye(Xs.shape[1], device=Xs.device, dtype=torch.bool)
    return Xs[:, mask].abs().mean()

def diag_abs_mean(Xs):
    d = min(Xs.shape[1], Xs.shape[2])
    return Xs[:, torch.arange(d), torch.arange(d)].abs().mean()

def block_group_sparsity_penalty(Xs, block_size):
    B, r, c = Xs.shape
    if c % block_size != 0:
        return torch.tensor(0.0, device=Xs.device)
    nb = c // block_size
    Xb = Xs.view(B, r, nb, block_size)
    block_norms = torch.linalg.norm(Xb, dim=(1, 3))
    return block_norms.mean()

def make_guidance_mask(Xs, block_size, target, max_blocks):
    B, r, c = Xs.shape
    if c % block_size != 0:
        return torch.ones_like(Xs), 1.0, c
    nb = c // block_size
    Xb = Xs.view(B, r, nb, block_size)
    block_norms = torch.linalg.norm(Xb, dim=(1, 3)).mean(0)  # (nb,)
    total = block_norms.sum()
    sorted_norms, indices = torch.sort(block_norms, descending=True)
    cumsum = torch.cumsum(sorted_norms, 0)
    k = torch.searchsorted(cumsum, target * total).item() + 1
    k = min(k, max_blocks)
    mask_blocks = torch.zeros(nb, device=Xs.device)
    mask_blocks[indices[:k]] = 1.0
    mask = mask_blocks.repeat_interleave(block_size).unsqueeze(0).unsqueeze(0).expand(B, r, c)
    kept_energy = (cumsum[k-1] / total).item() if total > 0 else 1.0
    return mask, kept_energy, k * block_size

# -----------------------------------------------------------------------------
# Rotation training
# -----------------------------------------------------------------------------
def train_rotations(Ws, clusters):
    U_par, V_par = [], []
    for idx in clusters:
        Wm = Ws[idx].mean(0)
        U0, V0 = svd_init_from_mean(Wm)
        U_par.append(OrthoParam(U0))
        V_par.append(OrthoParam(V0))

    if cfg.TRAIN_STEPS == 0:
        return [p.orthogonal().detach() for p in U_par], [p.orthogonal().detach() for p in V_par]

    params = [p.M for p in U_par] + [p.M for p in V_par]
    opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)
    guidance_masks = {}
    t0 = time.perf_counter()

    # S is sampled from the reduced rank r, not from H.
    r = cfg.RES_RANK
    sub_sample_size = min(cfg.SUBM, r)

    for step in range(1, cfg.TRAIN_STEPS+1):
        S = torch.randperm(r)[:sub_sample_size].to(DEVICE)

        if cfg.TRAIN_LAM_GUIDE > 0 and (step == 1 or step % cfg.TRAIN_GUIDE_EVERY == 0):
            with torch.no_grad():
                guidance_masks.clear()
                for m, idx in enumerate(clusters):
                    if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
                    Uo = U_par[m].orthogonal()
                    Vo = V_par[m].orthogonal()
                    pick = idx if cfg.BATCH_E >= len(idx) else random.sample(idx, cfg.BATCH_E)
                    if not isinstance(pick, list): pick = [pick]
                    Xs = slice_X_batch(Ws[pick], Uo, Vo, S)
                    mask, _, _ = make_guidance_mask(Xs, cfg.CORE_BLOCK, cfg.TRAIN_GUIDE_TARGET, cfg.TRAIN_GUIDE_MAX_BLOCKS)
                    guidance_masks[m] = mask

        lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
        lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
        lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp
        loss_total = 0.0
        n_terms = 0

        for m, idx in enumerate(clusters):
            if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
            Uo = U_par[m].orthogonal()
            Vo = V_par[m].orthogonal()
            pick = idx if cfg.BATCH_E >= len(idx) else random.sample(idx, cfg.BATCH_E)
            if not isinstance(pick, list): pick = [pick]
            Xs = slice_X_batch(Ws[pick], Uo, Vo, S)

            off = offdiag_abs_mean(Xs)
            diag = diag_abs_mean(Xs).clamp_min(1e-6)
            if cfg.TRAIN_OBJ == "logratio":
                base = torch.log(off + 1e-6) - torch.log(diag)
            else:
                base = off / diag

            if lam_block > 0:
                base = base + lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)
            if lam_guide > 0 and m in guidance_masks:
                Mmask = guidance_masks[m]
                Etot = (Xs * Xs).mean().clamp_min(1e-12)
                Eout = ((Xs * (1 - Mmask)) ** 2).mean()
                base = base + lam_guide * (Eout / Etot)

            loss_total += base
            n_terms += 1

        if n_terms == 0:
            break
        loss = loss_total / n_terms
        opt.zero_grad()
        loss.backward()
        if cfg.GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_(params, cfg.GRAD_CLIP)
        opt.step()

        if step % cfg.REORTHO_EVERY == 0 or step == cfg.TRAIN_STEPS:
            with torch.no_grad():
                for p in U_par: p.M.copy_(p.orthogonal())
                for p in V_par: p.M.copy_(p.orthogonal())

        if step % cfg.REPORT_EVERY == 0 or step == 1:
            t1 = time.perf_counter()
            log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={loss.item():.4f} (+{t1-t0:.1f}s)")
            t0 = t1

    return [p.orthogonal().detach() for p in U_par], [p.orthogonal().detach() for p in V_par]

# -----------------------------------------------------------------------------
# Payload construction (block-TopK core + low-rank + sparse residuals)
# -----------------------------------------------------------------------------
@torch.no_grad()
def block_energy_grid(X, b):
    n = X.shape[0]
    nb = (n + b - 1) // b
    if n % b != 0:
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()
    Eg = (Xb * Xb).sum(dim=(2, 3))
    tot = (X * X).sum().item()
    return Eg, tot, nb

@torch.no_grad()
def pick_blocks_until_target(Eg, tot, target, max_blocks, exclude=None):
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    picked, eacc = [], 0.0
    exclude = exclude or set()
    for idx in order.tolist():
        if len(picked) >= max_blocks:
            break
        e = flat[idx].item()
        if e <= 1e-18: break
        bi, bj = idx // nb, idx % nb
        if (bi, bj) in exclude: continue
        picked.append((bi, bj))
        eacc += e
        if eacc / max(tot, 1e-12) >= target:
            break
    return picked, eacc / max(tot, 1e-12)

@torch.no_grad()
def gather_block(X, i0, j0, b):
    n = X.shape[0]
    i1, j1 = min(n, i0+b), min(n, j0+b)
    return X[i0:i1, j0:j1].contiguous()

@torch.no_grad()
def rand_svd_vectors(A, r, n_iter=2):
    n = A.shape[0]
    r = min(r, n)
    Omega = torch.randn(n, r, device=A.device, dtype=DTYPE_ACC)
    Y = A @ Omega
    for _ in range(n_iter):
        Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    Uhat, _, Vh = torch.linalg.svd(B, full_matrices=False)
    return (Q @ Uhat[:, :r]).contiguous(), Vh.t()[:, :r].contiguous()

@torch.no_grad()
def frob_rel_err(A, B):
    return (torch.linalg.norm(A - B) / torch.linalg.norm(B).clamp_min(1e-12)).item()

@torch.no_grad()
def build_payload_for_cluster(Ws, idx, U, V):
    n = Ws.shape[1]
    X_list = [(U.t() @ Ws[pos] @ V).contiguous() for pos in idx]
    b = cfg.CORE_BLOCK

    # Core blocks
    core_per = []
    for X in X_list:
        Eg, te, nb = block_energy_grid(X, b)
        picks, _ = pick_blocks_until_target(Eg, te, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*b, bj*b
            blocks.append((i0, j0, gather_block(X, i0, j0, b)))
        core_per.append(blocks)

    # Residual after core
    R_list = []
    for X, cb in zip(X_list, core_per):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    # Shared low-rank
    Rmean = torch.stack(R_list).mean(0)
    r = min(cfg.RES_RANK, n)
    DL, DR = rand_svd_vectors(Rmean, r)

    coef_list, res_per = [], []
    bb = cfg.RES_BSIZE
    for j, Rm in enumerate(R_list):
        if cfg.RES_COEF == "diag":
            g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()
            coef_list.append(g)
            R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()
        else:
            C = (DL.t() @ Rm @ DR).contiguous()
            coef_list.append(C)
            R2 = (Rm - (DL @ C @ DR.t())).contiguous()

        Eg2, te2, nb2 = block_energy_grid(R2, bb)
        exclude = {(i0//bb, j0//bb) for (i0, j0, _) in core_per[j]}
        picks, _ = pick_blocks_until_target(Eg2, te2, cfg.RES_TARGET, cfg.RES_MAX_BLOCKS, exclude=exclude)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*bb, bj*bb
            blocks.append((i0, j0, gather_block(R2, i0, j0, bb)))
        res_per.append(blocks)

    # Refine (optional)
    if cfg.REFINE_ENABLE:
        rb = cfg.REFINE_BSIZE
        for j in range(len(idx)):
            X = X_list[j]
            def reconstruct():
                Xc = torch.zeros_like(X)
                for (i0, j0, Bc) in core_per[j]:
                    h, w = Bc.shape
                    Xc[i0:i0+h, j0:j0+w] = Bc
                if cfg.RES_COEF == "diag":
                    Xlr = (DL * coef_list[j].view(1, -1)) @ DR.t()
                else:
                    Xlr = DL @ coef_list[j] @ DR.t()
                Xr = torch.zeros_like(X)
                for (i0, j0, Br) in res_per[j]:
                    h, w = Br.shape
                    Xr[i0:i0+h, j0:j0+w] += Br
                return Xc + Xlr + Xr

            Xhat = reconstruct()
            err = frob_rel_err(Xhat, X)
            added = 0
            core_pos = {(i0, j0) for (i0, j0, _) in core_per[j]}
            res_pos = {(i0, j0) for (i0, j0, _) in res_per[j]}
            while err > cfg.REFINE_ERR_TARGET and added < cfg.REFINE_MAX_EXTRA:
                Rerr = (X - Xhat).contiguous()
                Eg, te, nb = block_energy_grid(Rerr, rb)
                flat = Eg.reshape(-1)
                if flat.max().item() <= 1e-18: break
                order = torch.argsort(flat, descending=True)
                found = False
                for idx_ in order.tolist():
                    bi, bj = idx_ // nb, idx_ % nb
                    i0, j0 = bi*rb, bj*rb
                    if (i0, j0) in core_pos or (i0, j0) in res_pos:
                        continue
                    Br = gather_block(Rerr, i0, j0, rb)
                    res_per[j].append((i0, j0, Br))
                    res_pos.add((i0, j0))
                    added += 1
                    found = True
                    break
                if not found: break
                if added % 32 == 0:
                    Xhat = reconstruct()
                    err = frob_rel_err(Xhat, X)
            Xhat = reconstruct()
            err = frob_rel_err(Xhat, X)

    return {
        "core_blocks": core_per,
        "res_blocks": res_per,
        "coef_list": coef_list,
        "DL": DL, "DR": DR
    }

# -----------------------------------------------------------------------------
# Packing ragged blocks
# -----------------------------------------------------------------------------
def pack_blocks_ragged(blocks_per_expert, qmode):
    all_vals = []
    indices = []
    for exp_blocks in blocks_per_expert:
        exp_idx = []
        for (i0, j0, B) in exp_blocks:
            start = len(all_vals)
            vals = B.flatten().cpu().numpy().astype(np.float32)
            all_vals.append(vals)
            end = start + len(vals)
            exp_idx.append((i0, j0, B.shape[0], B.shape[1], start, end))
        indices.append(exp_idx)
    flat = np.concatenate(all_vals) if all_vals else np.array([], dtype=np.float32)

    i0_list, j0_list, h_list, w_list, ptr_list = [], [], [], [], [0]
    for exp_idx in indices:
        for (i0, j0, h, w, start, end) in exp_idx:
            i0_list.append(i0); j0_list.append(j0); h_list.append(h); w_list.append(w)
            ptr_list.append(ptr_list[-1] + h*w)
    return {
        "flat": flat,
        "i0": np.array(i0_list, dtype=np.int16),
        "j0": np.array(j0_list, dtype=np.int16),
        "h": np.array(h_list, dtype=np.int16),
        "w": np.array(w_list, dtype=np.int16),
        "ptr": np.array(ptr_list, dtype=np.int64),
        "expert_ptr": np.cumsum([0] + [len(idx) for idx in indices], dtype=np.int32)
    }

# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------
class PayloadRuntime:
    def __init__(self, data, device):
        self.device = device
        self.scales = torch.from_numpy(data["scales"]).to(device, dtype=DTYPE_ACC)
        self.cluster_of = torch.from_numpy(data["cluster_of_pos"]).to(device, torch.long)
        n_clusters = data["n_clusters"][0]
        # Explicitly convert to DTYPE_ACC (float32) to avoid dtype mismatch
        self.U = [torch.from_numpy(data[f"U_{m}"]).to(device, dtype=DTYPE_ACC) for m in range(n_clusters)]
        self.V = [torch.from_numpy(data[f"V_{m}"]).to(device, dtype=DTYPE_ACC) for m in range(n_clusters)]
        self.DL = [torch.from_numpy(data[f"DL_{m}"]).to(device, dtype=DTYPE_ACC) for m in range(n_clusters)]
        self.DR = [torch.from_numpy(data[f"DR_{m}"]).to(device, dtype=DTYPE_ACC) for m in range(n_clusters)]
        if "gam" in data:
            self.gam = torch.from_numpy(data["gam"]).to(device, dtype=DTYPE_ACC)
            self.Cfull = None
        else:
            self.Cfull = torch.from_numpy(data["Cfull"]).to(device, dtype=DTYPE_ACC)
            self.gam = None
        self.core_flat = torch.from_numpy(data["core_flat"]).to(device, dtype=DTYPE_ACC)
        self.res_flat = torch.from_numpy(data["res_flat"]).to(device, dtype=DTYPE_ACC)
        self.core_info = (data["core_i0"], data["core_j0"], data["core_h"], data["core_w"],
                          data["core_ptr"], data["core_expert_ptr"])
        self.res_info = (data["res_i0"], data["res_j0"], data["res_h"], data["res_w"],
                         data["res_ptr"], data["res_expert_ptr"])

    def reconstruct_expert(self, pos):
        cid = self.cluster_of[pos].item()
        U, V = self.U[cid], self.V[cid]
        DL, DR = self.DL[cid], self.DR[cid]
        r = U.shape[1]
        X = torch.zeros(r, r, device=self.device, dtype=DTYPE_ACC)
        # Core blocks
        exp_ptr = self.core_info[5]
        for bi in range(exp_ptr[pos], exp_ptr[pos+1]):
            i0 = self.core_info[0][bi]; j0 = self.core_info[1][bi]
            h = self.core_info[2][bi]; w = self.core_info[3][bi]
            v0 = self.core_info[4][bi]; v1 = self.core_info[4][bi+1]
            B = self.core_flat[v0:v1].view(h, w)
            X[i0:i0+h, j0:j0+w] += B
        # Low-rank
        if self.gam is not None:
            X += (DL * self.gam[pos].view(1, -1)) @ DR.t()
        else:
            X += DL @ self.Cfull[pos] @ DR.t()
        # Residual blocks
        res_ptr = self.res_info[5]
        for bi in range(res_ptr[pos], res_ptr[pos+1]):
            i0 = self.res_info[0][bi]; j0 = self.res_info[1][bi]
            h = self.res_info[2][bi]; w = self.res_info[3][bi]
            v0 = self.res_info[4][bi]; v1 = self.res_info[4][bi+1]
            B = self.res_flat[v0:v1].view(h, w)
            X[i0:i0+h, j0:j0+w] += B
        return U @ X @ V.t() * self.scales[pos]

def eval_payload(rt, Ws_orig, scales):
    E = Ws_orig.shape[0]
    errs = []
    for pos in range(E):
        W_rec = rt.reconstruct_expert(pos)
        W_orig = Ws_orig[pos] * scales[pos]
        err = torch.linalg.norm(W_rec - W_orig).item() / torch.linalg.norm(W_orig).item()
        errs.append(err)
    log(f"[eval] per-expert rel-error mean={np.mean(errs):.6f} p95={np.percentile(errs,95):.6f} max={np.max(errs):.6f}")

# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def main():
    log("="*60)
    log("EBC-LLM Universal MoE Compression")
    log(f"Model: {cfg.MODEL_DIR}  Output: {cfg.OUTPUT_DIR}")
    log(f"Layer: {cfg.LAYER}  Max Experts: {cfg.MAX_EXPERTS}")
    log("="*60)

    eids, Ws, scales = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={Ws.shape}")

    # Clustering
    labels = cluster_experts(Ws)
    M = labels.max().item() + 1
    clusters = [torch.where(labels == m)[0].tolist() for m in range(M)]
    clusters = [c for c in clusters if c]
    log(f"[cluster] M={len(clusters)} sizes={[len(c) for c in clusters]}")
    cluster_of_pos = [0]*E
    for m, idx in enumerate(clusters):
        for pos in idx:
            cluster_of_pos[pos] = m

    # Train rotations
    U_list, V_list = train_rotations(Ws, clusters)

    # Build payloads
    log("[build] payloads...")
    core_all = [[] for _ in range(E)]
    res_all = [[] for _ in range(E)]
    DL_list, DR_list = [], []
    gam = torch.zeros((E, cfg.RES_RANK), device=DEVICE, dtype=DTYPE_ACC) if cfg.RES_COEF == "diag" else None
    Cfull = torch.zeros((E, cfg.RES_RANK, cfg.RES_RANK), device=DEVICE, dtype=DTYPE_ACC) if cfg.RES_COEF == "full" else None

    for m, idx in enumerate(clusters):
        U, V = U_list[m], V_list[m]
        P = build_payload_for_cluster(Ws, idx, U, V)
        for j, pos in enumerate(idx):
            core_all[pos] = P["core_blocks"][j]
            res_all[pos] = P["res_blocks"][j]
            if cfg.RES_COEF == "diag":
                gam[pos, :P["coef_list"][j].numel()] = P["coef_list"][j]
            else:
                C = P["coef_list"][j]
                Cfull[pos, :C.shape[0], :C.shape[1]] = C
        DL_list.append(P["DL"]); DR_list.append(P["DR"])
        log(f"  cluster{m}: E={len(idx)} core_blocks≈{np.mean([len(cb) for cb in P['core_blocks']]):.1f}")

    # Pack and save
    core_pack = pack_blocks_ragged(core_all, cfg.QMODE)
    res_pack = pack_blocks_ragged(res_all, cfg.QMODE)

    store_dtype = np.float16 if cfg.BASIS_STORE_DTYPE == "float16" else np.float32
    arrays = {
        "expert_ids": np.array(eids, dtype=np.int32),
        "scales": scales.cpu().numpy().astype(np.float32),
        "cluster_of_pos": np.array(cluster_of_pos, dtype=np.int16),
        "n_clusters": np.array([len(clusters)], dtype=np.int32),
    }
    for m in range(len(clusters)):
        arrays[f"U_{m}"] = U_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"V_{m}"] = V_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DL_{m}"] = DL_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DR_{m}"] = DR_list[m].cpu().numpy().astype(store_dtype)
    if cfg.RES_COEF == "diag":
        arrays["gam"] = gam.cpu().numpy().astype(store_dtype)
    else:
        arrays["Cfull"] = Cfull.cpu().numpy().astype(store_dtype)

    for k, v in core_pack.items():
        arrays["core_"+k] = v
    for k, v in res_pack.items():
        arrays["res_"+k] = v

    out_path = os.path.join(cfg.OUTPUT_DIR, f"ebc_payload_layer{cfg.LAYER}.npz")
    save_npz(out_path, arrays)
    log(f"[save] payload -> {out_path} size={os.path.getsize(out_path)/1e6:.2f} MB")

    # Evaluate
    rt = PayloadRuntime(arrays, DEVICE)
    eval_payload(rt, Ws, scales)
    log("✅ Done.")

if __name__ == "__main__":
    main()

✅ flash_attn stub installed (CPU mode).
EBC-LLM Universal MoE Compression
Model: /data/downloaded_models/Phi-3.5-MoE-instruct  Output: /home/daniyar/moe_ws_outputs_phi_compact
Layer: 0  Max Experts: 0
[found] layer=0 total=16 using=16
[cache] loaded Ws shape=torch.Size([16, 4096, 4096])
[Ws] shape=torch.Size([16, 4096, 4096])
[cluster] M=6 sizes=[3, 4, 2, 3, 2, 2]
[train] step   1/24 loss=0.4101 (+2.7s)
[train] step   4/24 loss=0.0239 (+7.1s)
[train] step   8/24 loss=-0.0025 (+8.5s)
[train] step  12/24 loss=-0.0115 (+8.6s)
[train] step  16/24 loss=0.0391 (+8.5s)
[train] step  20/24 loss=0.0303 (+8.5s)
[train] step  24/24 loss=0.0451 (+8.6s)
[build] payloads...
  cluster0: E=3 core_blocks≈6.7
  cluster1: E=4 core_blocks≈6.5
  cluster2: E=2 core_blocks≈7.0
  cluster3: E=3 core_blocks≈7.0
  cluster4: E=2 core_blocks≈6.0
  cluster5: E=2 core_blocks≈6.0
[save] payload -> /home/daniyar/moe_ws_outputs_phi_compact/ebc_payload_layer0.npz size=28.50 MB
[eval] per-expert rel-error mean=0.409486 p

In [10]:
import os
import json
import numpy as np
from safetensors import safe_open

# ===== EDIT THESE =====
MODEL_DIR = "/data/downloaded_models/Phi-3.5-MoE-instruct"
LAYER = 0
MAX_EXPERTS = 0          # 0 = use all found
PAYLOAD_PATH = "/home/daniyar/moe_ws_outputs_phi_compact/ebc_payload_layer0.npz"
# ======================

def read_index(model_dir):
    idx = os.path.join(model_dir, "model.safetensors.index.json")
    if os.path.isfile(idx):
        with open(idx) as f:
            return json.load(f).get("weight_map", {})
    files = [f for f in os.listdir(model_dir) if f.endswith(".safetensors")]
    if len(files) == 1:
        with safe_open(os.path.join(model_dir, files[0]), framework="pt") as sf:
            keys = list(sf.keys())
        return {k: files[0] for k in keys}
    raise RuntimeError("No safetensors index or single file found")

def find_expert_prefix(wm, layer):
    for p in [f"model.layers.{layer}.mlp.experts.",
              f"model.layers.{layer}.block_sparse_moe.experts.",
              f"model.layers.{layer}.moe.experts.",
              f"transformer.h.{layer}.mlp.experts."]:
        if any(k.startswith(p) for k in wm):
            return p
    for k in wm:
        if f"layers.{layer}" in k and "experts" in k:
            return k.split("experts.")[0] + "experts."
    return None

def find_expert_ids(wm, layer):
    p = find_expert_prefix(wm, layer)
    if not p: return []
    ids = set()
    for k in wm:
        if k.startswith(p):
            parts = k[len(p):].split('.')
            if parts and parts[0].isdigit():
                ids.add(int(parts[0]))
    return sorted(ids)

def pick_expert_keys(wm, layer, eid):
    p = find_expert_prefix(wm, layer)
    if not p: return {}
    full = f"{p}{eid}."
    gate = up = down = None
    for suf in ["w1.weight", "gate_proj.weight", "gate.weight"]:
        if full+suf in wm: gate = full+suf; break
    for suf in ["w3.weight", "up_proj.weight", "up.weight"]:
        if full+suf in wm: up = full+suf; break
    for suf in ["w2.weight", "down_proj.weight", "down.weight"]:
        if full+suf in wm: down = full+suf; break
    if gate and up and down:
        return {"gate": gate, "up": up, "down": down}
    return {}

def tensor_size(model_dir, wm, key):
    shard = wm.get(key)
    if not shard: return 0, None
    with safe_open(os.path.join(model_dir, shard), framework="pt") as f:
        t = f.get_tensor(key)
        return t.numel() * t.element_size(), t.dtype

# ----- Main -----
wm = read_index(MODEL_DIR)
all_ids = find_expert_ids(wm, LAYER)
max_e = MAX_EXPERTS if MAX_EXPERTS > 0 else len(all_ids)
eids = all_ids[:max_e]

print(f"Model: {MODEL_DIR}")
print(f"Layer {LAYER}: using {len(eids)} / {len(all_ids)} experts\n")

total_bytes = 0
dtype_counts = {}
for eid in eids:
    keys = pick_expert_keys(wm, LAYER, eid)
    if not keys:
        print(f"Expert {eid}: missing keys")
        continue
    for role, key in keys.items():
        sz, dt = tensor_size(MODEL_DIR, wm, key)
        total_bytes += sz
        dtype_counts[str(dt)] = dtype_counts.get(str(dt), 0) + 1
        print(f"  e{eid} {role}: {sz/1e6:.2f} MB  ({dt})")

print(f"\nTotal original expert weight size: {total_bytes/1e6:.2f} MB  ({total_bytes/1e9:.3f} GB)")
print(f"Dtype(s) used: {dtype_counts}")

if os.path.isfile(PAYLOAD_PATH):
    payload_sz = os.path.getsize(PAYLOAD_PATH)
    print(f"\nPayload size: {payload_sz/1e6:.2f} MB")
    ratio = total_bytes / payload_sz if payload_sz else float('inf')
    print(f"Compression ratio (orig / payload): {ratio:.2f}x")
    if ratio < 1:
        print("⚠️ Payload is LARGER than original.")
    else:
        print("✅ Payload is SMALLER.")
else:
    print(f"\nPayload not found: {PAYLOAD_PATH}")

Model: /data/downloaded_models/Phi-3.5-MoE-instruct
Layer 0: using 16 / 16 experts

  e0 gate: 52.43 MB  (torch.bfloat16)
  e0 up: 52.43 MB  (torch.bfloat16)
  e0 down: 52.43 MB  (torch.bfloat16)
  e1 gate: 52.43 MB  (torch.bfloat16)
  e1 up: 52.43 MB  (torch.bfloat16)
  e1 down: 52.43 MB  (torch.bfloat16)
  e2 gate: 52.43 MB  (torch.bfloat16)
  e2 up: 52.43 MB  (torch.bfloat16)
  e2 down: 52.43 MB  (torch.bfloat16)
  e3 gate: 52.43 MB  (torch.bfloat16)
  e3 up: 52.43 MB  (torch.bfloat16)
  e3 down: 52.43 MB  (torch.bfloat16)
  e4 gate: 52.43 MB  (torch.bfloat16)
  e4 up: 52.43 MB  (torch.bfloat16)
  e4 down: 52.43 MB  (torch.bfloat16)
  e5 gate: 52.43 MB  (torch.bfloat16)
  e5 up: 52.43 MB  (torch.bfloat16)
  e5 down: 52.43 MB  (torch.bfloat16)
  e6 gate: 52.43 MB  (torch.bfloat16)
  e6 up: 52.43 MB  (torch.bfloat16)
  e6 down: 52.43 MB  (torch.bfloat16)
  e7 gate: 52.43 MB  (torch.bfloat16)
  e7 up: 52.43 MB  (torch.bfloat16)
  e7 down: 52.43 MB  (torch.bfloat16)
  e8 gate: 52.43 MB 

In [11]:
import os
import json
import numpy as np
from safetensors import safe_open

# =============================================================================
# CONFIGURATION – Edit these paths to match your setup
# =============================================================================
MODELS = [
    {
        "name": "Mixtral-8x7B",
        "model_dir": "/data/downloaded_models/Mixtral-8x7B-v0.1",
        "layer": 0,
        "max_experts": 0,  # 0 = all found
        "payload": "/home/daniyar/moe_ws_outputs/ebc_payload_layer0_E8_qnone.npz",
    },
    {
        "name": "Qwen1.5-MoE",
        "model_dir": "/data/downloaded_models/Qwen1.5-MoE-A2.7B",
        "layer": 0,
        "max_experts": 0,
        "payload": "/home/daniyar/moe_ws_outputs_qwen/ebc_payload_layer0_E4_qnone.npz",
    },
    {
        "name": "Phi-3.5-MoE",
        "model_dir": "/data/downloaded_models/Phi-3.5-MoE-instruct",
        "layer": 0,
        "max_experts": 0,
        "payload": "/home/daniyar/moe_ws_outputs_phi_compact/ebc_payload_layer0.npz",  # compact run
    },
    {
        "name": "DeepSeek-V2-Lite",
        "model_dir": "/data/downloaded_models/DeepSeek-V2-Lite",
        "layer": 1,                     # DeepSeek used layer 1
        "max_experts": 0,
        "payload": "/home/daniyar/moe_ws_outputs_deepseek/ebc_payload_layer1_E16_qnone.npz",
    },
]

# =============================================================================
# Utility functions (same as before)
# =============================================================================
def read_index(model_dir):
    idx = os.path.join(model_dir, "model.safetensors.index.json")
    if os.path.isfile(idx):
        with open(idx) as f:
            return json.load(f).get("weight_map", {})
    files = [f for f in os.listdir(model_dir) if f.endswith(".safetensors")]
    if len(files) == 1:
        with safe_open(os.path.join(model_dir, files[0]), framework="pt") as sf:
            keys = list(sf.keys())
        return {k: files[0] for k in keys}
    raise RuntimeError("No safetensors index or single file found")

def find_expert_prefix(wm, layer):
    for p in [f"model.layers.{layer}.mlp.experts.",
              f"model.layers.{layer}.block_sparse_moe.experts.",
              f"model.layers.{layer}.moe.experts.",
              f"transformer.h.{layer}.mlp.experts."]:
        if any(k.startswith(p) for k in wm):
            return p
    for k in wm:
        if f"layers.{layer}" in k and "experts" in k:
            return k.split("experts.")[0] + "experts."
    return None

def find_expert_ids(wm, layer):
    p = find_expert_prefix(wm, layer)
    if not p: return []
    ids = set()
    for k in wm:
        if k.startswith(p):
            parts = k[len(p):].split('.')
            if parts and parts[0].isdigit():
                ids.add(int(parts[0]))
    return sorted(ids)

def pick_expert_keys(wm, layer, eid):
    p = find_expert_prefix(wm, layer)
    if not p: return {}
    full = f"{p}{eid}."
    gate = up = down = None
    for suf in ["w1.weight", "gate_proj.weight", "gate.weight"]:
        if full+suf in wm: gate = full+suf; break
    for suf in ["w3.weight", "up_proj.weight", "up.weight"]:
        if full+suf in wm: up = full+suf; break
    for suf in ["w2.weight", "down_proj.weight", "down.weight"]:
        if full+suf in wm: down = full+suf; break
    if gate and up and down:
        return {"gate": gate, "up": up, "down": down}
    return {}

def tensor_size(model_dir, wm, key):
    shard = wm.get(key)
    if not shard: return 0, None
    with safe_open(os.path.join(model_dir, shard), framework="pt") as f:
        t = f.get_tensor(key)
        return t.numel() * t.element_size(), t.dtype

# =============================================================================
# Main loop over models
# =============================================================================
print("\n" + "="*80)
print("EXPERT WEIGHT SIZE ANALYSIS")
print("="*80)

summary = []
for cfg in MODELS:
    name = cfg["name"]
    model_dir = cfg["model_dir"]
    layer = cfg["layer"]
    max_exp = cfg["max_experts"]
    payload_path = cfg["payload"]

    print(f"\n--- {name} ---")
    if not os.path.isdir(model_dir):
        print(f"  Model dir not found: {model_dir}")
        continue

    wm = read_index(model_dir)
    all_ids = find_expert_ids(wm, layer)
    if not all_ids:
        print(f"  No experts found at layer {layer}")
        continue

    n_used = max_exp if max_exp > 0 else len(all_ids)
    eids = all_ids[:n_used]
    print(f"  Layer {layer}: using {len(eids)} / {len(all_ids)} experts")

    total_bytes = 0
    for eid in eids:
        keys = pick_expert_keys(wm, layer, eid)
        if not keys:
            continue
        for role, key in keys.items():
            sz, _ = tensor_size(model_dir, wm, key)
            total_bytes += sz

    orig_mb = total_bytes / 1e6
    print(f"  Original expert weights: {orig_mb:.2f} MB ({total_bytes/1e9:.3f} GB)")

    payload_mb = None
    ratio = None
    if payload_path and os.path.isfile(payload_path):
        payload_bytes = os.path.getsize(payload_path)
        payload_mb = payload_bytes / 1e6
        ratio = total_bytes / payload_bytes if payload_bytes > 0 else float('inf')
        print(f"  Payload size: {payload_mb:.2f} MB")
        print(f"  Compression ratio (orig/payload): {ratio:.2f}x")
        if ratio < 1:
            print("  ⚠️  Payload is LARGER than original.")
        else:
            print("  ✅ Payload is SMALLER.")
    else:
        print(f"  Payload not found: {payload_path}")

    summary.append({
        "Model": name,
        "Experts": len(eids),
        "Orig (MB)": orig_mb,
        "Payload (MB)": payload_mb if payload_mb else "-",
        "Ratio": f"{ratio:.2f}x" if ratio else "-",
    })

# =============================================================================
# Summary table
# =============================================================================
print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
print(f"{'Model':<20} {'Experts':<8} {'Orig (MB)':<12} {'Payload (MB)':<14} {'Ratio':<10}")
print("-"*80)
for row in summary:
    print(f"{row['Model']:<20} {row['Experts']:<8} {row['Orig (MB)']:<12.2f} {str(row['Payload (MB)']):<14} {row['Ratio']:<10}")
print("="*80)


EXPERT WEIGHT SIZE ANALYSIS

--- Mixtral-8x7B ---
  Layer 0: using 8 / 8 experts
  Original expert weights: 2818.57 MB (2.819 GB)
  Payload size: 704.19 MB
  Compression ratio (orig/payload): 4.00x
  ✅ Payload is SMALLER.

--- Qwen1.5-MoE ---
  Layer 0: using 60 / 60 experts
  Original expert weights: 1038.09 MB (1.038 GB)
  Payload size: 81.65 MB
  Compression ratio (orig/payload): 12.71x
  ✅ Payload is SMALLER.

--- Phi-3.5-MoE ---
  Layer 0: using 16 / 16 experts
  Original expert weights: 2516.58 MB (2.517 GB)
  Payload size: 28.50 MB
  Compression ratio (orig/payload): 88.31x
  ✅ Payload is SMALLER.

--- DeepSeek-V2-Lite ---
  Layer 1: using 64 / 64 experts
  Original expert weights: 1107.30 MB (1.107 GB)
  Payload size: 343.99 MB
  Compression ratio (orig/payload): 3.22x
  ✅ Payload is SMALLER.

SUMMARY TABLE
Model                Experts  Orig (MB)    Payload (MB)   Ratio     
--------------------------------------------------------------------------------
Mixtral-8x7B         8

In [2]:
#!/usr/bin/env python3
# =============================================================================
# EBC-LLM: Enhanced Expert-Bank Compression with Full Metrics & Ablation
# =============================================================================
# This is a superset of the original EBC-LLM pipeline. It adds:
#   - Exact original weight size computation (from safetensors)
#   - Compression ratio logging
#   - Additional fidelity metrics: cosine similarity, PSNR, SDR
#   - Statistical significance tests (bootstrap confidence intervals)
#   - Ablation study options to disable components
#   - JSON export of all metrics for paper-ready tables
# =============================================================================

import os, re, json, math, time, random, sys
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Set
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs): return x

# -----------------------------------------------------------------------------
# Environment helpers
# -----------------------------------------------------------------------------
def _env_str(k: str, d: str) -> str:
    return os.environ.get(k, d)

def _env_int(k: str, d: int) -> int:
    try: return int(os.environ.get(k, str(d)))
    except: return d

def _env_float(k: str, d: float) -> float:
    try: return float(os.environ.get(k, str(d)))
    except: return d

def _env_bool(k: str, d: bool) -> bool:
    v = os.environ.get(k, None)
    if v is None: return d
    return v.strip().lower() in ("1", "true", "yes", "y", "on")

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
@dataclass
class Cfg:
    # Paths
    MODEL_DIR: str = "/data/downloaded_models/Mixtral-8x7B-v0.1"
    OUTPUT_DIR: str = "/home/daniyar/moe_ws_outputs"

    # Model slice
    LAYER: int = 0
    MAX_EXPERTS: int = 8

    # Calibration / router
    CALIB_PATH: str = _env_str("CALIB_PATH", "").strip()
    ROUTER_PATH: str = _env_str("ROUTER_PATH", "").strip()
    CALIB_SAMPLES: int = _env_int("CALIB_SAMPLES", 4096)
    RIDGE_WEIGHTED: bool = _env_bool("RIDGE_WEIGHTED", False)
    ROUTER_EIDS_ARE_GLOBAL: bool = _env_bool("ROUTER_EIDS_ARE_GLOBAL", True)
    RIDGE_DAMP: float = _env_float("RIDGE_DAMP", 1e-3)
    NORMALIZE_W: bool = _env_bool("NORMALIZE_W", True)

    # Capture
    CAPTURE_ENABLE: bool = True
    CAPTURE_FORCE: bool = _env_bool("CAPTURE_FORCE", False)
    CAPTURE_ITERS: int = _env_int("CAPTURE_ITERS", 32)
    CAPTURE_BATCH: int = _env_int("CAPTURE_BATCH", 1)
    CAPTURE_MAX_TOKENS: int = _env_int("CAPTURE_MAX_TOKENS", 1024)
    CAPTURE_TEXT: str = _env_str("CAPTURE_TEXT", "DeepSeek MoE calibration text. " * 256)
    CAPTURE_TEXT_FILE: str = _env_str("CAPTURE_TEXT_FILE", "").strip()
    CAPTURE_KEEP_PAD: bool = _env_bool("CAPTURE_KEEP_PAD", False)
    HF_TRUST_REMOTE_CODE: bool = _env_bool("HF_TRUST_REMOTE_CODE", True)
    HF_LOCAL_FILES_ONLY: bool = _env_bool("HF_LOCAL_FILES_ONLY", True)
    HF_AUTO_PIP: bool = _env_bool("HF_AUTO_PIP", False)

    # Basis mode
    BASIS_MODE: str = _env_str("BASIS_MODE", "dense_train").lower()
    BASIS_STORE_DTYPE: str = _env_str("BASIS_STORE_DTYPE", "float16").lower()

    # Clustering
    M0: int = _env_int("M0", 0)
    M_MAX: int = _env_int("M_MAX", 16)
    CLUSTER_FEAT_D: int = _env_int("CLUSTER_FEAT_D", 64)
    CLUSTER_ITERS: int = _env_int("CLUSTER_ITERS", 60)
    CLUSTER_RESTARTS: int = _env_int("CLUSTER_RESTARTS", 4)
    CLUSTER_MIN_SIZE: int = _env_int("CLUSTER_MIN_SIZE", 2)
    CLUSTER_MAX_SIZE: int = _env_int("CLUSTER_MAX_SIZE", 4)
    SPLIT_ITERS: int = _env_int("SPLIT_ITERS", 50)

    # Training
    TRAIN_STEPS: int = _env_int("TRAIN_STEPS", 24)
    TRAIN_WARMUP: int = _env_int("TRAIN_WARMUP", 6)
    TRAIN_LR: float = _env_float("TRAIN_LR", 5e-2)
    SUBM: int = _env_int("SUBM", 256)
    BATCH_E: int = _env_int("BATCH_E", 4)
    TRAIN_MIN_CLUSTER: int = _env_int("TRAIN_MIN_CLUSTER", 2)
    REORTHO_EVERY: int = _env_int("REORTHO_EVERY", 4)
    REPORT_EVERY: int = _env_int("REPORT_EVERY", 4)
    GRAD_CLIP: float = _env_float("GRAD_CLIP", 1.0)
    TRAIN_OBJ: str = _env_str("TRAIN_OBJ", "logratio").lower()
    TRAIN_LAM_BLOCK: float = _env_float("TRAIN_LAM_BLOCK", 0.10)
    TRAIN_LAM_GUIDE: float = _env_float("TRAIN_LAM_GUIDE", 1.0)
    TRAIN_GUIDE_EVERY: int = _env_int("TRAIN_GUIDE_EVERY", 2)
    TRAIN_GUIDE_TARGET: float = _env_float("TRAIN_GUIDE_TARGET", 0.80)
    TRAIN_GUIDE_MAX_BLOCKS: int = _env_int("TRAIN_GUIDE_MAX_BLOCKS", 2048)

    # Core selection
    CORE_MODE: str = _env_str("CORE_MODE", "blocktopk_perexpert").lower()
    CORE_AGG: str = _env_str("CORE_AGG", "mean").lower()
    CORE_BLOCK: int = _env_int("CORE_BLOCK", 64)
    CORE_TARGET: float = _env_float("CORE_TARGET", 0.85)
    CORE_MAX_BLOCKS: int = _env_int("CORE_MAX_BLOCKS", 256)

    # Residual
    RES_RANK: int = _env_int("RES_RANK", 512)
    RES_COEF: str = _env_str("RES_COEF", "diag").lower()
    RES_TARGET: float = _env_float("RES_TARGET", 0.995)
    RES_MAX_BLOCKS: int = _env_int("RES_MAX_BLOCKS", 4096)
    RES_BSIZE: int = _env_int("RES_BSIZE", 64)

    # Refine
    REFINE_ENABLE: bool = _env_bool("REFINE_ENABLE", True)
    REFINE_ERR_TARGET: float = _env_float("REFINE_ERR_TARGET", 0.03)
    REFINE_MAX_EXTRA: int = _env_int("REFINE_MAX_EXTRA", 4096)
    REFINE_BSIZE: int = _env_int("REFINE_BSIZE", 64)
    REFINE_RECHECK_EVERY: int = _env_int("REFINE_RECHECK_EVERY", 32)

    # Quantization
    QMODE: str = _env_str("QMODE", "none").lower()

    # Eval
    EVAL_TRIALS: int = _env_int("EVAL_TRIALS", 8)
    EVAL_BATCH: int = _env_int("EVAL_BATCH", 2)
    ROUTED_K: int = _env_int("ROUTED_K", 8)

    # New: Ablation & extended metrics
    ABLATION_MODE: str = _env_str("ABLATION_MODE", "none")  # none|no_cluster|no_rot|no_lowrank|no_blocks
    COMPUTE_SIGNIFICANCE: bool = _env_bool("COMPUTE_SIGNIFICANCE", True)
    EXPORT_METRICS_JSON: bool = _env_bool("EXPORT_METRICS_JSON", True)

cfg = Cfg()
PRESET = _env_str("PRESET", "").strip().lower()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def _setdefault_env(k: str, v: str):
    if k not in os.environ: os.environ[k] = v

if PRESET == "maxacc":
    _setdefault_env("CALIB_SAMPLES", "32768")
    _setdefault_env("RIDGE_DAMP", "1e-2")
    _setdefault_env("CORE_BLOCK", "32")
    _setdefault_env("CORE_TARGET", "0.995")
    _setdefault_env("CORE_MAX_BLOCKS", "8192")
    _setdefault_env("RES_RANK", "2048")
    _setdefault_env("RES_COEF", "full")
    _setdefault_env("RES_TARGET", "0.999")
    _setdefault_env("RES_MAX_BLOCKS", "32768")
    _setdefault_env("REFINE_ENABLE", "1")
    _setdefault_env("REFINE_ERR_TARGET", "0.01")
    _setdefault_env("REFINE_MAX_EXTRA", "65536")
    _setdefault_env("TRAIN_STEPS", "96")
    _setdefault_env("TRAIN_LR", "0.02")
    _setdefault_env("TRAIN_LAM_GUIDE", "0.5")
    cfg = Cfg()
elif PRESET == "compact":
    _setdefault_env("CALIB_SAMPLES", "4096")
    _setdefault_env("CORE_BLOCK", "64")
    _setdefault_env("CORE_TARGET", "0.90")
    _setdefault_env("CORE_MAX_BLOCKS", "512")
    _setdefault_env("RES_RANK", "512")
    _setdefault_env("RES_COEF", "diag")
    _setdefault_env("RES_TARGET", "0.99")
    _setdefault_env("RES_MAX_BLOCKS", "4096")
    _setdefault_env("QMODE", "float16")
    _setdefault_env("REFINE_ENABLE", "0")
    _setdefault_env("TRAIN_STEPS", "24")
    cfg = Cfg()

# -----------------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------------
def log(msg: str): print(msg, flush=True)
def now() -> str: return time.strftime("%Y-%m-%d %H:%M:%S")

def seed_all(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

SEED = _env_int("SEED", 1234)
seed_all(SEED)
NTHREADS = _env_int("KTXX_THREADS", 8)
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
try: torch.set_num_threads(NTHREADS)
except: pass

DEVICE = torch.device(_env_str("DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
DTYPE_ACC = torch.float32

# -----------------------------------------------------------------------------
# NPZ I/O
# -----------------------------------------------------------------------------
def save_npz_compressed(path: str, arrays: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, **arrays)

def load_npz(path: str) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    return {k: z[k] for k in z.files}

def _encode_meta(meta: dict) -> np.ndarray:
    return np.frombuffer(json.dumps(meta, sort_keys=True).encode("utf-8"), dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try: return json.loads(bytes(arr.tolist()).decode("utf-8"))
    except: return {}

# -----------------------------------------------------------------------------
# Offline shard loading
# -----------------------------------------------------------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r") as f:
        return json.load(f).get("weight_map", {})

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    patterns = [
        rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.",
        rf"^model\.layers\.{layer}\.block_sparse_moe\.experts\.(\d+)\.",
    ]
    ids = set()
    for pat_str in patterns:
        pat = re.compile(pat_str)
        for k in weight_map:
            m = pat.match(k)
            if m:
                ids.add(int(m.group(1)))
        if ids:
            break
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefixes = [
        f"model.layers.{layer}.mlp.experts.{eid}.",
        f"model.layers.{layer}.block_sparse_moe.experts.{eid}.",
    ]
    used_prefix = None
    for pfx in prefixes:
        if any(k.startswith(pfx) for k in weight_map):
            used_prefix = pfx
            break
    if used_prefix is None:
        return {}

    def pick(cands):
        for suf in cands:
            k = used_prefix + suf
            if k in weight_map:
                return k
        return None

    gate = pick(["w1.weight", "gate_proj.weight"])
    down = pick(["w2.weight", "down_proj.weight"])
    up   = pick(["w3.weight", "up_proj.weight"])

    if gate is None or down is None or up is None:
        return {}
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard = {}
    for k in keys:
        shard = weight_map.get(k)
        if shard is None: continue
        by_shard.setdefault(shard, []).append(k)
    out = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp): continue
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks: out[k] = f.get_tensor(k)
    return out

# -----------------------------------------------------------------------------
# Calibration / Router
# -----------------------------------------------------------------------------
def autodetect_calib_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return cand if os.path.isfile(cand) else None

def autodetect_router_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return cand if os.path.isfile(cand) else None

def load_calib_X(path: str, H: int) -> torch.Tensor:
    z = np.load(path)
    X = torch.from_numpy(z["X"].astype(np.float32))
    if X.ndim != 2 or X.shape[1] != H: raise RuntimeError(f"Bad X shape {X.shape}")
    if X.shape[0] > cfg.CALIB_SAMPLES: X = X[:cfg.CALIB_SAMPLES]
    return X.to(device=DEVICE, dtype=DTYPE_ACC)

def load_router_P(path: str) -> np.ndarray:
    return np.load(path)["P"].astype(np.float32)

def _maybe_autopip():
    if not cfg.HF_AUTO_PIP: return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", "transformers", "sentencepiece", "tokenizers"])

def _patch_transformers_cache_compat():
    try:
        from transformers.cache_utils import DynamicCache
        if not hasattr(DynamicCache, "get_usable_length"):
            DynamicCache.get_usable_length = lambda self, seq_length: int(seq_length)
    except: pass

class _Collector:
    def __init__(self, H, E_total, max_rows):
        self.H = H; self.E_total = E_total; self.max_rows = max_rows
        self.X_chunks, self.P_chunks = [], []; self.nX = self.nP = 0

    def _take(self, flat, need): return flat[:need] if flat.shape[0] > need else flat

    def add_X(self, hs, attn_mask):
        if hs is None: return
        if hs.ndim == 2: hs = hs.unsqueeze(0)
        if hs.ndim != 3 or hs.shape[-1] != self.H: return
        hs = hs.detach().to(torch.float32).cpu()
        if attn_mask is not None and not cfg.CAPTURE_KEEP_PAD:
            m = attn_mask.cpu().to(torch.bool); flat = hs.reshape(-1, self.H)[m.reshape(-1)]
        else: flat = hs.reshape(-1, self.H)
        if flat.numel() == 0: return
        need = self.max_rows - self.nX
        if need <= 0: return
        self.X_chunks.append(self._take(flat, need)); self.nX += self.X_chunks[-1].shape[0]

    def add_logits(self, logits, attn_mask):
        if logits is None: return
        if logits.ndim == 2: logits = logits.unsqueeze(0)
        if logits.ndim != 3: return
        P = torch.softmax(logits.detach().to(torch.float32), dim=-1)[..., :self.E_total].cpu()
        if attn_mask is not None and not cfg.CAPTURE_KEEP_PAD:
            m = attn_mask.cpu().to(torch.bool); flat = P.reshape(-1, P.shape[-1])[m.reshape(-1)]
        else: flat = P.reshape(-1, P.shape[-1])
        if flat.numel() == 0: return
        need = self.max_rows - self.nP
        if need <= 0: return
        self.P_chunks.append(self._take(flat, need)); self.nP += self.P_chunks[-1].shape[0]

def capture_XP_transformers(model_dir, layer_idx, H, E_total, out_x, out_p):
    _maybe_autopip(); _patch_transformers_cache_compat()
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=cfg.HF_TRUST_REMOTE_CODE, local_files_only=cfg.HF_LOCAL_FILES_ONLY)
    if tok.pad_token is None: tok.pad_token = tok.eos_token or tok.unk_token
    model = AutoModelForCausalLM.from_pretrained(model_dir, trust_remote_code=cfg.HF_TRUST_REMOTE_CODE, local_files_only=cfg.HF_LOCAL_FILES_ONLY, torch_dtype=torch.float16 if DEVICE.type=="cuda" else torch.float32, low_cpu_mem_usage=True).to(DEVICE).eval()

    layers = None
    if hasattr(model, "model") and hasattr(model.model, "layers"): layers = model.model.layers
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"): layers = model.transformer.h
    elif hasattr(model, "layers"): layers = model.layers
    if layers is None: raise RuntimeError("Cannot locate layers")
    if layer_idx >= len(layers): raise RuntimeError(f"Layer {layer_idx} out of range")
    layer = layers[layer_idx]
    mlp = getattr(layer, "mlp", None)
    if mlp is None:
        for n, m in layer.named_modules():
            if n.lower().endswith("mlp"): mlp = m; break
    if mlp is None: raise RuntimeError("Could not find layer.mlp")

    router_linear = None
    for name, mod in layer.named_modules():
        if isinstance(mod, nn.Linear) and mod.in_features == H and mod.out_features >= E_total:
            if "router" in name.lower() or "gate" in name.lower(): router_linear = mod; break

    coll = _Collector(H, E_total, cfg.CALIB_SAMPLES)
    attn_holder = {"mask": None}

    def mlp_pre_hook(_, inputs):
        coll.add_X(inputs[0], attn_holder["mask"])

    h1 = mlp.register_forward_pre_hook(mlp_pre_hook)
    h2 = None
    if router_linear is not None:
        def router_hook(_, __, out):
            o = out[0] if isinstance(out, (tuple, list)) else out
            coll.add_logits(o, attn_holder["mask"])
        h2 = router_linear.register_forward_hook(router_hook)

    texts = [cfg.CAPTURE_TEXT]
    if cfg.CAPTURE_TEXT_FILE and os.path.isfile(cfg.CAPTURE_TEXT_FILE):
        with open(cfg.CAPTURE_TEXT_FILE) as f: texts = [ln.strip() for ln in f if ln.strip()]
    tptr = 0
    for it in range(cfg.CAPTURE_ITERS):
        text = texts[tptr % len(texts)]; tptr += 1
        enc = tok(text, return_tensors="pt", truncation=True, max_length=cfg.CAPTURE_MAX_TOKENS, padding="max_length")
        for k in enc:
            if enc[k].ndim == 2 and cfg.CAPTURE_BATCH > 1: enc[k] = enc[k].repeat(cfg.CAPTURE_BATCH, 1)
        attn_holder["mask"] = enc.get("attention_mask")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.inference_mode(): _ = model(**enc, use_cache=False)
        if (it+1) % 4 == 0: log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS} nX={coll.nX} nP={coll.nP}")
        if coll.nX >= cfg.CALIB_SAMPLES and (not cfg.RIDGE_WEIGHTED or coll.nP >= cfg.CALIB_SAMPLES): break

    h1.remove()
    if h2: h2.remove()

    if coll.nX == 0: raise RuntimeError("Capture collected 0 rows")
    X = torch.cat(coll.X_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz_compressed(out_x, {"X": X})
    log(f"[capture] wrote X -> {out_x} shape={X.shape}")
    p_written = None
    if coll.nP > 0:
        P = torch.cat(coll.P_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
        N = min(P.shape[0], X.shape[0])
        if N < X.shape[0]: X = X[:N]; save_npz_compressed(out_x, {"X": X})
        P = P[:N]; save_npz_compressed(out_p, {"P": P})
        log(f"[capture] wrote P -> {out_p} shape={P.shape}")
        p_written = out_p
    return out_x, p_written

def ensure_calib_router(H: int, E_total: int):
    if not cfg.CALIB_PATH:
        c = autodetect_calib_path()
        if c: cfg.CALIB_PATH = c; log(f"[calib] auto-found {cfg.CALIB_PATH}")
    if not cfg.ROUTER_PATH:
        r = autodetect_router_path()
        if r: cfg.ROUTER_PATH = r; log(f"[router] auto-found {cfg.ROUTER_PATH}")
    if cfg.CAPTURE_FORCE or (cfg.CAPTURE_ENABLE and (not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH))):
        out_x = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
        out_p = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
        log("[capture] capturing via transformers...")
        x_path, p_path = capture_XP_transformers(cfg.MODEL_DIR, cfg.LAYER, H, E_total, out_x, out_p)
        cfg.CALIB_PATH = x_path
        if p_path: cfg.ROUTER_PATH = p_path

# -----------------------------------------------------------------------------
# Ridge linearization: build Ws
# -----------------------------------------------------------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate, W_up, W_down) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    return hid @ W_down.to(DTYPE_ACC).t()

def ws_cache_path(E: int) -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{E}_ridge_ebc.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        script="ebc_llm_enhanced", model_dir=cfg.MODEL_DIR, layer=cfg.LAYER, expert_ids=eids,
        ridge_damp=cfg.RIDGE_DAMP, ridge_weighted=cfg.RIDGE_WEIGHTED,
        router_path=cfg.ROUTER_PATH or "", calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES, normalize_w=cfg.NORMALIZE_W, seed=SEED, device=str(DEVICE)
    )

@torch.no_grad()
def build_Ws(eids: List[int], wm: Dict[str, str]) -> Tuple[torch.Tensor, torch.Tensor]:
    per_e, need_keys = {}, []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk: raise RuntimeError(f"Expert {eid} missing tensors")
        per_e[eid] = kk; need_keys += [kk["up"], kk["down"], kk["gate"]]
    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))
    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = W_up0.shape[0], W_up0.shape[1]
    log(f"[shape] H={H} d_ff={dff}")

    ensure_calib_router(H, len(find_layer_expert_ids(wm, cfg.LAYER)))
    if not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH):
        raise RuntimeError("CALIB_PATH missing. Set CALIB_PATH or CAPTURE_ENABLE=1.")
    X = load_calib_X(cfg.CALIB_PATH, H)
    log(f"[calib] X: {X.shape}")

    P = None
    if cfg.RIDGE_WEIGHTED:
        if cfg.ROUTER_PATH and os.path.isfile(cfg.ROUTER_PATH):
            P = load_router_P(cfg.ROUTER_PATH)
            log(f"[router] P: {P.shape}")
        else:
            log("[router] RIDGE_WEIGHTED=1 but ROUTER_PATH missing -> disabling.")
            cfg.RIDGE_WEIGHTED = False

    Xf = X.to(DTYPE_ACC); I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * torch.trace(XtX).item() / H
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list, scales = [], []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws (ridge)")):
        W_up = T[per_e[eid]["up"]].to(DEVICE)
        W_dn = T[per_e[eid]["down"]].to(DEVICE)
        W_gt = T[per_e[eid]["gate"]].to(DEVICE)
        Y = forward_mlp(X, W_gt, W_up, W_dn).to(DTYPE_ACC)

        if cfg.RIDGE_WEIGHTED and P is not None:
            w = torch.from_numpy(P[:X.shape[0], eid if cfg.ROUTER_EIDS_ARE_GLOBAL else i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0)
            sw = torch.sqrt(w + 1e-12).view(-1,1)
            Xw, Yw = Xf * sw, Y * sw
            XtX_e = Xw.t() @ Xw
            lam_e = cfg.RIDGE_DAMP * torch.trace(XtX_e).item() / H
            chol = torch.linalg.cholesky(XtX_e + lam_e * I)
            Wt = torch.cholesky_solve(Xw.t() @ Yw, chol)
            W = Wt.t().contiguous()
        else:
            Wt = torch.cholesky_solve(Xf.t() @ Y, cholG)
            W = Wt.t().contiguous()

        if cfg.NORMALIZE_W:
            s = torch.linalg.norm(W, ord="fro").clamp_min(1e-12).item()
            W = W / s
        else: s = 1.0
        Ws_list.append(W); scales.append(s)

    Ws = torch.stack(Ws_list).to(DTYPE_ACC).to(DEVICE)
    Sc = torch.tensor(scales, dtype=DTYPE_ACC, device=DEVICE)
    return Ws, Sc

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor, torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids: raise RuntimeError(f"No experts at layer {cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer={cfg.LAYER} total={len(all_eids)} using={len(eids)} eids={eids}")

    if not cfg.CALIB_PATH: cfg.CALIB_PATH = autodetect_calib_path() or ""
    if not cfg.ROUTER_PATH: cfg.ROUTER_PATH = autodetect_router_path() or ""

    cpath = ws_cache_path(len(eids))
    if os.path.isfile(cpath):
        z = load_npz(cpath)
        if all(k in z for k in ["meta","Ws","expert_ids","scales"]) and _decode_meta(z["meta"]) == ws_meta(eids):
            Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
            Sc = torch.from_numpy(z["scales"]).to(DTYPE_ACC).to(DEVICE)
            log(f"[cache] loaded Ws -> {cpath} shape={Ws.shape}")
            return [int(x) for x in z["expert_ids"]], Ws, Sc
        log("[cache] meta mismatch -> rebuild")

    Ws, Sc = build_Ws(eids, wm)
    save_npz_compressed(cpath, {
        "meta": _encode_meta(ws_meta(eids)),
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.cpu().numpy().astype(np.float32),
        "scales": Sc.cpu().numpy().astype(np.float32)
    })
    log(f"[cache] wrote Ws -> {cpath} size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws, Sc

# -----------------------------------------------------------------------------
# Clustering (kmeans++ + hierarchical split)
# -----------------------------------------------------------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED+17)
    R = (torch.randint(0,2,(n,d),generator=g,dtype=torch.int8)*2-1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]; row = torch.diag(W @ W.t()); col = torch.diag(W.t() @ W)
        feats.append(torch.cat([row @ R, col @ R]).unsqueeze(0))
    X = torch.cat(feats, dim=0)
    X = (X - X.mean(0, keepdim=True)) / (X.std(0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> torch.Tensor:
    best_lab, best_inertia = None, float("inf")
    g = torch.Generator(device="cpu").manual_seed(SEED+999)
    for _ in range(max(1, restarts)):
        n = X.shape[0]
        centers = [X[torch.randint(0, n, (1,), generator=g).item()].clone()]
        for _ in range(1, k):
            C = torch.stack(centers)
            dist2 = torch.cdist(X, C).pow(2).min(1).values
            prob = dist2 / dist2.sum().clamp_min(1e-12)
            centers.append(X[torch.multinomial(prob, 1, generator=g).item()].clone())
        C = torch.stack(centers)
        for _ in range(iters):
            dist = torch.cdist(X, C); lab = dist.argmin(1)
            for j in range(k):
                m = (lab == j)
                if m.any(): C[j] = X[m].mean(0)
                else: C[j] = X[dist.min(1).values.argmax().item()].clone()
        inertia = torch.cdist(X, C).min(1).values.pow(2).sum().item()
        if inertia < best_inertia: best_inertia, best_lab = inertia, lab.clone()
    return best_lab.to(torch.int64)

@torch.no_grad()
def relabel_contiguous(labels: torch.Tensor) -> torch.Tensor:
    uniq = torch.unique(labels); out = labels.clone()
    for new, old in enumerate(uniq.tolist()): out[labels == old] = new
    return out

@torch.no_grad()
def merge_small_clusters(X: torch.Tensor, labels: torch.Tensor, min_size: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if min_size <= 1: return labels
    while True:
        K = labels.max().item() + 1
        counts = torch.bincount(labels, minlength=K)
        small = (counts < min_size).nonzero(as_tuple=False).flatten()
        if small.numel() == 0: break
        C = torch.stack([X[labels == k].mean(0) for k in range(K)])
        for c in small.tolist():
            idxs = (labels == c).nonzero(as_tuple=False).flatten()
            if idxs.numel() == 0: continue
            dist = torch.cdist(C[c].unsqueeze(0), C).squeeze(0); dist[c] = 1e9
            labels[idxs] = dist.argmin().item()
        labels = relabel_contiguous(labels)
    return labels

@torch.no_grad()
def hierarchical_split(X: torch.Tensor, labels: torch.Tensor, max_size: int, max_k: int, split_iters: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if max_size <= 0: return labels
    while True:
        K = labels.max().item() + 1
        if K >= max_k: break
        counts = torch.bincount(labels, minlength=K)
        biggest = counts.argmax().item()
        if counts[biggest] <= max_size: break
        idxs = (labels == biggest).nonzero(as_tuple=False).flatten()
        if idxs.numel() < 2: break
        sub = X[idxs]; sub_lab = kmeans_torch(sub, 2, split_iters, 1)
        a, b = idxs[sub_lab == 0], idxs[sub_lab == 1]
        if a.numel() == 0 or b.numel() == 0: break
        labels[b] = K
        labels = relabel_contiguous(labels)
    return labels

# -----------------------------------------------------------------------------
# Basis training (dense)
# -----------------------------------------------------------------------------
class OrthoParam(nn.Module):
    def __init__(self, init_mat: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(init_mat.to(DEVICE, DTYPE_ACC).contiguous())
    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M); return Q

@torch.no_grad()
def svd_init_from_mean(Wmean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(Wmean, full_matrices=False)
    return U.to(DTYPE_ACC).contiguous(), Vh.t().to(DTYPE_ACC).contiguous()

def schedule(step: int, warmup: int, total: int) -> float:
    if step <= warmup: return 0.0
    return min(1.0, (step - warmup) / max(1, total - warmup))

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    U_S, V_S = U[:, S], V[:, S]
    return torch.matmul(U_S.t().unsqueeze(0), Ws_batch @ V_S)

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return (Xs - torch.diag_embed(D)).abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    return torch.diagonal(Xs, dim1=1, dim2=2).abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    Eb, s, _ = Xs.shape; b = int(block)
    if b <= 0: return torch.zeros((), device=Xs.device)
    nb = s // b
    if nb <= 0: return torch.zeros((), device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0,1,3,2,4).contiguous()
    Eblk = (Xb * Xb).sum(dim=(3,4))
    P = Eblk.mean(0)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

@torch.no_grad()
def make_guidance_mask_from_Xs(Xs: torch.Tensor, block: int, target: float, max_blocks: int) -> Tuple[torch.Tensor, float, int]:
    Eb, s, _ = Xs.shape; b = int(block)
    if b <= 0: return torch.ones(s,s,device=Xs.device), 1.0, 0
    nb = s // b
    if nb <= 0: return torch.ones(s,s,device=Xs.device), 1.0, 0
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0,1,3,2,4).contiguous()
    Eg = (Xb * Xb).sum(dim=(3,4)).mean(0)
    tot = (X * X).sum().item() / max(1, Eb)
    flat = Eg.reshape(-1); order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], 0)
    frac = csum / max(tot, 1e-12)
    need = (frac >= target).nonzero(as_tuple=False)[0].item() + 1 if (frac >= target).any() else flat.numel()
    K = min(need, max_blocks, flat.numel())
    mask = torch.zeros(s2, s2, device=Xs.device)
    for idx in order[:K].tolist():
        bi, bj = idx // nb, idx % nb
        mask[bi*b:(bi+1)*b, bj*b:(bj+1)*b] = 1.0
    if s2 < s:
        full = torch.zeros(s, s, device=Xs.device); full[:s2, :s2] = mask; mask = full
    ef = float(frac[K-1].item()) if K > 0 else 0.0
    return mask, ef, K

# -----------------------------------------------------------------------------
# Block energy & selection
# -----------------------------------------------------------------------------
@torch.no_grad()
def block_energy_grid(X: torch.Tensor, b: int) -> Tuple[torch.Tensor, float, int]:
    n = X.shape[0]; nb = (n + b - 1) // b
    if n % b != 0:
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X; X = Xp
    Xb = X.view(nb, b, nb, b).permute(0,2,1,3).contiguous()
    Eg = (Xb * Xb).sum(dim=(2,3))
    tot = (X * X).sum().item()
    return Eg, tot, nb

@torch.no_grad()
def pick_blocks_until_target(Eg: torch.Tensor, tot_energy: float, target: float, max_blocks: int,
                             exclude: Optional[Set[Tuple[int,int]]]=None) -> Tuple[List[Tuple[int,int]], float]:
    nb = Eg.shape[0]; flat = Eg.reshape(-1); order = torch.argsort(flat, descending=True)
    picked, eacc = [], 0.0
    exclude = exclude or set()
    for idx in order.tolist():
        if len(picked) >= max_blocks: break
        e = flat[idx].item()
        if e <= 1e-18: break
        bi, bj = idx // nb, idx % nb
        if (bi, bj) in exclude: continue
        picked.append((bi, bj)); eacc += e
        if eacc / max(tot_energy, 1e-12) >= target: break
    return picked, eacc / max(tot_energy, 1e-12)

@torch.no_grad()
def gather_block(X: torch.Tensor, i0: int, j0: int, b: int) -> torch.Tensor:
    n = X.shape[0]; i1, j1 = min(n, i0+b), min(n, j0+b)
    return X[i0:i1, j0:j1].contiguous()

# -----------------------------------------------------------------------------
# Low-rank (randomized SVD)
# -----------------------------------------------------------------------------
@torch.no_grad()
def rand_svd_vectors(A: torch.Tensor, r: int, n_iter: int=2) -> Tuple[torch.Tensor, torch.Tensor]:
    n = A.shape[0]; r = min(r, n)
    g = torch.Generator(device="cpu").manual_seed(SEED+777)
    Omega = torch.randn(n, r, generator=g, dtype=DTYPE_ACC, device=A.device)
    Y = A @ Omega
    for _ in range(n_iter): Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    Uhat, _, Vh = torch.linalg.svd(B, full_matrices=False)
    return (Q @ Uhat[:, :r]).contiguous(), Vh.t()[:, :r].contiguous()

# -----------------------------------------------------------------------------
# Payload packing (ragged blocks)
# -----------------------------------------------------------------------------
def _block_store_dtype(qmode: str) -> np.dtype:
    return np.float32 if qmode == "none" else np.float16

def pack_blocks_ragged(blocks_per_item: List[List[Tuple[int,int,torch.Tensor]]], qmode: str) -> Dict[str, np.ndarray]:
    val_dtype = _block_store_dtype(qmode)
    M = len(blocks_per_item)
    item_ptr = [0]
    blk_i0, blk_j0, blk_h, blk_w = [], [], [], []
    blk_ptr = [0]
    vals, vals_i8, scales = [], [], []
    for m in range(M):
        for (i0, j0, B) in blocks_per_item[m]:
            h, w = B.shape
            blk_i0.append(i0); blk_j0.append(j0); blk_h.append(h); blk_w.append(w)
            if qmode == "int8":
                x = B.cpu().float(); maxabs = x.abs().max().item()
                if maxabs < 1e-12: q = np.zeros(x.numel(), dtype=np.int8); sc = np.float16(1.0)
                else:
                    scale = maxabs / 127.0
                    q = torch.clamp(torch.round(x/scale), -127, 127).to(torch.int8).numpy()
                    sc = np.float16(scale)
                vals_i8.append(q.reshape(-1)); scales.append(sc)
                blk_ptr.append(blk_ptr[-1] + q.size)
            else:
                v = B.cpu().float().numpy().astype(val_dtype).reshape(-1)
                vals.append(v); blk_ptr.append(blk_ptr[-1] + v.size)
        item_ptr.append(len(blk_i0))

    out = {
        "item_ptr": np.array(item_ptr, dtype=np.int32),
        "blk_i0": np.array(blk_i0, dtype=np.int16),
        "blk_j0": np.array(blk_j0, dtype=np.int16),
        "blk_h": np.array(blk_h, dtype=np.int16),
        "blk_w": np.array(blk_w, dtype=np.int16),
        "blk_ptr": np.array(blk_ptr, dtype=np.int64)
    }
    if qmode == "int8":
        out["blk_q"] = np.concatenate(vals_i8).astype(np.int8) if vals_i8 else np.zeros((0,), dtype=np.int8)
        out["blk_scale"] = np.array(scales, dtype=np.float16)
    else:
        out["blk_val"] = np.concatenate(vals) if vals else np.zeros((0,), dtype=val_dtype)
    return out

def unpack_blocks_ragged(pack: Dict[str, np.ndarray], qmode: str, device: torch.device) -> List[List[Tuple[int,int,torch.Tensor]]]:
    item_ptr = pack["item_ptr"]
    blk_i0 = pack["blk_i0"]; blk_j0 = pack["blk_j0"]; blk_h = pack["blk_h"]; blk_w = pack["blk_w"]
    blk_ptr = pack["blk_ptr"]
    if qmode == "int8":
        blk_q = pack["blk_q"]; blk_scale = pack["blk_scale"]; blk_val = None
    else:
        blk_val = pack["blk_val"]; blk_q = None; blk_scale = None
    M = item_ptr.shape[0] - 1
    out = []
    for m in range(M):
        b0, b1 = item_ptr[m], item_ptr[m+1]
        lst = []
        for bi in range(b0, b1):
            i0, j0 = int(blk_i0[bi]), int(blk_j0[bi])
            h, w = int(blk_h[bi]), int(blk_w[bi])
            v0, v1 = blk_ptr[bi], blk_ptr[bi+1]
            if qmode == "int8":
                q = blk_q[v0:v1].astype(np.float32); sc = float(blk_scale[bi])
                B = torch.from_numpy((q * sc).reshape(h, w)).to(device, DTYPE_ACC)
            else:
                B = torch.from_numpy(blk_val[v0:v1].astype(np.float32).reshape(h, w)).to(device, DTYPE_ACC)
            lst.append((i0, j0, B))
        out.append(lst)
    return out

# -----------------------------------------------------------------------------
# Payload runtime
# -----------------------------------------------------------------------------
class PayloadRuntime:
    def __init__(self):
        self.meta = {}
        self.expert_ids = []
        self.scales: Optional[torch.Tensor] = None
        self.cluster_of_pos: Optional[torch.Tensor] = None
        self.U: List[torch.Tensor] = []
        self.V: List[torch.Tensor] = []
        self.DL: List[torch.Tensor] = []
        self.DR: List[torch.Tensor] = []
        self.gam: Optional[torch.Tensor] = None
        self.Cfull: Optional[torch.Tensor] = None
        self.core_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.res_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.qmode = "none"
        self.res_coef = "diag"

    @torch.no_grad()
    def apply_expert(self, x: torch.Tensor, pos: int) -> torch.Tensor:
        c = int(self.cluster_of_pos[pos].item())
        U, V = self.U[c], self.V[c]
        DL, DR = self.DL[c], self.DR[c]
        z = x @ U
        u = torch.zeros_like(z)
        for (i0, j0, B) in self.core_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B
        if self.res_coef == "diag":
            g = self.gam[pos]
            u += ((z @ DL) * g.view(1,-1)) @ DR.t()
        else:
            C = self.Cfull[pos]
            u += (z @ DL) @ C @ DR.t()
        for (i0, j0, B) in self.res_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B
        y = u @ V.t()
        if self.scales is not None:
            y = y * self.scales[pos]
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        for a, pos in zip(gates.tolist(), routed):
            y += a * self.apply_expert(x, int(pos))
        return y

    @torch.no_grad()
    def reconstruct_expert(self, pos: int) -> torch.Tensor:
        """Reconstruct the dense square matrix for expert `pos` from the payload."""
        c = int(self.cluster_of_pos[pos].item())
        U, V = self.U[c], self.V[c]
        DL, DR = self.DL[c], self.DR[c]
        r = U.shape[1]
        X = torch.zeros(r, r, device=U.device, dtype=DTYPE_ACC)

        # Core blocks
        for (i0, j0, B) in self.core_blocks[pos]:
            h, w = B.shape
            X[i0:i0+h, j0:j0+w] += B

        # Low-rank
        if self.res_coef == "diag":
            g = self.gam[pos]
            X += (DL * g.view(1, -1)) @ DR.t()
        else:
            C = self.Cfull[pos]
            X += DL @ C @ DR.t()

        # Residual blocks
        for (i0, j0, B) in self.res_blocks[pos]:
            h, w = B.shape
            X[i0:i0+h, j0:j0+w] += B

        W = U @ X @ V.t()
        if self.scales is not None:
            W = W * self.scales[pos]
        return W

def load_payload_runtime(path: str, device: torch.device) -> PayloadRuntime:
    z = load_npz(path)
    rt = PayloadRuntime()
    rt.meta = _decode_meta(z["meta"])
    rt.qmode = rt.meta.get("qmode", "none")
    rt.res_coef = rt.meta.get("res_coef", "diag")
    rt.expert_ids = [int(x) for x in z["expert_ids"]]
    rt.scales = torch.from_numpy(z["scales"]).to(device, DTYPE_ACC)
    rt.cluster_of_pos = torch.from_numpy(z["cluster_of_pos"]).to(device, torch.int64)
    M = z["n_clusters"][0]
    for m in range(M):
        rt.U.append(torch.from_numpy(z[f"U_{m}"]).to(device, DTYPE_ACC))
        rt.V.append(torch.from_numpy(z[f"V_{m}"]).to(device, DTYPE_ACC))
        rt.DL.append(torch.from_numpy(z[f"DL_{m}"]).to(device, DTYPE_ACC))
        rt.DR.append(torch.from_numpy(z[f"DR_{m}"]).to(device, DTYPE_ACC))
    if rt.res_coef == "diag":
        rt.gam = torch.from_numpy(z["gam"]).to(device, DTYPE_ACC)
    else:
        rt.Cfull = torch.from_numpy(z["Cfull"]).to(device, DTYPE_ACC)
    core_pack = {k[5:]: z[k] for k in z if k.startswith("core_")}
    res_pack  = {k[4:]: z[k] for k in z if k.startswith("res_")}
    rt.core_blocks = unpack_blocks_ragged(core_pack, rt.qmode, device)
    rt.res_blocks  = unpack_blocks_ragged(res_pack, rt.qmode, device)
    return rt

# -----------------------------------------------------------------------------
# Build payload for one cluster
# -----------------------------------------------------------------------------
@torch.no_grad()
def frob_rel_err(A, B): return (torch.linalg.norm(A-B) / torch.linalg.norm(B).clamp_min(1e-12)).item()

@torch.no_grad()
def build_payload_for_cluster(Ws_norm: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor) -> Dict:
    n = Ws_norm.shape[-1]
    X_list = [(U.t() @ Ws_norm[pos] @ V).contiguous() for pos in idx]
    b = cfg.CORE_BLOCK

    core_per = []
    core_ef = []
    for X in X_list:
        Eg, te, nb = block_energy_grid(X, b)
        picks, eff = pick_blocks_until_target(Eg, te, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*b, bj*b
            blocks.append((i0, j0, gather_block(X, i0, j0, b)))
        core_per.append(blocks); core_ef.append(eff)

    R_list = []
    for X, cb in zip(X_list, core_per):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb: h,w = Bc.shape; Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    Rmean = torch.stack(R_list).mean(0)
    r = min(cfg.RES_RANK, n)
    DL, DR = rand_svd_vectors(Rmean, r, n_iter=2)

    coef_list, res_per = [], []
    bb = cfg.RES_BSIZE
    for j, Rm in enumerate(R_list):
        if cfg.RES_COEF == "diag":
            g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()
            coef_list.append(g)
            R2 = (Rm - (DL * g.view(1,-1)) @ DR.t()).contiguous()
        else:
            C = (DL.t() @ Rm @ DR).contiguous()
            coef_list.append(C)
            R2 = (Rm - (DL @ C @ DR.t())).contiguous()

        Eg2, te2, nb2 = block_energy_grid(R2, bb)
        exclude = {(i0//bb, j0//bb) for (i0,j0,_) in core_per[j]}
        picks, _ = pick_blocks_until_target(Eg2, te2, cfg.RES_TARGET, cfg.RES_MAX_BLOCKS, exclude=exclude)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*bb, bj*bb
            blocks.append((i0, j0, gather_block(R2, i0, j0, bb)))
        res_per.append(blocks)

    if cfg.REFINE_ENABLE:
        rb = cfg.REFINE_BSIZE
        for j in range(len(idx)):
            X = X_list[j]
            def reconstruct():
                Xc = torch.zeros_like(X)
                for (i0,j0,Bc) in core_per[j]: h,w=Bc.shape; Xc[i0:i0+h, j0:j0+w] = Bc
                if cfg.RES_COEF == "diag":
                    g = coef_list[j]; Xlr = (DL * g.view(1,-1)) @ DR.t()
                else:
                    C = coef_list[j]; Xlr = DL @ C @ DR.t()
                Xr = torch.zeros_like(X)
                for (i0,j0,Bb) in res_per[j]: h,w=Bb.shape; Xr[i0:i0+h, j0:j0+w] += Bb
                return Xc + Xlr + Xr
            Xhat = reconstruct()
            err = frob_rel_err(Xhat, X)
            added = 0
            core_pos = {(i0,j0) for (i0,j0,_) in core_per[j]}
            res_pos = {(i0,j0) for (i0,j0,_) in res_per[j]}
            while err > cfg.REFINE_ERR_TARGET and added < cfg.REFINE_MAX_EXTRA:
                Rerr = (X - Xhat).contiguous()
                Eg, te, nb = block_energy_grid(Rerr, rb)
                flat = Eg.reshape(-1)
                if flat.max().item() <= 1e-18: break
                order = torch.argsort(flat, descending=True)
                found = False
                for idx_ in order.tolist():
                    bi, bj = idx_ // nb, idx_ % nb
                    i0, j0 = bi*rb, bj*rb
                    if (i0, j0) in core_pos or (i0, j0) in res_pos: continue
                    Bb = gather_block(Rerr, i0, j0, rb)
                    res_per[j].append((i0, j0, Bb)); res_pos.add((i0, j0))
                    added += 1; found = True; break
                if not found: break
                if added % cfg.REFINE_RECHECK_EVERY == 0:
                    Xhat = reconstruct(); err = frob_rel_err(Xhat, X)
            Xhat = reconstruct(); err = frob_rel_err(Xhat, X)

    return {
        "core_blocks": core_per, "core_energy": core_ef,
        "DL": DL, "DR": DR, "coef_list": coef_list, "res_blocks": res_per
    }

# -----------------------------------------------------------------------------
# Evaluation (extended with new metrics)
# -----------------------------------------------------------------------------
@torch.no_grad()
def compute_cosine_similarity(A: torch.Tensor, B: torch.Tensor) -> float:
    A_flat = A.flatten().to(torch.float64)
    B_flat = B.flatten().to(torch.float64)
    return (A_flat @ B_flat).item() / (A_flat.norm().item() * B_flat.norm().item() + 1e-12)

@torch.no_grad()
def compute_psnr(A: torch.Tensor, B: torch.Tensor) -> float:
    mse = F.mse_loss(A, B).item()
    if mse == 0: return float('inf')
    max_val = max(A.abs().max().item(), B.abs().max().item())
    return 20 * math.log10(max_val / math.sqrt(mse))

@torch.no_grad()
def compute_sdr(A: torch.Tensor, B: torch.Tensor) -> float:
    # Signal-to-distortion ratio
    signal_power = (A ** 2).sum().item()
    noise_power = ((A - B) ** 2).sum().item()
    if noise_power == 0: return float('inf')
    return 10 * math.log10(signal_power / noise_power)

@torch.no_grad()
def eval_payload_extended(rt: PayloadRuntime, Ws_norm: torch.Tensor, Sc: torch.Tensor) -> Dict[str, Any]:
    E, n, _ = Ws_norm.shape
    metrics = {
        "per_expert": {"rel_err": [], "cos_sim": [], "psnr": [], "sdr": []},
        "routed": {"rel_err": [], "cos_sim": [], "psnr": [], "sdr": []}
    }

    # Per-expert
    for pos in range(E):
        W_orig = Ws_norm[pos] * Sc[pos]
        W_rec = rt.reconstruct_expert(pos)
        metrics["per_expert"]["rel_err"].append(
            torch.linalg.norm(W_rec - W_orig).item() / torch.linalg.norm(W_orig).item()
        )
        metrics["per_expert"]["cos_sim"].append(compute_cosine_similarity(W_orig, W_rec))
        metrics["per_expert"]["psnr"].append(compute_psnr(W_orig, W_rec))
        metrics["per_expert"]["sdr"].append(compute_sdr(W_orig, W_rec))

    # Routed mixture
    for _ in range(cfg.EVAL_TRIALS):
        x = torch.randn(cfg.EVAL_BATCH, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), min(cfg.ROUTED_K, E))
        gates = torch.rand(len(routed), device=DEVICE); gates /= gates.sum()
        y_hat = rt.apply_mixture(x, routed, gates)
        Wsum = sum(gates[i].item() * (Ws_norm[pos] * Sc[pos]) for i, pos in enumerate(routed))
        y_ref = x @ Wsum
        metrics["routed"]["rel_err"].append(
            torch.linalg.norm(y_hat - y_ref).item() / torch.linalg.norm(y_ref).item()
        )
        metrics["routed"]["cos_sim"].append(compute_cosine_similarity(y_ref, y_hat))
        metrics["routed"]["psnr"].append(compute_psnr(y_ref, y_hat))
        metrics["routed"]["sdr"].append(compute_sdr(y_ref, y_hat))

    # Aggregate
    agg = {}
    for scope in ["per_expert", "routed"]:
        agg[scope] = {}
        for m in ["rel_err", "cos_sim", "psnr", "sdr"]:
            arr = np.array(metrics[scope][m])
            agg[scope][m] = {
                "mean": float(np.mean(arr)),
                "std": float(np.std(arr)),
                "min": float(np.min(arr)),
                "max": float(np.max(arr)),
                "p95": float(np.percentile(arr, 95)),
                "raw_values": arr.tolist()  # keep raw for bootstrap
            }
    return agg

# -----------------------------------------------------------------------------
# Original size computation
# -----------------------------------------------------------------------------
def compute_original_expert_size_bytes(model_dir: str, layer: int, max_experts: int) -> Tuple[int, List[int]]:
    wm = read_index(model_dir)
    all_ids = find_layer_expert_ids(wm, layer)
    n_used = max_experts if max_experts > 0 else len(all_ids)
    eids = all_ids[:n_used]

    total_bytes = 0
    for eid in eids:
        keys = pick_expert_tensor_keys(wm, layer, eid)
        if not keys: continue
        for role, key in keys.items():
            shard = wm[key]
            with safe_open(os.path.join(model_dir, shard), framework="pt") as f:
                t = f.get_tensor(key)
                total_bytes += t.numel() * t.element_size()
    return total_bytes, eids

# -----------------------------------------------------------------------------
# Main (with ablation and metric export)
# -----------------------------------------------------------------------------
def banner():
    log("="*60)
    log("EBC-LLM Enhanced Compression Pipeline")
    log(f"Time: {now()}  Device: {DEVICE}")
    log(f"MODEL_DIR: {cfg.MODEL_DIR}  OUTPUT_DIR: {cfg.OUTPUT_DIR}")
    log(f"Layer: {cfg.LAYER}  Experts: {cfg.MAX_EXPERTS}")
    log(f"Ablation mode: {cfg.ABLATION_MODE}")
    log("="*60)

def main():
    banner()

    # 1. Original size
    orig_bytes, eids_used = compute_original_expert_size_bytes(cfg.MODEL_DIR, cfg.LAYER, cfg.MAX_EXPERTS)
    log(f"[size] Original expert weight bytes: {orig_bytes/1e6:.2f} MB ({orig_bytes/1e9:.3f} GB)")

    # 2. Load or build Ws
    expert_ids, Ws_norm, Sc = load_or_build_Ws()
    E, n, _ = Ws_norm.shape
    log(f"[Ws] shape={Ws_norm.shape}")

    # 3. Ablation handling: override components based on ABLATION_MODE
    if cfg.ABLATION_MODE == "no_cluster":
        cfg.M_MAX = E  # force one cluster
        cfg.CLUSTER_MAX_SIZE = E
    elif cfg.ABLATION_MODE == "no_rot":
        cfg.BASIS_MODE = "identity"  # will need to handle identity later; for simplicity we just skip training
        cfg.TRAIN_STEPS = 0
    elif cfg.ABLATION_MODE == "no_lowrank":
        cfg.RES_RANK = 0
    elif cfg.ABLATION_MODE == "no_blocks":
        cfg.CORE_MAX_BLOCKS = 0
        cfg.RES_MAX_BLOCKS = 0
    elif cfg.ABLATION_MODE == "no_refine":
        cfg.REFINE_ENABLE = False

    # 4. Clustering
    if cfg.ABLATION_MODE == "no_cluster":
        # Force all experts into one cluster
        labels = torch.zeros(E, dtype=torch.int64)
        M = 1
        clusters = [list(range(E))]
    else:
        Xfeat = random_proj_features(Ws_norm, cfg.CLUSTER_FEAT_D)
        M0 = max(2, min(cfg.M0 if cfg.M0>0 else int(round(2*math.sqrt(E))), E))
        labels = kmeans_torch(Xfeat, M0, cfg.CLUSTER_ITERS, cfg.CLUSTER_RESTARTS)
        labels = merge_small_clusters(Xfeat, labels, cfg.CLUSTER_MIN_SIZE)
        labels = hierarchical_split(Xfeat, labels, cfg.CLUSTER_MAX_SIZE, min(cfg.M_MAX, E), cfg.SPLIT_ITERS)
        labels = merge_small_clusters(Xfeat, labels, cfg.CLUSTER_MIN_SIZE)
        labels = relabel_contiguous(labels)
        M = labels.max().item() + 1
        clusters = [torch.nonzero(labels==m, as_tuple=False).flatten().tolist() for m in range(M)]
        clusters = [c for c in clusters if c]

    log(f"[cluster] M={len(clusters)} sizes={[len(c) for c in clusters]}")
    cluster_of_pos = [0]*E
    for m, idx in enumerate(clusters):
        for pos in idx: cluster_of_pos[pos] = m

    # 5. Initialize bases
    U_par, V_par = [], []
    for idx in clusters:
        Wm = Ws_norm[idx].mean(0)
        if cfg.ABLATION_MODE == "no_rot":
            # Use identity
            r = min(cfg.RES_RANK, n)
            U0 = torch.eye(n, r, dtype=DTYPE_ACC, device=DEVICE)
            V0 = torch.eye(n, r, dtype=DTYPE_ACC, device=DEVICE)
        else:
            U0, V0 = svd_init_from_mean(Wm)
        U_par.append(OrthoParam(U0)); V_par.append(OrthoParam(V0))

    # 6. Train rotations
    if cfg.TRAIN_STEPS > 0 and cfg.BASIS_MODE == "dense_train" and cfg.ABLATION_MODE != "no_rot":
        params = [p.M for p in U_par] + [p.M for p in V_par]
        opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)
        guidance_masks, guidance_stats = {}, {}
        t0 = time.perf_counter()
        for step in range(1, cfg.TRAIN_STEPS+1):
            S = torch.randperm(n)[:cfg.SUBM].to(DEVICE)
            if cfg.TRAIN_LAM_GUIDE > 0 and (step==1 or step%cfg.TRAIN_GUIDE_EVERY==0):
                with torch.no_grad():
                    guidance_masks.clear(); guidance_stats.clear()
                    for m, idx in enumerate(clusters):
                        if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
                        Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                        pick = idx if cfg.BATCH_E>=len(idx) else [idx[i] for i in torch.randperm(len(idx))[:cfg.BATCH_E].tolist()]
                        Xs_ng = slice_X_batch(Ws_norm[pick], Uo, Vo, S).detach()
                        mask, ef, kblk = make_guidance_mask_from_Xs(Xs_ng, cfg.CORE_BLOCK, cfg.TRAIN_GUIDE_TARGET, cfg.TRAIN_GUIDE_MAX_BLOCKS)
                        guidance_masks[m] = mask; guidance_stats[m] = (ef, kblk)

            lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
            lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
            lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp
            L_total, n_terms = None, 0
            for m, idx in enumerate(clusters):
                if len(idx) < cfg.TRAIN_MIN_CLUSTER: continue
                Uo, Vo = U_par[m].orthogonal(), V_par[m].orthogonal()
                pick = idx if cfg.BATCH_E>=len(idx) else [idx[i] for i in torch.randperm(len(idx))[:cfg.BATCH_E].tolist()]
                Xs = slice_X_batch(Ws_norm[pick], Uo, Vo, S)
                off, diag = offdiag_abs_mean(Xs), diag_abs_mean(Xs).clamp_min(1e-6)
                base = torch.log(off+1e-6) - torch.log(diag) if cfg.TRAIN_OBJ=="logratio" else off/diag
                if lam_block > 0: base += lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)
                if lam_guide > 0 and m in guidance_masks:
                    Mmask = guidance_masks[m]
                    Etot = (Xs*Xs).mean().clamp_min(1e-12)
                    Eout = ((Xs*(1-Mmask))**2).mean()
                    base += lam_guide * (Eout/Etot)
                L_total = base if L_total is None else L_total + base
                n_terms += 1
            if L_total is None: break
            L_total = L_total / n_terms
            opt.zero_grad(); L_total.backward()
            if cfg.GRAD_CLIP > 0: torch.nn.utils.clip_grad_norm_(params, cfg.GRAD_CLIP)
            opt.step()
            if step % cfg.REORTHO_EVERY == 0 or step == cfg.TRAIN_STEPS:
                with torch.no_grad():
                    for p in U_par: p.M.copy_(p.orthogonal())
                    for p in V_par: p.M.copy_(p.orthogonal())
            if step % cfg.REPORT_EVERY == 0 or step == 1:
                t1 = time.perf_counter()
                gstr = "" if not guidance_stats else f" guide≈{np.mean([v[0] for v in guidance_stats.values()]):.3f}"
                log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={L_total.item():.4f} {gstr} (+{t1-t0:.1f}s)")
                t0 = t1

    # Freeze bases
    U_list = [p.orthogonal().detach() for p in U_par]
    V_list = [p.orthogonal().detach() for p in V_par]

    # 7. Build payloads
    log("[build] payloads ...")
    core_all = [[] for _ in range(E)]
    res_all  = [[] for _ in range(E)]
    DL_list, DR_list = [], []
    rmax = min(cfg.RES_RANK, n)
    gam = torch.zeros((E, rmax), dtype=DTYPE_ACC, device=DEVICE) if cfg.RES_COEF=="diag" else None
    Cfull = torch.zeros((E, rmax, rmax), dtype=DTYPE_ACC, device=DEVICE) if cfg.RES_COEF=="full" else None

    for m, idx in enumerate(clusters):
        U, V = U_list[m], V_list[m]
        if cfg.ABLATION_MODE == "no_lowrank":
            # Skip low-rank: set DL, DR to zero, coef to zero
            DL = torch.zeros(n, rmax, dtype=DTYPE_ACC, device=DEVICE)
            DR = torch.zeros(n, rmax, dtype=DTYPE_ACC, device=DEVICE)
            P = {"core_blocks": [], "res_blocks": [], "coef_list": [], "DL": DL, "DR": DR}
            # For core blocks, still do normal processing (unless no_blocks)
            if cfg.ABLATION_MODE != "no_blocks":
                # compute core blocks normally but without low-rank
                X_list = [(U.t() @ Ws_norm[pos] @ V).contiguous() for pos in idx]
                core_per = []
                for X in X_list:
                    Eg, te, nb = block_energy_grid(X, cfg.CORE_BLOCK)
                    picks, _ = pick_blocks_until_target(Eg, te, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
                    blocks = [(bi*cfg.CORE_BLOCK, bj*cfg.CORE_BLOCK, gather_block(X, bi*cfg.CORE_BLOCK, bj*cfg.CORE_BLOCK, cfg.CORE_BLOCK)) for (bi,bj) in picks]
                    core_per.append(blocks)
                P["core_blocks"] = core_per
                P["res_blocks"] = [[] for _ in idx]
                P["coef_list"] = [torch.zeros(rmax, device=DEVICE) for _ in idx]
        else:
            P = build_payload_for_cluster(Ws_norm, idx, U, V)

        for j, pos in enumerate(idx):
            core_all[pos] = P["core_blocks"][j]
            res_all[pos] = P["res_blocks"][j]
            if cfg.RES_COEF == "diag":
                gam[pos, :P["coef_list"][j].numel()] = P["coef_list"][j]
            else:
                C = P["coef_list"][j]; Cfull[pos, :C.shape[0], :C.shape[1]] = C
        DL_list.append(P["DL"]); DR_list.append(P["DR"])
        log(f"  cluster{m}: E={len(idx)} core_blocks≈{np.mean([len(c) for c in P['core_blocks']]):.1f} r={P['DL'].shape[1]}")

    # 8. Save payload
    out_path = os.path.join(cfg.OUTPUT_DIR, f"ebc_payload_layer{cfg.LAYER}_E{E}_q{cfg.QMODE}.npz")
    store_dtype = np.float16 if cfg.BASIS_STORE_DTYPE=="float16" else np.float32
    arrays = {
        "meta": _encode_meta(ws_meta(expert_ids) | {"time": now(), "qmode": cfg.QMODE, "res_coef": cfg.RES_COEF, "ablation": cfg.ABLATION_MODE}),
        "expert_ids": np.array(expert_ids, dtype=np.int32),
        "scales": Sc.cpu().numpy().astype(np.float32),
        "cluster_of_pos": np.array(cluster_of_pos, dtype=np.int16),
        "n_clusters": np.array([len(clusters)], dtype=np.int32),
    }
    for m in range(len(clusters)):
        arrays[f"U_{m}"] = U_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"V_{m}"] = V_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DL_{m}"] = DL_list[m].cpu().numpy().astype(store_dtype)
        arrays[f"DR_{m}"] = DR_list[m].cpu().numpy().astype(store_dtype)
    if cfg.RES_COEF == "diag":
        arrays["gam"] = gam.cpu().numpy().astype(store_dtype)
    else:
        arrays["Cfull"] = Cfull.cpu().numpy().astype(store_dtype)

    core_pack = pack_blocks_ragged(core_all, cfg.QMODE)
    res_pack  = pack_blocks_ragged(res_all, cfg.QMODE)
    for k, v in core_pack.items(): arrays["core_"+k] = v
    for k, v in res_pack.items(): arrays["res_"+k] = v

    save_npz_compressed(out_path, arrays)
    payload_bytes = os.path.getsize(out_path)
    log(f"[save] payload -> {out_path} size={payload_bytes/1e6:.2f} MB")

    # 9. Evaluate
    rt = load_payload_runtime(out_path, DEVICE)
    extended_metrics = eval_payload_extended(rt, Ws_norm, Sc)

    # 10. Compression ratio
    compression_ratio = orig_bytes / payload_bytes if payload_bytes > 0 else float('inf')
    log(f"[metrics] Original size: {orig_bytes/1e6:.2f} MB")
    log(f"[metrics] Compressed size: {payload_bytes/1e6:.2f} MB")
    log(f"[metrics] Compression ratio: {compression_ratio:.2f}x")
    log(f"[metrics] Per-expert RelErr: {extended_metrics['per_expert']['rel_err']['mean']:.6f} ± {extended_metrics['per_expert']['rel_err']['std']:.6f}")
    log(f"[metrics] Routed RelErr: {extended_metrics['routed']['rel_err']['mean']:.6f} ± {extended_metrics['routed']['rel_err']['std']:.6f}")

    # 11. Statistical significance (bootstrap confidence intervals)
    if cfg.COMPUTE_SIGNIFICANCE:
        def bootstrap_ci(data, n_bootstrap=1000, alpha=0.05):
            means = []
            n = len(data)
            for _ in range(n_bootstrap):
                sample = np.random.choice(data, size=n, replace=True)
                means.append(np.mean(sample))
            lower = np.percentile(means, 100*alpha/2)
            upper = np.percentile(means, 100*(1-alpha/2))
            return lower, upper

        routed_errs = np.array(extended_metrics['routed']['rel_err']['raw_values'])
        if len(routed_errs) > 0:
            ci_low, ci_high = bootstrap_ci(routed_errs)
            log(f"[stats] Routed RelErr 95% CI: [{ci_low:.6f}, {ci_high:.6f}]")
            extended_metrics['routed']['rel_err']['ci_95'] = [ci_low, ci_high]

    # 12. Export full metrics to JSON
    if cfg.EXPORT_METRICS_JSON:
        report = {
            "config": {k: str(v) for k, v in cfg.__dict__.items()},
            "original_size_bytes": orig_bytes,
            "compressed_size_bytes": payload_bytes,
            "compression_ratio": compression_ratio,
            "metrics": extended_metrics,
            "expert_ids": expert_ids,
            "layer": cfg.LAYER,
            "timestamp": now(),
        }
        json_path = os.path.join(cfg.OUTPUT_DIR, f"metrics_layer{cfg.LAYER}_{cfg.ABLATION_MODE or 'full'}.json")
        with open(json_path, "w") as f:
            json.dump(report, f, indent=2)
        log(f"[export] Metrics saved to {json_path}")

    log("✅ Done.")

if __name__ == "__main__":
    main()

EBC-LLM Enhanced Compression Pipeline
Time: 2026-04-23 09:53:23  Device: cpu
MODEL_DIR: /data/downloaded_models/Mixtral-8x7B-v0.1  OUTPUT_DIR: /home/daniyar/moe_ws_outputs
Layer: 0  Experts: 8
Ablation mode: none
[size] Original expert weight bytes: 2818.57 MB (2.819 GB)
[found] layer=0 total=8 using=8 eids=[0, 1, 2, 3, 4, 5, 6, 7]
[cache] loaded Ws -> /home/daniyar/moe_ws_outputs/Ws_cache_layer0_E8_ridge_ebc.npz shape=torch.Size([8, 4096, 4096])
[Ws] shape=torch.Size([8, 4096, 4096])
[cluster] M=3 sizes=[2, 4, 2]
[train] step   1/24 loss=-4.7225  guide≈0.922 (+22.6s)
[train] step   4/24 loss=-0.8138  guide≈0.827 (+67.4s)
[train] step   8/24 loss=-0.8611  guide≈0.823 (+82.4s)
[train] step  12/24 loss=0.3989  guide≈0.815 (+82.8s)
[train] step  16/24 loss=5.1294  guide≈0.822 (+84.9s)
[train] step  20/24 loss=6.9194  guide≈0.824 (+83.1s)
[train] step  24/24 loss=6.2806  guide≈0.838 (+82.8s)
[build] payloads ...
  cluster0: E=2 core_blocks≈23.0 r=512
  cluster1: E=4 core_blocks≈121.8 r=512